# Nutrelle — treino em GPU, quinta execução (seção 14: recorte pelo prato e receitas corrigidas; uma única foto de cima)

Notebook gerado por `eval/treino/kaggle/monta_pacote.py`; **não edite à mão** (edite `treino_gpu.py` e gere de novo). Lê a saída do kernel da etapa 0 (`nutrelle-deteccao-r5`, anexada em *Input*) e `listas/conferencia_r5.json` (regra e bases reprovadas na conferência a olho; sem `"conferida": true` o kernel não começa). Baixa as fotos, recorta cada uma na janela do guia, extrai as features do So400m sobre o recorte, treina as configurações de 14.5 até o orçamento e grava previsões cruas (fora-do-fold no treino e nas partes de registro das bases externas, **sem pontuar nada; nenhuma do teste2**), os arquivos para servir e `servico.json`.

**Versão do repositório: só bases liberadas** (Nutrition5k, Flavoria, LeFood-Set, GT Mensa). As configurações que usam OmniFood8K ou o Food Portion Benchmark ficam de fora e o manifesto diz por quê; o SNAPMe também. A versão completa é montada fora do repositório com `python3 monta_pacote.py push <pasta>`.

Kaggle: *Settings → Accelerator → GPU T4 x2* e *Internet → On*. Instruções completas em `eval/treino/kaggle/LEIA-ME.md`.

In [ ]:
# Parâmetros (os padrões servem para o Kaggle)
import json, os
RODADA = '5'         # este pacote só traz as listas desta execução
ORCAMENTO_H = 5.0    # horas de relógio; o que não couber inteiro não começa (a ordem de prioridade decide o que fica de fora)
SAIDA = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content/nutrelle_saida'   # no Colab, aponte para o Drive para retomar
EXTRA = []           # ex.: ['--so', 'ridge']; ['--dominios', 'flavoria,mensa']; ['--retoma', '<pasta>']; ['--espelho', '<pasta com o zip do OmniFood8K>']
# as variáveis de ambiente NUTRELLE_SAIDA e NUTRELLE_EXTRA (lista em JSON) substituem os valores acima sem editar o notebook
SAIDA = os.environ.get('NUTRELLE_SAIDA') or SAIDA
EXTRA = json.loads(os.environ.get('NUTRELLE_EXTRA') or 'null') or EXTRA
os.makedirs('/tmp/nutrelle_codigo/listas', exist_ok=True)

In [ ]:
# treino_gpu.py (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/treino_gpu.py', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5<IH;R|6KqPh(bq-BQwKy4>SE$m+hy=@aif<Hp(x6uSRyx75<tfLmlr9v8=rA^vK9U_wKSxGTYDxj-jWATV6)EKC(Hm
YNGRX;bmU+;6s%e2My;AdQ+%8O$p?ZCbjDzN$>!xfgl28Q=I#TvJTAmr^77~D@Zu{uA8j}=Ah_;`+(Wh7Pf7bceFsb$RGZ$;$m)?GKB?9<~?!B1vArY#MT#@
mIAho^Z=^{$g_MwVIU6R+-vlNl8?BI_hHwb%$T%+eNqIhigtVv!`j{?O7Q+YhT=3|SaqgO8$j!W`>k|m9zhNv0uDp5Xr3Q^6;J_^66184ImAO>Ppp50eUOUl
VD&>9$J+8n9FytF5)CM~Sc#tdYxtF`hWZz|<d_JaC{_Y4R^}Ij(EC{*VE_;CXsX}VhcYbuwO8~%Y#jlw@lc?7#bpik-7eRYpnBU+;&;cT?6a+Jgm6*e!rBs;
mKwl`A@b7S_1st7`0v2=*3SJ@q!Ng+Dz436NEB|wU%m8E96_)<P-l^1)8Z?<H?6y_C|Pe<q{q@wqXQX!dJ#C~>+^DJ`dry)O_26bgb{w+1LNuP?4__ckTBNH
8ZM%F+=GRyv@U9Xriv?F0cc*hfgBhv0A&Xfo0dg?gCwhr_Nv2Cvv2xm=Kg~d9_%egGaM2gHrbpRp<p_^7`FByodDM#@>~_1{YUjb^lk%zA9{jw^h9RP*#)bp
rqu7<)>u@JT`MoDkSEXn+zcToA|eb4jJ7QgSU)TBk`U1mEvwZ1d1Mo_5dBC793jB&#7;t7sK70M5}=QTo5t;EWX_fVIw*rY2fC=vCunV^cn(e~faHkK9yb>I
-ID}rWTL!B3<fDBL}uT+Am2>PTAH~+P5RN;$qv?)0KjM&1k&KG3`=wOe3?b46*LwXfw_$!Vv!Z7;tV|2_ZS5IwTQz68?1dAI~=u$OR7y;MGV+<P82%<so7rv
-GxjvyNMxuznP@-EK#>sRosEsNu)Cb1xW%JhNWLu8%%20`kE<9yU7d8@{j{|D6Hq}(Sxywk?hubJq9N4gaTO3$1JX%$W^`NwyMy(s4w1|3h|Jjie6d4YB~Q2
of6stzib(J%>po^qmT#yz3z>W;)88Gf{nX}+&Ym+aeZ+^Y?!J%A~&m$b6|@`AExp#Y%ozq@OA2TY%RD~p1ZZHQ6C>a|BD$+pI7iK5y#?tG9efCY4~UAm^w?C
OGEcuiTXKYQwMR3oZyhdL5qLcA%K2niCI?4&)(T|pJhn6IO}y!0v>kldb($B;WkYdtN}v~tEAWpqbx=NNUYM=>y_!`HMYg#tL7FsfKJGq9TkJH1e5+gtwYkp
jB!kU=|OcOdtkrqqD4H-7p5(EjZpXT8}cI4Pql@8Z@^u1d^XQV7C?Yc!3#W>99B+#I1A2|Xb||1>LcZ+tyg?kNlqg$-4WLW2R>3T$;d_T8HR&?CVMY*nq^RB
9yS6dF*U;T0wwZ%7OvJeg)K_p9ilWw$RyGtU91@qI@yf$P2civ_bRp1ys7pRn>B@~7yShI4Q&?SUCX<DVwkStv)|rJBr?eS4bd=He@kXdP`5)ax7&I_Fw-iB
2e{{U_hvt`P{yBka#iD3@BJIeaP8g`>8gEpxSl*+5bB%-u<>JkfGN!F=ZW12Echb0!#=a69`qoE5I=rHr8KO}`TOXY;*%}e=b47B*N0mlSXAZZFH{e=cd~}X
iAW=wv$>OC1EHtIqXxG*;?FbQ8uLSHZ;!tRXa3b*KAmyB_R_J>J`~jFp}$TN>ziKC;Dl$!ayk{@&-oWm%JoTAuOz3Pd2m+YksbbT%oa6Qt0XCJy1V3&OtYd#
n-R0w1kM{WRJ!S|LL8KftvrA8E$E<sjjMV<N8Ye>GKMM{)7{QoFeL?t87cO<sYGTba<<wZH2IMl%p<p7^ai(hQxey)wuGTd+DS&SU(2vYV&;<h@3yi&$^(jo
;K<E9ZE!>yioi(FNzhsYy|9ScmyQA12T6`Vgi0h1?o}T~@JT$k6_+;-$TFHtOK;i(>@pWz+*2z=Pv-1}+-R~rcK}LyVPSP1`rYkS<!}%dGRF(65<E>n#^0xr
q^$WO91}M%uGWupdWcuD=m!o;D;6l04;1!#w53s;#yv{!X$#7G*gm@eKLjrZfi@sviNY>kKGQ0EX&`b=1WwLGQ)zG))!P9xm_%oQvBJSy>``!3V2GQ}De~ns
>O4~Hfw?uXr4mzU2!d_H$*NQuyVAhh*jJkj0G-n~5MhQ!9M}0jgfs6Vdy1Fv9byO@<t1j3)H6pwaxaCC-+MS}6S)8KURSZC9{Brq3oH)=l!?6c9ajp-G4GX|
0n}J+0yAZGnkM_(Be&M%(6`i^Qw*6UGUT8Ev*ql(Sc50Bm}apCPfb?E%rNgPc%?b9TI7udl~+O~%ltQ%_)RQFW~x|>u6p>FUb|Vo%&@V<XC&Bk5(^*O<p!9o
qg5)q<YbL5g7jb|is7$th$%LF77QguoV#U4x6&pgXD2+F&Oo&VBSI{<%^onm8j}cCc=|Iyx<oqfOi;=4@uvlCw;x$K7~al{Xmy`bHemR5Zq+182Y+qt>x8N3
hI^hgk>=YA!uWIEw|O!wlAV9)FT4<$r5sn%&Otwt%o;FK)qqWrZOAtpB3}LlfuJPdw!m@vdq__VP+9MZ#F?|`OPt~83=+tk%|4(+sU#GkWk5}1>!)<m$8(7m
f$}tPq0E}CCs5=H4d^<7@J?nWIu;>5l<cIL9RSg{{a%}Fk6ouKs-X=!_wcHhwADSa&aq4X>t;b#i5cWt!J9jm@2i^DWFqS1D!^uuY8zQ=auvHTVaPa$S}yh(
W~AXBDLQRItIayu6FIn7ms#emS9@jOc|r;zm5Z>%@mxWSEMW?l!}T@V2!{|ExV@e3D4aG9&8u<&+@ak|(x<Q98xR`7d(jF<7g&&H2BtrAs+$0{{C)DABifSa
e9`SVYzg2Y#6=oSii#YFZ;41DB%MeyUE7!ccJsM*i4A<2W6Vf(G<4z}Ob3YeCxbf9w69tv`LN2y@Xf!+bqbne{i=+baOMq`Kmt8N5hzH1;{6kk;EN%a;g1{|
a#~RSl<bc{%c58GEEAA+XQ9)m5?<Szwmdd~8DLG_LaaUq$$80SkQtsl0e`Dy&z&}n6>Ma50P$(0J)}7%Y3QekKn`OGx-gYaM3?VKhen{h{trnQsv?<*FdwIH
=rVa3X$xf?@O}n5GfjS=RS%(k%WS+C6~5(1yC|$AD6~oG0A%*J&VegTF4}$y0b^x`?W%8cKX@~pZFEPD?nL~z`)oXf;wty&=<xbXj?=>&tCrubyxEcIC|Bfd
GbJ`<1|FPmT5j4MAT1=&X3{V66OPf{o*T=Lce=l=SBlrN7ixzX2J+vdCNah@XbESIoa6>Ds*I(b<x6_upyX~1PL^%D&YK%=wkgWCu%&y~x7|t!%j^px5u>9~
y>O-syynWAaY5O_$||3v8EOolx2|!vkk3?$`!CuuTpX)m=b<UCJ|-T-5srw`9v)p?g;BAHh5T&B3rxc|v{qA;ZuLLhm7<cJX;pUmBkJmyg{xYC0U08v#1&4a
n=co>OScnAAE!54`<IcXS?Jpp!U~T@1mmgmdZa;O^Ej0b?P1Bu)K?Oc=c6B-9*N|fio%eYe$M}kAFD?6=zf^ZV?QjBNc7hcOJ|-<W0cZy(|R7YRhYw1!$j&h
NHx9Cj2rU}e+i1v5=v_#IhAP>G}Ce3Ni}-iM>^Um=!w-7#zbQAA~Q!kV|mjVSsZUUh_!zV+}KS*-pCxB_=j0)lbHz!hv-vpoXdN?SIRY3p3oRtOv5WQN7K|&
3MM_#mZNVCQ0ZJt?-g&pdWZ6g!=z)61877IiTlewN=JYOwFoSsyZBI1?DJr-eI5Q8mUyM#xfVroHq&FBNQ_>Z0>#vU9Ab`(;!yZm!%t(OWux*LV}|2$1^d^i
Wdf0BfNQDKPEY>?+>(DzlgK|PG2<I9wY00FJ(f1Rpqh9{L_mX2*2*sR2>-9zgKJEnjuNI`ez5L3VBJGYEH^u_e|sYdHDvEtKw5X`05UO;<w~F=N$z3FKXt=p
$}Duvxj!2d@yE#qb{|51j?+di|Eso#_{x_kZg*nT-F>MeboZk3vcf4InH-lE&17nut}YqlM8-EOj4|lj91);hzF<TpjyMa+AP#TLV<BKTa0(xfA5qCDbq-)h
?1Fx$13|sbL&>?W6?S!L6^7bev@CUkx6dv%+k?!i0Wh~gADi8Z;2c6Z{e3k8Z!X5AHplI3B1N2JMQ|dtFtlIu=PeJ83OE|8B<|KPpdF1BHKJL>wr4kfl(-p}
s)FCd3mhqO{2UUwmXXntFOy(Od66?e(7nk)Bj!K!!ca)4g;4p~_DFqoa$1(W2X03@)w^2No`Bq`zWJf|rDS%Aq(JVh+=KK$sDj}v|3kX1BQpyni=?#t^k7{z
df{2oT*z;R<a}q317knR@XVi#(CHQ=mC&2xwg}I{)}l9$pB-)0?-?*1FVth9(x0?!p6q%JBYI1h{rxo=-<LZFOOTyZEGW5E1hY9PrHg$60Z5!9$b#?(&`I`e
Z$Hus`h2P~pS{;7!%y}NOI&A94t%3z#DcWee5<AI@d7DE@#2F0Z!W}GhGRh0@5)siV)c)1=-y?#A5!<mM)=e<q3SI?<5wt<CWQ}Td#b7Ye=80H(PfRW7y9>{
(HL?Z?Ze7_IsB;R%L1Zfp;{l6U_ah9@_hb~h3h}Hh08@%X6B*uQnbdo&OwB`mKxU?)J41hw!`Q+%x^L3gz;H^kJ@F1Fw8Mj-#H7!hlL}tH|6yRZA;PoVNbs}
SbvO*{WWXOziXgR#qyCLB-%5IMV|bBu_<+htExp91zj6=^=GC2KhB|hzQ%CxsGZk!0|KFqQ$}5FY4Ve+Yo)4GY~Fns7=F*?$5&S@enXsym7LN2BycT)IFa0B
b09fxoUdZ|$^V@ex63fTH&$c^kP~uAGW&sa7NuFRL1XPAyB1Bz@{Z<=x_18OZucI)zy}%aC>+!b-(<#m?rJ!R+3gL%Fg%~-&u)`80Ci>oqjZ2_GEdyIGKfzE
RF70Q4cp$(y&|xTDpF3GxD}X~LNi)#VCHJiF>X^`U6fmM>n8Zax@PSQF`NE`jX-hnfZ7rNyF%~<^50yh*PJF^I_^$=Jf|WoyjkQo)_7XP`qi~S-QCC;=20so
3;4|E8fqv_JHg%%JUD|uOTh=&(l%xqIt&xewinQ(gfD@4CKhfBzjQ2^Z<~Qbk}l&5qzvfdVu<m?c0K~z!~9*?UGy<J?8Ai&A1u)2-E{N;Mt#@88RM_bg2hoK
B3*lVioma5!>tQW=Y~TP%i&+4vqY$Ai|J)#ctMslp{L91T&Y8oI=vHnJfCn|aGO=E)|VV`R(aPcyUH&>x*D3P=7Ph$w8}*l<VG80^gY&4;`Op{Ax~9HPHmI?
jPUy(;@m6FQ$o*1G-~RIeE2ktaGvy9!7UTcCBpH>usY9~bW(i6h<TsuiAWtID66?bVW$I@OqXbiQ1?W3DQK-LaxOtg@ylTNxJbdTK=sTPY?qjXTP6w7dWDNB
gu*Y_B{!ml%w*5aFjL>ohG1%g`IX4o6Y0pB(&H225nOpGSB$5^na|oGfABGyVz7L`QU+^toLS^=ZdoW!mEi_6L@$1W#WE&|1w&W%JT02ViLM%b9rinhn#H}W
vLQqXKAG09+jWrM^c}eIOZ3hSu0hAaJRehpW?spHAw~NtBk3jZ{)aWYhN$D8XZ|&2ZZ&4|Vd8|2Do@ti_#eg4G+541e(KH>19fNz=6)H5c>xwZEI(F6TA^O=
DX^Z7DVXp!L9XV3a09JvuvjyLh^zR7Wx{p>h@nw~Zy7k})bO+OR*R!ph^1D3Exm3uY5<)v)C2eeziuTze90q;{}b$5+cbSM&#7VSXP%89R^YN|J5nmkBuPkU
(JHliWI?Ax3Of5-%vH22W%*U1=zhf*A0AodPMg<t;PRhjWVfVe^iCH=lo`4s!;`w$hg5RuuGDbxY_OOG5#KFh8aWzbZd(UnN}0QC(hEt(uA^G>x>UYIOh>Q7
OU3^&Gf4ClGjA=I<H&X=s{kOkE|0;5_rpAp`AplVjkC@Iih4d8ihcZKru1u-KG(!Akf3vO{bt9zhO_Lrfk#(xw~L3*VM}SzV1KJN()zJ#GP;c7H-0;Ocpz~N
FAzxHatSfcy?W-Z*?#6s82ph^!4dkWa%$KW0XbU&^CBx;kgty9lef7aY=%EbL9Q_~aoJy;Dj!zhJIO<jr9adj#<;e3LeyG9f4FIiqmwpxHS$d{&(UJ<JZ9mn
MA?6E(zr7v(l=nHwtT)izqvlR@XPI*0&7|>>axX4+H-?rP4RS+41a$L0*W$ShH?vSMjO2_=m0;Hu)SEFL0(%>;szw+p|3SYV43$tkoZX22SR!?&LgsI<NXsg
Ox1aeYpe3gu$CYqyDm&Pm1po;2}RBMK(a~%*^HDy#bWr`et2gbyrsKsM%c)|;~<aZd<XM@Gx>SCoFwX9I0CtTbmrH6pGv+;-dta7?_l4*SQc}qHzZ=>d?szC
HFydg0xsC@JFJ_@<=?OBu|Y0}Ixu**8Uut#F61xw`Pk)>z<aKIB#Bp{xoPEnIu3SpEF<L9!Qe~APTnNa7A%WcuZkm&UXpGKG=73Q?2g@GRY;cfPOn3nm}p=w
lq6TG?J0YYfvf*kIzMJW5(@J~f$Id%&nQsu8z4prAl_~;cbke7F*q-|gj8-3l`v8E{2{U4g0z|A)9U}MuK`Z<gE*Az7~etsHYeO;JqYKHvOXPQ=*{af1&wW6
gCdPaj0zEQEh<T|H=cXEwAe&-v_(Yy9v)p*rnegXqTNt5Zfc{)kr=UG6uLJ9%kR?60Wi%LXm&2ag`_XpkbVt^-u48aw;i4%cb)bdbLIBzLrudoNLV`5K^7J_
(x~e#Rs8Pvh%!Dr5Y#kx*d^=9`OHT*`vpX7&PiGb0DX@%gI{srKCf242FuC=1?%8g47PG+w{l{O3<+7GOR{Opk^d$j8>E%LrUZwoZ}Dg78tJIjpDKpab=CSF
psL&=F*?~75w+DB{9dttd|`{+Ys2fuF;jS4s-P%INH~K~_SG&0K6`K6eZ+(9Idul&01WF_!RvW=kx$i55(p#4T}%afy!Y>8zy*Lo%d>&!YzZwP>Zlwb&9&BV
0>NrzuqXN8R@Qy6n4Kt^BKwX0`(~w`%qZsvqG*L%bY<&O`0!bVfbmXPRhYLd9duo4;fzGEc|6kD+^3<Dsw2mv2iOjmM$o*8VwlxW*uF}4)9;2<4(E5~9bQ}*
uzzV9LKuP6bufoeeytLi0#Gs~<YW_yqq|B@U8Q93l}4C1JBfY7IslC`7OB*tT;iApM=(B`{lEtUd-F77v4FEFrM*Lni42nU$n)5N3-^&k?^%g0<va9y!pT#x
WB!@brH7?Yn2n4&WKZXp@U6RcWagcdoEJO}{W`tWz%Y&?X7v66TW&EF>*h#p<HvfFR|4-S7uX{C=HRS0$RGaR?gU*7+6a^D2B}#!ik0j~ntqW43@5ilyVvXx
hhdRS`t6|o>))5hJ6Z!rqI<nV9a@y@$H^DDX?5k;79(3i9SXD9WB^v>_W}Ir1d^j&XX7lLjKl4F$FEwvi*N!enMr;KW8uWQGm+kQ7Pe}5c}b!<NaG+=G9yq{
f1l^*txqBFU3nrFCob@|=A<RZN9?C?bF4F|b(^SjnSt&7FWMPaa~}TglX;Xaf~p$y7*_xox_SmR*5ECO2@OsVy#tr?4%NM8OHFi{>B}6_EQl0&I(qF!3Jz1j
4YFdWg+X$apjZjJ*3(f)9Ns-bVnKFO+#lo8b3<9%{Cxo9X+!zpF?t1ZZUh*%L##*ff0se|#`#?<+3)4iNpWBiwsAT)jdk<bK$t!2G@^(yp&n8DOC*QzaZ(9<
;hbPQGeeeGbvNL6*CpFyq$T*&u6)^huXFhB=V%1K)WiKk0eiy**@k*@VzbB?^!U`^QoIn9C4X;i70HZ(I<DDpx{WW-zZ6Q>IZT&@Zw*o6blG{m4vWxYBQ0y$
P#9eshj?*z*rZLU+)Za9PSXq$UFA>vH%!)WJiEQ80$Y)o%%xi4rh6t?4Z$m(Z>H__3~4Qv<1@oKKXPQdzIk8G=y%hW)=aDF$S5D;W2sbf^GuF$e;1!5bicxU
$rXSSshLjL8aUpMse!zI{}!l0bf2<0z6QQxRzG-F-O0c0%&*o*<L+!m@$McP>bXKT4~-NCYt6~|Lm2iS9nTHdw_sqj@!a%Q^%bhmoS<QpgcD$+Xd;;s8OKLI
h$5xCAhu+xrg^U2>|jOWb0I{6yybY`nY!YyVBf?<cD`deTcH%vI}T%=M7BXR$^j1R+XnMe%0GtFkkU$s{I5oH51I9Z4O&gu@TDq3FHt?_#F`emMgZc1y?0#K
M~V5~Yu<}mz*baJM}$~HRK?+q<CKmwdxCD*ky^BjNAkqr&ylkn0mb~MIT-qU@F~f`kZp63mMmJZO;V&`-M2}P%(0;5u#j>dF|ijdm`#_xSZSGbd`-&E5d!ol
AXCio?^_qH0{F}aJa-L^3rIY^2#&h^W;bu7vP7D;e`6U{<PXvtU}562SmKEQsmz((b?%xp3CO#yrac-7B=e$fcS+e!Q1S)YVt3Km`O!|ljGaFFnFg3q(sZw6
-9(%5+oP3PfGq09Bg$jtL2Ny&F0vD<%lmI+S+ig^9D{UsF87ZAY0D?WFUJ&+wDiN(Ms!^0psPaDwrESgne)_w%_ycySSFE@0Hz`rVxS3dpe!)fSq<7SS)#z*
Tm0F;k>xrPw`p9g3U`E`5phO+E$uK+q)du`?mPB(=p6?v&LOLl$L<*dN4Pw4J|BQMiU<3TQ3BCclQmeP2yw1vJJIGLHeai@RHcSXT;gVNi(ucD*^aauCb7>k
<E;18_Q|d=x#?#lnUFpw_>PcB+fyxm(VZziav=1_k01(bXSaSJut}JkygSOcpzjXWKoSTQy_Zkz4xAWWF;*%|+Mp%yN%9zxQWzzF_iMVq^N!GCTm5f21&o)O
;%9ry9pD1UtTZ_HgzTlHhA1Lp1R{+AO901iwVz@A(iI|9qFHWF#aA+MB^6Ir#s5|aGxyyHJz=;AflBu@B`Y+v9dSp4GaAJ3IAhS&TXxwEHa9__9K!2gIYK<V
qK%c?lQ<$|@(<jzl$%7raxOTb%q(cES=&!MU~CsP<>S34ZtJC$4FzwTXPUL>>BpuU4Gz^|e8?PO*qXKGh-A13Q<Gfua@3;G+c)o?%BPzmruXJ}3LCZ3t~6Z)
R)axi?U2m#2rW7shVB`+gKH<Tnd(duIN-|h$A`?HFC^8rL(|C16;h^2ho2W|K+`%j?73MUqN_T+Q7i^{g7zPu!nBDqpKbwHbnqc(tacq>j*+OD%J^DRl`yU8
a=Z9cs6|FL*i&8qETcfT!i&yZ{&vETT)qP@eJqou8wL)=DCLTVw+5nI#;vtq+bb1H(X9gBO8Fy$jZooF!A`RZvaL*eP7<jhKm-h9@T%&{)ufJ?)$f&>!OfrT
wkjm~L@bO5-fA7u4k%tvQZ4dHAMISc=C{bUR{VctR-guv8@zVP9%*&0g&jV#lm;U(7fpd<SQ@ajB3TBcH3^bkQz4n4h0m2xK=mn)PAd;W%g(s~I#iVvX3HGH
RrQ4wZz=6ViAUK?L+j6?_Q^c$YC_~+t3hTDJ&)KaHLIN4;?-hV&%IJlFwP+}7g@;307W6c4j%aVr<xFcQMLnn3QG|2tZLyvM#OGi&{%5r$U6VA^>6!VRBfO_
yi)z#*mS&L@8ff49fV=55@Az6&H4qz_5R_dxN|~c*Z3re*bE?q%ZXX9h415(|40c5rSZwGtA5bv0%wTK1kDI$?YBA9VSCMvCNMNdm?3Fj_YTF0(JX3T&R_dL
t_}Mm^Y?WBvwC49J@viFO%KBRSq%-)pbs#|P7l;<k>tmI@i~q*0H%8*<o9zR)RZJOT(lcp=P~T2tWB7RGbkDhl$rRP9(ZYCKykw?!K^tely{V}r)*j!sJ>L}
;<?_5p@vE@PO$K-Hhg6)Mf!waP5sI{{L?5f8ji)QxpwJ9=2fzQ!HhWp|AVFucKSFf-S7GjfM$#b9QMpAkQ@|NNi?T{D&;<G850sDT0<GYz+5|-(41}f+dni+
6B#p*Yl1veEp6>qL!!g%tPyFrVTttM%E_NzN}$=?I5u^A(_eyc&{rH_mzvfM_6I;aQQ&OXej?&rSmW4uH%2nji~c`KuZRFm9MeoZ*1g{Jt{?iyNv!ASe!6D;
HkQynA9!`YcxYJqQ$|CEFFSZ%VBxgYEIDeFPbvgn1B8SW@WGRlqvBN4h1;rFFv3j!>B&LSK3x+OKq~@{8E`I|0L%BKKz_QRi-z!2E^z8Bx7+0?t#O_c^qQWA
oLJ%l2DY(8)?$TI#oT+>w~M!Q`MdkcmgA&4UPWUA7VIB?Hk#=n?#e-r#NfjjV&CRZfM_3QzrjWN?T1^}vm`C+|KdoQ;K<b_r*zId&_}VP#Yu9NeDeY0+ix^T
_4Uk6R%Z&E9PyZYJ0#i_IP>~^C{M0b1Bp%5sLk~$T&n2VUW$NzQ7__eqJ6~9vj$N_r%VW!(%B`XoyEEDA0H4;Xs&4x*0x7>x!U};FJ1NqM|a;bCRLWuR!9&}
MhJ{q`3qpukRllGABc`(f~&lkAgm%MCdb0d%5vF-MGHN@gbhl@YT39Q7m4f(;OL|~?S*@CA}yI6{(|>;Fv2U@5&4?#)i%CvmS;B@9r;io-K1jt1e^xa4<HtD
E#K^S@OzUmVf_}qdx)mr7FoeEhb$FmLuJlTiX)-~HLM83`2#(dKOJD&1{QecHCWQ}Irbuy^(!H5kG~s^ka`5kdQ>#2TzeiII?etopuqBaPsn8+^`eM~DWa}f
cK|kN<<E~^>Pv+5G8s-}PqK1`9)Qo&F)7jvVI{5AqR`>My{>K;)+XK-8nF*N7V=QWTmIr(DcmgZDJ6kJ6sc-I37Lpo4>oI`x}qjW#=kICqk;|}v8oqqTfOO#
?7pwttMUq!{<pq77I(>Gy^aehQcgasq+3sAkTg0c#hRN@{*`bj^+&j`(0=gfk_mj`LRCLG6{&LxRUJ()7%s70*w&Rybe$xyj*#j<wt|O0+7}KMU_QBudT8K1
p+2ju;E_@B{2IBrnj(bOx7@fWk1Qgs%(#p(7szr@LsqmD)&qOWes3nqD8Nq$?hZijBRYU8XAkccFCdEI3&Y0C&x-+Xeq<Wj1%>0vP4-2=9#&#%*}VSM!Dxy{
LlNY7fANp^W6-*i8)~&*=e)LS0z$&TLa{V*s1hpJnB={g)Wf%9f|Oh?X%b~4Buk1$_qkN%UUx6mR4Dg8^tLOeuf}C7jN<pJbp{o$-cR!egDqcUNj4{_#;*Ng
Z<O1IIQwK!b_s!izh~TPUR2FTa*^7+EU?oNuSbvN!6L$@6*Cvf-@oZ?#6UgC<#t=`AI;UTx$04!OmySqR)Zke#&~!y7bR91Nz!7<O@LB;J5%N~9gK5zy5uMS
nnjywL@{PtNV?Ue!Lh-1>6dcB;rEB{mLd!2D9oX2OB^*^ibe|7N;LUTw%Xf(T>WGHiY6Se#QJvz8sutckCIo^>v2(cpPxDcgh7OI{_9FSaaLH_jdb3cK4b5$
WRJMaNiq3$m{WJpQ*Ivm@*1_CYp#)EjvEp<3lS=vAUNtJ>07EP8<rXIYskZxH?&RmCJaPvHUK0*8HLupdu3$^-cdiUl^@yPrB0^j?g_{5;@Z&W4$nvJGI5vP
pV{~c+ZhRN(l4S&=k->QV%c&X!T7^_2Xxf3yqJ!68N#QOGqLv5ktHxiHddatP-4lk)kO%>`-i5g(e$2QEx|6j-9&jG{E;4trB-zR{ZstJhARUp#Vwz3Iq7+i
@;#y6o1o^2*FS9m{{$ogN^7(;g(`x=6#`#?;rs38yLTeP2UfsAAuGDUsI#Z~7^k2`GTE?Tp$kSRYLnZN(K^%8zp&A#!!+{9x=1bPtK(?Q7a=QbKij;8o`TMg
+L&FvqpYUCQ^@DDB%V^chE)n4O~E0eLc`kW0q-mgOFEZ<I^ha*{ZEy@@H$n0QikcbAyUc8)Auy(Y<{ZD{^O9i-i(U4+YyPt`r9_{(S+DkTtBLOfq}4kw;Ri4
(vAt4Qyn7`g%%n4HcPUs%m@{gw;!^G>X9cuy<$70Py}~6=)exeJj=c&jRCPEmcc;?K%<l2`ZIij*{&6#M(~3|GNz)^PKQP}?Q6e2rV1-l@38#?)F8HMjc}3#
P(|ZEDT>D#i|AI(dIsieP(;Ovz|e1#DH9Uj6=jBljD=T1UC){k7e!J5d*+WIi`x80O7RM37F6&z#VVsOwc2DUV$5}zv+rLgx*vkGlMuZBv#UFIS+tNvbACsE
noNxTyIC4hM2iR(Vo8L^94Qk*0Ecy8l`(l4f|i=t$D(-ge4vUg0x|{)u%fWbPS$ChI~aI>H<2v8o@J!IpWhbXhs9NGpMG%RK9F$g<+Sz-2s|6=$1d@}_qBIc
L!E6*QOx`Dy}~iwlC!qMrp8_`<Poc~+2%)>cq1y}MXJ1BdV$w^;5N3*=$7p&l!N&0o|=B@16Ck|0pCuXL8ua%=LibI1*o=GCT2Y`_JpZ77^H-G&X^b8>=c;%
z(-A%YtFcKRcsHurA*qCyFbWC+?`b!c1CV}AZr1ow3rL%DRjscut4v-&+N@J2u3-GZ0%fBDQX{FT74kvK*rz&$Y&|eA<4mKUnVNU*@lt_g9CT<{i;I2N^lgs
Kx>HW7csh#_w4}_-2-9`uo=1QDTmV!X56PnsI7VwDDt^8xXzMa9UiU&Ba=^Ym}=fiAqZK@hAmI?&h}x??klDnj}2NCQxTi_&?0xIxE^(;4#RtGF1lJ#*x*J-
K6H!N>k(@>d*R>AK06plQt~ZDMR&yVaevdaJ6Jiy&%S0~iA3oB+z3TB{*CLV;6{O08^4Kk`hL^4AGd@Pa@fz@B#viEtQRYHA77%U(RE-HE$}DwN8P>R%k8l-
*C}O8fxy?{@t>oSeH)21s-oX?%98|%C|n{T3lwffk?I2P)>GWh(OW|EQzJU~I(3FKngDp6kc-D;I%T*70)zj;Y;qttWa-vh`CA&4i;EF;Mp#zK_ZX$j+j&vC
DKu-O-t57?KW+dg?yMK^x!P$m2f)i?Rb*l-4e6jdY-QpqJukK2#l*g;w>{9t6|SXKIfddzR(i7rhvqeWn)57`ypr9hYC+gxLi^Z@N-`r@6rMHy<u%inQRAOO
&Z9Tmw3!4=xC+fOgQ>fo15NeISP#U1)|_rc_5Z_zrPd}lm)Dx3i2ywdB?W{ewvLtDKNH^vb^<Jl%_1{wwyl~>)d2>YD*%m%;MRXcTmyPET6b^>)nW#SX=5$K
0S0j9ZYC0V+jOd>su7vIStD_(tSVHy&8$ru3d+Upiga_9ZRew56yEn!D)l7OnSDO^h7@8jB+75Z`n_o8q!yn5J<n|)aX?lyt19OiwI|dreNZBJlCV@H353TT
)X`fFc$)Jc`fkXNw0=Y~KOmphZ3i2VJ6wRVcU(ggy0xmBo;#O|!pC-stZnD>F&;W&slyJK+!1BL@V6IM7S=^5CfLIfba=}QJrd*8`4Ntngrp(!|4mS4;Yu0t
JyUYSo8IHCjs`CLHfTEqXt9&Br9;SIdUoBqA;RNUO$_Wjn8_cNQkBUy&5XnA=-EtC9VxL;*A9YYY~<@;ZX`-Ww7Cx-G|bcN5kFl~v~<+S!U>gQ7n5@LK5Gm%
tQg`g7LrZY(AY+JumqNT=Gn9umLyw2dlWpd8m1Y~*x+Z2o^*<0PA8ab20a}vr49@B28mbdi{p$aQEY<nwO>8$l&|vFlFieyUzp4RcbXG``kExF+>#0qOB^#$
`?y1RrxbO^J9299v$OsKw0RG}_AMKS>Dv_l+0!HvbQT)AOg)7tDFT+P!tqI(x{H-M^X5ztzwEA8me}1(%SBPsxPmzr5Qry3P3);C8}<m;wBE?nZDK+%!(2nF
Uv$<*UvR5fytn7K?$=zk+eF@`{&lljPskVAAv#~G8h$pCGzC(AWPA+9`vWLK2=fLec*VonS*+d--$~&0N+mzCrbzy*4KT?Mk;OL>E@`{Zf`9@qLOJnb=)j8$
o;u#UY&n9g{;zfkK6cxFeW=eV$LKEvDtQ&pA{<IJ##Gdq=s=KDJ%w$iqQYX=ImS#l#VWqbtDTFn+uek^sb#oJ=Fns@{<PYOO^3Vdg_CI&@0~q{qzsYHPoni4
`_Od=?2y}$Yb(PkptGpU%yhJlSipW)7^L+l*SuSOM|CilSx|8*+vbN-s!bX6PuyJ&mliSHP&V4I%f*NoycrXYu{l0z+(!`XmSl*QlqCy%XIl_sK2l};QnW~p
Eo;Cwl^9)&*Y|~H4{Fn<FLC6Q-GiBfknT^V(Wp#&=?p_;b*v=NqNYyy@XoC0@f|ZauHjY=yT<8|=mo&BIJ{n=g*I<Ub6eqsp)(}DU=Hjb-~Oenj}L0E<uI?p
fh;3{AEJ7BRRYiA$RrDp+R@z0-xs&U-OB^_RwBWwSho5eZb5(3hm+gf;6Yk@XpWI_wsxO}mg!M``?MsvYonA1g%G_nT3(t1MZtom8N76zlK20ozRjcwwwRYi
ESWTJIx(OyA6uXNGER~@^1XVvxPUFg^9@&X%}w6&zOcp*SNnZ9L7Q;)A~=_U5m=${SYdohA&LgPZPRjLi4Wj+p0o=wpLxrh>(m1G1iD!MBndw28^O@y_XTbh
%?$dZm(VCViXQ)0>$JD=n}F8id})d39LR?#G11Ggv5A;6<&}<~+NouyGJs?~`%*^DoL%<*1ev72VlRlVUAauUq;sidc#Rbe_}TdCDcc{?qktrLOlAXqy@gr~
CZS_o>gR(*bP;9?of1Z-Qki#IL-CEzql5x-RSPQa&~R=K;ZokG+{gZUyWn9rS~gwwi$04?{qi%fG}vDSc&U4#Kc~B2p%Z;TLgu&L!iuMUFyWU3Su~jGDOpmn
_m$%}M;jfG)__M=`BsJt+-<n7fPhq1oYHrZrq29m&9X{Iy3fA2q%Pm&=Ah$1qGEQN))}U&jq5Ua1hdWsAUE}22E6B?_8qMKSTS72Z<helxERv_ZRh^~Xj0h*
l0m%H7Ad89LIpiGD7>F+P_&lz><}c;g1FS{n1vB?T*Wvy)_0F7gs|RaEJa}ayCQUYUNo;_KM{K5qlma1(S_#md3!D)76<4*Snv#_ucOJB-t7`?LUOS&FS#_#
2W~60)xazTN6Mha10#%5`+}TKU9N|%g%;L}t2JqFRuVd~6d7CPDQP)^&UA(C<g2g}FEo)?QQ9!v9N?wHREtW_)S8WYV!29igd0ST+YQnh!NVeV;87uAvDsg@
;gV95NU_AY9+pLWBwZ-z)Eo^MZpA;=aHwIaspRSnn6GIPtvCES7`U`Dl`zm}oE?FVwrnU#T52IH*=a$$@D~$2Q_cCN;hgp4fvd4$*I`T+%Q0|?=-~$71POG7
aaLv=aIdN=jPmCe+YgiAt%jHLjLm9kHyD07=CL>Dp!E-8Ku-G6{6H&XW$$*tF@Qsx8hL}d#^zQIg)mAeRd@^za+7e`K0q8c`O7e{Qx_%ZSCs)b@3_lhmoN1%
F=t;5&R{P|XM$J4!OeG=&2pb3hYZ~X0TKl~!B$gZ0w;K(>n~QHquc^qXh2J%?TdyK)o<bG_Em>w^a(i|v8m=&Z~F{Gd>KZQ8hdvkTa&{AO_(lz4IT;Yh`JC&
n=q2;pz4=%h2`G{-QX6+xFqL+FY_mjdX=$S)f?sOy*kxKg7Q{}ZURi><Q-1BKz_+#SSHVH769H|qPIXD$>!PC^!Ofc{nj(fs*FM3NW-V~;mm^pnp)Q3xu4Qv
N@}G@XT(IXwZ(TH4%INlgdbfhZN>>T$ol+<JFRF^GsR8u<zFuMQG#~RE*b?xHra*_3Hqt)2`9M(>bG%g<T5d_^?ZDuoN+SsrFaE)r=8ajfG%Vk>1W2Gw2aB(
zeRa&IX#_pBtZG#bm9kFq|8<}lcgM!pC4fi?H*|6A&Pyp(iTwQ<c0O)JEt8G?{jgG_jny><K$dtnBPw7z12&S`m~Az{1tI_bK8MVc=}VN(Cv3^bl|zdw1q<e
w^H!eDuSYhw3p<zSa8j9a8WQVgDGlj-Gvp%gx<IYllgPj-&E3|Eog64vQ7DKyaHZEgk2CX$UF-)g>pZqX5z;#tJ*ey#Qh1^DShe)2Td+V+%hG}u0HIDRoc>V
V2K6tK+vOMKw~qmumlG+e2R4G(y&XtY%e%NG~)pNvtvGH*0Is03hy}RybrIoE#iXFkv1@NY&`%vh)mTXQb}Mi@^e@%;LVgwGTrkT>K=MM8y3>I#V9}0_;_G6
!)-8dl70znyP<!3{zzF044HfoYoN@Bh_CK4I;IcHrh<GNvoU_J6vJ6l03dSl;kW+6MvnZsgZ$VZxAm8(Ai4sbXg~_>a7R9gt~pt20?L@ofS?!jc7`SS2Xf$s
{$t&XhK2_TcMF+WQ=daQ*$+nc^Q>ODTZE4_Y_vkh4}T@2jksu8a;r3#V3gr~QO6I$pZ#f_N3$ARx=8Jo6G_e48_4<{s13@hM<+&bkwV6wVd$U$MNy>ElMXLk
u4QZ;|4sb|DOAJ%+45f5W+fa*X)**<Yw-$ChqqorK9~}wrhVecG{r|wK)E05yB*8!w<Ly7>UT!qTC{p(`;ir}>}xSrggX8k<FnL%Ss&mhn=&+)MEtEo$_iXt
f~<dtEXeXq(UvBIqvjn|b`&omwrH-QGo_=2`wRXS_)ShKjqyxXf1<px4@4J3?DW+ChRg9k`;0Vj*%B~|@-)jtF_!_5XCic#qb@>pmc5F)S|?91QKt{w{U|M)
DeeO_)5_?r%k<11ep(Z&^<3E%AS&fQRUdX7CDer5cl?Z}I4b*Gpeu*%r}MM<Kp+?pDk4hZ3sy1x1@_uHzx9={@lql!X^^y`$o-%m>Upkj66P)r3HoN9*ENLO
-45BckbCtNg4CLT=a1P{AG=}fkn3&Fc0rDUi*+T0GPMh?Mr)m27kutC2sH+V)FOWaL4}&rEq9%&k-Aa}QYsBu;&xiuA!JCHuZECZ>AWv}D@oQ}LP5Sr!FaEg
%m8Noa8=bFlM|nKJ5!_;I8%8bWWjVXe=ARlPuYqF99P?9872obISZDw+mG&CfxsdSg`Ddg<9Bck-MN9+sJ&!*c-d@ogFz(g84hFdIG|UDdSf<(gStFA!_M*}
1d>^34bw^`j?FbuDqnPPevVf>Q+>JtK8#37k>6XF0Jm_KE(;_Yp;8vVB4rE+*iVmXQ@dGnFj85g6#M+(SEmk7^HIpGn6om$2gNL;Jaq|Zw;|SJ9x??BcZRkU
Si4mnIAt0+(^W@TlFSs5bHpSMxxIG`e2u}4LdSXC1y~x^*`e{9E$6|Pzx|-<IKm<NLE0jW<#I710~*l^zRh;JCQ$m3SC_Et<>jsS4`sU|>T9v_;+n(HrCeQ`
S5d)8M6Y3pl*1PYIiK#<zft^0>m*{HtWhjJqQgkR#WU)K{*&%volU(RvL>d=4_)d7rL%{MucTyjBB0IeS>0l5B3zQ})75SZ!ogSd;U7Vs(asWoTs`!YYzP<V
1vwEzsLA1ArN-TXee9WCK^k6K2vj;Rg;MvC34oM~Rx0N(N12Kai7Q?WW5B#V9z+;_G;e!6&6a{qiy)dhMii4&;;4*>W4{UB2$^ZRr*dYwD#4w|+z}3w+r5F^
8MeRN2U8B14TrsDV2@S8iukU;fcH-jngzSV$Q{3Ex!pJDJN_0QD2=Sr-lnKaR)yW4%$f*uj9fbT&!0dsEfwW645gR(wXr*3y^fsm(k-bc<9$$vT(E1MZl|=Q
{V3ai%7)`!QCgH~??k(j%<g2qy6+{_8Tz<xW(=`i5{r@VNEKq*lbHycnf&^@OayP~rcugC;de^2^Y(R}PFxr9F4UNsS5uts-Su*5zLA|)k)+f_PRJ#p-f|DP
S@)C>cO{@*q?{RlK4GxN=U^~bmRRS69zP1rACiW7$8#OdqjyKSHrA}(@h>mUN^4kZIjE%L#Ks&sxH(pOJM?;oCyp6bj_h_;vq+rp<IV?1p#I0Mgi{Qljna0L
$u~!>elMRi=7^q4_N_NGIAtr)=&jy<Q<_O>|93Q~@r2}P`VY%+FGGF7O;GC`#-uJ)r?H~+XkCTH*az{p`5t!5v|Oj!)TpgCMc0ix-n?zI(1W)IRC?T>Wf5bP
#r)iH=jy@7854cNQ96i&<8dU!+9xUipU(g9f=%4gqg<no@~PYvlEP=12ag7*h--H0Pxt_N=!2T7<L^hJy=-N<a^(F(ulVND78CD=6RtCKo%!7#ZqyIDnFlAS
#?XJIUzi=RZ#DzjA{Qol_568~;gXPYo9~v^0b&bc+`dCypZ|m)93viEB=4v0UY}f@J5Ug+)0MGj_|`6~-=lK*+9GZkpO9WT3~?^S1m?N)*XX`6$p8?!kLA{>
3WPpFLC!gelPXl5sE!GJcI3&8Of999BLO|`tA~K(!JvSF``sP^^e%$9c(wuFnFI=VzjC>ZlXQS|x!ff35oWW=p)6L8TL+39$3h|ri9r*S4p2vcO%$YeKJ1Ep
I&}Whazd>?yR!y*mLKh?L1Y}Q!MRHn`wTsz9T0G>saXEs@Il+yvvN(*tpN8{26L`2Wh8O@ope9wR<<Ij;^HbjWSWZ{W->3fb%gQT1-tdtBuP+=4VD}4V2S*=
VeOG{Qp(6nD%<RIeYMaxCxO<KqFS+l$W4H~;)Xuk46Ud<Nv=@TW}tt-#d80$`eIWV2J9t=xH$Nvho`5t1smk$usgJn%CW2w`PxDzbE_)mygTGS9Sc+kW8moR
5%~e>F+}$eSm+~~NLJd`E|}kThlNvsYWqgBy0*}W^`zx0bk7$@S?}W2C>_b+lzIK>L@hNlj$ffFP)ENC^yzB~9#L9LboB8+@xJ+_T%>=n^0Y$c%i6_JFSd}J
mb?oASx<AvL^LtzzhnpsoLgvv5LIZ00{g+)X7n%4Hq|pFoSXS-(Yh>8GY`%%w3rYW+famdOY>3G?l#mVUm36mZ+LUA?Y`^U(#(z_*Sif*kY2HcDLY)?SPj&H
K!FqFIaeM0*@;af9#6D*7VPL)-4cVJu}VqlcbZcB`DX(=)TO3sb#tnytLCYp?9h?gl|GiV>@%hONjBz*LR$gGtX&M6yoX#RyF-qK_#ayih|Y(`;)D_=D#Eu7
(dA{k?pe~l9IG-2Ql2QyBOYpE*<2iEM=|I_Rt`%M<rHL8c*Wvv`}QX`prpNP$S7H+XI|3G)jCV!VdM{bTxlJ9(V_&yz*2(LaYJDVHDZdn>qS-Fd@lmAPD7e&
ZBUL}VtjZZ4!vp*$qr*Z8Gw;~VGQD)?1s1vBla4|85R^J_t%hOF!;kp3oonBR3A+h@oHeZh+<-haLJPV67a@=26xhwd-uBR+nW!BU)o-GrXK8w4$Ds9SC6*6
E9Yiwc#Gx-;6Vb!i8Sl=$ta3@&2!Hentb&3Web<yVxs;Ri8vpRJjeu#yO)Sdk+#p?Fe(aO%T~~KpANh;3_lPQ^;p;Ghd0S4{Go((1LTl7WZ))+@OqfYFq}K5
e-CW85lDcT4R?$%;r%41*~(Ei>n8@k)k{d5jcUwr^nMgF1Uv($#8V;#dmhWh6nwIpsT%SrGua~OMt<*Zj}YLZzIu2c@>I2AVQY!9c8m|yH@8viqgFSKWTLF8
_5rkGe<djREWV76vOPF;`}>Pz&^K?WHDs881$riyBG=(=eioMR$%h^noOXd~X}r^=U*pcklQH4R6*4;~N-T1r_(=pcq74%u9LUCVOo^}0{e5hVONx&Vx`maT
s?W~3OvJ1qBL2cWHJMM^rpw=qfgv7k7wddMcFSqVvR)&Ss|!Pe%8NJ(M0T~GOkLB08*3T-tI&MPQTF;4r88%rh&c7J5_>s&Dno)O2kwq3GE;}ihtVnC7xyQ&
k6S*)qx?>GPVK!;ixF}u)Y*!?kz1WAzjG6D2GcRgZtQN2No{PpEew`^_F4u?Js9W<Gi=t7E{UH-M9rU3A^b}+Q+3a(Azpujw4lQ7^I}-p1KW4Vd}_6o`>U0s
_mml|(sZd;O3bqM&8jHFJyDu{4AYPayHDalciF2+&N0<Nq(`~!oU@C4wIOzFKe3(5m>Ua1Y5gzW>RJJUetAn>Em-M5Z__*rUgo*v8h<?X$pwnW<Yk!ZNvC|Y
Qw(9-Nv*ALqE%}%WVUAJJArOHk#xd}2A<QK0$-OOOgFKKlwSbBsP+n>QQzCoiGX02!_tjNYdSjE|Cx&7SqrZdVtE5o8oxo6t=C)XTh!N@C5L2f+(b7N=#13R
98T%$+6OS(6<ISbSC=}vEAB9Dv3Y4t@X8a$6G()e&583RK3F>aR&c>x3oQ1AKhc1&a&=p=Yo6&&hv+ZZ`5O{}7_fM(Ns}`wVxsIYO8ZYYmIDSeN(cl9#qSKu
_CA6ogx3!xt3}z4nX*RbRE^(==Nl8;)UqLhWpZ9cPzm8ND{piEC4pywxAU5g`xLq{teq;>=n+jupf^z{>@s1ODze-?$1vO$>tPlh9%aJQc`NV`lH%L~fjG;Q
+#TNf?WHFLKfp|%K#4|v0Uhl`ww@1IHIqkNbv(f}{)2$+`(0HKYqYQr<tTc_Ta9FsY)K=ktv#)gEdwIQK+Qchph`Z$J(}u&Zor$k30!NzJ9QcLm|4|nOsy=~
gYxwl!dCK;18<JHymWn_CsQ6+V4;LvCU7sJtpWR>e-Gl&3l^{&mxTi}B}`UGd*5<^>TjiO=_#D>t=*t(HI1t5`-kiGoI~mgP0|8;bN~5B7?xbpi1Io|@L-vw
mEkNoraJ=^#c?O?73np9I@?7AkXzZHoWj1zt~VQs896XY%3gxwG9Tm1sbZ&hiC*xTd8cbeqOb`%)$7X%2Q~{g{mAHCuH?Aj#zivCtQZ2%9aFceZ`mp+4E1!1
x;<win~8@oN~XU;8+AhYuCH=;1zCas7bf>fNGG`8TnpF2aTw!X+*07(7}V%h-$z5-;2UvJssp3F=cGjlzjs5sqXH+day_?<V4n~2X4`akE&9zunGsAm(?^8}
^)5Yq00B;j-T98W0x~v~lZBB~XgA$_>0F%5SJua^aq`?N9Hfgn8dd7@VCDjE0h;N`>x)XI1cRoIsRHesH$hb=Z_u54EAVmEHUVMw!84zj?he7Kl_5iw6P<!@
M2V}HZQS%PCx&#Oyq=+vnTMk?S1A@NnGILRBv`lfqyWOar!!`9bt(_>9r0tiO6Z7rgCe54zSF`?hwihIdLPJIdX~1Jydjii>_VOk^p+9Q1X5NMTrN2{?VC(M
4*9(giJ%VEel%6>L0wo&p@lptf#?I2go;N8DA`e=r`~-5tcKc!?NbDiMUaTo&Si3U`BKBn5VK9H2cfB(wiDPD%Hq{l7+Wig8VXhhE?D-dA?$E5pLcG&XXSEy
5gt43=^IRVQ4py_M7+E`eh^02pL~<HU^Sw9H9{HRL!ltYqCUvL@gZWfxS6$v6odELd(!;8P`gQr3t+~;`3>>B(WA?X86JYXy{co!G$R5Ws1y+xEgSFVe&9nS
kCY59+`X!ZTCZJb$ew~@U94G{0ay%KY|~l#BJj5ofHe9oz%O$VBbvM#EKGvv?hLtK>HF7Et<UVOCGQ7*@I?IhU_&;~2h@aMmtIK`u`4_cM1Fh3n4QS(Czop;
Vao!i6W49D1ZrcJh~e=(jCjCTogWusdpoQvK2=eA!)3oF2MnOH@hZ|dtM{eXcB4Yonh~NJ5AXeT{@P))-fEQlN;5YYb3M0$Sok4L_y1UG>+{4!6DmVM80ENj
mylz^f^4@<g1KyrNzYF1*BWq9w?hsKp+UJKhc_uXgR<#!S!X1r1583a-W^4*s#?|qM_asrR-NA{X3ZU`Y7Mj4^^svS+O4p9qg+-j3xcuehW=a&I6f#u_mqWb
0tF1RJj<YwQJz>@1SMTyqMjjHWFOz5e?wnDson4Z_;mY|1+WjyHyZH7X2q4bWn*$8BPAP+OV&&8=Hw>9LUCnhU)KaLu{B}T(iPORvR=c&*R|!r1Uq9`0O$YQ
4f>H(?#`{Bx!<f#@4eAal_^|VTF3X4mD!E(lfqpdm1emVVXY*D(}jD-kXrS}0IAuyoH))j$;<M^$B>E}ilCY;b>@zxr|A7_T&0Ja(qrChCDo?Wunut0dk>&7
l9R1#2L&Pp?3pT&AK3hhVCGcBE$|?(D+2-6YmYuWRhcy^1x2IcvWv*>O&++X&UI6iNWM&?{wQpMPDxUo?8-6%f#rv-IJGG^QLKd7(nO^ClULVORi7n-Lgn(r
@xj(P@L?(n%HF<~@MvBF7aMCt%7oo`IzT<_10>|y5C9MLU*?L!-o!XLx98I!DcZU#zmwh2s261YbNZZ<+R`EVCbwnaJF{5qJe5%HMdHvw0_|G-xWhb34Lcb8
AV=RcQi|}J8O27Dov9Eyw6V@tfl#p`<73%z%<bfO-lZ8Oi+@nWFonJZmux(^ftFMHw0b$0!NnYr5=CCQ4Sgb^&iPIEUm?U(0GN5p(RLrdTC_o|HKrC5NFF4~
@V-VTnu9q{G@F{hAxvh}>9T4*WIf<{Ze+j|v`-+fp-ZviFq=1JT;HSBd+gYdc-a70(V0dEHJ^emGFEHY$}NvNYuMf&Yd@tup-xRA^7ym8PlpP_sWR5~`eeys
rtRPmOv`9gs)E>yY8&%wQCU~%kLApVfFs!Y*jR!XXEz=H@u=@`pmO01=h)68fFP6*Qjcn#Y3DEwKN||K9P#@sWW!)@IghxvM^7rZzruL`HT@_3N^XLqs*(OJ
e3ueH>3}I%Yz?|+TPT0lUWYhu+8UI<B}#>)G|{XWh>!1@qNJ&qgEiB&SAimBOFx{btezrL?3IsjcqsbEblABkT7PXq6HdD3+#zss4*C0d3~fdNNeg<{AW`7t
S3cQES>nWF$o?mHq9YYYZ|yQzb@^<4NUO`w%l#bmuMqX>)0oSt068wuz^@?T;smH1#!G^f4Vy+}^SH@D5j6h|1UKk1SI`bZakk&|={(i{x&7F6APx!GkYc?h
9^h?R$axC0z5t}H$nb6o=6o-TN#ThcDos)ZfqEBSv7l{Aux`*Oe460H4pC3QhB@{fnY7#i@WK#YxVY@scAra|<zlz;th158>V8yX)ItSJ`dkixrZ<21WG_@w
l`30sLQF?O%<6T2vAsI~lM4&OW%5;Z=A`xmRFzuz(vFK!omC+E=zgmXiW!>3nxkwSvTm0WTpi%*BhoRohO{$6bx1Jvs+FD+Jbo7p_-oJXqu(m}ysCT=o6hbB
J$ru;fZwbf4(R;}aBS03gS8tZYmF64|979ksMe-K<AHIeer+(lU!as~1Wiv>*SbsBeS4HBl(fsS9W@Fc*sMD4XoQ9P$=%1Te}6(45zYbZi64-|%>69<%kP*O
#^ljDr9||jGT1KG?tQ-=!D^TH*uyPpSJ8|yYV{p*H=sRMLJ+9KHL=sAlC8~h6d*5t{J4u`1Kf8<f1~e~W`@7fm879kVBU>cMaYb(4WlGdY7-~X#imE)0|kLX
wB8neo>(sb^H%N|YdGCG^Wcj#eR!i^I}j6f&+^$vTS+Na+4hA<H>#t4f-7X@tU%opypGNHv|PdJ(YL>E30&54S|y2<{u8?joEwYU+Q%A_x;F!0n#6$!$dXsc
<RS8Ek@nGVTOr^;7dZk)6H;9#mG)_7IeL5=si(ZEhqz~VbeGG2^!|M<b{wGJxldY=b@4!38ddHSd#s9vfEihwE;6Dg{pX~N;XUYoJJuFig343rmb$7XO<)%Y
in0qG*)@OD;YOt-ZK<>ZoyArrL&L5AYrLDRh#4=<(n!0#Rvkkn?So4A*tHe3b)yb$ep4F}CWa?;MxX+Z?SWS~%Iv7UIj})*dZVDtPn!=a^c`zG5N&ehu)$qE
V~3Pkl1re>#`v`YmdtB#^Z53p6R_1M`JIJcC76bJ;LKBLVuiqtO>l5@)eel~nB%04cmO@2ct7URr3Y<UP!XgHTfTF8(R7MHURy9iJxvfhi`ib>7ne72Pb#>U
n<eZ!y0LVK@=1LAr09Wt?KUSeARHgw%w~4S?2rE*p(i-JFaZgX=IjKQMLstsyOJ@8?w)62A~4#=S2xQHbeds20wprZ<PsHwD`gu~8F1yb@+CweB2H`yyBSNY
BW2?Kj7#~<NV;JVm7V}K+OHuu8H|tBQdF1c=-6x$SWy@ehO8`QNYxU_z8U1Bj`etkz<u{shcBR!n=Er5hmF8biouR-yXav#^^_=K>f3|Le|Wc=kL04d{~BMw
<&@_;HjI&hL50g0==t_{q94$`mcw$6NF+d4xPcJ&1*Ke-L_i{oJ7dfzby_UiR^HV{)W?F7H-`m7nKQc%$rQErm;I(TK#>@({fE@(8z?kPH|?8#;N}iR(kQ2+
O@0e&H>vuQgkLl>Wz73JlHPFTDo=bZhWYavmU!};j=yz9D=fUacD+L`RF2NA0)gk&6+c~G>P|GNTnRaxq($Q|hFUkh@C{P6Ni81{g+zw-XBC_<_SiCeSJFvK
9CD1fdE^DhF0fE3G>%vJUl%_V6j|>R^u`gx5-ahAb*0{{yZuqT_Ogem6i>0|Vzz^jsK+vO311E&<!wdc0T{n>ehVM4DB%XVcAi+(#$o^wMEZ1YIJN2L3rWsX
>SLNuJy-QTJte{l`47xlT!AnGEU5*vF?YsTMddyq<R&h*3t}y4a9IqaHJdaLs0mfX!&ET)es)D^$M5Ex%C>MVoKj=qNB%mzrX7*om`N7rn1Rv_OEvk@cDv1~
brWs1PUts4Lnpu<pC#!X!d#>Y`2%i`OA*`)zR`<G|8}5&TE|j$x_nY1xOCg@fFgYf(v_KezS`R3q&p}P(oP5|-&0=N{v=q`zRT-6{iM9JU!Y$6H7!l!iG+i<
?*ld9+#+j|FaZ>7@*lf>(!X*gPC`!Y-ic%mv>D8!rEqjT5c>EyZkI9!s^4!S+iVgvS1F#G%kUr+hzor#J`T<kaapM@sfi=dHP!I_)Ug!32x-|@%z>H|kQy;j
PI0|npjZOtTP2Biwfo#BOy<Qe`!5%95g06=0_ekMwcT5uF5od?)*YWnB=P+(1^tfaacYPVIwpVUbeI2YDRjt-!s_Sbc{#u2#t^a<3#G;0lnYT+1(EH_*&@0+
^xlW#_OU)z9X;&_E3fC84*W)o|E&Q1Cmykll5nU3Ef~A4H%sSA`@dQ2F>;a#`Hhg`B-B5u`|>d^&C-{xiCBS#6mJUZ4#x&P2`(b%9#!B+dHBfMKlumkMl0H%
!M*yF9Mw_*$G~@k3<W;L6TO*dd@uomHu>d#wg<kL1MmP71)^8~CNfLI@O*}iR+Ww+9Cx%dTh8z3+<MyugQrqJEa5&L=XLc?gh9mV@I>?n`W}kYZ1B&&LIS@Y
BteOxs(bHVx6}#^(vEX&Xhh)UEYzLR=7p4XvpnRlaKSaK4v!R*pVOw!q;9S6pawm}o~y`}S5ARR_vps(zuCXWa)%(LiuaYb+XR!JJEBG30*8u@-{VEPDDzU$
nFS1jE6LbIK&*1GtWboP?HLb{>A@Z_u$naji=!o|zhg+Wqr4y(>qL@fXGKp-%2w)Z#6xjlj{_^B)h8((-*=Ic@x1+x(J)!zsGf!_*lU;;xt0GeCVD@|Mkzf;
KsYmgiSHoTtVMi;Qxl58#JJ~G+(((D1cqo#wVj1puy1hvJyX{xNxnX+*uW(r2C{OENh-pPp57oohB$VUFzVP`Y>d>*RGvI(D`k?D1s1fxVf(WPL!w{7Gq9X7
Q7$#J6Cx$^l4}c)WpH?(ZvQ4A{j8Lxs}51SpHT_Eg5=x84T2@ScAx^xv-&LppS5Ypw;ho>Qw#F{dO?w8q^D~-mrtF=eO+c%ix>+595{#sPa38=%ZCT_o4Jpi
c|gz!LvYK)Q&r|#^IyjEW3Q$=8oqAOHO)~V1ZF1ij4yP7V1(W^l?nlrAp!DBF+)71N)DI=8r3cHZxcZ1SalP_AKFK|y2}C`zAsU_ivOiMJ+TOjr%TD=2%HZ9
HcJ~UnPjz_Q{!&D!x^FWv09@N(Y%E0B{&`LSSYCG>2igO@)2+B>%`<_jgO)@p-~7oAq~>fh8ev?8`=P|UI@cnmlKt2vcq^tgyd4xrJ2e!aOQKrf<{i#Jj%`)
(*_focqcR#8#PHqduzS2r);K+7vAh~B$8jNQ!t8cQijknCa2zK84+u+-72%>dx%Y)RVD6gwR~vL_B$rsD-WYe2LKVK>vRvE{;VzwvBH&;uyma41A+DatXxiN
e@1=i8U5w>pgT-zregY&za>0VwaK1r^v4ho)fyIesx0cJBBW`yU%J<-t!+;G$^@*0+rFx^cz74&(NHEdixo6Dc_1wphe|G2i#WnJC%0xC)lP2M%%Mt-*3JK@
ZMcOgnh!$GzJsXEv}p?0^GDbD&<vzf)^<5jJjD2UCU*9|!l*t({&%{b+I_yQNPvI}@-5NaGaH{4syN#qaAA)%<mQnD#N>q^=Y1Mudf9l})xs~5BJiVg$CVA^
ni6L=2d~qNNu__!Xok9HGTv^kPS+CKKvqFv!Jq+!0kHH&-_SLF)Pl7Hsy&D@(;4o6-EO4@tY}DBiA0TW&`|9dV8KX`{K9#|jiGDX<65M*?3!IB6(0{6zEekA
HBfE0OhUHv9g<>d1(FXC+fTme@6)Fab}LVVB*aj<Y4RXJRl7sJ@0tpeJfDnbD)2UEtP=%B2rk25`XmelGcIhTb9@%2XjTjg(UoSpQE~C%-l-$A6!I;mfvO2|
g1V1!L->tsN3mtHH=sYvu!Qu%yFlK{$;67z+W((!MtQOX?W2d)?R2TS2qB#b$&-57Z-<iJoO8eP4<ScOx_g@c_=57xT}r`+Y<Ydz+w>vL+Ft();wH+XY)lry
v)Dh+k_c9Tyo=4b<WrE`8p)yMdY9cQ6G{}JP}v2`aQe)Br<NMd`)PSBQ9I3^kpO`GvQ9;P$*IM#6_rj^;$LQa;=xYVYOEW$?@8@p7N{=?yWlp64%V(p;b|r4
hdD1ESV=!j!P5*XkxzvB)Y~h-ALnt4EL$nPZYI}(Uw{=)L|=gLlLX1;wRvI7Kf&<4o*SmoudGl)W*Xu+)LDx%@5@UEVa@AH7q-MtZuWW0%#3zciPJhjQmIU0
PkuY9gD8cNKmqQ{j$0lHILs0bY1-1=s)-R<S@hU7B&R$@#oQV1!?rp;ard{@o_sol$Q{q@H)WmmFjBke!&{$@e;ZxrwhGcWp2a!^hHd?cWWU|598W5%QIJ)A
v4RgvmoY4-6MuDv@0K5Hwuy}%&ZC&A<6|6%qv`domdq9meDG>7n6zw6REn2+mfHoV*qN+73)`Y(5BJ2F@EAxWYm5F(;1w7^^T1<nH;ikW*g&KtJr`q`-7%$2
pO#7Au`anYB>;s(M|lX#kq%o1*be$a*5Ds5$uwiR>Q5tB6lgAI!slQyeITR?>?{f}Y9BY}sGeE~O#%Y%5$+45^tTPH>|7GstXG$3;N|?Y5rZ2#%|#MBT}(t_
>9=EBJOuCGFi>tHHb<^zd~BXohErpiMN-8xLSkoV4{aXo7h@9vPS5FzbaPk-qR`ulo_67-4*D`C_`lH}JE(c$q#Cjbi}Y7+i!>F0nF^nAxABrmW{Ng$j-3RO
&h6`~VTL39@dMZ2RZMvFu~J)8JV9LtQjid|PY>?rnolYhOlCpWQ>*ptkilW}*B|$WqX01Ls-(as76Vi*8GbfrqMtX;eqplIITQ4|)?0p8EC|&Of$aL3#=mz1
9_U8I3eU%PF{>`m%){2k;T79zLi>3c<K+BJXC4XV9-Jsst2zROgRAnTJz8HY=4+ah#WY~j8IqkAI&9l-_FKs3Q28ysodWUJ3a{kl{^j>s;9zk?Az<<}RQm}s
b#pcZBk(#K=GLUD!gG&z+mF?1m;*R?t^2f&gODN>+AFo663ie87$6V)5@?A3f>|~7Evi`c!1Nr$3L8ecD?+lgs%KyU{lPq-aP|J#j7mdS|D_|_Y>=5Y7BjL6
NsU;B)~*iavK6tMs;`TZFD;1Q$&CZ*b+*iBbQ&S0EQO~5EJ;5iMk|COecCcp=?+;$@}~$%#!aC2iS37M3c!^@00#>U#(>>xw$`v4EIL8<7sey77WZcifuNzR
qPKG1M;YJ{JmcRdU|M&rs^wrGrj+-h6j;AL?^<?q(^ZXAyXtCyRn;<-t3u?L<tJ1~PN$5mrd33SppygH-J@vF{1g5!eH^^Z_qcjjy+JI0WcejX)jY_YY3Xbs
nN0Tc`Z%P+uY&GrU%~UvlL3b<wT@9-qLHnTi}GJTAqBYJDJyNq?|D0cb?8J;nw=`|*aa_c9et`hM~etyar9aZNBWij66~0RWGt_kR%O?a#mWgdZgs~#tRx6(
QK(N|<61EL-4Y2M@U(Ai-`;jRHpbF!*@E*c1f%-#LOM&?qmo*7O;=Kvnq$Ym#K%DF;sk74*=p-FPJ)AtR%nXNdxP}GC;Pe+bcoF7V%d>w-S2|#H-AjjLuPAx
`3<4S-(|SY%=%09cXyX0L8@A{q#_WIwYW3Y5+gFf%IM<Z!vqzsK2uYRladQ!!o?h<`F6^WAGXCruPlgH|1>#dhSAF~9u?Eh9KpWFSEuvd2OsP`8`X<LuGbIw
;bjpljLK&jqNGF9(%65OXWd#a^Dp)9om}V<CQU0ED(&QUDAu(BwB+xE$K4WYp#^A+;8<*Ujg$(tJ!8cJg(&7S<L9{JR1xh7BkhDPKknfZ;@JRv><|KU_`pK3
QBCeh2FMAdRF&e2%mCikY*WG=VagoLH_C>exf!%+?@m|mePZC7<ePlDCs*<&oy=hROm2f)$noj4MEHa9x!1o&&f|SG7Tl@at1FLrv!5LJMU%Ephu3n*MgrF(
bXe4T8`ZBcN_z_yHvQ?$rE}lGfZnkVM?q;;N~-9hsM~o3l_brkeLY!P4T+Zme%fytv{m5vBax6BUNEd2hWEPBNH^<@`;mh!tc$lyO9$GM946ZXV<Gp$7zl(W
%w~fAowXqmNb?s>H`<t34Yl|2=0%k-_O`SiQoeKIr8wXix`)$~xw(L~Ui5GEkO{UsRRw!#NI3!mXPM|hazW<x(LYG2<Jg=L&^V>e?n|brR@%gsKRMXpDZ_t~
(t8*g#Y1IFBgt%k@>gc!$(Xs)QlArQei3tI!6ZIFs3+g0>hErLIC41gaX?4z*uVMlM=(h|fAgF9^kWjjBGI*m%VrW_`=s|@CYDO*sVMsd7J`O!Z5$vs)vvNc
!7rp|NrzF9tR}zC5Heq-TO}lRiYt#ZVnHE@4Wc?E+${2vH1dP>c6|xo$@><oSZ#*<%@A9s8Donx2{-)mLyVdS$RKO)9z_~ubfc36nV9dWwf3}{M=G2kwQR`B
RzBd6PFqVTdl)fBwUmqShXVDwN#cbR(R`Xts44wxkj_6iX5_5pnZ2rW=(CSeg*zf}Zbi;q=?ik7k>9KB@%ps$!F=GUxeJeafAvH%jkLXLj9+jr8q}_{(Og%D
ZfhkiXO;SRx9e9yJ`9?l{z8i{&MOzr^Q{+*(epx;v|kgV6T?4GR~n-ru0S5POk_$j5y*WA#}9rharP_15{H-GX`OHbI*zl<z0O~MJ)lY(5vxvB&_o2_8uqy5
=cnuUhx5P^RDqg3`_E$F9RXi?R4WBJEYd<nTKaGm&RPMIA-aG}Ax~o=TJw+nRw{qWoFI7xPXhzN;R+TAq4~I2a>NjYqb%`GIw<x_f(v8}mwOCe!C)7N=ctF!
AS^{qn2&gYu!4CikJ#*REClw}&*cavcH(iw4J$CYRZ0(lCm_Hsnsc;i4M?aXYcz6z_2xD<{X=_~9!vpR)Rdl*G@pt}4kYK4GEEVWx(tBJlYEs8w+QEKk<W~C
Qm7jFcI3@81ZAOZo@Z?w6w!kHC?N6tcRV)uy}fGD@y=nCZ-d}1pOvId{;8w%nmb_KgX)^^t<eKP;qU1YOa1`xkC7^xeZ3>USE=hIDaMaPWm^)3{y$FZmU2&H
nnOg>T8Oq{)Lv2sMcBnXfIO<5YDi%wAWx%Sg)0&os>HdQ<v^Ykm4rx=omh52pl0@kyH2*++RSq#G${9CB?8Xc56Jijt<MT1<rW%IKi_|RlzndTYPup#O-hv2
9e;`>as_?o+6%wHnca($2KN8a7^79sB^WANK5V^M`VGGM4K5L6qj+Ql2t6nUT30>FxL}KUIpBHK&2Ap6A<Q_Ym=nMyTIqB2zBRP8Ud@rKq5*zkcT0*g4~cBz
3~{&q0fiti(Ro96Sffbc=BzD`H8Z5~&NXO&0f_b0+nMcZojETc{Fi@u-}den*$lG*b$SAL4yQaoGs=&+Fa9=6fP8{s>foMlfRwNRgx*m4o|WtuFOeN-y0IrC
D%D%Bn=ge=PbO}uY7)s#PG;s(gviq-Bs|UF(rP}mV8u%uYdQMJ@OvSwCI%re&3j^fhqM-RH3H`o8NbiOP0i<@Y_-<VP)`nuSRcc<vic%J;jp&1FuBTJJh2~5
^`(Hlrh3502A6dB2*A*U)1t_9>gzaSp6YdxweopDek+_X`B2vxX{~AylCcKWD8gc>()m>AffH*tm9?GYxT%Ql!gXG-KfG@sTf-8_;<q7!AYm<@YAz9$*I1o)
-P_V3h>C*tU*7Ym*&^T;Iw)5~C;hw0J}8=LQ9!9GXa!iM=S^apat|!)PP-V!welmRMk+_@CjL^vK}tD3ix>%?pBKbx<k}*oqx4n(v8Zq?icvOnL^h{`Tx0nP
;;r}GGG)~=z1J-aAGA5QwZBFUnv$;b%t1ZtK*NL^0MxpL9W+*{D~)B_!9sJzHR|mKiLpG?%u!$>s?esjoDE(Ju|RfEca~SSO^9XSrvSTBm__UY;Pm~}47hr$
A16M=M`z0Uqd}hH`1gOQ-_$Mv+7Wg~T(j^tk5-YY5sQx#6!fjbgsK~fr2;%a<<4@M9=)I9Z06;vhQ2^&nBnZj@aNx-Tg!qqOVpoIL7cdfTxb1<Ww59lBG`H)
4yF5DBemm@VpE$U9+oL+5hA4Rb}kL9RczLEKMN3FIa}$gy9ury0%Gd0VQ2}^5napt`uING!@Xu|4rJ-74c2Na1d_E`sF}tBz+Tu?iEM^Oes=!aPu;i$PFaNz
JoVUKWfMrhS%k$0MvA(SUZ0aZ)E#xQ*#cOpUjyy+lk9AZiE`5imp2#socx;MWJM9EvC-?uhNDxN61%Qm#-C88Z(tCsAUGmsUWWMr=7?>fBz81**wGj(9Qg1y
`_DJ)>ANtYglEokojoWZEa`G-OrVTWK@jZD+m6^_i>WwY=QMuD2dne73%W0+c!(uf&+*@ws)Z%BY$e%nN2pwGieQjiFd==NH5`?L=F3h?ymJ5_@L1V-4x|Y2
6*T$vJi$2|0?fWSPla}Ai7kT^H~Z$-W~i1`8;W`a{Ag%EERrJJazt?;#k={4C?sw&s-W;sdu)Hr6Nh11js&q}L-c|de=$Fdob<>js?9uXv3{f*X$gg3xz0kG
Fp|Y!23vMb{0b-CG14nUCarn~h}wSBcu2_#5u5q6+R<(a-B>YXOd-NAjd&pQ%+9igR~gpy6cO~hW@6kp$sPO*a?UX5D!{$<LldB`S0FVodEwKEJ>a>rVnZk=
9+*x7k8xT!jFfrmp>7*T72%MeHV9y$-cx|H=l??Ao~xrXRv|hwuw5p}e$fKm%EN`5$QLR`NM&`x<&wLh-@I<cAi7<9RzC@v#~tU$+`WMtloUjY00`I2%%9TA
%6rkmc%V_F2uC>YDovo8K7|z&Rpf3}$=y48-JN2<iY#PQAs*;{baL+DhRG<~KFQ1_k75o-vy>UOM^TBV-34G1l0171Dp&%jIp<!6?h5O@C<;VLMwnnqf7iOc
@*ngr!CV!V`WvL8>)j-x=W408)ckCVKc%E`2*6ZBP1SCn-L11`_$;$LA94n#BD;h6ze7|UV_3#Fv3-0(pr6`0#5ubn)NA=P$aY?rREAn@>m13UqXXLP{`0x0
JI4xY(Z-w}ck@T-yYTI-H{-gB21UJKr1={B0F@t8(XjY#uBY>cw$pK5X#Y#(|9TJ#eLZeE3?E!ggE&Tafd5Tyq`s0BGy{s|<&yzAt{txX%M^Zg-UvvGSa=Ko
5aJjr+W=ruCu+Sw?2g1O;35`ATN~G_>1%4?x_YtckZObK6G6hJNBj^%zPN3!_jUKJ(Kl*lkRdoamI|#u4A2#F)P591=Rs_EhlF6e4}6Q8+z-9lLh47L1nM)I
e84+5CWwT9+4~OGWe=4oNIGw39;|&i<~xJ8F1{s?ZiLoo`>u#X&CH^grgJ0#p>t+^y1@1g+G(5}%xoDOhj7q4`I_Efh#BSIF2e-pbTTQwkBp<T-DjfiVyxMs
>}w0@pirFl6Bw+BJe(O_o4ptFh{?s(MEyFay%}Ou1udg>i;c-p3ay5%tCmoi3UCuA#||$uH50FvF^h{%0pg$Tq$<)GfxubywTHNohRL%cnR9416*If~spHEZ
ReRT*%Tk{^C7hODK;bT1o;90Yx@ly+&h3>BD{Qu}5*e^Utzx^f_ym)9o55kH7}CsixPR#5#TGJhFK!wtnl0t}h`;z^OKI*J`w!DaLwNiA4vYpC@%KBiyNGF4
`zZA+hIewNgNYNIk#8{ehV{v%f&XCGndRirN6ixkq9}fg7IoAcfGR`wr>X{k;oZ&@HTX!C*k6Ti&JKMf#>pW$-vi*FPhH16_1T=Np>^Ul*PS0a+5`rLh)LXq
3u8gRv7A@m!bh!%YY-;#!A5slTF>m?{(yrNnOIM#{SN~9aE(uHmw+p$-c5XL+A<eW`KrES+tg$e=*pu34}sI3z%S_J+A1~x*aGz({s%fSsc*BCBmv2b7z9>_
u&#0UnQ%M>-#|AldYpp1ha*xm+x`y#k0Zw`TwE=9(;d-^pO+*gW>v!3_-NiQ>WL?vk8+G2$X(g`=R6|0s3<NExZjf1fGE|(`N-=#->2z`^ybRjDo#M_NB%!|
-&b+e1n<06R$-`b%mQRvjP8ZjltJ<~xUCok-E4W2*Dx8T>nehvg9533CZ7~U0Q#t%^VmOO$fAKp!Xmb8UrhIN(;8xUl|N+;AGv_)s8_5m(L>EUA<@{nHO$#N
>8;}OV;1mWq#(IWrny>#0!~4vDbh)uA%Xm6g))9BtPso7sQcuzu^U--ZZ*6>Y0UY^JS)YoQ3!YVhpXzU*X*$d$R)|jToMb<GII*BOgxPO^g@a7KH46<5WJ|3
6~C74BTJOu>axQfy7?jBp1pz~Ma_x+=gje0q3F&WLrZ9k>R{x9IInX!6tZ$9v)VRny8tn8&Mb(BrWya==G8_!;U{87Vcu)UAaVy0%Q7_O*s>C(({OX*mfYtJ
sQJEpd^({9?mo3jV*%i;H-XSZlK*cgJi{9}qMEfiFtc7mhq@f4fbp=bwFE8N$VnOvuhi2RKjK<Et8+V8T1|XccRrMY%lRYf;1LM!u;DqM{6=I=SZIV%wR%*+
CgY@;h}+i418l|S(BL4NsSIF-fnmS=PYniF;2?0U(*E7%Ib1hS5h$eNh{I#=I<3``<@#mL5)QlF&kg^LB5x1gmM2%2eLMj%rC)tUHbQS63i4@Zrd2S%$I&NL
VghXTJ_;htIl`dwwTO42dlzN_j~_InGlH--`Bp+UpP(RFJCBEi*h{ertapV1KgZeRA0V|@_>-6r{o`ZXmOW)HusWBwz3@+ez$tK=lQ2I9ezl$ZYwhY-aS$A2
r+Nzh5HX#=o_*hS@!YZ9guI)!KVVjZ()YBin7==MO@22<8Hp~cXotS%!;7mW4H(l1WP@OeH<XADYFrB|r~k_*w#5@f(Y7ezF#M!c7$j0HBOqS-siCH$5_6pA
Vh}?WtrfeYQ%!Y0e*nO8>IYQ*L~TZ9O&GXK64`R^4oN5nIyfk%gy6|#+0px0R_w+&_sYegp=<*CzfFq<?G@b$S{x1`Ecbq1I|82cTcc9VO%uwE67n%3i10yB
NSX3mW4Shf=jK>~jA?I7mkJ5FhZwscawG(<t$DT`as{U1BLWZ%A~@>-?#FQdP-5Ym%Fpn!GFAy#I|ic?E;>-4f%#RfmfAZ<i3dxjl)b3pk8IB!*krjnx+Ldy
u3T6mllH16Ru);u{IB%%DhvQ}`rU8T#4SBvNi~LdfxZdHSc)cZz_NPC1LSM6tsP4BCaQ(hOhZ3`HwBAP^^h1IqJ$bYBGvxT1}~duu;Guk=0Y5^=mKGZHS{LB
>(tbd;x#Z;L{;ZPuJTS5Et^Nc2cY}xvjSq9Hwd{PM{-=tHoXE&(~trZ@Mn|HGMiyTdswh}nClZhNC=@%%#V;`=sFR32SvMi9g;TuQ*MvyP;g5J!|9Tq^0%h9
ky=9<Er<obRVNH0%BXSeV&Sq_YTaYx>GfvHbWP>;-B76I$6@Y{;9~hrw-@%kqIOs)VxL>mXXz8d%S34<#Ka|5+<p?^A_R#vMT9RjL}&+ye@29B-?<~8%M%}?
;qn65rF=G7E{YwFI9eYryl`zs2}x0migAytn=09YnIZ(XL6j?Q!CCsHp#!Y4rZkgr#gS&o9MwqBh8tkp`B01%GrP^iu)Zu5$|72PQg|P>L_ZpO=-apMT$f+0
UvR}Ow``&+??mQ1q2$}IG`JAnN<RzTOwvnn%I5mmnpo?Vws6fBmUX&*Eqrw%x*pkK3$DmD2xn<QF0${J#3#}rR&oZ5qxV|JxEphNHQ>3tn*Xz8pbC+;Wfg+6
#}DaA#C2lIGZ*gnciTw+R9Bb`T>5eub-KV;<|UaW>p9;(eZP9#8O8ZDQF>}6bSkZ=@h{A!j)2F}+xU;V>t+V2^rJ_GkJ1>ku*=tFEl0`$qSSUtQ{<bnn?FWG
PyDYe)x%+fBn2br@J?@13Uuh+hWrFtiaAgSH9DA-+9!dzzUEkK)uDf$mT^mmcQ*%|ul*4tl(V3_!EXuklGC=T5aAHRY03_=@W9WNqw{>2LbV1y_gnT33qW4c
y?EWtpn*$d**ei-HmvKtk_r!R12rnqMEAOr6}k8P@nH+F4q<N{f?*-Wf4TspzBxUiSPRE+)k(W}atptKNUZdbCDWN=#Xx~nv(W|#e%r(GuwV7TOEEttD)>zf
w8E=1Jxv#7P4$KN)fmD%0@?fA?A+CRXAqjSn^~t&JwB7B<0NIx(@b4sy*j95ZgQFaOvOH*3GFty^a$X2%Pn|0?PxVz(?1l(0u=ioCh%_>f(Wr@g@6XBj`9$M
brTN=xZyMj`<CI5AWZ7m{iTpv*c{L+c(-wY;;}$|0><IMgw@~L{vLWatG6SDG4t0-8H&D8aylZdtKAGAgCQ>$>O6f*#@8QN{{7+F;q=9N^QIr!;reynvBbV7
>{}t+#e=O!QHCaElf=ufTa7AYs{b&oJ`@ss?x!%l+{?UDZkmlCRly!>N27^eqdWjSOy85uyyky7;c@92jzgTXd+zF=aHR)X2H6{|_r8$V)Ci2^&0C{T4*@o5
@(Std5NQtEdhf0iHaAZfV#+MfD1fQ<isHmz_oF|RixZ8ye|EFLy|u}K%D-^tv@Br!r0`m^d<CEP`ms*cE&VAT1??F`(naoq7U&Tw?x{NjLGN-U`3D(xgfm9=
wd3VEp{(*TGbTGhY47UPmOK3C_dH`%J{!amhMWqTCsx+;y3w&-cw13K@1o(gd#bOTQ8cF<6G<SQ-}|AkuaVPR(32m)@<Ygus#jk-ul5ABLtV0G!q__&xwx_w
K6U0Hs)I(?4bqA)<;5l>pnxX+p;c&-Pu|kzb^+OgYOxO+&BL)B)!%>W$hm+SI$G8xTlai;O-%B!_BWjEr{2rYl{>^HdHP=)iY@2bsa8U%n^UGuVIxX}QW>{h
aSOcb5Ti8Q@bFUrZ;5OD_Z66yF&X+~U_RVldJwLh$kP~BTg+!Dnum_#eK0f{Bf&GL^)G?%=Fljhn_5FE)vD{`;MLPY8g4c6JB4opOYq5wnsOaK7Je^#SOgS1
$K;L+0-OQ8T#^oAda-f3R?hwrF{F9JovPe@G5cM)!yZA+-O498?nJ{WA6v37;gqyBSp6&qb6?avIZI%+LEt);9&-Sio7u}O!-D}(gjW{_4deb!-gc<vj{u$v
FPIc0$7trI1VK9y8pgh4E2brSZC>?9V1g^ZY7Z`&I%`$cy&Co$v(JtM>g-7Kg`$T}@`_ePAh-68P`;9CZB3hV1vX`jE=G%*{YuGgBbA>pTaeWNm-uUY83ZQe
LZXs-kioeDXzhiy(+u_{c=JQd(cE_F^zCKZeVd4}Wm9MyqVz*-G^&!jeq27)bV=Wl0jH)pJMr&1+dA#Vs^sD&DvrIzB2(^ev@ygd9W<dLQg-g@RC^zt&{``D
&1{~96e`)Z3scOB+Saxi{xZ?c$E_ym0@|bNGfBo7mS+d)V&HV|_M98r`5$w4^AEi}9?LO!3EaI}p;<BPsKVK@C(VCl*AbN(5w8X8VwpKaoXo!y7(>VA9y$%i
WydmeJ}@uSyuHkE*Q5^-rrt=p$!K{7>4DE<6|Rhd@f^WAJOKSJk$C9*{G4Y-g53PM`G3v+(+wS}8eLy+<xNz5TvqQF)BQmUFc$1wI6h%cNp-a|H*9dgaZeZk
+&ry6tMkRSaboHQ``nM%Qm@%o-uP_!0fd#zCAjBly3-Eg>M;oel$jadmfN2{pz2Bzu<OB{XsgcHL$_J;j}=5ysH#P7ySEI+<nw28b$XNH78xLA4d10Ui@7lW
E_w(%u%s+IapOKVpDC_r8Bo8+U@cEN#Yk-14MU)9IxLIX3-M6n`<prNfvSP{t1B&h+wysF_4k50uQjhpa<X_YF%~22!q_e7%u$Og>@OYP_vNe6ZWuRt?4F~c
QwD-1=tvYCVCG$f`Wrg#?=R--#7%<2Ol}~p17T4~H)+#?rQC|u2QCRC9%G{}kWSvS${!L<s<4lWM|dn?C2pXxPOJ}E^)e>yVP7NC_ZpuAk8FQ&{RE8v&o;Y(
7_?ptR~QV^=C;SujOu~dGn6N_t?M4seI7?YyTeusVtgq4#rB@EXbX4{Fe6$e-vYK~xG|<#3&p7`;Rc}`LY8RXn53(JF{SmpwuY%zc2D$*Ls<(B>L?WY9mWlK
uUl9z7Ij*_0Z84ah47Frf14rQHNuoSfV{by&(4zr1n5#vx_=wLKdwctSR+A|-b_6sCK#oHdIKljOJ5yKl6zZw2>E(KPjeA8fY8;&g^0M_cM;fAQ#lUs*bDaN
5RXb)fw<N|y_|vrz%vxGw_Od~tbr`7sk)$RrQ6eLkY*T7S7<u_9=kzCT0XBGmC!7f`q&isa>vP2wWE9q_fO?Jui*?-oyP&7*$&$gR<dO;R*Tr`2j$uHYLh3+
+Bw~D1M+Se-EqZ@?0qe19caOCCUDN4`zg^*O3}23;_1#pcT!pUw*JbXJ<zGv@EdRzhgCIzr<04Hcw;Ia!b?$T_JQ6G4%ybCXX>#jJdZ9uPpFWOY|LM|Dwkcx
c9X=nIK$M^Ljz1jdf?nBWfZtm62SA(Z*LvY?uQb>%K350*<kH;XguS$l$Dk?Gv&MBUA$?@X9CeLGnP65c0dai7KR&qPf3f95q8sQO|}gM)@QR#tYv&o=fGRI
uGD~&icahEx-O1Bt4^|UJmz5I#M?`bep8?c>avd4ngMpK7H(B8QtT)kP};y$HptRg=_5+h6DbY(?-AaQ#|Nb|&jYz}ADju*w0D#ajM!`7y~{UawII*|A<Wy)
;!#*A(?WNieC@P-E`}Kt80uY-Xzc#hoO#6|KVXu=N1<S#>7pp4mP|QeMA*_@CZI}Uhk+o!hK`J%2dzI!d#rJ-Q`!$J?~ICms6ft>5!MY!Et?~bl^=h;3kY?*
azGCY6mdZoaYt{Kf^yteZDSDh;XHy19S(}wXcYJqB|NHgk_s$L81O&fa9e~~)M~F;Lagkg&Rn+^*VX~n5)dsXhjYB7sWQ$)Mktaf?P2~J$pzV!4YK1$gEjsY
_+w-S17;I92CF8-z3xp^?Y{q@>hO}GKabBs!N3E81OPcTK)zw?o<r&9NQpAMYp|uaUu`jqO6W$UaRTJWEf925b_340#y9>^uW)*8<B@7mJw%MTm$MvpS=p*5
v1cd;plGFl-0V_6v+9HAk&A6y<@%xAaN@~%c1GOnO(;QT+?&aLhtQ&P#x&|(`vA0siGk!orzk@DNDP>zTEb77p?XFH9jvm|K|oL}F*`S~RSk5~5&TwDps^XM
I}0!?&>g_!|DyuD=Zo4)pT*X98>r0A=|GvI>vRie#)`E*s8cqT_0(GBOXxdxswtk(;0pqTD~B-ccOszC@78j>HyHXIaPEIERd7sn@P|KPZ_i$v)#(nP)Flk5
u)iL@f;qVx&^k$?EcK3Xb7p4tadWA#7_cscmObLw0jYp{d@6>&dWtqVDzW7cxLHOz!u|e&Z2y<ya<g$pWFY?9O6iU|`68Rr&6tMVUy=0R@w4ahx(al$)*iuh
J5P)8d^xZ7TE)LOi}j^qqL9R(Y+^Vta?N=o_2Z1?7nUYwJ-Zpag$c?{+chvxWZyo)K8qPmZ<8A*vBxsCF??H5gPF-OQm9fMj?Rkw43b#{TqEFP795)HQUu~I
txtU75=^Op_@6&C7g^)0QRhX7!AgluG9t7x<^}0!@)gYK+1m5A`o{V!Ayqwg5cYUy>KM`*Mqwgg){wk7SMS_470*z(Pg}Yf@!QHb5>p|;khVj;C>s8Tm)v&g
WD5zujK>xu{Hx{52jvv4UpntSResf0)eeSO;w0x;?rS`uA3TS=jBU3|BZ3l?n)XNXqv>9bit{?Gkhtna&*sk)M@+cJzYTVDW;WpOy*9OhK`d#yG<Ra8%kPW0
&pR=8Y6<1l<`~_?%1W`Ro+oEEyT+k6T;!`f0lO4DW?5&ds_1-|OUmi6{RQ>AL*5+0{vo=*B4{DU)Sq*g?m2Ej0Z}|-jQL;E@L*~4-d2<Mtdm4BE{2C6o3wBp
m^kv6uflc8K7a<5bo!9uaCz~hQ9Z$$c8Es69626Wl<Z+&=wdmsB2U=9hwCjum?9nyH$+{}g#BO^;N(BLXvQSJYYi*>%I?ks_1?}-Waxbtg7<86@S18|D}8MQ
4Idb<+zcTvcKkf31@IpF(T}lvdQL-bF`AWjTp4b8r||NOi=BAT4|rrZ!>Bp!VxOdvo0ZnvqU*4KafEiDcR2;|3>M}G5@6O-t_QVc7(i6nizOt1kF!AW1SG}A
b@ZQ;4Iwh_LG6u7;833>!y$p^310$9BhwU_<FXxP+k~fo!X@CPRz{ppnShHB{FP_n>1lisTg)%ZBI1qGy|2Ty03OvgUfkEsfyyeoJpeSoI)&_bcGVE9qao#j
(eMYQcM37g2kf1g47zg*5E9nmCl?(|rZxWgba*IdtY^EIaWi>Rr`+V=U=57mYoA8tZ05{Jzf}#}5-5;1H0HNCP^RP~v!~XnBiR<A-H^z5!!-WFCBAaT@HNrA
<T=J;#7Hz81E}}|e|sDd;>=OcmGzWT32(2EpkXdjF}|HOb<n%kZ)3=};&qN4DvdZa=KIsz_|`gzYkoxc%XhsAo5(uBO?XGdwqhJ934oR$5HPG?FmUs(ItU7M
@CUpTgvy^5vpiZNa#a)31aRbWwxZ2GKhVO%MoAccvbog{W{mZYJ>{M+`K61wW9G{IevPonm*5VKGDK~%bc7mk{P>N{E_L^c%bhAwwu%Ht=IXfD$CyPI>*11e
5ez^2g@JgKKKmE3oX;-g;jx+z9X5P6{21dLIfo6B=&V{w-^l`>HeOQF8h*&_pgZwe;YSP3*Msb5ql$E9$sa=T_Z-F<(&;U;Gy5a^qJt9YjqCNx7FcA%`P8dq
n}0{fhA!AfzvFS5rM$Bw(=&tnz$YbqtCE1Axju1SYl#FEMWL}@SAeH}h6#~sq$-N`_%s|KSA5<H*QK>gs@E=0%!g4???^P{^Ei`%d9|h1D!bVt1haH8sMn)h
I@UWXFvu=FIPd=mrj7$xYZqKVhrD5W6gr(0g6?_Vjq)=Vd|OR}d387BH{d=a<vFG&qtAfy#aMJXF9}wD`Ph_*5}PrgHfIA~i3?@Af#%rJI;O~V;b<QQv(tC>
-~UuhIiN}KLq2Ko8*NE85_t*nfpTJM`!9+O`1lsUoXzNcE#K{@zAWDZ>jNd<I-pz*CK)iQ@b5RB({)U6lYGMLL!{UQpRsonaCaEU;^v9QTUzu}5HLld+5*9*
C+pt%my`$eUr<~KCON|j2z8}cYuhiQ9fV5`Q(6PWg&>Mt+4K6U;BCGUjvT$V`PKTJ^-Oy7+YkR#qzX7DNOVkc2L=5H=$hM9{{<UUQZ$brZnpZmHKN>$hCI$7
!(Yb+$^l`MadX{%i-IovG4QOrQRAVu?Zh&pl-<H6gLv<M3?YK+6-dz9Of_NgEE_B{E_y0;J}rwz^#3zzNFc}icQ9)*zaR>@Np`6PmMKZIh8l+%5BPcg>e^c>
sLy@*$yQ_6CZ-(j+luNv0%UX9Xt$=o_IiGD@5GxCcn7A7<S4@=tAJZN)O-<udU%Cv9-kAwKG|>*_`N3oW1$&v5a1}*xAQ=|S0|`n-vs119JAz!97~q$hfgP_
@8*aGVf5wYuQjmGhVFpm_VkDzTjxlg3Lb(3pET0u$7W+vPi;2WG|lA?9i~Th1YQP@#%z$Jp`^to@|Y1#hXVoIuVG&J6z-15{Sq`EvtD{#db1Jw?7N1XZ4HCb
MX&g>jb3!-VK0#M>SRDrBDG?5cwPj#DBe+UiLJhkT{vWmTT(mKR9`s=1YVVVI{i4NQ2!R;w19<80OkRmMvGC;xpy#AyXFIRkyFi>r!BgBWa_D_#uFi{)z<Kj
Q`#V;Xw?>j>W-Lq3V{&M^iY(ACu)kjD|U1;5ghjQebU-cWBD|5?CC$D#M@A{4juXPm%4cfUC;=(HBmn>;JAa9SWX65yR=SvPV?I?AxZ9FSt&^0mwrVmGBJ%@
FbIXAElFy|@}@ejFfzdy*t(atUW6pAVWk;jY7ZZSHz>TgJ+CSb7!2hEu`7%OwvQBIvlrtLu4=HX19;0LQZU>zomi*I)FTj@hkf&8u}(mL^y9kgecuWo-hKF*
AZ9Cx-Z_u#7=23-(C9P>9r4eHhj{FW9`A-8XsZxgu1~O^JNWDj(juG(K-sBxomLw=sQCkLb)r5;lT_tS4_rC9jV_A4zGD#S60wJx0D$OVN5>jg)deC+-4p89
%^fN4d+c?w+4jI$?LsAU7>hm@(!(DUk@%x&)q|$dg^9##?>BEOb~fXLpoIETR~v_x@0P|;&efZhaseirT>B@RV}*vtqguXNKXYJdnxDwAm8muZy>)T$l|WTZ
<<b|T@g6L@A0?f;Rz^YwkGY)REAXqkwf#h&%7Q4ie3Efj$V#DtZJ;~Ye63Br<r}v+c3Sq(xCo>`^?xyz*SUb}wjaoTMsRKY$}geSwZuE_9(VhJo1V=3K;9Ob
!fsA%=qWp(^@9Bxrbd|4Kz`xfkZvql8lX1leO!eARjr19&U~Etk0NlyqoPVb>pH&=TYi=0czE&G7BtYf=npt7P6JnLNVmgVMOX}8Ef|wXE_m677)W2%z2jLc
xS<>Gv_mu7BR&(vzH|Gkzj$}@nboJN4_hVu_{WNdO9kzMzHa$%T7>rWQ$j-U9iMDQ^{yORa1u6CTrq^V_FJ+mJ659nTrk;!G{}R#VA5E*3&#X0tbL~rxH-y=
NE;GcPCv{JdYd8~4L)zz#)L3`-QWPwi1C*F;2}ut6(jbJ?N}<95byH+dy`wEQ9grtx?#o!=UB50))tP2E8eM&??DMDaBBP8LO@ICaDF^?KR~ZXFNN?zRlw$E
)_WKSo=Id=yXn$dgAR|aBXk8bbB40)*u!v4{knyHpmG2}q<5vx6DbkT7d=x^6U|)vXq99Iydl43`JB5Y{?Ra<2za*d$~6VP&>_2UTVxRb`_`UgY|}e3RoKtM
u>&-hO}0&18I+fLqeY+ky?x$VY8r{nF&6U6M>vOLTmLGG0N_~4*z}IQE6h)q-D2dADx71z$c>&TAT3ev*u6qz(R|MzKf00;WV&}|kxY-oTKyvP5LP9xO%7M>
uT(_3bNzX)fC;pB$1x9wGT8)N-%QS`9wIF?uuPTtFrp-zap>_{j&c(R(J!>U^sJ~LqO0JtPx`8(xz6D;yUsbas09<bRYCKLDf#!fkpgAF?9GpXMQr7o5&8ss
P;-w)v14jkA^zqiylz?4qE+bq8f2n*eh+<H`N1|@w=|B~WXNhl{~72WM{MSb(RzpU6Y4%Mlw#(@GP0iXy1Lxx`{$;f0T3i%T|}8&yK`seUjRn1U&`uV7thT$
EI=B)J}TD0vHU2{t~1K5HftH|DYLeZ#cH~Tj)g1w-;Fh9UC=Ft7*+;>ncE<B0YGi^bhd~W!2jwnqhNZ2qMcuR=aCmf)h8}~>J}^o$@@k1I%PEdV*T<*xRQwF
as1Sp<H(GmsuTS{e%98KZwvJjEG4gh6yTsW{``~++_NDOb_v@`_!~%JwO+-@@j<Il(uiJ23V^wOsOMp-Prp;3<Z6*y#aZz&8CzY%ntk2k*O82Z=uZKdc|&^F
gOwQ`MB}4Mp9?26xoqcds~izQ85(QPufOVIe^v1W(;)qf{)xOdoX!AvVn{Kgxd74V6nE6B?l=&QT$(p>pE7bYBERlo`1Eij!@mRNcErZpx_fXsVrg0o#fA1;
rL>f}I6@c=B@sJ`YID`&N6tHpL8>^LDf0g3=VLb64lgwV!H*c*&dJXjgb}>vh*(#LZ_UEo+iEp`kH5YlAh=M^p|*=<)-l9b3Njy0TD}^%@6l0n@k+$4GuN@}
pewZzepgJ;QPBQQtP~YB?r6s8zc&q+gm}hLkOfAksXoNo1U*kn16<8-E_i(BT|Znyk?ZGoH{rwT2nI!p!8~UH=LWOY;DP>G3`Isv8$&goib&3xe_ugVG~vf4
WhEO2jm2i&CGLUA-+zbMa2)YYTd(FYPwO67mMLFQ?GmYWB^2vQzi&I4K{q>cmM~8&r{X$<hx=4TUSyS6boj2ej<R^`U!hSVc}iHAm`!L&m&W6Em&fUoH_GT0
`T)USiJURv&|A7jI!9l*WMPqpsM8ano+W<|HY+!Jd_y(pMB>`_VB>L#I1t@x?zf?*^G8PdPcZ_BJAnC=3F@UqLHBPpBSKo4W{P%l-RYXKm%CEqh4;Mj+7|CL
)WcYBbz&i?8%LN;F?%c>VM6&Lw1x#1KRs=cp49<yW#?qs)@U21fDigExjs2I=}i<Cjf4J4ZOpzDRK!gs=yKdf+v=rmS%NR6zUAa4Iur&1U{I@}0h#CrxW+se
z!#vv+yPR|JA1zZ^d#R3pY{_F3)GjTY!w%SKhJ{T9OM2e!ZbUAbeaYS68#L7%}{fRS8Hfx$N(9D*s{5wyWwrPu)Zzg{hy#H8=kDOO?7Ul43Nbxt;(YZk5&C#
9HpG!az&|rHtalB<NkBBMWv;U&ZSHx`gLC1pi+etpLf%@MWkP`lp%5J&^IYX>sdTJFTTJ7n~{{Ey#V#r!|ST{)8V5{OrT;L)S1m%r-xCK+{B&AkGMWi)rT7W
r~58P`GF}nsd@7J_m>`r54#_z)AbJBzC1+bqO3(mEwewKzmB5-b$XR~h8}1-!nvP$>Y)x)OQHbaRD1`9A^hVanp;`!FDv=rB}=M+y6az(N_X<!0~(4~!|1&;
?tG|inSn<tN=(nYQU_Vo<~jhKgSMSQ<NQ3V#KvOW=uDDkKY~6>#(K?ziciBQ#rbeMVvt;W_X|r5ZhJ|peJM`Hoj|)_;Y6sUL>!j=EJQp~_$-+MV|8F+;>+_?
u2`p75w_3@;EGTV2xXMk551E1trr>p9<5;UYDUys?}oSZqpIZ^uOX&22fbC&U{8D`A;|Y~2iFtMvHs<Sjpb403FVJ?fEHR#UqB8S(wYqrj94_v9wm=YUZMQQ
1GLHp9BM4?J@%sZX$7w?wmt;O)Bpx~fu#ptEsMQYP^VYY-Q_Pi9KvGJA;ZTzRb}&ZkwUJ>DAUEA^4%v15b;Wgb!EiHK|ZED4jFEbwbKLan_}0Z$w{Rv>BM+h
cWb$%58V)%z9%YH;_%6cVRI{=|E?Gu-PEDlbs8m%UsHqH4fGg*dr>yA6z)-%GPDf$o<)>7b3|rbW4K;_zG4zZV2xWNok9_i3_JCEXO2;k`Yp9z9#fq~DA@zx
9Xah5qYVOJsxaemYx~8=CQ<`w@XQOUbz)z%?~`*Y7p$s5e;OaV_971!Rgsurhh&mP6V}n*<ghe-8hsCjpn`d?!H!Fjcna;|jEEa)#5qT`@!t$KwP~2Nw6iBr
?<5GDC|p1}48V<x9FH+f@uv+}btrINplrhDJs-H#jL9)j>lQ+miv$ILzjYR{4WxvSVp}?$Ej*`c^SVr`9YlrVw@j}biZ9bhSAaW%du<6>IE1M76B*Un{Fmt{
!Ki@**2%X?IFsZlfv{#U-y^OU@6;R`O^P0MNefYhD7b`{K<C?9ag>`zqKYdl9eW#^G?1F_S;0O<o>Zc+Op(GzniTnBS6@CFXx_XX)>W}&nryU?5}~+hLgzs?
PKET7I+ai%nw?SsDxgbC7$s>(6&u>pgAJTZ1L-mg!G4OrBv?Rz=Jf5yoy<L8jLgIo7Nmbtvf#m|AqC>T7_}2{ltS%;<!<v!!mO$en651Li#ZBcM0Pj^*fObs
-j2RxfQKJ5ID}`-{MOkecCHucw!gI)JnF?E<qxhqQ&$+^T^5}BmDZ(V8mEGFXQaOSoMiV4VUA7A?=gBWw5CNk4QvcRR%MW=fz27v-#xhakVfq&zXJu|w&yP%
@)o1>=;gm@z<n`hrOwZKYC!b;Ix5PUkaRi#u!xO<J?@eYdHv39?-eHQ-g8|<IjFDxf_C+yyM9AwLq~c19=RwR`F^j@)cYQTMKD<JZw#2TRS!w9gJt~MFv^Ak
WH@@A&cZRhE&hTP_iJ^~mk2IGQ`^GmZ-n99*2?(;r#xp@Sw9lAW3t`F_~ix>7d^XuyE3++y~J(K^QBguAbOFHx6X)Nf1KjDSG`CXzji{OB+X@Q1|3dpXzcVo
et1$(@l8dbeAIQ_zx}H+U}c&XtiDb71mj(=P{M6OufC8(C423s*|Py0l6Ek#q!#nIHxLv%)=6G5;}c;8Oz0tpOG%?JIR_{MOPplZ{9>rsUxjD)P$Gh!A05Ui
3(r$JEb|R~VYaq?AgWco<`YVPA_2l-DlX8?{e;FRg`d2ti}u#RrH-*^fnD)PU09|u6TzblEI>??;knzEGlC9Adoj8Fl0>w!tZKRcAG%Q_`e_wkE&fwgnjlu^
z$Pjucr5)+WOGvd27Nt{vm8vXP3SN7V4NQ+NH|aqeO3B#F-nNF&qgq{XBl$>?t_afL<R-7V(@^bG$Z)1Wt>P6B^^4??1cl#SfHuaD(M#xZi8~p9)da`W3`*t
=Z!xGTChN8k!Neb0U1}@Cv>Up2Wd?Q=s#@Az{s!ur(ai+@q_utS``0>--tSlebonPjd%Ds^Z!^%`z(qNj%JrEd(N~RxFZ3>dow;jW`}z}*{CEDL7>s@?rcKy
9YdoUpPQu_>c~Hs99TmfSO7F+RnFQ=<5N2n@G>BJS#pX=tGAeVS%rE9)Gh|8*qD)ntE<WGo$;o&%Y>2Znh%&tyIfqEqV6d(dw9F94y9kcCm}j*tm(k{GYhFe
=b#;pU-?SzjX((;HDPY(0}QLpo3R+=58P`m6X8A@Iz1lcx$Gi6GBf4Iz}N|y`pkW6MhcjTdNI73C%Rsgr=t%VBX7=o#yQ|-jQ*lTbO{{N1Wi-8-dkuU${e@6
h)g^<!a?592;Y?rlDAX&C=wvx$)Lv@o5svj0-dFPJmRgQ>UD!OcV53lmGm8!uBdL^akOP&SFC>_9S_d`G8;Q6X)6%ikTaC0WW7t71hH)(Z)1JB_v4H007qJE
g8c|w(<9r9qNCLk9qRDGL}HwtnyOhc)5mlbKs?jvD)&1*d982Q_*K*?hKoqs#Y*kbOiP1UkVtj7>OucFHn6UugZ4hppSM&p9+>80y3x%wLQR3(G08H1Nn0;w
mASR85-6}14&;Q^e-H|Bk^~qZ;m~>43K>~70#vcJj<ZAypRycLI0t!N5CdLIrn@ph{MeKCQ4tL*Vxe)@b>LczN3=l#E{+FhN`DAUK>aL<_i42+xj~*}bO!%W
u)U_$VhQ8nv1ynX{a+3kT=w0jc4m6Ww}#Zhx=bp_iB`hvb@PMCuJk7&y1X_b{lsh?O_<puP{;YK$L*qrB8eel#0;;=<2aB?x*go>0%0P@KNsc%BU@_uX0y#y
rD2zzE_K8=Cim(ztWuMYq@u4CXYt$kM>QfS5IU@UH6?(Ky7{Exl0=FX?mYEAyX0Se@Hqklu7(5sk&Z6<Z!QPHyPsUj*GH<Hq^N-tWrg)wYs;=yeEDaMMlYO_
?JBx!Ng>ik{t=f1_-9RK>=Lkv>OhZM@ykI<uLG=0hHiwuuK56V4=#ok8-B?c<MqQ~+rB5!*t}bTWjz00?EczqRG4!PsJoRvVESyBO~;kjz%IMaM%wxMH90z{
l?P4eL8N~?PeF*8x^$glvH9;}>*8>+u3lAw_$bnSPF4Tw`nE8P%0#eTAUq*NGR-QcTWDrY(AKp_<L?ddN>ZHy3b=iaPPP;$omM{k!n$I-`6#!EFzHROv7u-8
NXY!i^%q#>=Ax))#11OJ-N6}hkwDGRZmkt$e&}fGE7I?R+%!+$Sj)U4NmWzB=}A;<pVS!=*66Y4(0CA-WSra709;c6f!1?MF6@8+RVWXx{Gyv4e@7I|riz5~
S8tWKb|YI@l67@9<4}M#TAx~|wsEikl^z!k3)=*qWP4?`A0NYBUZ4t{US$qXt?@UzX9Vd-uO|7|CvAJlNKd8Jx}HQtoIc*ep~bV6{ef=)pHnlJ{7YZt_sd*0
=rMX*Qs#wuzh+W~i;y^7;BD?{f07Gkch&Fv$XZ2PS9$sbRo-#$K2JRGD?eR)gqBUisPUF}TFd4p1dtV{xv|_7<E|fI74yfiI0s43^|fmbs?j|uh61CFeU~pF
G;%SlWfwfu%9ox@$*VqQ@7uzk3erm!C&~hb*oYF!d3cTA&OBlv?CP1G8*xYh;jO`3_7oHnqoT*Ki*2Sh34ZOOblvYOWNdKjRu-rzVnc-M#PSUnim6<>U;?#B
AJBH?q%-w6I6YQ-gFc`Ig#nCBZYj^;NP#=CmYVm0R3NB$Yb%7*8eWfmTlu_6p*;&O47<mlbcA(^1%?@FG#bQfBVI#?n2N0jTp58=nTlRqg+YLx!ZJDkiO|#A
{MK@W%|{nd3ksEptP->CtF*wvStWAfxHE)szP=!zl^^p2VY8cYB#`Auir0$_Zls6790D7aY_8#MS;d-$t41XJ*mdhpvWH3HBs0n;OS)L=V@dT!0vs#?SsC*Y
zA0N3DASUl*RQ(SnTd@6KNu;YdjFQdKa$Z)!pRcMxN<|$nO=A8{}3q&E7Cf?vWz)s+Lo?}Z7YL~pn4f?HO96CzQ+Cb2e7PQ5Vd*VvH4*jFU-OXcR<AofEBI<
T>0^ruIjx?1BkIRlkJu@QWx19ce6F$x$*vk7lR=cMBc{}<yx$CxSt!=cnv?Ua&J|-BP;XrZ5b48W2N^`dp5-VPzy%(;IU7|?hi}T^^eMuJP~cZqH_<Nw8hBf
l_xzGkV3?S_TOobJD%hc%<MlVcH1(HG&F-{a7{u1?{UdP*H)0Sw6)N;gqS1JLJ~4&<+Ii;P(UVfW#Ep)GlA50X9pm$sI@7c1mrNimqwXlsT0buLkkYnCQ&3`
Fi1$@C;@ybpaK03w|cUtbWvvNbM?=9RDx`K$MEcpsz+UpaWeM&+$R9-w-v7Hyb<S+F|#(c8Z#A-_H=W3r$K!W=VD4xb6@%{wNDq_?85gTG%5st=qHUnDIzzi
LWZQ0n1Jv97uM(?4C43QDveK|9O;&ZW$9It^G#&t_s@HRkKqAs*$Uk{p?-BI0p~DO?R@8UNn70Xm3JCyKH%*!&bo=LuT+7)%WKbFJk@=xDw52fE9>TD9L62I
Uu_JNBc}Dj$)Y=44m<fLikjS^DI+$T@Te~5IE4Q{y!G-H0uU6ODzFruuC=acuHSuk*;!Wz<1DlUQ5gm^EU1fo^wqduSwQ9vFgqe(DyMMYh*fuL>J_5;97(LQ
dHY^<8ybD&-R!c&9){L{#a!`->n18Uah5o%^xkz%z4sryA#UoSY|gY7a({kIHE7jb8-K1Aqroxmgf=s!7{q(UmqpjFF>N}YCP<PNpU%u&+J$VJQZAZe6Lm)Y
d&v_0C>F}JwgWB0_v9-JJwTy<<tuZCdO!3b=6Vw8F_`cPawny*QyzVK!rR|=v}3C8fAUy!J4xyhI*jkk0W1wNveawlL>K97Kh7Q%>_4cbGyk|cq$TGM7)Mz;
+V6==vuID1a0!TgimGQQp#|K>eO=6$?2?jHIS88-2kC)E*Unw4x;wW9!V@*-SH@wJ-h~>Ewykrok6$UZvvfvz;`qm`_H99gddBc7p7=q>T9=wO5>2Nx%c)w=
-4%F!`>HWgS_jpi%g6Rt=PW*-)vnejC1rkuND&bIE9VnW<zr#-A*S=c`hPMil5wg9gL4A><-3S*qtYj(bE#Zc6KH42qtF~kPr*yOazqCt=2P{Lxk}{!S^NTz
Q3fa#Ltb>#D{f%|4X408l}4KH-_8tuG<{Z&79Y0oNl%gRv6Ekf4%Y69?R!e9NW;!o&G^1#OZ#HqU#;u*<sRToygnjP?RtI*I_PH)Z2>ieu3U9_Iy#Aw(f}?#
v5UD`Bh+9K()IM$xo|ulyW1jdCg>+YaJdsy2;%AT$o)(@vILEW$aAgV-iWVc@UnC%It?ORVJ%d$2XXe*CX=c(E0?oP2y$M;aPVJ#-Fv*ITHUCuxQulGD=(ZW
8wqJ;iFqLH=7p;BlwArNJBDJIooOs^7u_#|R6GH}i?h@~h9s0bamiZXua(rC`_4^t05rQ%usF;_O>UnpaDB3;<e-@Yl0`}#gc4&VKP@r}O8meg<4#*OuH-Kf
>TUn8K~wiM!~E$kj(MB<XoNiY;m9~cyMtc1+x1ZPN*&HZSYH|0G{1LgKkdQ)5bQYn9jgL<{^QEn5*d;-qK&U1CLMv5(uw<jhBPRY-18rr%xTGBg=m>fh%rn0
_<IAkJe0lteIc`tsd}Zu-HFY&MWyE6X{O-ZoFq6KPV4-cuE{L$v&;sWE~jV5P}LMMW*Uk#=3u^#+`_^1)e*FWbgF+;o}t*eG(?@XvmNqNDR9lS51YO0)7@2t
e&v+qE1HFq3e_{x#0HXdH4zWa@+iSn5HJS`zzjn5{eLJ25APd$=jn8+d?yoh^@i2lmI-0I-NadyMu`pIGHiAVejr?Iq0Lm8%#YDmF#@l4m?Mnbt@ol!O#w1<
CZo2saV))yhO);*3K>0a;pm2#tr9&u*v9EKXs#QB-}yYz9D_O(QHGsn=`%%DMaqk%2)fo|ee5WM^PYD*EW0i%D3hyn&Syb+-`lDDyH_);{?l6&y$0aN-&XJ@
_VZi>{0QnqKn@OLI;Aqkyb)!Hb1JYmM(bI}oDkyE1BYB^f^#$xo|7P(F}H@|2(MkydBG56j?aUOW6ZIGeDg$IoH&KV+nFZ^r7|wmN)@)4)NPF?*|ykP$({Y?
QH|9<pRMVqpDPdq5Q9-?+L4rH0|pv1rWIJ#c9q)CLk}I^JYD4md!c;EE;NZ@qr~KOiGP51@Iszp5UW*HJU%Wg;N`*D5B@l`kr(1d5Jr|AilLgg&@M3kK8|0d
6ZfKf<MmQ6rcHJ{HKfv92iRx4j_ll?D7?&zrw&RKt>7L?a2$V6AV*=CvTMaYXDQoaGUaLui4uyAP}aY(AdJAmLP5fjpJ%N?$ZgfV41~>lEa{7d)#zfw=M?&x
_}HA5Y~E03nle1MFe(a(T)=d&(23Yq^oA{9o&%@ig@{Vo0%)HP;o`J$8R)kbjC!s-4mL!(6ypv(FN|XrbP^vZksU<h`>NwZI$v}!=Piup^wsz^V?N>pji-`-
B9#97W;dPt(IO*!c&;^(qg~~(XPjNQ4_BDW*fQFOnM#>El4atcZvQ3~x&q-7Atkk*YB1fh5C{%1Y+}lo1AX8nKGr;npq*bikLUvp)IGeff5|+yfs51-=QXUF
b5$g;q2?v7xcIoLhfd35t4K+h0*!WUIR&PoF#sPYP1G?lJ<X-aptr<dn&S{1KnI1Z(?|xnvP^Xz=-t(eScnDZvDk-1fXn%E>)x$qWFwioii(xuVk!b*-2K>;
yAJl(LDIY1RComvL5(eL4#S1RZv@>o5WpH{<p>Su>1>~X9B8+W{)$LZrAx0yh`tRu(mB~Zk?V-R|41)`h?Of;6MGMBYB9KL`4E6Vd<G9kxw?L+lG;f3xxe29
BG2ur6^eH)5M^@*9D_%cD;{&+RGb$Xj<~c0;)MzxY$9s*+Nwuhv@yPi$Eld4eR?IIN(e_@mlB^fny}Ma-D)}#rmg?bVB)+)ERj<+G&K$pH9#WneOv)9K!vCL
-%TQ>7|`c&&+Epg2ECPvrC`1%g@3nl!85E1GuGP|tMU!^M6*1<KzR<P2I2@!jSW#@?b)>9+)84@KRS%B9J#(yEA#WDvpP1OVq5IgMK3pHbYpY5K#X}#hanBO
piB<EoLW}J$*xAH9^r<B`wu?&QWs;@e6NC4RP^!Ewhp?OV)*|=DR~mTgWwQ*uu3<Ub0gS=X)N}pz#FIu>P;Cp4&~g#5bhVxTqJHc8pqrsJ*w;Q4)i^$*|s~&
a}E?io1q=DE$+{)<2H4&89_sY&2Fse3u=7y^{|=x@-Yg~LCc_K;A;cN)GnTBaErV1$!N-B1;o)tN#ye?l4dd&CmvAF3JpC4)7Qok2@+`9n1fz>9E=wU2CUlr
EhpoLsiO0x)Rq*&vGgvWF`^qGgv|0SFgf+#jZpM4EsF8=yOv9xt7G?Op@Zpy39LGlOO0bzKu^_Os~uIWr<3!WA444R%6RytlSH>z$pFIpc!=bQ=CI2n2KG1a
+h<zulOTjgrLG}`PI}i?j<bdiVY=xc&?FDuWJc@Zt<oe@kav9VTus?z>?aEAbS^m(ucH@0T&*E^vOVmx%EWk!v>$=4ak8y#z(P#RKINA#t+~-84=`q_Iibnt
W`{G7@pfAi2Ep4nPm0jBTkzw_#5%0>+hxafw4bK69k*ysXJ0%1Vom&AoON9IKU2_mk<Z0UxbP0A+p!HfI|Z)Lq+7PJt;YwR2S*{SW6Q5SZ$ihm0hl%KCE#Y!
#*^6Z(C6_0sLMV8t+S5(HCx)-*o@R%{F4_NlCR7quUy)4HdOXvR`{z@K8;&ZkgT&PNU4SGBW83$=O20f9kLu7F5z^6zVo~+J*6V`{0fjDI-O}W+x?iLrhHaw
ioDqb5|}jQG$6fZe+z=<$N~9KqzSbbTQuXJ05hhz^;glXE=|zC0dO5ls0RmY>BPni;5Rf|k8RdyRN+5f4ppe~X-Lz<Un|c-I}s)oSV{i{y6!}iSHXSEyRQ}!
!vp^qy}HxUGPIAbv0i$P-XVn&2$QMtL2H|^M0SP&h<ZQUcMOu!vZOkYwx5=$x>=!ps$nn)qF|^$J3846!BrZ=h2pb&u~e^xCa6MR2+xMeIwJe-b^ILrM{uJr
4HjQ-QIJF2rDzR+A2m37G;;+ba#h+;@ENg?m1<D7O$)lT>!viliI_`zR~ACBP@wXMhx8?C^INFL-hL(D(u*L?R;8Q&l40S^ls;s_EAa~xNdUNR5EO1DH^=$)
TvkhRmdQo&YTS}YqBn1FHVcn%pS|>aa*Z|oUecKfWlR2-SIl^GvXf|9x4q*a-E>+#FH4ZRW8^&0_N2XzH?FN3LA0HAB&-C*1$Ecw!$|Q1q?iE4Agr}(uY|%c
eT{*qOUAi`j}v)j`)$+xtIryc2Ics9K68OrXhZtrD(cH8!7Y~gABqE#lSLvvM7NtnIv&?7o{W`GJd45v*#J(cO1+RESII^-I<jE@g$82$a4T?+`4uw)zVQvl
HGN=&_u?ROq4+-DT;hunN5#|{2q(>SkY7ia{)=&$+3xZg4D<p=d1aH<B#y5xVxtUQLzaxcnDFC4<A-w|i*Y0V%e%CHA`m7=FGQzr%HM<&f>N#Z%Wu#*G}W31
uK>Y7ZQ>i<u_7u)b9#-CCX35i*R27&4p=Jx6jdpx(r&fnOQMLF5)f=@EKVGQ;xvX^+>JL-Ivu^DXOXojnAhA7DStH_isx}Rj!6ZN%;U@iL<0q;NJB@>*+bGP
GkB2e1+=AP5?0%9x#eo+b&+1BgHored~;vyO>ug#B5$xmg@E+=6pH)ywb0a*dwXIMWgibANEk96Z+H9p_=wz%EWCK8C%`_~w2;Y?|6FO<Cs2_XS8=SbobVXV
`@P$y7_92Ah$M85AaXZGjCdqoQIy#8nYrQ8&(jU#^%+d?kFoe_kYZpMmXfp`3wWcSe+v^t^{A`M^B``Ya?cG$*?-k8!hOG@;$_Dv_>c|}MDAZzxm;XbnDZFh
g6}i<*WXJ|-v*2n7;_2f741u#b<z>##GnGn<r$zUQ1qD~bPX%@LrFIf0Y0Wu*E3Sr4FDzSNsHPp4v6y=F%5s|Fg<U)gRofHT@mOh_W71P_<#p<w<NHNQp;>t
X?ol@m>P<?Ax!|1L0hobSkR+`3?20}ulqKI%ULSW7!ZMSDD7Te*@uh%VB5;5#heLseSgoaN;qN9hkpq~;`O{!1lC=a!Mt|Z-`lf^6U%V>@jMBEEX*Tw1smu$
vDy5r1^?AMkr-cy;f*9r_+aQt5XiCtgUt^77E7%I>h%%G(8aDpCw3i`UL$zM{bJ)60-l_~c_^BU+7gOWdB66-9(G(EJJn~=Qu6l3vSUm|bJLD0anIT%VMuHM
BC3G1!aMEih-Wnl-SG#ts?8m01HXLrv)`^@Y1T#oW}|I^LcTS3aJT%fx>{C+Lcw?7B}8Fqg&Lg70X!}H&$|5=zI|Dk_2>5^okEQ806Br2spdB*Jg!A9Y3F5y
rt!}JHm7mto0&3_<mSK~EEjZzYBchy3nD|8l=814m*P(@=-z8K!~c@`WZ7Nhjv`ZabYzWIuBI`V5JF8=)LU-w8)*#bKHQn{v88C2iF5p<d7wCbW&aEG7GP`<
$6vJ>w`xZjhFk-AQI6&<<~v$Bf$P#yX(Q`fX^7@2-UD7JN_6K|F=!!i2?(px+eXU1IHg6;2ayL)lI1%X4L1{4=kte%hPkucQ?o0x6{GyGFm^Kp^`xHgIBk`C
{|84>)!a_)=_)nC=}lg$)>yncwJb15pHx_wi0|mlnQTl#$Bs+VKlsL|WPi2E-gy&<zy5N6L!coLz66C5ZSDCekQ>JM94cMBM{Pysy0ySlT7RB`x#yOG=@u$t
b}nyVE=B6`fF)&YKrrk1SZX?EZ}csgo85czYAoO{*_lWAbMZ!}eu@BUfZ4G*UYy2T<u2@efbl|+67*hKy=1g_da$HRU>FzT&MhI-d9-6DP%dvpPDI2fprwI`
VWIxvy&ts0#2l*@Yh8;(i0)wLERuL>Pr5791ofC`@Q?g33dyg+I3Q68hI9hR)<Q><Zdt-*oqD`0Y<yKlJsrn0nBnAd){KA{nMV!i_ReI7Z$}@f7ATUzo>2q|
B_C$Y14kPq*6uW|D3|92<f?Au)gfQoxBC7wO*k34E_{a1icD1D|GGZ3K!$-R@FF+(Iw(9mau<mW@OhS~;az3>`Ib<A;LH9oIH5J<4FF3Z^v0TABUc%QR)g=p
<wct8Uu_OBL+b8BaR|y*h0=t3CTPk)p^Bf%Ga!<3m%=!$Z=)<zbAf6vbK=nu57*uRsd0(G`kT-t-|7?mwM|h3wKS|o)g}%WC*;RRa0U_M(#85;k+-+HW1d>h
qqD`kF)3feX4K$Purx@{r{j3Y?%fn$KZySJeqHy?sm6T5<}h;Q#M(;56nd?m6B+d{)=PDKy*qL4^0QKJ9U0(_F1%@h!Uyb(>={<%Ao+?$$e*8YeuYH!=t^iS
@8z?znv&C}8NOUcM|8X5K$bK9kLRh?i=eWcPK#)q$&A^=>BWy!?oarCa7>+@EXz0e?iKN&$Na<`0-(is$pKN!XPf5BqFOXanOwCH{(Dd)#;!-p-2E`_+0o+f
C%@ey`h|)P^4&#3bv{?$q>qa25)g7cUEj{S&-pW@7k91o7j^}}TPO=H*!sRT3hcN@LUj}Wkr__W9=i86w52Pc!J;*jpeSx&+W;iVM41;<54l~%NEaeSf-L4?
YH+8FS*#cctd244$*sE!zjcI<S7?{lps3%)U+S&=DESC6Tpgx_YBh7sPhJ^>Yq!Aqdxse^;a}MV6e{Mg35AGZD91M#VwV|bNYMhyIDd~Jos<}3vY%BtsvHjJ
l8&CV3hiM4>nv{9f`lZ!T3~>1VM>a85pnDT#(wj*eoN(Q*A8-EW{F^Yp6Ws$CMHV4SykaYO7fDB#t`)QH%#8Xo;1QJY@{LYFV{#RvR1CluM#;avrRGoz{!I+
K>}qBP4}rwwq8Vec<#Aq+slzzcSoDZzG>R52zyb^U*eE1J)x~6l>+roCJ}IQGI#EQ1JsZXAtXqvk<N<pGvpohbK_@_)y#N&FVw3|aa#JF>)<z6%(UTLz(CE%
=#UVJt0_7%R_^VW`48Tf87j8EV;M%KdeTiE%@b~_5~%?Zzqn5oa05~}U7E*^j~^EAQTZ|}tl-a^7&wzT3oeMLA5ypRSi8fwQc`df@G4*6sL5=vpcVabP*K?z
WxmD?dF+-2?;AzKN^X_40{>*s>p@WWI=z?1CNUt8l5FXfb1dlF<GI9R+{yfn4x8hjupt_pS;0e4elrOovfa(ZsIvZotxC}Nq2T;?4Cv-OG({3+3Edf)wColy
l=qB;_KxH`v!{mi!&eIh6AOq+FmVWLJAV)Y<jWZ?G^4Ch!piwRX52*%@cKhJ2l<LYwZALO>cvqw)1Hx?Sj+M(eNEjL%+54?vc{h0YYJ~Fj=R$!3XrJ(*ZR$C
_ORf$96X7+J8K2YZj|Xau52AWsVRggk|EBM9H=+IV8mKKdyIso>ID6FLG|mg1aZj<^;)nzd-xAa{6?5X=Q9Zz!6tQ0vd-kC*8)O8m?gg@C7>UDwLYW_I_$CA
01ZM0gAAU8xgU+jiSH$EJ(}_l519&>zQpqIgB?eD0;lYT>myc9$G?-OFtJC{R+t|0^_#EecM+;?%I&Q)H1Z>!Bm{n8Re8|4%M}h_J^-pB+*px5%|=hbDqB3f
dXd$nCQ8kjoAj#|h&H*vDUzUXWo&B6dv!@W&hFGK!)xRoM2yvF^d)bRp0U0B!(XpWAm#PjGGF9Jx&8fVT_{@Mi_a|OC_8es?XA=b8W(L`5Q-u3CG9(N@QKj6
&`G&1@@eITM>*3-4J#5Ifh3~-{LIQ3q}M5(Hj){lnPf3Sb}PaB#FN_*pvA~e&K^_OyL@=Ig|}dSEVq<#p-1>vAa~AD{<>X4W?yQCx_V{Z`pxdHcwWGZV!ee*
rSIG2IVsKqBn>M#i~bP`P>JyJo1Y}J3^KqX?3=B*viAj19q?r<bJ>!!?5|7d2TJ<-m}F!t>}v}aZ_;|1Q`9(`N6r>PLw)rB1qXExkI@Ta#qlu@`lfxL1l;lk
lhJslwZp)X*0U!B@(gHz&YRU=7OJ3t<ETD*q)I!3Zr4w#%{a<Stm3??<y%`<U(01K00CgPLQI2{#z!DYW&KK#1=nSzFZDT+Gpucu7T@$mpa@EXqf<r^4ba+^
JNVW9luu0~wpxVdVIhWl>|maeJ>n3n)?JiI5Rrchs^=Ly;ZQE6G2Yqm3+m}LfQY)}br@EBBm*O)qV2UZCt41v8bCaVN=9G#Y9Q~PBBqJfpOpJp61jUKa3oqo
iwYN3otXm97j`IuAKGZRnf1YMoJcDXTA1oz-MyY4m)HlJ!a4OddRq*e&2vEl74p#c<6)E9L<6xoAUzW|np2YF%+gwD9ZD}w;s#WEQGGMNK2fV|vLqEBp{LBg
4u@|Dq;WZbVn+U9qJoNAd$K2m-aVR;4La_`^gr#(+vgqo-RSgVb*Mx)>g|AuH$#Khq1w*b5+DFQRn}nHjbF9OENhn^?IX84oIEjd9t3$nH&kI>B_6=z02(P9
ytPCQl^!kXVsPo3rzl3q^`*t1S*RYn*0q9eRR|KXZvb=&sUfRRtUtG4K&8zP?QR`6KbP{bg&eOjkH5->5>i~(aA848XQv%|X};)YfcaHgfw|_Y;P@hhg1B?_
AM0@vp(0ZDj5Z+ICV?0q(ghRM>>&s+Q0YKcceScfK)!okac%y<Ah?2K@U?r-XuSSCePjyODd2~x6+JbzJ_a1A7rjE@o4ZjlC(Q7b6fd`Tx4m>$|0N7z@@;9(
v~uU{WsOwz=d{Ng2nCBgZF|_-C=sQW!gdP)lrb9%P9KWu?}h`-CmMu0)j~CuZf9msoteBfF*?hDm_#O8V{=nqkUeMj!HE#ZQ5rlV^D;=P{RdIA3{9<EQp_%H
;ouEO6Amc)I}pZx+WVK*)OPNj{9e<QH&sDxh*W{!Sj>tZ;YXj#mF$C62Ph-b3(~AMig<;V7mCjkA)eH=25c)PTK4P^n}&=XM=JhCi<-pUbEp(DSbhveaZ938
S7qh_Ttzi98u;zW_pxbiaZwD%Swi<r|A}OkB1^=WjO^dYg+2b@>JE1X2~Cvv?8Tx(l>`%wwW+-dtRnz{J>X0A(h=8-F*(;7v0K_>K=4*bJS4PcTIvSINX2d9
L1kV2|3>O)&*$tB(a={}bR^2EBaf~j^pI}8Nc+EJ6ASMta`pKqu#PdogZK`eqhj5*%{szg$H@u`X{=-V6ffJ+Zfv!Rli(^=p%8gB&XkKj)%@(3LAw;a)MXW0
1ze}Zz+&GoR}C$r%nt2$xqh~oFg^L9QEzr{=QpE!LcYS?ng0kw+vVz{iWIm+5GP6C?Ez^<--!Sm)hYofn<rC6zK|0Uge+(}Hd|e<MT!_<xlPTJ3i<4M8KG&B
6(UJb7+LZFxfxLKoD+UR_vHW$VPq+*&^1PGkd|sfz0B{-IP&bvki4`g*3lXUpN*Z%^y0U?cfTN=Y8q&G7!xFq;y#G9KBvhR&nWhLz+X5jtUhd)Zxtta)!s0<
Qqw7w{=r#8sh7KP{LiAF>R&`6jYH=ACob)}#c=)s^(JT{E!Dt2>t5~8TzspKB4o=KdvogHaawkuz8bnXdv-D#-I&U+=+w|E-~v>E$doY^9L4e_H!+8q%qI|b
cmzxRMl~%f{{r`R4+vId{cAfDlMeKAa5Wl%tN8FSYEr&P5kJ9ni)ZrV_Wn4}W9?3*HaKyZ{TREcR`2)kTOslJYyo1_B(a{a#feC#E*z+}o+@iwwYM}~?+`d)
1);aODi##+mmi>L;>y-RURi$%Py_#eCVMLPb~sXcU)TZS(BU}zG*PD<t~;0r&bLMgPWl>#e>sT;NoS8Biv7!UK+<%{3z=ouC8Ow$L`GF_`0O=i@VJfrGc7z5
1qmjoRX})(Cacr%hRKNwJ+~&<Ljr7=d}HWbihhWA@DJh#^bHX$&Gj%_eD8<_a;@hNsFRb7>J?RjSemW)IK(kw3NU^HaJaDZFR1d@;ZPpJ>ujCu$rQ`Tdykv_
?F9xC8bdpS;3KHTCgT`etMb1AU<ZY^R=8;W$?#*vHU~rDCUiDZ%Sf;Lr@0#uJfypFF}(9yAwI|bDahgwPpXMTd##NUU`c#;nQJ0$>@a@39Wzwo@#7DvwG?GX
G{l!V(3HYd+*75=nlTz;bEl@w>(nvS18RL3n}NzV?wH4E^npbKOd3|L`j7SQ+%CG)tFS#3EC^FsVZO1(Rgefbq%}E`vAGh3gE~;?cRwGK?=-nw`5BJy3d^W!
OT!|Vrq&p-9tM}(fk!^Lx)g8u3xHhpNPZBG_H;Pf3O!)vUlO6FykO8l&{cB`aW7CRg*XcN=wxz}409ClQ+P6C-c(nr$0qtr<J90LiAgh|&I~1M?I=Gz95==N
9L9A+k7C;Jv9PWt>#He>+G&lEP#|A0a=HzU?CL(s^WI(LC7@z*4f%F0mX^0b3-G(B^ZlvxMn8)*J0c;21$0?jn9{=TVQ{jJ%sxIi5W^nF>F+l^|Nk$As*-<c
U7K)S7OSDAw_^Ggdf8qt#5>mlk@&-}AaIrj2b4`$$m0|_QQ!Rwpz++jr0RN3eMZD@c+*TFeGD$$pX?7r4mzhjOLw}-btzY499FXNruhv#TIHg(?rs+i5-1Kb
{Re_6-;R)y>;WxISb_AiTL=keX&}7*kh~Znj4{G<s19G^y3ZINhczNd*Yx5^nVXK1({V&L#EEd5Foq=tRsvK3e>;2V_qWAdwZ?vSLoj;~Ney1lbksV~Dxuh&
Coa(t%<ByiIRivfxk^jD4Ns<Uio55DD@b{c(Gu5>)DhBrVag;IzMFLFc4cGk)*qm=E#|r!AE}*uCb|vug3kHDYQS()E3BP3@R=(Wx5&&Iy(F7t0zxrG50VUJ
-35l!_z+S&W9(e-bExod`i>CkxB>|@$MEQ;TNfuRpHyDGV@@RlC{{$4xa|$gaE@g2zZ_emg4!?-c>Q+Q8ZvMPBd)5IsQB2e+M43iMoE*!_gfKZ$hy2L#%Ex6
aFBt|X~z3rN9t2<FCwvNypBFVU>FPqh#m)A4~nN}T1|(xr7h0fJ_Ib`tLhXOx8BAsz);clU^v>*>7~FzQR?Yn0ki>l@qaGE5N432)W0!yS0}ofek6TT;vt22
gCzNmUPFAB!ngFMPUSTQO!l;dST9yXuZ5cWLQrb0_Gb~#Uh(5)TZnl@3=2ODLj;(W4E2=DP4Rh+&5Lv0@^xB(WsLX465xiJuhI_?j!z0ilGvbP<<M&>@~!t6
_*>$)Qg+?3)V@rCGI68LpM0qr8>Jh}jSwO;DKRZU6ML8$0*PiHobV|YuAm)>;ZYE?w5=>RnFGG;E9%IEQ2!FsK$woQOJuI}$tizq@+Z{nzS4SdB<K#IRVfUL
dzr-AUDg6-(-XBYYqY7E<2z#e4v0v|8X*3X|4pZ3!hfRzv!o8efI|2AK#vF)GL+IB$23pQ`uRQkqf+YUOKrPoO%q5^gvTf3vE+H_cFCG63So>UK-Ez9Y1@Az
mxQLLP-$=pGEeFODb1X8?IRN?hFsdxlE1i;mX~N-BBHVGoY8js|4IEuCw7117}Em>6Dvx7S&Bd?XW(7PzM1)9?4$mxsK%9_YX1+LaKNs;Or`5wOsfG+39M=w
%}_V6%#$Yq^l#_!rvU~$ZLr02%oFbOjV@PJVVxk4o^eExgE|AYlBzW8b=PF=kb4J`3W;J7=2%!4hluUpy+|Kn4DD9Q09mwa3bgZTe}m{yg2QOR(NN#??j8^P
J9rV#RUAY(KPDb3<*MtSoXVR`kg3dN6Q{Xjaw`^?QnR-<kq_2-I+#@mZ+Njhqp7oeth*fD-y6^DzAobW>Y|assO<hB(-K>~O23H<I+r+$&1}r)HH^O$`_Z&W
);Wb{SV&gk;ps>*pchQFeVQl)M3eJ=UBXr3ibBT_*gjPoyD9e)j$lnNAt!o9G+l)JX<u!0G7!5O9i(DxYHq4W`4tL|EDkL4@ip^MykIT>WLxRln>}qw&(GKR
w!PmEmnKsX33fZ7%4{|xxA#ASia<7<py-iSa3Kk$8yRv1;5Q@46E>2U>+I$)+VckS%aX_i>!?@S-EHwxR%O;(FK)453(QkmyBx}A!f8;aRWQn8He-^<;(HQ^
SW21nkz;?7$8IA%K(vXNdQ@>~cq9U85?xD?RgYc?TgdF-8JMpSC*m`Jo$r*VWS41}AqBzAXRTDp&q+8|V9Q~-%G*3Qknb#{1`hj)sKb5<b-xS(s@K!cBpzVD
-0WyxS_}!+!OBu9O!U%4c94CZmlueSqFTG$XzdJHiGH6b?&G@R^qBFI1?4&ZJ}VwnuKV&<`EqZv>^&QaZ38MtI*2F(<Cj0Yv!0^B6W`9Fq(RtR5e+cEV7FT*
Dfo4!lc~#*B_T7a4IBh-e7y?BLH}ZO+E4@CHM(n`Y2(FWOUHXS?dJ5eAnAON<U#2lU!57tPE4$KqRobj4{st;<5?fqO=9Vfs`WsIuZ?JM6v9mS^@MS^MX2nN
U>~vLh&~1Hl{@-Xt?Q7BJ)FU=*=-O6x!5Iayp{_stX}=uJ2_Lhv(&``lSmR9PpxN+C=v}{T4~jF^T=#~;np&v9ooTdh$@5jC^!Su2XsDV0*^YjcvDWgo5F$9
w}d)LH(hapkl6QR+r3XwQz*f_$dr>F)GLJruZvgk*{D{(uV{Z%%zZJwj7!dF*KK>E&90^uDTKT~UXw7f_#mNXx}y{Q7-pJLcATe-ay*hoCAwP2z07?s9ts>q
73<ml89C@ggm=&iiE<WsO;32Xv%|IL8>2OkEXrpC_!|@5{JOKumdnl{{YF74>FvI)p{@_50!$}E*1xIW_=ej^RC~(~A0u;%M6XH!6Z!jSgs|B!!C+*>Q<3`7
9rJ=Dfv3ffoipwSj9G2MbB5@gdBEfR2G835As4Go3(?y0JGi-(&5AZbL5!b#9p-k!H;7e$Nrd=whZX-e^ZPI#yMApmXf0t$uyyr&yO~cYg-oIS=(dw7(bKh9
q{5|flNhV0aHzmKUoWA%JQ$Yi9L-==G&i6+RY1^O?lcfF+b~IuH$F2dXHce^e!ba(&5bkxeZZ=E7o@SrZyIN!i2%Mi&r(Euax=1;t4Vb*RfL(I=!#&LWoto+
XwpL`8Ga>&qNKnwUf+DB;Y)me=@xP+fU^@0^o$?fb>JTlfO@8$>qPCLlR6QzPi+*JG^$QUEXp04#uo*c8Dqwd0@>0|aAs=zy_y~Fc4($z;Aj{ugp~fs;yOJn
v(2mhH*zs^`mDM-XBvLxXeKcElu1yGlW{_X(Uc>w+nu)U;ZUZBCvB^5xrYn``dta2AbXSz?chHWGS0eOd-FtB>NL$datlLgViz&O0LBbGFMtQ0(m1di#aCOQ
Yx{#GrOUa5&=I%A99}p`5)}Z`aAH>@Wu~#^?LqV-$O9OcLTJ?JesNP_4VKy#1(UM#8Ky5@RV+Cw5i6ls5Aw8&611|kQg2KqKP10rhM6v+Hm{L9_{9eI&6MKR
L2_WH&%3i?lQv^w%zJWs+yur-y-F`keRNzQtOtV^z{3MBoIMxy)I9W#&M}mkh?d&7UGFiDo?N;MJEt3dSs3F*GoXRR?|m^@n(_*Q6%Bi*ASTC4d)9fZR{7v+
YZ;<<**b<=Mc!<kJgD5)oZYh*EMzq#a+Co8kq7?L$NjISG6eRPQ%<22E6JzryKkiEf#b*OI~kR~dTAEB8Rm<M_GeWz7`PAH2v4GBKp|R%uBZb{5Q*EenU^-0
OH>1Pp%KVIsR!Unw#`wVAMFw~sVq-y5T|x@-`ldz9y?TQ8fVO=H*ykXUdJxAH5mnd!_^0%M7@xxi#cYG&&V9-q?1v#cCC+MgZscD8Z=JBrg5qV6x04xCM7TL
PF@NnZP6o>r*R6t`Wq7&XyW|14>EF|Q9EV=eDHrJr-_aygZfj#xcLRQOJVH|M9Rv5%|CX_Bk<EtHAVc>QB6<K9fJw|(QHG3=du1s@rdgj`jbyCP5*lL5-`x!
?2?AU(}U+5#O|!6_`5CxFKVx@jGog6K1Pjz$FZimbBxNv(a{aWO0cm_I@&*&o)jMC<^XEhb7ho8Zt=h|Q(i@zmYWWFiCG*SVf`7oAv})#x!-l6RIZCme)<Wm
tZkOzidUaIUdKHy`xsuuQ#m!R35`hrej~f@q#cHIJCg~dz?zhaHGsIRC|0*zN2Fd&>UBP?#j)qpaJ1Rp)EaHwX#2Praneh)O<WFjQICIz(k$z5#;EI0OsMuI
x(F+5<p9v;T;Kx1EDt%WL7@(YgoOZVE(A-x-&7HFSH;7+Jpr)MeYK1w{HKzO?M)TxZ8p*(N830QRMfhS%VG6Sc`e0%HzNqqaN$R^O28}K^w{L;->N_48t_Is
iTp+5gF&LFW^qK%n3&y<4y8Up+E%u~xQL}WJd%cR_}CpT=q{vn$JE(wFqI(mK#(?V?*lGZL%L&`fa@S^te`&q_)x+m53MrI%zUu11r2Y>k4`TWLel2bRQ<4&
^!x4tPGvyUFdRpn_8Y>cKaUmuCpQ{r(e3=x<?!34HwKp|w-|F6O~I#Z?wo%cxo^5*J8p}~(1<_K5}e8A)@RXC9BS4#+gIc7Twn>nNPh4^fvs(*r2O{H@~GRb
9+mTt^5RnOEBQ!5Pf6{y)cQ<dUCE%=P+ort8BD6kGL$9g+|vt0w2>0NqpDIKxO@;VGv@ATo8!8xZky#Wcr1nOP2YTEY}S7@1I{w>Q1JrKFoi`v6Cxt|)wKpj
%K>U!|4*!Lg(W|~cq3wA_$zY?S|vJUbfCl9OusFm$QPd%gAj%#vChP&kelmg7+dmZKRiCC@NZpl_h!pU9J&3JqYwF4Pu=CqzF+EVOb?orr#9I;)euFAL|iy<
g-ern$tRoie=^7{>%l544*^teAF)z*4s=K7k?Vt$u>_Y&^yS|VoOC5)cJgU%{rlgq3Vy?mXm2gu3fbS7PGDmy$j@leK5W2W^k4#=D}S==lv+>Q;|ag(fiL%Z
)r%Nn2w~}hYlO{|d}!2EQ3#&C4K@qCFucVViOpbVoZrRdS%f2GZeo_n%uWhQVln52jU_-vgynTPMYM^&5z>qABQ_4wi5~Q*)vk}Y$Zr%SHfb}1;TUr{+YE8B
oGIRm@YLc$Vw6Du&J@I|QFN@OIs0xsYZhFiNSpJglqun@QNYIY^aA<RYp~u$<DM($1)z%K2k><i=t$F)8_1OAF?$m@;O|lGy+>L1rBkW7kp%8gSU2w#M15<b
c>tmna}M=>u2dk<(yoL`ODM32G%1E!n;UZfR<w*rt0a7$LRe8W&WzyS0%pIp>T7*dV+Zk`KzLs;xZAhpvA&tUK?mx7{Me3eUDW9k_KV1?p*~1LJmRuAum#a5
un^eRyR$rozlM<%y`9Vp?By&<V=n@|*CO4TBJxvkaIhUKFY3T#Jr$tUfy=yS#ec=bSUNhp7G!ue2y?%y58EA)*?Kb{)Ue@)3r*)bacFiMV|^CCn&+dZLAyYR
viR%tdI3Q`;S;Sg7*3g(J+BX!E+9{XZTnTd2FDgV@;t5vDPM)+6dM24<;;rq0G*;Sa4oHS`G>n1gwXV6U2Ph^JV>QWdQC?%)5XdYT_j7V*FJGr2cm&>F6|_g
e+mq2(;;sv15x}vD_j1Uu&7i57Wk|PNNX`M!AUV1bF&FJ7gvof`9EuKU!#cIx8K1MO3M%Xl_h%;l$^%1V_W&z0N__!oR{OQntwhYih~D2zhKZ+d5g2+X9-;g
L(V*#AEM`suhdFxMHy)0mM)KQ+f@*+_$9S9Gf0`x11|NDBNL(9aogp@ApJ`Avb+)!aB}!MlP@HksTDl#h|sk|&DzLzG*4g|dv{^hp-ololm8J>5MzgwEY*lj
AG9y1b$Zi!p90Bt@QlyiF0=OQjy(ft;aS?m%A1>fvpDLoj^I=|neSwcd6MELq0J^8$nR%+er_IAjgh{(X(V?W3cCS;ax*zlyam$s!XIw;n7SRNw&Euib!J7!
M%$%ccb@ci;a_a))EMdT+|!m2UrRo&yDB9Rg*%odi-Bwr;2%duYO}aw;+B&zf#AGL2MWE1K84dbTh*fX0bVsy(_jT@mVmn(2d%mrXUO~xnJw5QMa`ITX??y#
+8K$q{L4}87c-?pSO)`|>j(KR@_E01kDSwp!|S3{r2N~2T#xwhNh(e`l0Q<kslcw;TMxOd(k&NUpZiEx&3dMM;#G4+?<A2W2^ZN&K<H0cPa$S@*c_8pB7E*G
JuZBdn!0?I*(ST7Qs~!i3DQlX?Hg0?mCtxzQe=ZZ3XwW;r9uSkMsR88`lFX66{F*RsrLZ|WP|(#!vC9gD*<_CoWgNuM_+W(OE<Ex{3=d`DAf^9apZSf_N8P7
IeRQec6mH<ei})<`hKBosLk|ebT8VcNN97}0BpTIegAa{$Z5~C>lP?=J)-GDv$YaY$s}9J_Z2GhXMw1ODU?O<kPg0x8hWVPbQ;rnWy#j!8kO79B)_d^vL;ik
+67p6`%i(Q8oeBPiG=&cqq+|l;|($PricDuGbGyYo`Su%rEwt&$4u478^KB4xv{vP%2|V2ApvG$FgjHgHmJ{CUH|>ZB}OWSt1kop1qvkKQY=Uofbga@CZ0x1
6#?1dsqrQ#*hHeuOjAB3j44PgM0VIdTwlnB&FcyuYJqJnTySKK-Uu%AK38q*B;lvH_!M;lA~tflc_8YXJ}J?}-WBhUDWINjGiRh=zBV-4B`TOA+}gcJIvEC*
Uu$9gsoC3cC?w>{pfXquau0E!GPy>gcW1SNijdmxdbf~&udw_v{*irL=_2Cux{$F}do*D|RIJLlP8`fd$b<=}ocJXS8ZT~O77)mCvI{5ZGO^h25<yU4=!#nb
LF}y|4)j5}pqd~`HsB^enU9U_#u#NeFS1LRaKRzoJRGb7?Hg*L;<X<L)62Q3fQ)Ya8nlAHn{Pw-@`_j5*qCXnEYd40XC%3gR`|;PWGdSWAlHu;)fc3|y1%)M
tefTBF|BUh>akk$HKwpp9bS@;gl9S7+tX2-Xz67dl!5^>%_XiqDr6|Pp=UKDN^@Kw)gili0K#g!X75N@Md8CRQ;;g&>uCK_b?}jz_zS<#w2!7gc~&r(x;u?z
OWUB-@^3O6kv#YFeU+<N3OtL=DEZI5vqwKP(cI!N?&w2>y$qX+%!2%fQ3l21{g&%RdRptJ-w{9_NdGEG2RQxSF1K9M@=qaW&#*v8;>q<-Opx%oxXG`;ZCijU
VXGJf^Gc<JfDHIe*y)m!l+8)@lXx7VPm7OPj7S55iW{0ANs(U%Jk_S93TQeJsa33zViQ`Ogab&u7bGXd*P#b&PdGP$iqLaM)<PwMmBdGKJ&Y5#zVZ3gS{&5g
m0{4PzDItDXb9Z0a7gAp_sYo^bl6qI(J-XpvU-ZQR#HG+X<`?s+4WPoQOUbllX4s)hlapz{AW`I#7PSqSUHYAkGb1(xVJixV`ar-Tf^8HEd&-iMFkQ786}kM
6eG`gdeuqH34na(HgDn8*|2MBVkgo|{e*S1snMI2{GVv`PlvKbI9~{Mz5o+gydHduYs)3cdUeBWXCEC)-PMFEX5L_bk-e%hai1WlgQPbJMhS=SMo!dn8kO&j
oaLPnSH%brbx~x@fth*t7O3NtjrXew&3j@o@hIn|l{j|XCB5Cs?G4=-qHPGfXPW10&FKLhIH+>KHF9gI)1Inye`vMKAm((GqUGG@>^xk>3wwQm=8?;dqHB*+
F#iC7ZaZlv3nFfri5M?OA94Y_RD3|tBVmkZXl&+#ZLAr3Ou8Bx$$Tox)Zu|sKOyU@l7KlojDhKxr<yM#p7sDrYi8}Z_ZwsvrZP7LwHwTNg=O%Hl&XgyR(4po
B<{}A2?j0;U?Bt}pn8;%7~x~v6fnO2U@zwJYL|@AK`iUaVG=swX2{d1U%btiyM!JMv%@=$T&1I=<|tL_N>$26G^I7CoMEK7QBvp#^J5<|w@C?(K>>p@d;+h}
W`WwTm3!S}$Ahpsj;Y{<Qb?q1g{njDHCxR~5zrj7Bv8VEi&nJ)^qBKX56OsO)OD6u4;UY6_mQ2midh}$0yP1LC-%#TMnb&(WR%aqWI}L-HqEN43JYH;;MEw|
0o~a{UY0ebHPON0R$8|`@V{!Yk0<@`BA_d8#eyKw5L#E&mpmg;Ma(XUw=TU&QZ`fW5RZqVjXYQS*%fP2?-h6kxeE941@uY(zT84oFeqTJKeSsmJo^SmE(<<X
`4*^`I6~vaDSPQWSIx$s?QitPN+%Tec%o`aI-w34({$w3En?u@Z18#V64EThW1Sq9eQCz5^3_abD)LS1$Q}r%fv*DuPrK-}fQAHHfDX=9WcH50TN<PLG?CaD
da1dC@5Fp2%+FLBf5h}+{NO%R(_||$zz$CmaJ@2zdI*YyHBL`^Y|q$@vzbFo$4{+ZM(ogbUX0OADob)z1L^oJ4*i1)K&hCs-m-qC92t(lA)XLR9!$hZT^*B6
oj;PbP*&Xg%%JPOLk%f+T<*;Whv~b$DHZF>Wr`@pijDC0Lf;ZK=7!}T`jfHfiO(!LW4KNOb^3PZ3Pjs>>Muc&?n7PBu6KkCj~-YrTJtxw=mB7`U8C0S3yH}w
Ezp6?SUQ}bds2)9k*nnCi;Tn3wGq?j0q)i&C`xElf{@z7Xl{raw_#Fzq$gb08}s_Ffhw!HB3GHqmy<|+NAjFjsz=B$Ey?@Un_s+I5AtW9^<PefJYvLsA6lW_
CWpCBfBr^V*w*dtJpX4@;jQ?&1P{2>C`F2uV+yVqu=dwTy>9$O7j8+)g^wLcecc{5FV_hr93|vhLZzeqOFK?o6>c&gtWq{Wu%w?b9pZ|(f<Z7wo)wAuAiED`
gfTcI`aH~rf=Wn4sv^!;^2|*n`L3HGQK!WWZT~XO8TvW1IjQM_Q~9{^c~Kt9iPb#k)T#MpZf;63D*pQU36D*FX~n0{@e!=ZBMtM%u5YHL%bkW8+?MCYtGfS(
9($?S9za^@z+sJy$sn&kE<|XMm?MXbZMoFunEK~(0mZnEj7&35Iu?2+SpOPn`{Coo2t?kL2vonEJ2|qgBDmytACAbx5xkEu>BSZs%TS!K|5O$(uW{chsJG&8
(Yn_IFPBb;*<G&N9c|@4*Vfw0Y89PiS3%xC24!Q&e)X>7o|!d3w_oUNgAY4?PXjWU*!F{xjgY>615q%IkWIHy;cE&{QJ)!Rr@_?-P$HyOqpj{egq6{UOnUk`
)6LJ-X_S4mw@D~jTkxwm_dd~<UhiWAG4oJk-K5B!zchr5w>Lj&KHE=O1(A@$SG$A-!TiN4hnr>bPFlTOx0iq27#R;g;5IoliAX<Bq|q&_7woo#SxRydgh3fo
`V_tbb1;%_a#>i)0`u>6uZeF)w15wFXSOXe)w;9F(OUv2hk4t{`kkJV1#^u@G2YLyQ;&G2Me<CcXE9F|67xvEbJ=mV$wIDSv&`69S(HPjkQSnEtt1diiOwI|
Uis)gL?11aP7^1!n9|nCgzZlvs7ts)5vs`EByBSNAWQ5jms*6KmE=q`cItZ@%I2z-HiyvhRm#eJ=M=x!#Z5<;cCMLCem|F)?@z+qfv;DSij>vP8l9B70Dp5)
n9-wk0$n=HDC>>^!>B5Yh4tA3HwOWkKuGBLDHB0JwDBg0vek!5YTmh*jk)49GR4~|Vu*VAZ7M){pb0U_1!;A(@}(X%Vz%9Ms&;~>vtB%=u$x?6jd`Oy!Q(d~
#0%Eh0-w7}SH9zVuzhN#i(_Od59?EM!20pMc3a3&fPQ>-yAveGUWqtqp1>AO=O$65gyUBW7K}1!C!G;%V!<0e<H+4UzX$$dw?hXZu-KLeuyi_tXkvTl$dSw}
;hE1lYd7$*XM{0*d;gL(MGkHO*u7MTnHj7a+|MX^yH~UVzyYzERvkv-$D53jn!G0kUFE`vsRb6Rj}{YtW|GkczSPF4YQv~n;5pG$x`@luPNr@R0BkCXEolWs
4l#RSYKlNoc`YN9-bpZt0r#3J+BT^5)QePj!G>UNR@}HQO9o9$y&f*fucXhy(fn%c^I^Pg3p=N}#W7(6OUTn?c&nf#IySvP4LbN3e%Jd4Uvn-1FyHvjVuTAx
2z}kM@Do{{stZBHEDxb6(^Ro8fr^vYFE@31l*)ns8YZ#4Oe0xe%uZCA7QJ)TW?OIABI&oDdn5Y=y?l1|#l3Xym!eAKAJ7#e_+rw9wfU#6TUXbe57XMhxJM^D
qY;IsuYQn?DrX*?Q#TZmr=$(gpm9`d^FCvDIvrETEn~jZh1Ieno=&iYjhKO}?^1^wTVt$@fZ>^^R?sJsiN5|yg<>SL6;=_u2{P_$_I@endKipNT0EhJS4AnW
(yryb?@+_WqB3g8%Qq2Fc(onKB){NntFWc({v+q7BW7c{nE?_o_PllKx<oOMD7$al<ay2i+%%dzfp~LbiPKn21F@TF<xxe)U=Hl|<ft(?kQCmt|8RgfCRxv<
w7@cT$t7^3;`4|N%h$67KMBjx-c36>B<ppcV#+VjL4ASzb|jZ+fPTQ{mF4ChfQkaZdh3x-xq$pt@al;rzOKo>K(~W#N6#sEbEZo80SEc19mP21;p}%$g(zre
Xj{6*v;8$BoGiA3RI++!OFlD4E_i{FxF?EdCPY!$&edAKS6Exe)8rwT_$HQ3i}nxDVoPKs`&E6BA;1}){^xk@6#mPYx3ao&a(E_^n^6j+Tt&A|jFS`<XwvJZ
8OD4+EP9w_JK9wxBMUApI4Op056}>l8h?Y}#9M+VhE$bVQ}Rpem|sstnR{)ulPVaq@CBr=)V;*%2u=~VmN#%rR=zJ|x5Q50{LsxVdNlvm4+c0V4ML@X(R!Z?
48>;LeE+xh-co%wM)4GkGTkd`3&w@)j=B%WHlUe&lLlL(xy$YahM$P}6*^Yfc=dPTD_kNABUn%Dx&l(Q1`oJ30RC>)MI$jN&hO1%2Wi=6?h=4oA+vmAKx<LK
M44#Q7<01Dz(H1(aHY%C5?F)gbM&j{zofdyeXLg*wT~<mFQI`KPF$q~q2h70=Ispj>qA?;M6QCNKOV^2e&*+2*8Muh$L%X3-s0?CV;XVp@RDMtWe*!u*N(@}
3=i13cHA3QXGvm!%gUk;=wCvH8*iLAP1E{$iR=QmnFd8vq9`*UjJ4mjAWvWuwWkF_h8~fI`G%K);(uttflhLCd*DeultDO+ac3t+#aa@OHs5&<Y3a|?bxM|3
>TE9<B!r0A+ZHgj+>}C*O}Sw^{?X}Y@AI0cMl>xC4njId4l>rI<VB>Xz>f1UX0*)Q3;R5mYYO2;a4?-FmRR3*HFE*z*`Ohwm3fbB87jhWTTs6{;nbOQZc42S
gNFEznR*PS-XWfmPmdO?MrM`ghVlP06hIF5&{hU+Nh6hsyqbh?<WZ8-R>KAzDLr#jehtPw|IRypiAXPtkU4(mlUhHVjO+bf9Ka4{X8<~*#&L+dqDC;d?W^B^
fFgmA6si8N5bJ;AU6+7bN;j<J8X+HH;}1OXON8r1))bHp#*n&dRZuh-?(m_818{3A-(e!sX#&5qRzGEPR38C;VfPNH2Fu#~V<}i;qNrG@L_OL)$239lf}ThL
zhZ>p#8M4D0Zex5<@gJyNN;u7gohuE+8{V7NsX~Hc=vwAM<$A2wbr!8V8Ou}NN0wm!l(%M-u^HTfLxH!mjlm}QzDMG&%?bSU;NDIi^X%o%v=*@`n~N#e1G?o
_=9gkqZOGhabq1##O;eik``s=>;Q2$y(xc62|xjT{B2b<0ZzWz$e_rEyVSp<HtN}r0Hf4KuWm7}?)>OwNI?EMJq}l^O0{8ARx94LXkBpf*GA56^$_ugo{@YI
SddqV4*HD);_pMTaTG8^y2RLj4c-7{gD(T0RNd)PMwTnhuadOKB1Uee&ed0X?^E|l{q#SFb2f_m(;{osf>d6>$-Mu$((TsH{QgKUVrPls64C8drH)QuLL%1p
i(d(8YAG_kFc7P14j}y|gT+3mlE;gz3x$kA<!bAeO0s8OP$Xq<Yrc<>m5K-Jkw|mTk4fLejArq2y$iPoA3a5iHh*pK!EyOg{6lTK(6i25@$XS3hQgf~?q4uj
{DMouPK1Q~a=r=b*EHTKEBVMxVjUAh6~SB_D>+A~wku$|51B|>luVj10g>cREewn81Uv{?QOpfp{OfMybX-^gRTCyZKYGcn;NcM7vM6#T`W6~1JCI+j?6K%6
Q*Waop6q|)(!4Fs_*h5^;rXS&_U@3elxyY(l<$EsOmG1hMg(<u?z2#~FAo-ZgYT>M3C#f$nhO9`&St*Bu<N94XoTFp3-0cP!D@x-^f&+ED+$bme%76Z74*9U
5)%bB0QAYv9!v9nQj&HV&YedBeH}O6daG}!hWNL&YC^sI?|Jid`CaV}z{Hy-0zEwKf=Z8vN>WT6!Kizo;~iBh`;bftDX8}17y~7l{i{D#9M|D=>VXePt-(W_
TuZ;V1}E4=4+S~2`4XsrgQpyWBx5p8fUYL7nq;!TRF(3y`x+?7Ln`#e^<#x2g*~P~wHOGXSUymtrYGTEwLWiZR-EclTrz>Rlk?EAWZ-xrNV(boy(8+@_U++y
2zQc?sKoc71~>+^07It~!E1NwmS=PsWy7Ag&-*Bj8Mp?S2Im!cQiQhz2)Nyd^4K`wrsY>}DIA<dO@a9V$6u~JQP%dtUO?zrePAk_VR(q}$tR$iU&-)R_N_}b
MZ0STz~s{;cKw0=;i#iBftxC1>jB-4A*{MN|3``82cB+@UfIGXf3!7fo5GTAIxnAej~Wy+KCrjyHoHWh!iMX$=nhyr(djPU3?Y8bnG@#b(igz3y8&E&7>oyV
L!E8tVpOzW>Qj0?ps}!S1TPFfF*RxEK44e=(1|@HQ0z+7c<lYsNF{hfNf0@!3HU)t3<o4-gZVLoOYE5N%~CersE|Ce+QZ+zyny)HF7BQWZ0Y-UMEa$=&LILb
sL!qst1k1Qm`fy~vm_=AErN`9-6VPtG~FYEEyV`3+ORj6>cf-bVd<9OiTVCk8h{Dt@6o)orB$vhtOg2QN+ev^%1wW}WTK1wS|e0bYKNc-E03laZ<A)1-x8sn
j787W!#u~=q&WCrpAD^j4?H0@xU%4do1C)YvTxQwSB|EnZ#G@L2r2;~Tm@tDg(&Q#8dJBppTf)QTI^UlH8|Ce@fqE&@r(Nu9x|c%Db1|YnLjqcm}H`mad0t>
5o<4L*O-hg&H3iS^(!`Ju7xHV^5vzCWYELl*&igNqMSZ@kHor;=Ehi=2v#Re5yP3qpHkym{8vUWY$H{|cNbzXI=bsf$3)321A(#{+G4ZLbsvPR&!LE!396zL
m|GGZ6bQCw!$*&eFAW_^yAZ}<xtVm-BjtdrgupKg<k4Tp`Vg2oY^z_V5^>SYGmOZ##bBq(q&R*qBUrLkCuxV%v3R}?@4x2xzE|Jn&VBoXxv$7T4c33E`7OKY
1<$lZ#A8~dFl(f89sIXu1HHh@qsijKZ|#uUCszrm1LVw6Z2o#|77;JXEIYfR&s4ntQ1lFGSFgu6Z~J-)I)9wBFQ1LQKHYlGRo;%z#Ks4o(f^VTq3-v2B~<@o
X!+VyzmLJ&yYHCFoDUd=uXt=vQA0M3s@is_W(<<nGgMReCF&zoJ|$>WCGRKhbax~OQw`OK<-wtIoAeJqK>d)BKTgekLBJyr8H98T>=78syv48Ln(Yed?4SD{
D2wH3kQI}(#q`-LoB$vD?h9)r<+%!s&@rKd6Q!x)^@A(d#KlS_TBna#EoFdxL4I<F9b_~W6;#+v0Gu1QpNzeP)8K>Eox!I@*;k#3oT$f+%tV&8<tl-O&tU|w
3Y=vbSqr7Y={*VP6^so8XGmG6t|k>V)Z{pV@b(u?e$gANpxO(W>CeURJNGRQM0hq)6+=_k&qd7mDsEA$Jpp8Nnp-vy7i=vWy(e3@$%g!q=74UM;t7I@lwj^L
_m=r}?B;3VL}jU+$+%8O>Q>QuP#0Yv3-`)<9O6h^iGC_L6tYNoQbIGE_-4?UMExb|O-3&#)KHZf;+C|wEBi@IzXF;90bV4%W$LHz&5nZal&fBCy?6o_zgJ@M
qW|VlH!{gc*}5(Y%LWW@MVqKWeU}+(FW6FS<4}vZh5H2^z_gY+n#o$yAjI^#Y)KcRrkJe?!z)j>QX0*&TpoXOnX6ZxKg|DMHTo6+r^K^p$skF)sSoN-3~L17
bemzDRJYs=w8T?g3o6&$FTXDCi2mnSy*K(SrM@(uP}IKxM^$<13={qr;P%gXrZ6B^^cRtXQKL<%)A@B7mL%Jh{@>aH=3akeI%@g3G`}9ypP+aAgVKfrdPMxZ
CnS_sRv(Glmg?9D79?IiTXAxJa|19zRUdF}CxisA(|*lY*z|l``Y|%2QMf$yC$5yNqWF?~Be9k~y#PWt_a+KHsfbFbM}}=Odn@ncFfrMo=&chtGBrgwSqY$a
NvPs4%)>pCfF6fRvx`oNfp}1bBk5jCSyVb6L%@`F=3aY}MrvB5_O*t<xyyOjS7vZXveq7O=i=OAzbDGC1;0fNx@J)~>oHWZq;T@nnd|nEe}fUFE$2fIWy3X=
3R*6!MCKS$`csulu`70}2dhiDMCb)OqUZ3phcT6+&nI{7H~O(LX8s6^7!)<qcOwvb#3q*@Nf>YxPg7g)?gnE)JCd4lAw@cf6FmJVebZH&>YZVu7V1V~fiCay
3JKU@gs&TE7<qE@^b23kjHx)T>Uok*8Z%_?k^e$!mjybpG@5MrWy~Otu<b=}=l^qEG-1tpb2Gs2n&X&EqKt%mF2SKw+s62UBB9>X?hu$TLe4+^f18*HgeZ>H
YkA~vsGZ3qh6H_K0gF(k|J?H}jVtiYwjG-p0k^e`+SEcz%+*(|KXpm6Le<hB0M{1A{JojKelq6X%i!~v`nDD)y)J@3+4kk*rim&7DF6A(?I$mO*a*4O^&OrD
sPXFIvmxaw0zWRr&6Xn&Z;k&-8NOxPAW+agvOaiQzqjKHKvgC7Ad)Tir<3Fi4qBVBTgbh2`6;uSU+%a&kZ~bMk&xaHCq`26>{<f7e`2g25W%CtT@3+`&rbR}
ZBQ2z&PEY-<PKXKi!PE!QCZzh*N`GQWh6AZ@o|l`zR<}~Ef;xe5i;N4l!yt$7pRs(Z@8T0O0<FH)o=W%8s?0p?V(A$zP%aA7I?)Ld&2~c3mse)wV13zYwgN7
T@+yl`21H>=Fd284waOj__|>Atxcfy@a!B(_&B;<&CAByyS0{*@pi}Ibv?%g6iovs+rpsd++m(SVb#uHtvR7~QiF86k<TfznFWOY^HmK{0@W*Hq^9N;=IVHn
1nM^yA>Qg)<5-k8|9$T`FYi_FPylOjJw8e`t;Ur)GA^gW(KZ;oV!T|!kT@nX0Zxmgh&<*n(c55TLxHR%C;vvZd4|-!29iL%i+LwxmIIo{iVy0wM=KnLCquSt
)Iq~+{SvhT!DQ9;MOcG|Ou?64d|2NLfgs?8QU%yWDmFa$K%~!!|At!r-U1N?aZYlh1bemwc?dQS>OAi-thTvSCKz~Sb^c|D7EI_dSua0LF7<}bM<pjbEU{OY
#?$UP|K$=>NQ<aY%b1Q<2=To>8uCE1EJ%YaDqFp+r%cw9y|ATMsxG8Yu;}%&6JUm>0}ip?agoV2uCi-+cX-Z}uz}=D*V}ZRSVRgRtYYq--5m)Ndv0^#QFlTj
{QUj=^u25QgHNStYLJUCd>o~*1CCw@xAD@Lfb&~#D*wVdgNG9sEg{`-N*+fGqDx+njX^T-`F^vN(8<S}ullWkxNl2uP48sCEJ$o1>#7aayC^$sdMKnw^*#xC
34F!hANphm(w1a390V~oa=c1>j_EZO%Wc$fqe4xHqq(fo^~#$}mK{Xn(49)Wy9^5Ek}r)A4Z)_7fOg%@)*tGp#8^~Uh^nU~hpS6(KqpibPNPP`>7sZ+!SPGz
4Z@HYIaFcLp_C(FbWsII^`66qLbPN{e;Mv`ziMb$(F~<e=v(Ff3nLI#qd5pSS8=VgRn*>o(pP5N2hdaqo3(>(Vw|#dhu$o0ODMq-(%@59_A9?!6VA<d`YuoQ
pKMzE*L?U%(e6f>rP$nlyBaKm<D39#WH4eGUbdcw+tAz2yk_{NC$Vstm`|6P*g_7HYSk7gI`Uw=0O6=EIEiDu!kzL~obZ_aKXFhjNtgP}3op=vNxUcydC)GT
X%1XDq18HVy1YQT?4|8TJ}=>AgHQi<IOm|y0tG$d!v6zn-qea99@6}KYG{rI1VX`fvhyXC&P@-a_}$-fqQieA>78wPp`#0nybKf=$-MPiGoe(kq1IlmytX}2
gRYGs{6ySfGe=pqH#8$`kXMSDr?9n?Eozg`$7ia~D=H*589CV(mT11s*?VvDx2^z!BrC@t<jkXFo%%t9832?kF@rqQ+705i9R3-}eG`t)%o3H;1_f$Hrn<Q_
%Ny&hG&T;m$sl6rCl;_?KD?JHbYFZd3JE-$O-3jD3CyiwLG)l{k8$2F*3F?&z;I*Am;QRUmhgU-xg%yoB5oj4>29JVbEd$J;MMhzoKQL8j+|(#>v9<9s6#3n
J!PD@K{weHSg(Ft8gWxgmfZ`<3f5C5UGBb4jH*^Je&pSmyu*U0qSKLAKptKQ0l+?J(tyvj%mQ484M8&Y2%mF!%5IWFfK$mioF0KI6!56#y$&Gs2+j3QgwSl$
8@~?7Rru}2m;&r&%Of-3o<%`o^s=7vTl3>QLKSKqOBwqqoWl4AISMV4UhBlOF0P>iY|PTfKRb1HxZ}H(w0keGT}?@Oi-)Nly~m(nX5{g4<g=40Xn@_1;TR+@
VLtNAz&CkATnuIm&0%W<X&Jjr2HS08sq#!P!EeSQtitM3S9wRJp%A4A2gwqD<h;6vZa8c}TF9dgs1&%N99hN=sH`>^8tS9HQD}`gx8Mi$6j3s=1gOX7<CZ*_
4Quy<5>J<L@ggGB6p`3EaQ;<KEZxr;+#;tCYlGLDJ0PrXk~R$Nd%fE+7Y5KJ6qXGyufPj!5e3i$=f5uM_7HZ9bJ%U)yvo>az9;&COBhNZ6o^Sc6$zxNqnTU+
*}FBRTBLpc@C(I!7FS2$1WInmqkI(jf4LM{<kVN^jmX&l3$JF5Lc!L%JJxSqJ$w|^$V(74Qw!}w{WN;15wilV$2bYce1g#DFa=b3(4Adv5=p0Q`HB4#qyVQa
QA<Y@57^g@n;ReW3>KKtt1do<e7&Rv0fIzxW!$o=y2vnz*zu>pk2HgnsNs@qi`$-#$e~Z}hc%!Rxs4`Ao%Bn_3m#YNGf%l@r-$MlxRYO?q{F8;U?9K^+7ghp
bvqAxevWugjjWD<_QhYBc<?IdXzY~-4vD#s`q1Lp$C>37uYU@jf0bcm9au?gM4VrA0hOS3X_u}P5?Ido9%naA7|1Shk~bWy&SROdh^EOG<OwcmHtmgP<*E%~
IM@toRUuw-(<Qi~%u9~F+JwjNU~O@cOdbHU9T&w21)xtJRz)D|aDkQUAmj$ZKpnnYx|q_tZyY;eDgupmql`M^507c3wwC8=y+=JY8rI%fK&tm4iWym67#dBh
x!Ls$O&~7X$F0;^PQ~>KEdfkBnfMjlW9H8}E$DR8T7lhY(L^$<vl8M|;BpiwLyVz#se-Dt9}p;swb~ftz~H~29Om}{R9fjxk;R>%%6Os!Qu40Ddbb<I91Z8Z
M<H|camUBxZb5FS&7@Psc2-zmIt{Wug!f1T7GD5DPT2SAn~Wp;j3wOP$G+Voc4aOq3@%A?6kK+1TnXWEB_0-Lb(B}4O2)aP$1bq?s9rC}+A3etVqfXcI!^+o
M%psj&4oHXjIYT{9jsm^ftS}DCdlR%99_G8u74Kj`>ElN7!wZWEB<~*M6T4QNSc}?1`;g9N=xSlZQB?uZ7>W^qA#x?n;#$ID*QRC2iWv!BM=d-NVh~6w?2`0
Za_|}&x7jka*1HRQ)a&y4meGM?JuXZOuHzFJFZb+=i4s%yWRLD$Z{>o$|o|MXmg=Vc_T9%V+RTzb}~ZJE`wF)O^wJ@^Rk!@yZZacKf#rDc#}`hf#l_@D=${Q
X&`$TS=`uOyFy$v!ODDB9_G|yFn6oazyN&KVG41i-!7U)i_(QYBbfn9_ZC2Hccvf-xWRU8bd@t-u6zDDyb<NNA%$fzsFv*G2`R1_2>)Oqd4n7AE;@EB@Ol3(
ne^edXHfz`p~Va12)*BG<GQx`{kc0is{khOeb_g9=9@@8SMu!!*X6WjV}OCJrP;n%ru#l78)7oce~O_Baa1G9j`Fi}0{j$Z#d)v>cRP{aJA?z}_poo^<KFRC
B@Ks7lqxzSTSQL@Kk8mD#nGo3O$~QM9LPpaRy_@sm(&A2se1ba{J9pirt&GlyVYY2%0WHtYB*cPmu8Yb{cafbBVXjUEFHTZr)RPLi(?HDcMZLhQJMi*WKACq
4^`!?#^@%ltN--FBe*>tq*%$SG58O2f9b;L^r5d;-0Jv{P`N`2d%^BnwAy_R_G!u17o~gXQ>Jg6mMlf_34mr5z`O2U&^eHqUq5(Nos)?%V^3g-I(FD931P(t
V)|upkh<z^;2qr;L1k`{|KXA$fQ~H6+v3kn2fuW4tS6a`TYs7I=pyl~TSAuW%6Xi~at&_*pQC9AjAOq9zumw-by}iKA5)g6;V$6M;F$RU$b2Jkl$nxDX-HYR
gS6aFz>mu$ylz$M`eSLaWWFaX)dAAdKO`#?>C%LE#T)*dW&C><&%}fnb8vOE8%XuLzx3KDkhnaMMh}o#32L@L-*$A%m2D4qU^*A9A+_{kIFWaXTz_qI6k(hV
B6(Nj^~C{R6XQ_1F<R$Ch8SmgcN%4m^c<wi=1h!_GARu7`d2%4v8JIApMgy58EBIX9@8#|hn`ha9;=}Tyop!-Op3*V>%J&5QKL8Dy&TmFgfOe+!z6d~YLnGD
gIytbIJQEnC;=IWPRmE&xpQ$Uk+qj7`RK5B_`=Y2v=F**PYd*);Ch`2dy%sM5{M&)b*q-eT(xi}qJlv^(?5A_wNTPAhN_wB+KQaJjrE)aS2fAwG4~~{QwbPV
#FMG&j_1?_4<ahQtV|Hx_Clcj26L_Xz{IpsG*O)ex?|}%hgTiP$`wwenut!=@&9^5^_s{11pbmlPIlqA{3!A0oGjQtsaLAG36?xrUT5!5qtb6x&I{z^fNOJ}
6`QQut_ljbH4JiJoBfbeC3Y+D=!JRwpMwiND}IEZ0h;N{+vO}W5P-cj!6XiE?JBgp#uiM}ih<2qjuyo{iA=iuuSXpIuxDlVdYjh#7sx>|4|qsu+dwc7Ko4lS
t$KO-RV^k0lK!8vtvL#z<Yfl&ngWB2MyBaYxR3^Dfv|4f?OH3tJH2rM!h06}BmKo;Q4jt<nqE9xVdQQV&cH!s=!p`(U$NbzcoaB^8Bgs*&29AUSoRJTrJrum
e@*>vQHjt5`3X<iviJ9<9ADfc556K^v;;B<c2jP(d<q&@zyN!EX{IWCx-LnNadC9E!R;&+9p#Gy`jR=hS;Usr;6Pjxi-SBTRRx0(F{|`--GEi5w|@SI$fM8f
p#~D&HeKp1Yka-&meK{ZB%$Cc>i=Ea?l$X}?XEcJd3d}n`ZvoKixXGF*MbPPlF_xeh$ecQuEmOR%c;gz|LhVbHs$Wmb0NKShH>~u&-#ABpUurVnG)3SSIdy`
-BI0IOZN-&=@yF$806M=3M%L24RU2-h*VVLkrgr&x$Zn=t>HvMu7Y2|e9MRbW{L5RZgz$U!UO`YogXZEez(NRz$_h`NNGcKCh~|4y+mV^kd)IRPj)Vyi%Jk>
UYB@{$@~M=dCngBZ8#$o7)#EI-N3Nc-EUicBO2;2kObE-)}(E7Nq^+sc&W5c-!0NH3j9_ua(SRPKP^;d3KK4#=5-HF+%vMisnPI$mFk6}V~@-^=x_K11Krc)
be9b@5tl^u+`*_rp{m54(<=p5dZXQ@zgpIgH|-mt2_5`Rs1qu_BoRmIpCCR)eq?9uJ+uj$1W{Zgc`EXh_Pltwrud|T94PCFc}$muwdQ}}vg9aUSws;#)s5E!
f&xtAIjGp1Y94pBAEhYL{H?8$_PgpUHM*~Zat(A&V=@D;(}4sQF$hmR8h&kv{U1rv22Kg2p>NtxGHhLy90Z9@8&NEAvZuzjz2UNuQvW=YuOQiFvZJW#@-Tdm
T<BU02eaKFhn%%?6jy-<r%!acWqoPN6N|4bY<yqRx>2a?VNAe9C$j$JCB009T)bwTyohguX2|lAx}u{&)ko6poH{v)AQzb@iB3D!Zr<xW2k-&CV_J1_A57D|
L<m)YJ&9vgP>R&U+;G}Y&N%A|1<wb`l!%NHy*~|a=u##$DXuz=fZ0$xg{}{q)j}7yDe&_<gU%|cl=D3rbpdUy;%w<RJIs6duH+AkU<4%`>P1+qfCz@Nj0rcn
1y7ZkhrqenjV*!m^hI1hY;Hex83^QGH3G@>Tr_uDW_pM7MLK@yD;BBc+A%<k$>n%gf@nj2L_~T9=Gv{OB$I=xg+gZzjF7ZFcpg*?pn6a4X4LN^e;mEXd;7&H
koonVtJ7eoTh4JT@!dj0<#1MHrDXqlYxf*`ccdmAh@PWnxzc~v1Jf#H13RoM4DF>~((PT*n?11_4VyoWA`rk;Kfq~JJzdm7fkF!p{LjPn<!Nv}&|Q4>D4?J9
Y}WDN@yV8Mp<&ex;7d{dlYbEg(*i#pqH>LbS9*}MW-imU;+T1}EM>Hx6J1eB@xAnSZtWHte8E6<Rz@azFr<@$Os{DsMJ$|=`E6I)=sL;4YJJO(T<y~40M)GS
ryqO^eKz^^$mswZ0BZ4MB8pR?HuJRs_)&f4+vK}X>lna3cSBM`Vah#s?d^g*Ui%vS6ojx|zCMZH#eER5o(p1$QtDvXwfdf=6hw`W&D2h09gFu62y!BT=BPjH
|HHPxXJs(WGvYrw@*)vglWc<*_es^Yr%~l4#9py%pFY*c5LwJ+`*5yKbXJs5d)O)-f@$_Z+^Y7aM545CTC#_Y>FLvYqDMT9f3lH%J?ijGB)&skdYE@~`2zew
4;{FZY!R9>>th}48U2%LcnkfGBayoa{+$}Nn#8*3r?dX+E*{s<yN%c*{(LJOzM!n)Cdd}Tu>hJ#my=qS6LBU|I5+OsNxV8Yf1A`%{5HX+3xhE^Ri*sUvGq-D
enLA=3YG1G0Hdk}iX2oShwC>;8=VBIKa0_2*D|WQidY1~KX%66>QGpQQr>S}b&(f<loL9u{)@NmaMXQ@86<c6IOr4$ob5mK@t<oN*0Oe4iz1>z>4o^l{_q&@
(iJM3$N4_=(J1|`kGs^)XN}LxR%dj;x3f#CswJ=AOz~5~@NC8BE7sF#+CpTIpt2l4(KKCT{tBh0dcl?WO=0oHz1f`>3<#6)x#v0O)u|W~&n3ohtFpA;pN}-)
#n|{t#9q@&1Pgx4lZuyjnu;ZN%NauF0-1m`>>D!-{abZ0>{g&&y1?F~-3YS82EJuPtGn7hMn%m)RHT5E37>6A2sh&`7|ziU6b|}FcN#xV!qr9PR*X3mVY87k
^l;Y?o-_XMhU(CCSh2edugbQ#xAkmY_m?r<AhO1-o--S3b{)FVpKp`eSvlLmu_4@{7@0_TJ0Vn$DFh%M_qF^_^BGPT_C779jD7;KVZ!`$YER9A-qwbo|DAJs
?cp9Spp|6%V*9xBW;ZT${XKt^_1th8$MI~Gz3Xx!ZZ3x>-lNMY$v=jI&7Dq{_pDPXuRwU2c@GUG>R)*)ELSwS<jc;l@T}g3mkR~xrHIoLx0o=F2AHi;iYW!H
Y0lV_C3e%o<{*pG4ZSoY@lk>9G7C)`gqL?v@-n6na2aFv-<<fTj9;To>4Vn}q7h;dy^p&BxeX<)s0n_8jd4Q-wQ8a285*+<r^IhiOiV=?Xhj&9`H|Ktbp!0r
P8-_(%JGBU_&kmrgrA-^hi89uWB`)CH6qa*GUb8;Q*MR;?e=680L@u9SCHRuE**c~T^zy0;dpS7NP6wBv@U<T!Da0Nrgq6+B%k4jw*<gG0d%8z%+mq`^-xN~
T+i`)u6q)xwubg*Q}K(UxXoTMGdx<mu57uD*;=TypH{K_VQb^I{ouK`Nq4nPI7CqW(!gB|reu~?4U^}F#h1Mu!Fhe#P#<{*P%lX=y3V7Mf|$%93<NP+3#kLc
&4BUKQ&#^#SOI0-s3_GSxm~VWOIxcG#T5iB?t756%UOer+Rt?!b8(GBgud3v^5Y%#ddQc>>6=mk_%~Qn(Ozdl=CA4kw<-Gc-}9CLu5EQqNtX<@fhADtVHB=O
gf&4VgE!W3zZA}L<r4>e&Xq!Pzn0EK7AH^Uk|GHdY2Qva0CdHn+jm6(;ZLcttn4P6y{qvS)s2K?^nB3t$GFD$hTsJ|G*4<P<hYPnhBV~``hpMh<}t)C>@72K
FQ>kEuZCwfl%U?oxj2tfI{Vz@=0x3IOGt0M*#we;);WKuW}dh2z$D_vY^--EDPT)wT8yPoh34I+<L=9a$`l*CTx+egi<kHtOq+)BZ6cIB5K$&28KH%fY+p$g
%ov~zJhivSFM{5VU*cAhN>RK_?B5>Gu-Z~S1v(B~ftGX#l-`_e+K`Yz<U1eh>=F&t@dkYV15Jm&lgwzlAtBjpR;~a)rFv@jluIX*^~2U~s++$0R^{v*eB}3&
jY?w+nQCAjH?+ui5K>4QrLYAYUH=uN(!vXv-&<`|H5l@Dw0zXOCqh0rFEDlHA%~rMV5IzX<}Elv1H7zEdk(zmSP8pF-;_MO5RPn333Herbj8*8cFN+7#=u>2
H`9*1Vt{4yMXEK`S5kd4e6cA$N1%H|!)sBbOL2gSlwId_ga}pb|7&(=1-X4(063745wG5H8va58CI??E$E2F|rQ>!_sOFN7z{^QsI~>g?SVm7FNJ9U$>9Q7=
jR_b&+zy;agL-d^lk{w_b*M|}WAv2hIrs5Nq6Eg`SstpxSo_%2>+PVIIe8x@NSu9u<|XntTGP5wMUjJ`Cb!Oe<@QB#pcqmACHg!n+m*L7BbDr!S$220;bH!)
IA)$;-lKBSbz&!7OsWejJ$X-&M774dAJg03JB@XxUZUqUdX{!IV68o0B?0;VG_pen4)XXWX-C=K%zU@Y1g(Y?f8I3IWDweH!;GKDm1wtXp*?j~rSUgU&UR$P
XtbqEK`r`S1I0k;ys$yzZ54L~LC5b7rFk$&^u+hX)WFQ#8%P|j$G&h`psU47pK>Eu>n=!`EMQ}!wf@wm5s8Rw0T~`*Fu@M(<As`eSO%t$c;<#CauuT~(KCvu
=6ya>v^I4-qRFvIHfRBNLmjksV?Zjl!@G0ZFqZEW`MnS2=u{O1#YQ6U<&Ji9%Vb%Sw)6bhNGx#zG>@72ZrCa@-Oa+dxnHoLaM{bvF41B~7(hBu032-M3rb^n
2hyBDtjqGffxU(Rz+Dn%pKV)Fl&u<wny?+w(n6w+cQp$=S|!25D~GqKtU8Ho#VdQ~l4{eyiAiLCqKTuA_&2vkco2+wd^Vc^)cv4B0DxnynP&ZM9A`vS)Yjd9
Nb|>9OU5+{1>TVh6FI)$Yl$>^*rGrZ*=o%nJsC^eu(%`F>@uNZx({(4p10GI4jq&MQ>I$Oi^;W-2<fSvZ+PEPC~9i^yS+y_jz}y;hi4<CT8*joEaM|mA`Zd2
!E7?2uFpr#w>xmo(Dk>h@vP!rqAfRIXX_fLN<GJeenc43q!&lw6c4KVu@?-XN1|XC*3bLV8dR`tG=2jXRd?%8#4xyE#0fu3vnY`4v<A3N95DxAhCBwo5denm
<tI)#CY|HhzD$@#bC(46u8zpETqk72r>kbXxhf`5#&dGUFi~ls=N&bsWMtUwwx;fd76K1@D`&6*lw#=|+QPy_VH!j@f%yU#D%UD=LoU90cI=}~Uu~ZPvCpU~
@CboAJp}Lh=0(@vFhl1{wn;x`L#<K23?CMra{qNB12SA79;;U6V5D>Z3NyT^!-Lxl`k~fZ*|5A>N4p3zDF6i^DH<b@*e6rZ!~U_%o(RqfPa}SDAu1u)P=T5l
nrAN68YE|zjT<(j*R14`24mCDLNMeCRzZUmFr>Tip;y1fd7?CTGOoCc2sEAPOhSlv{Pr#yeG5oUav;?7#*0!1+tg4570#TxQoCF{Rn|VxbDy;Aagz$;|M&;-
<|w5Hum3OMg@C{y%%NnCHKG%=XD-EEz8KElFQk8ct>`iGHG9^a(bxAhFgS25-_%W(5Z3x<^;4mo2H8u6U^31LAiBNovQ>3NHJq+tcn+PlsoQg03fzf?&iy=?
q1iiS7tj(DLB*+W2(VavzKNf??WfVZgsvUa_a0y#o_hytcjMQ5alhg!*dw~Gsk(P#d9oZdvBe?`CqROV?9(ghLd!Hr`8ATd7j5TsAq_i2)CULpnHrp=Ez%-N
x(N5rEzq0}>=EaT7c2<LF0p4w&zhTH8asa_gh|Ya#@PnY(SqqIn~iEdj19TdT&Gk`Rs`GkD1$DCMK<lwD8T4HZY_a*Iqgmld5r&Lrhdy9QdGXa8svFOLN&}(
<Tvv}V3ECA`xICbz&fT0oo8Jf5*M#oGI$*X!Ndmj=Rj~!zBUbnJoBZpG?%1FZ)u;5fr+;w<;Cm~RT;|Ai)OtqrZnr1((Tb|Hl=1uB}95O9<pDIWQ)E1eyMf+
T74nXkOTbwWYul#?xsz#s`sk07M_-Xg${h$bO9+Du`nwnDpm&Ga}zoC|JrU^JMy>XCwyX`&~4f4G=4jv{<-TaO(ce|Rlv1!)mFxPuxw;3Vi<m0><=%5H)%2v
W%`K#;q2784v87x#e12_*b#4M35bEdi^>@qQvy<aP-@6XYP_tvm2!9v&0ia<f8Eh~Nyd=HAcc{+pxNvnI9&e5qja6s_}1TU9f8>=gPm6Sd21bwnWm}wtVr>#
4S2qtf;a_@RI-bPk0`~Mcn_PSm^YCI#Rp~8zlxFeXEPC(j_{56|L#e8IiljKD@Uw+A?dX01t3hJ9M-6Sbj%9DUb<RRN^d4-Z7-z=F6F7s%8_lSa*g{SN-4DN
V1=rrcKyS~6Yrq4921|cn?v(D<9PHxou2TNw*Y{l8qSpD^0EU`Kw%YRv8H$3sE5T4RbyBp8gIqac4rbCoGC9GQ1Wd&)z8fk6IG{b04fI_sC_}#wVuYwxHMlb
w%K_o3Y;jfc~0Ct2p6Ni8tPM9QxK#uc6fQO7~U8$6Zw&+i#vG8rp*P3$yO$oSS<q8V8Py>)E!UBv~rSvIPH{R4-Oyll(BLD44H?=*>BK0TN|&+f+c2_C13w$
g=%ttAuUMicO&Z-kq0z0amwe*gzw+8@#wHH_&DVK@2E=4jKY0}UP~K!VRnqWIQC_X%fyzLyoj@P=O>*1d%rtEx6GU4_S`kr5r`|#vaORB?A{pl8HG<8<oiRz
pFD8sKc3XUV@j18uAS7H-h!GrypQ@!IfA~oxX@3FE`EndhZ>ZM)yzzCqt`J9k0ZHy#Kc_+YPZJ{B=iU<s%U)!(U`<nNzaseB|q!0(;rJjbjfOcbLS;$Q*yNF
{mnKsn>3@F-ad!(Oox+xNx%q#r~tNm6W1JYPktyyj~3Cg{p+RzZJ7HEslFC>>2BJ6l`cVo5m!^0BfD;$^1j{|4@S&m9Y1R^Jb4}$T`)mkOkW0RK>%?CJr<Sa
X&IF1h{Cp9ZOuCl8hQZ-sL7{5S&>mZyaQli+_yYsPBw_e<MEu4r)$JBYHu^&Zc_A@k2HIv4;pIXgCflUpg*J^kTi(oG57VW^Hl^(#PJs8Rl5$-xlkcyqCQ;?
f{pNDDgCI)eNs!9$j+i-hm*CUG{YAtnY$9J;fay*U5Py51b)ad-MtlG$C}byzobfEs<&^aKIaV-^L4gPO)bc?BzsUyQpeb(J!i=*Ydwa##tSiF(nWF(Eo`TD
zUg3FcDyhPzQW==3x+3MlPg%E{hGOng#PZSz7EVtG!Jz8<ex3z*^|iH-$KIDI{*?n`0@6Y=af$&ax`J#LbWP&+AeJt;_^gL4YCJZ!ObSMt<nT%1}MmRcKfe-
LugmtUKI=0V)S#EwLam2U&!7#={d^`1k%!w)Pw81e@w|%3-+MVA$;I^CiFVutAx*LPrMG(M5<flzU_mNI*>??h@(o7i~iFd;ijNWZwy<P^HkBc<INUm-Hp7v
XZ|Ruk4^RaQV9FZHG_wgkI)muL+!U5Box;6X)y9Bdb~-vaa@JAn7|<h5OxBbO9LzwJX_ZbCeZHHH9cq%nu;it`g%pi?@DR_<Jqv0J_%=_<RR`FRw}^C#1f-Y
87qZg%{JZ}Fgn~C$3`^1@=mae8fHM~RZxmUTt%D-$9kl9;{8Sy?__&^G+sN;Z!^yPi{xRegAun(XV`OVq}z?3m5vMAAE~t981D6`n7!p#`^SUAihdA}`3_Z2
y{|U}l?0o*WoH*d6ROy0)C5OX?*^&;I|KXmp6K;;`obIjJfoVc#-Sqan+=&Nt7eODb1)Gk65t8iw1vPTN#W<xZ-nEvVvZ5a)~ur*(!1}#DOMQ7lu<UE^^Zaf
bJ@@fy@GI2N5PqP<eIpulpGatiob&38)?41>??&Iq6?(MvR}%XpNfZLZaeq9EGEs>h4}iJx#07$f-e2L#ya`94OOVA|3QHeP9d$sR-+{DmOqpqmH2ZmWeR5w
ZK3VGVa-0lhmPg!_DYPG4xRDD{k%Ns!N@QHAhw17RPSSEAH>0!JuJGT8f*Pmx(CS&j=oenZ7~UJjw{{1_DUI(Bvw#ScEgM><Sn`>9X!DmsWAmd=nVjlMDlr!
$eXxW6Uq;@#tcpSEn3tg<|J%P$Uvmkv5K~vT8m&X^3S#NzUE007Y|fnfXOl#Uv<@!%k-FVQS8#vFU*}$)kBpQNq`|;ij@mzW)|9`2FpKKq)KG1Zxw`wA_GJI
)PqJfFIwq<2Y}SN5>Y+dx8|7~=r(up1Q?s|%Fe(tktqGvmn)NY0ntN|w-+$%4sdQB$gu7KP2JGXo_&WZ^mm}6x4IR?^v314$V1{R+u+X4oGCBgFBolQg|r}r
eT)pRYj)Gz8sUs{gG@+biRXxiBMF|NflZkBR5zvGX0`9&hnIx%raJr}oX=d0C9Tc9M9TrhUi#UbO1b2zx|Bh($N~R<f>qWTcIxFU!Y^)*RECr6dfKW+Qh$k_
-mcV%$N5)$YD9J%vQg%E34hN-dfg15pH0Q8#fShr(}+D`ZC{=V9Ao>kjng4%)#8N=xF-mQ;zGqlW{kr~DE5G69{9Vc@Cv#AF`nt<sPrnfIHx5RKt7QC1n1?c
J~5a7C*l42GK{R~kQCVmh`z%TEg!i&4$W6I7)_RW`5zPSNS-<x#y$#~4_;Ey{s%lHR^uatle#p!LZDe~TN@oLpFlHgF~|)AqQ2^y85YJF<p=-LNvR`MVPW9D
-3^ymMt?tAit+VSTZ9cW?P8(}Z^Z*DrxW(F$|c`IEdaX;bth4I(_!>cjhs@e4xn+$F#{+Ekk#nkA!#HD2_y)ZmWI$zsZqP4lLPCI{k||k`hZDf`A{#KTiEx=
8d?f0(WI4;3o~_-wC&(WCaBQslNq&&v4=-LIk3xP(f@=O6&hDIm@dYCNNlKK(x!s%Tp7G<AkFHqzJLzw<7~HVP~Cm!KS}1w&8!SKuZzQ(axRpI)hhlkvCY#C
Gq*24BnUH;S{b;K^7pbI;ogta+Rd8eEemiEpE_Y0pob;G<SXcol7b4*9W0I-@_ap`G6bkN2HxMDVptTy1CRw=sd=;6urNxjWKxbn$=$j9hULrUef~c>*{YzZ
IhGCa@LP?-Uc_t;Vx(JGed<{@{$ErHFo8R4g6{haxoVlUI=tYHp>xB`WgCa0-$V@}qk`lO1UVF@pb4~N#akg!{!%5D?CL<f%_>;4Yez|wz3L*%`jR#Pf?$_T
(R4kWUy|S5q8NLFe8y(eDvt|wxkUZK3|04exWJpK8mufUTTA29CTs$2cL1f3!|jZzIRicA7aZsm3r8baV)+%WLx1?J6?IW!T^1ep$e6r+yTf$)YoR(L<pxs<
Bs{{9Lhjn<5udO24zM9@+%|(~b1i=KO%E7|wSs}CgcZwxxH@c)Hg0NabGJ$BCAp};Z<-fDayY!(Hc-#`MX-3|{~PsO2ae0G05Cp*^^)Uv_i_)ZlcuJa&QfZP
gjXAQhk%)K4(l2XHfAk)Bc)DmWOI-71(5;kw!MmYd37Z)jvH4fdJswVFkH@??XWFoJ9OP9@t-e5PQ6TU%-<h`v6j!G?{N+XQzs_CFxUa~rn^gpW`m$uHpI0R
Zg<RUV(sah9p&f1)N|6R69#^f^vothYiWkyt>6^ORA@^Cxp}wWw`c(1u_3*Vrvq?N%>|*aP;0t`JieF#+IP1nC__=0aaYkm1&O<tpp8ehCWVd&4s)#l^8Wvl
`Yu7b>=Wm$J<|3TU=-iM8udZmY_7M}|C1GpbNz3RLS!p*?sRmfgb2!DP|l-Pa#8eECv4*+^jy=8j5tbCrav#Iwf05xiyZSVJsQ`N{v%{88<V+?ndV&Ss#kyo
m0+tZsPboc&8pA!oojCAl)i?&ynql)(Z1PoJuRW(7KW50RgB8|sjoh7H1b#vrO_mT#V*}V^UH7q?O0)<^{7x`^X{C~&Sa^nvezN7bC=)(l@lXdH!R%mND55+
ix9O@Y1u6QOFY^~xlm^<A}y3}wc`j%+KFaU*4_0sA_(Y*Szhg};04F7`jxb`K|krPjq%Fz!N#@D576li-Cc^q?JeRA<PrTeIe>~8_Dp7^4X!}5;>vA{6jn(y
_PuL<4Lf$mrEzv6KL1aaOvKVoExkd&dzuNoc+e4>t`lz~EX}o$Nl+c<wv*e`T=H%3ySEooD>hv;F}z$7?_S3OX0N}m(TvW-{(0(@-#tU>W81H{gv$7yhpU6T
TDZOoWO3DnSY6^5Q320}YsrR@!a~lQf2Zd2MeG^qqurU+;^ptZWXQ}sU08l271FlTnaH8z+GJeIrnywbjq|9^A0qdRP=BQO4X<0J6^T;f+15l6KgJ}xc<^J#
`crz3a;fH@;o^b-QCnMD?Gsn1#7%=WkI9*p$SVu7Zd?tve2L0}63?BF*fi3!;eOBHKMr1<m~4Kle;Hm!+`FK}57V7b!`Ckldh3Y!)|ZzG#(hb8&(Sz`J|!U-
NCK6KYGbE%N##z*)9@ym^6!m4WW(;7J_=^q^Rs9xxU}jsjgQ{%N8rTB!GX^kz@1OXGyKXH*qqlGaU@G;>bCFqj=|xD|LK)qFr`Pz&ZEAny=nBXuiGK?(ixk$
9<Sk5zx%_A(Z_2s;%4`J9=m*Q+Rk1qs^sn>34a|*#KPTOhx9Gi-0joB-DLGUl1xq8w`e;}HPH7R)d#fHf$_5j!FVaXxhj9CzXbgu4fb*M2vY4JyxrEZOLP=D
PG^66r|I<hXwC^#xHcW5jn;Q}GRHGnX6P|-v@cz7Vn`W-Fb;e()WKqot#z8O+vXTOY#3AzW79R2v=NV4hPq26$?8IU5_{h^G7=rY+HTiV#0}9es6j`)+<F=D
NZQ^>st4|~?Bzt`ksCLQAi~bliNon}6v&}tro4?0A}@!tFaS*dUpOX@T~_9Eb6OShZ(BWJ9P=Jmpto&|SBT-%0$X-I12u<8i=$cMdIY9tWT+(vaX}{O2k0M-
=vLoR?Z5C&81D7l@;1O%j7;O3`N|7$`w`BYSd%ITn;72>8a|YpxOL_DD~ezXP@hR&Dn7LUTk%l>-jB|)fEI0NCESLK6Zo6pdv$9io>RtqHPLtz14JN;7iFTh
WFx^3GV>~I^l8QH2CLD*-(zdj`}NLc*Y;($G5D|VB`0yq)CJ~mfUGOom5XuOSO<eM{}AuIpdMy@J?g?yejlJmFL3dAXko)QIaac3!4KEyw$PoT=i%FF$^1O7
Kx!bwF;_Lb;ydpc(SO5s(t=0MhkJvh=T0;{QefF+QZL@+1`@ZEtx+eTiS?!<Kq(Wn1R7|Ju-``eIIuS~MWwK7(Mz(+Gnwqzt!Uda%{Twh0#%6gCyW8Hg@tMk
97@n|luJP$eJv#nu?aN<jB+NmOP|Havbo^0jrJ*irG+KYg>q{L-`Sm>IRZN4s~^bpBn*!Wn~a~#w(@%f?SuB<oH|o~d3zugvkO%c8aJ<0xW@;}n^@V^Rpr{i
#z@y_I>JeK=u0*;a1}4PBfw4(JwD;MpjfC;V@un+ft!aq5S?LuA)5#rf2BLI9I0@KuF1kJKJEDyk9JfHU?$M=t$O=e3g_o9p1Jk(m)31bH%jITb5E(4(v!@1
?2V~?Xn^&F=}bN{3UHQODaff+sP}KSp7Kv&)w@CR;fybFq~&>7hzjb*>Ti~eQcb}FVQAW6>3eiTAnL_N(<+`q$MxQO=H$)zV8**tod}m)?=qMime^SoVLAJL
&H=|4J1nrGhTfa#P#Y?i>vfq<`W=+({GE)qd2S^O<?bS9k~cAPg@j%1z9B~d(pt~NLjdnjH-I<wwTm`SS+en6{-2igz;QA}&g^Y-u9A`pRYz34Ept`)rq7sJ
bs}N4>|rx(f=D{SoQv#4?UekCW1c>!#@_$fba-&6+``XQe0SV<PA{0?ZQYCSo`3?U^r)Bp)Ck6{LsEs7mtnQMdv`F_i3sIY{GD<00v(ePCIsnG5b_+QT{Z*@
<d}9;_e4b$Wk&`maW5_scEZ*Bdd+;6{OM?sOyML(cO%19Oyss|&Zn?e9I+fk1s|V-YN299?bs~HTTXoEj(8a?sX+9ppUE%Go+B7*qH~O#?g$(H)`alJ*_mmq
(KWm&2L%Ou+ZRvrD*>&QmzqQ?QMDx$n=3APL0*3sgRK-K9N6#zCHO1_^GZ{hAA=aG*W1;!(Rw_&lf-I78+XJw;&CCmk8SBi7l9a=xU5T2c`X_r&QjxDMm4pk
1&lq3NPxc=s#~h!<y0>XVu1me{F#UHz_RpH6~w+O-N5`5jWrJq&RMWZ2<?m=iVu{@tklIWe(?Fn8qh~M2MZ8;%$&<oTr%A0Ohwxmt0=9MhM+e?e>-th><U2t
g3U1RLQ~9fuCXuck{q6xcD0yaxmV>rKnR0t>oTAs=k^@U4=!gj9<+l$ahbfh?H{^jKO26i$@>x*P%vD2nlE0gMTu}OrKDH!r1=eVYyiSBm*BV>N{dSjq-(+X
5N@A4pKmwf9x!?ARq?XE_wgp{dAYc4l)y`s+=;sfXoxt)tQYKZUF|T{7%(|N5=sOuX0S#(vUT~^GhymD^LMy;L@2G;t<uo_pb&6N30FxJoz#l*n@mX4u_QBO
awTOYfi?)4Cq-l$<s^(>wlDz!+OXG3+(1vVd0|9-k>HWbIa5@}V?TAe{-fs$(N4s=zUM<fN8{YFIZ!)8xB4@z;-)2jq=}i{Mmt>DXjuv*(lj7ac*nT?&R<bX
Te33;C^y>CmeiZrtzROfy|Zk7(jBN0sZY0LkZiQw)pqvt*n+){*{rlS4Rc&Y`l!e`ig}zAp63*mBF-l1(3^VZ$7!Tfe8)TRy1$ZQ`;v@A(AIY2T&3+}a&xJ(
3w{B5cJ6)JO`@MegVV!}!}&3HuFN|%(wk6p&--&r`Cn)1qo^uAxPVB2!TTapW;+ncZBMtVG5Lxueq2IlqTZUUHMe2}AfgR<l?Ni>b{Et9idg&Ag|D7!j_e?g
?9_cpIu{5Uhha;l(r5;4=|&6mX$ns|s`Rysx9%HUnibv=PK>?6I^Snad~6^0IL78w*L<jOi1sXne8tT1=|P59a6Kx@w1)2-y`ll6gjy_ywYlkfN0`I5@i;NV
e0xZbD`(>i9SJ2c_^1;gQS5F=F_dr$${!mN2ld((fV_rzkXYoK^%(`>F}iJJ!LoQlL$g%|4lE=rtHyni0AwvyK3Rqgo`1ycED690GsCW){?_xt5sa)DfU$*@
sAn&whwfsj;KxOlcaN`hNXeqr(-Q8a+Ea_jUqHptA|a8RN^5&@Wj^QDg~0Tc*wIxEbiiRqPzt6G-wKVkVcJ`<XvLrwbvK*$&-pC#DC<krJ`n3VvQLX|c)N*1
O3peFCeppVgGQ=lg~Et3qJc!F_{!E&l(OUc;jG!NQ{}wl2aX+_@W$7!B4cE*EX7}my87wTc;3OFvFQiakxx-l0d#_`_|ULr=#gD*JcC(RcXB~9LIH~I_>h>V
{XM5)H#5$IQD_8%F**h&LL++Y^P9|nt6(XZ?36;j2aNtPjIJdpo!Aak5JL)7Bx$1&s2rej*GU$=51C;0rdX}jhJE8}ihnNk>OYnvL5b$|dhXbm1VXd~we%y?
>YpRy8_h62(Ut+paG7HdLPvC<VA=mI5sbAY5(rWiv6>2M?ltqB1Y!M)k4c<Vb0tD1)GGLD13W(1z5j&Mh>O*v$bj#2e}w#ODkm<ul^cvm#YHNHT>MtZkxulk
^uL?oC-|R}k)HA@xeq}altVot*imoniN!Iwg*}fS?0>;vSmoP=4}{JW3D`G)ndz4>^z4!Bt}u1;UwmT*#9ou*{1aL^ucHsQAqSEqV#Yp76L#Qk6LES2SSDPW
+xH14j{0%ncD%Ja-IY&3##(xIQLM5d34Hpo+UT)YBaKW}=hfO`c~zJh>^@iMW)<`!CF3sAH>zE-)EZh&?G<uBd4%*1>ej^~(G}g3y+QOP=}%IpmE!cgkeCZ&
+d-iivUy!^^t42GjshrpYzG`Qd&2%Ly^hk+zCH2Wc!)}yOgBn@O?oiMNULYffAv;`_@Fz_lHrcR5+zWZKO6r#vwCB18Ac8rHFn%v-~>I5|I6CNTynL@^;F8#
CSR3OAsGX^RSdfWf5^2ve;gF{>_x@1?S+ocwjFC#M~zh~1<oQ8oIEgjw<f2wpi+7+*&ZkXy;7I$La>0yZNjknE6%wy{FksEl6(g?Cg;BCNRbd+SHWQ?)5O=b
2n1vgOw!9H70@EWVYvbg`!7Gx>Yp}KOXiFXV2oF$!`s`LJGr(>5z1PQF+b^2C>N3Z9L`CtDdCI(-pc%#%ts{<$o=>&dSc*wsV4)65;8)n(iEpFY6RYt!6|ni
1D;Xp&R4JJ4rLXwR&>|shYfX2B5ke8LJKS4PSUV)5d!T+i?3A%=YLOAN{EzhR^)=Q*Z4nXV+&6z?<l%IPI4e7M?y9TYHVde6r}tws|f(#6XYO_+(-1hk_H;g
7cHDGMaMdN$@>a_N){PEdJ|nSunM~A=4wtBe8>dd9YD`2%@T`XP(x^;hX&Oxur;sK*dPqS!I!LV!W7)mRQ~RB%ax6Q^+ueTbz*a?A0N6D2y_S~3u86JCzz(A
U}zVe>Ee-(Mw|ODmA*m+-iwe-V;x*KzBHu!E4Vh!G#F$|g!aqj4BWV7PIV(%yV7JO_KE!<Q~|DE#KOoV9KZ$<BCvk6YEtL`-OOg0S5cE$nRyFMY=;978>G$>
W>_mF5q>mXDU+56=3Zm-PCN>TRS<nk9(g;=eKBOWy>j73c;}&dvsrDXl>K!Z*`0oDlGk3}i5NK~h}nPm;+EdVjU`ky<lWa=W^|BbI}!7xX2L#aVh>(|LB$ga
kI`)cGiuY1Sa^X|uZrj*=LKT(jOX<~&~qS)o_XU2%wdAORg2&<n{0kjf6@sSb55lTNsQ~UtC!mqdp_97w__6RAUvx;dNk4Tz(DB6^4!~f0Y*vT(NBmwO!V`?
e~gb&g2r2|7;=!c7RTYcXGh)Vn)y~~!>4Y+)}cO~UrP!9O2Hs-Ho9s&&z(-8vPe_kPOhp4z^|sNl(ZrJ9NcgNfv7TY9Creo6Vm7PzM@fF_n96QP7iKkSSj>$
M<9Zha5z_?V?3`(9D6WT>kwJ|^QYL)0)eFbLP9-i(;+DBO>OAY&PB&dT9NK|%xxi$IV_bL@`94y;V%!hFskZ}$@cr<m@khsd^vs`<M}N63(3lIvwuU_i02>|
#aee$;is5CNw$x7LVST}a7PU2%AgPgMd=2PM0gk@bo}D7wY^7UbZ~0|+MY7<rv+$RSgl{9kWoJ%OtBq_+z2+)m%?BXsT+hNpLvCpZ+G=BVXvZh6pJCj@NAl#
bR>6c6B0R9p(j%ya4>y7U;a_GTWAvO&e^qNYI{JXtphD%FdS6<wKF(_)qPjQqXLgPi3<tdUQwVIKK;liekuZKl549zjuI>wLFT?uqC+y75kh6YVa&}5=lkqu
>y$4Ll`*o0+B%D}-JE|7!WuULlyi83-DLVol2Kkw$&Ju$f(RYU6E>2<@VRJu93|H-pw=6B)DKo+3VsA}Onz~$5p|laAKYcYCK^k6BOe5ps>|K9r+v_usWgkk
h4ND%{+SnoCeI2bj;l2IUj}IMJDmHpUBaq8iITb(BEb51jFEdYOX&#@Kc%-YNjt{@b-6v@j8&l5Vk~K<6YwGR(%ErB1U{tK=D#Oo)eV2OJ|6e~P)7O8<`P=|
x{xn3bthjKtYuafDReWwfl6}8eo2R59<_`DlEHDR{mWYc@Zla7;-FrkEkyUnR*^v-8V0$dgSX4{DY<&S5zf~9?6%vz`Ix|z8J)mI9D6S(^JbICf7j34^6;Z4
YSLuU2Q%k>g%U>l^0y>vcXEP!K^1o{dD>+QiE2OJ01c52E=_186ew@PBkH<R`tsx~{WGe)vaN38w5)$g9rH7)(pTD3IBr$Ym+Qzkp(Kp%Ff4oQI&YteK%e$1
`rk)rJp#UPDV!z5pTiQNq}AAdF?O^kpy$C-d-Hz%>GA8oaM~NcNisO8J$Jcns%l3hT<Iqo^Me?>t=nySIsL=tO<7FAk8CD>fozIKUByA=HabY75j^QIin5+|
{Yv&`L?N{}jXKT)*SJ0}q+Shb^Lj<)pOAlB07`BfnxOG+NYEc%aPIDei2j2PrlZqxSBE&68HcYJ^+E*k32Yb*wA80QZ<VRph+l?L?I=3MN|S<MOT0)-{s<gj
M=LGI2T(PT{dqa0We}PPWlYqW=FcmTe#`KTZO_^0SEcodx7c;qBM$M}+M&S7qq?pKa@ja6AvPXblAG;UVse*Y_xu)VAh4!fUW>>e@6>S$-5t5bX>*6pMeQvM
5I#dvI!`k8W_*hi@yI2o>i&q#6zPK3kX#^1E5CpP4$jT~du9vL1KuFw_&Dq*Hg+=att9(ugb@`x^+-mxEat)n)K}PRAFukajUSaaROlHIr{hO~3j$mvLs*!M
m*axABVc@4Z<xpV@LzH!-%F;L5O1kqo~PSn<@FAM*%DA$P_T(9zZj@U=DZ^wk$HS!jU!(H=vsOGtJHT%o*N6X(xvy2Z<cJHG6nslJx4NYO>*K^{uCY4#mu|4
X?K{bdIUHg#~40vFXmhO7k2<ckIzE4DHmll%PVm2f-o^t7rCC|$>i}VNKDNCPodHTSKB7>MV;Y|pvhYp;P<k%#H{0~7z#$p!Fyu{8AoE;rf^S^lUk^yqb9?C
4j<uB>hhw;f1Uri1dDHD3+9@**kdfmJ2V{EGOr!GhpX1OY#dfy6rnWYiPdI~hj;t&QV0egB3CVCo2naXF}LjrW}?2Y3A~f+PCAU5!$X4!9KBEW>?jZ6_b}1d
7am?Gu(UKv)e2SQ^zBWvl>De7RCX6zkIy76w^&v;a{S%YeKunA?^Re8ua-6@%O|cK%GqpVKG2kU?^e@{l%F^DjH7+Ty{2=v)C$I*2KY>e?AdYUM}RG_90ofM
7fYZ@(7|iR;w;@ZzR&|MSv0xRdX=EzolYlCCe6))nWk*#$>r@iYwR`6z9zw!g;`v0SW;Gq$2AR&B_J2Y+Y!(`T!Y=7s8(3G!*)7HXpqMDn1_b>IpSaoELv|D
@k(ZTqTm;EgNLTCU&#gyF9XF{{S?l29w4*v%=p9M#{qLA?pQv<KKwhR=MW7k#%^>PXYaM*MKnlDggz5L*)}L6&P4iq-!ipda{Wp#n7gEpJEK^YxYY?mg{0;m
-gWi$;*N<yA{-uD>#|kh@wg_cjExdoth3@1_73QX&pNqY&(5nRv5u5{Wr;;@UM;^am%}P*@woe<t7u@IDd-tX8ysC~+rICS4<4GeA{z!b%jAMnxRgSN`J_OW
jer%U1FAUfOaEfK*;7G)iv%zW)h^GO5LIka2)J9WXuo<+N$!oEw@xwmc^yGBWq;vFlrItw?7HyknV?H#|7Nxf{y@1*jNG`rE~mEi43T+NWZ@q)gkA;0Sykyy
!~QR^o<TqzVwlCkL<OYffQ-v9TY{$LuUJsd{2G?Mk$ai*=kcFBgqR%53E1%4)b^%&Ef||zkbH3mA8JXh=1MTa;}!GJ2u;rlFXM*3j9xz0o`|@!w=~X@&PY2o
Z!h1v`+iA;6QUINU9dN|mNMYfGxJRlt0tj!2tV#3>#KHv$-zDN5bAHN0(iG?Ezki3GX_Rckniv^UtnHgQ%#m9hgj>D8AkD?$EQB%pmu4~4ja9aU+#4ekqw$-
9sIFnf@{e&dF=|-jHe&Y#%!7?$5YgplxEXV)&z_hCRUx{6AMLbGVe#j_C`q3zMLyF0^4)Fl$N)Ri3w3A&qVl*`M9?B;)uP=gcy%qQC{eZ_2^MDdd;5zGu+pO
m(F3xYcZ8uss(H#p(Cz?V^eUFA=-i-`~)N_yH?VPnY+|<sT8nKfP-fDh90^f^9)^Yh*$iTXRY^|w#3>4byG~Y@mIep7k<lquWDemLH6k%yV@DfL7ylz64vSf
80&ajrTuko>dWHtac3Icc{wY~2&B-vu(DqVm&KgTvsMW%MO``cuv61k(mi9n)^z$SzTTOaE)F>7!%Y;1@jk{__)OvcEN}Tc3K0~vEjbyn2O#wzYVOttdVs4h
9RSLePZw$pf5Os?2{msD@!Ro`(uFeAnL$0J4RRVp2tjWZPNHb!+8U`aOw9r<^pD4cOI0YRM5X*EOR}nB)eEU9?3@Z<$nT>*T1ua|VK5J6Rogu@U!6zWpPF&B
&W&jU-csR}Brf2zHr1!bJySuB4y<dolk%io%Ps)&c0V}6oel`P^300+<Ld=J?Rl;pj147T_+hmj=i)p))LfxuAgA%Y7jfQP*`Ne_l%C)}2R7d+KH^cuU;V}A
hG~^Kw<-^kkHA8S<$sacm;qt)A~_~wh^$$gll~td1;2Z+GXd~y&3E3KE|2~?2ik5%mP&1qwGn4N-Ge#ZpjjS<y~|EJ;J){<!52CY2wk97L`a&)mFRTx_NwFA
0LlXbi*cK()zeC@CW^HgSXCk5nO4GMs2MT~WWZICAV$lc2<fe0CpnYyDOqz50ZVSak1<~yBDAxdKp@o&5FjP$5dw!HLd52_pS=g#v>VDRI?2g{C{k-eKnz|q
nxl5GCxMRA$S%GUp!Aio=?iIErU{;~M5}@~5T(7=^(M5}mDfz|dOM`*3$B;x<PXBSDZ1hImRcEw^ZE;z$0a&Oss{Z29&VnxQi;s$Wi^vb;7K!vxFHA#XpInT
X?IF4lyE`CTqr^ZgkNFP9s9ZgP8j8m<*3}r0Mz<cYdiQy{l-nL=rA%eri2n)!6#cu<MCtOc|C<?F(Z@%0FF9<PqL+0P(~9v04;g|Fh9g>5*-yiVeFx>q)WJ4
Xc0MO&>IRDj{)8wgPh@d|JOqP&o;`}OAkn_Ok2L~4|rQrvMga@oxFe%Ov9W4q3*;TP6SR)iT#*$i_&7G<_tZ8*0N+k`%pA<A+e7NM;~o+c&2XB+$J!a2<O6*
&@qrJ2`vCfy1+Xv-DBN-BB`Lyf9MQLK18Zm(XR=99&G0Ipx-)>9<2M-rUJRnCTq7Yd}n1pv_CnmHvIZMaGvs0Mjoj3I5kho@T!8R!=^DYBK6KZY8l)g0x)>{
Xi_syr}VzF&J}|i7K-aZa)zj-U8U4ZyXIME#f<DiMXIOX1BP@N3pD5s(0NIlyO0|XeqVrESw*&9?f`Zg=_UKE4$56st7by^l=pfgN(YPCc_ynDR7EkJ8RY<s
cl$9?u1tExOlSTDFBm*JJqb7%!8yN1g)H3>th|@VW_|G7R^6dhCyL#b^zue22@0NK{v;DxMYo2>P?IQ8yN(IcJ1I<+6&NyJS)F#P02Cjw9zanpXSR2|>p5D7
b9#rE)(hZaYHxYVY|oyvUKp!(dfF7cZSk%CVm!U_N838uFkIsKrX{1<D_pFP8U3{alnYhzp}(dCT9<<P7?_qyU3fvmJ4KuF2i0Q#8Qu157V3@;B5K^Z7EY;U
@t=xE6M&kf$=&0S@~LzJK7Na5D1u9&$svdjD$JBTc<NlfFA6(eF4vq#%#`1_sg6V^U7Rt68=3jX6P}Z^kqnYrmu?URQIBfX3#gC~yF=;2^|flodyp25QHJyy
tn7;WsEv~mS&TD9Ba2XF^`YMwkQlR`W919ANe;cWPGNP!ExomMpg^z;nT)&l6X7)bP{&T;LGxp4mBZRkE9lw5*5t@e>Z2iRWb%{J$R0F5ai4*dI!*iWg>vO`
3o=<sk87cpKMO#WvGo5#Nc51r6t2yI75<yGe>}v!sJ41t3QqiZWrjfd4dhX8D1%bnFIHf|NM|L7@yG}N?uv$t7@YhydJVowQDT0{5|u~?*6<AtGLfz_BKE1B
Ga|d~g2uQetxA+FgD=jNKn)#Y9*73S!7BIr8W$UsvK06fw&!1pEBH_BGWJtp9IqXqMF$W|;~drgfCr9{KwJwz!@t3_K$UpBKphu3NhG&{#=kpgSl))!;Y8ZI
nR1;@9869+V3f%9>xY0CDN|<NbFTo`Vfwt$es0AxLpha6%!Fs>zwPh3GpCW(x}M5~H2U!S&5ieFD46!I;5XgnmL6`}-r)My@P-d!MvwIFS+%3KZ=~4e*>!!A
fb()+V7MMuZ{h?)K|uf)a#q)rW#C^1HE@$9hE<4SoaZ7)Iw(o^t+ZBbs*B|N30cp|UvNf!Hw2a5cH<UMC$}nlzki`RTxl#B(nHxOd*91J62~5cQwUSsr9n{%
#Pq<^nI852d1xH)YYBchVAAc<Nye^_OZJjB#)I&7n&&e#noK+najrtYsj0VQTPu9Oc0O|DhtjV6H1clD01Snn<~=rAB(Y?XQX)c_SY&npXmAy4Pk;!Q+@o$@
XR;_?=|j<*2>Rat;Uj3FduH23oztEW_FmCtO~I+_-LYXRLJm6sN{os7KnDLX=+HOQ9gO!=X9$^O-6STOqo<zuN-jlq4>*%O9xY&z2<%~DrzrzLxk_ky{Df&-
xOLB4nb43)J~6%pkp`z?0iF~>M$E*am^C$J&}#SfJ;F}^e2bo0?6CtvgONOaU1PdZU)P~sX9D$a?$@n60hSv^snF%pJJk{{PQlswAo^U4QDM+>x>d`wBpIQ(
)!7A4^DHQ|gW$tpAv@e}2KMfmU3#QFq1TbJ8k%wtq->IIx0n5wa0CTua8R;5Q=}fSrPboS!^ie3S*F*^@Qr+jD*R~(EKek612Z5L?OFN|%KiIo=Ja74H)O6h
-iWq(3hTnqBbFG_E;x>|0k!#WHKDD|GTeJM*x5h^r`XZ}Oq=#Op>1h-&hZ2;oHM#f&)}N-g5Azt9ut(Z$or@`PH6$Ljd8S)C_^G2R)cCp*m&U5;6}P4?jmb*
EyUkg&St`ipw2mmZ}T{NB24_GfRFVb549Ah_735Mk1Rw9f-PLT7?_wBGr>t$jh+dCUCAjDS=(7f*;KS19D%S=C2RJ0p-G4lS)NNX2z#gFH*#X+8J&lNIZ-8j
(IP-a_T7J)iHqGZGvh&UXh^SPaBsi|KTbST6vf<{?JM=K9Kf-eV{&*f2Iohl2!MlcX0_yCS3ZwwV}&E@6|f+4L--#6tfR3vhg8UWoU8)7xWU~yqTJA#b}k;g
qW;PgOk$Ar*%|&h82D@k4rI=^Kpy<KcpvGs)UfxWwfQ`N97gV=Pv_!aY`}7^$U+{wk*v?%dvI1@Jc~+8l^k=yjjD^8eu~nGr2yP_Q!Jb~XwcWuYt57Hzi$u3
LA|)1s{c8WFdJd=%tBAxIo-rQL{c%eJGD08gyX;sC+v{(0iP<?_3=exk6;lQx2-M?yB_-zOx%Rp?ir8zDOszs6Fv2_@e`dpMKh+C(^581xrQ~%8JJi3slTxL
)bhmYI4P#=XcP6t6U0TgGl=nTF!xNj-{*5it#f65UU`mk8v+FX)ggF)0=|#@rki^4a?X*<?tmjdm<9u5L0%yc-Qq5p{9<9LD0T5Ubm=&T-!8BGJzFh&Gswi~
xC_dM7C6ZSA0>$IV${qYJkWzz=TJ^?7G&(y=J`)y%04y=S{VU|Fr!Pt9jHU@TxMM4#{GD3o!tG#CO+;xom9o=w+J^WUJqoQmCHXgYqwlLdXEP)r4z;zlMLh7
G2nRI;hz8I#F$L-+B9jaP#3gacP8&jWs<#wOhWR*ilQx7yCVke9||PybVmnNUZ3k40MdUV2-Skdh>l*?W52Lc;sfyLr$uz#mbs4t2MPq=Wdb^}WB+($p~kjH
X#xX9qR}+>rtCfxm$Y?|tue!j2mG_YbLMpz0Na<^l4(VS=WVY-c2yjZ2e|W>m{AN4*GFH9d*7x=&w30Xq8k1-ZTS-Jn0&wrN-X?YX%3y2sx>WYr?$}h$DcV4
cuu17Rkz4!UK?~>2lH6&{_|9xvHtmGbY^}CCr_mGLNyWySpo|N>2!O01sRYhGMb$S=zIhm&7ZHJa<*gr);q2{u3%2sTll+vr_+a}xJV8q>aF~4TD`>cKvq8h
!!z(C4S~)3nE`j*9L{f9yUuAgHD|&}c0jiAcc{2=*kDil88T=TsJ`mnJ2c)8iHeiQ5Aug($Nmj$bgVhm)cqLK`h}w#j3l~ht|^0Hcfa;_q7B`cvT8GC`u&|t
&#U>E20y?G=|T&784wnlVLG1$W)S?1OroW(1^aCA)Y=?nUm~V|T5#H+@x~M~y4-$a47u+{f3fjdfJ-qOb@$dy%gBk)JIYY1Wd;|T8e~LRx<yClWB<bK&d{O%
Gx0G1-i!h#V4&s2?^(Yq!b!r||4J9z!po<}-X<*5G+>~UXKIEkAKb+dG6tgcFM^1+IPXd>rN8RGrvVurI@#q9pN*6$jje?@Ji6LQHQVMLhYgI(99zr3KGwx_
edVjP-Zu1LrV);du5O!%lPjtmzH<17?&iE1$p-VT#x{}eAa(uR(qZ5bbgh*CXYiRKjZp3?2OeNM`q9s=Ns2qZx#Qiz?T+5LbwR6h+s<%%Q8hYqHxQ3UBeV(`
dvfh0ifyMxMV!8N!OCn`A+srWccln_@8SzR_M4|ht4a}J_l_=4aa_>=V!`378D805tD7YIT0PNX$(7M`pyo19hO+jdK8mH<g;)kAR}9QsR>`&EcshZom&bm1
gS#WHQJ=zXX>B$-8({evN7}VuHVishm3DU<TXxW1pcs^|0hTJW)6`}$oJ4w9dl2@yWNm+&KB^aAP`5^@MRImhX%1(KY0+KPIivbHX;+bgMplP?Ve;S;8wVq&
N2UWd63M+>X7bX>Y9V6mUU?G{RSrKuO~4Xf+x-{5Ff;q<@HS&wohOrmCrk{rv0%AiQhr?{gdv`-p~nzuwRk-YlU_+vb(f;I?y@T!4&!X>JDt=G8pK1QF&t#w
W-uywS|2##Sz!LR_#7B!jzh_6C8jAY5aO#Cc*0DwMtXTp))(bjK=JAmQs^i1pAQIJwy8AmA2F2R#0<@H`2<c^HcuVH4J&d-a~o$ILeolD#+fHNyY5djHbrM^
DfX<xT&*9GaR@LT)O__l<g7-p19`|I1}z$$@TwCQ=ezwq|D^Jq*-;4yvapSIf*>>!Q2_Mb$HVIJ4YZUP!c!P7I+~zN1SNVpk>jZ>aq?``q>k+#dTXw~)!hZn
_IlUW^o+|^v)J$O=9IY3lt~`wTlE(OzH&nshx_cHGUPxKaQ#SUfBSJk0?y!`J&WQS{oI@eMhH&DBhhxEOgs$~KMzbE1bSMI9ga|sRH`5%><$V|6ta01#v~fz
)TYJp7NV2S02Iy4Nz9zZWOtZOsOU2WMA*{-Qj|Jzy3Kj<8jz|Xm4`516Gj-U=GinO`tlp^+Y_`F#<OnK7oKAW{Jza(r@C1VmJMjhpYFKlii`@U*RFS=KHjQz
0Z{;KlP7+^1>FnbG&g>WfR`T-ptIU&8_^0gMGP>pzrA~hzC1Hj7M{_XxLX~#l<UC?ea+WsG-<Ibg*RjXhf7p?s=^uNDV{44oQJViA7Cwx@7H%idGLkIOC8^S
Ry~y)uyVNDH{GeG>@%++YGWKApYv4-_C)G~n&TLlRJ5k4*uQ;Ji(V&jvZW!*EqK;dv_%r_hxf^51l?O1ddx<V%{;{uP&uVAxCQPoSaQ33ZMBn)?1CpS3pg~d
W=^imXGKI#Q<t*fz6Z&{U66-Dk#I9*a}QR{OODsBi;29d$fsp#Z*~7)SRH9{5V<QP1dEr5h-}9FA6omY+-~YQQ=K#?NIyyF%+{k0d~w%~z5&)F!Ytn``}!~A
GRpRrFOh}o8CVEH#a|)Y9hB53!)I{-&^>9Oy-!jGWN?f=TV%MY+6CZ{yZox&LK!eC2a<6?QUG?~=AD@X7FhQ*nLs1%&Kyfp8y6e#Dzd!^31e^%McrQ&jMk(m
86&N13ziV_LIZ;VL1!09RMY7KQ{I7yQ(uCZLtA~BR9p(qram<neAuo8EBB;C=p9p>xC{A`x6zZy&QKX=PxdK#ii9R2b@~RRlL@}ufaGk>uziofn_z^76R;ep
r$Yjaon6CU%NAW*%<oDO5fe{~v6hAd<<hEeU~)ClK5DxMkWclp!~$t`y1#XRh%{nm^mqaC#SKx2f%u=dy-;gd`a}G?^f1M907|+VFJ$)Qf8+F4;xFW)AvFI`
Mj9glv-2L|5B}@{RkHP|(wDeKv`W_b^SQtqc`6do%_Xil`ODB{7b_j06C9&3f{fFN^0_a%goo(u057Vn9ypph>xc`NI|I!w(_|9Pe8ypGoxj{(hFqYux>OAF
&2D+`hvD+eI5Gd%2)EU6BiRlbM>1PMy^n)F=KZH=t<v$&{;>vIP8ETtDOM(tM?8Lgs5b^KNfMN#UZhCzR|M551WVuz!JgMi;)xy_<k+m<;d~!Pvf2ZLhwQd<
#tFwMrv>p*?nDXPUvHTi1w4ROKsa&dVKn-}tT7LU#T2&q>XSmfWpsep2K=~YAl(BH&mN(dvc~lvPzgED(AsjxK)LyWCA13sfW<2&RwtCDY;{7A>U>=BYDfW^
3y}Y;mAHf!Y7>#RecOp5sK6uy?;rWV8a;by+5t-iESLT8J`5uALol|w9k6TBjr94%uXC$LHy_7(doJykva3`7Aspn;!%n2C+XY&AJt62yMS8@=P?c7}onw?V
D$wJ-5#u&t>|~Jdi^eTy^0+Ewnz8eCdvZAB%N{4kC2#R`_rbxf{&Ep6(-X`Z!!Hhsq-o4O%alSuV_{1(Rrecg<l!njaEwKV&K~$b&d8_BOj~r4KQR7IaTld)
i>I~Ls_;cihYc&vg9G_di`$F>*;CHr!Z({%j`oL=i}oJsNM^aoAH_S8J7sEDUXbLMMIxWVQ$rJQ44uV){@Fhm{|PC;wDss%=-*2Uzzu9q8P3GnX%RLDl$yNB
6J!Y)D3)535mmIE61Ee`#Z8h2rS~61*s~shp>SMjb|jU+Uj8W2>Hv-~Xk^kF=~Yq%?NzRR*}D$1D;s2nRR2(&M{dE{;mh(-ON!;?^BcEI<Ld+sa8x}q(rA15
nTbVO5>96*KLb!$A%crtJ=cx8@5renu-Rd$0tOjZ<aqPk8b7Y56*hr0Eq0fbvhn78DP)r|o6A}WWH~i6)|H{hw>BN=A3oVq=G!Ce@a-L*XUNC<$)->UcVt}3
x2<e-652?`Pxr1XQBnHbLlJE2-^hX*U#d;gZ4HC)DC@$-h{*p9=O@`S;Ji~yp<RbQ9427lLvVN?$7+PQ<S2x|E-58<uy?*UxHJ@)AL<VKo>nHBC4}!+Mr_DY
(o?dwEIi0><tS9o)<-iQ$vJ_;x(!!(;-+<7K)MB*Q@B;IW2%5^0$FFlFdT%gMr(T4;3e(C@3GKyD^~}VyZ@5!V@@_zaC@#v!|Zq;`;REyqRt!dNrQC0M;yu*
rW0do!J?_J)JHKF{X@q*hghp|e9au2*vzG%0gH5%zv%}?;F5%lT$ja{=cmSwT1TA(*6oOPOIk9O#0(i6o`tk}v&V$IVdCpz6)}i6e;|;^q4Hs-q);A@gg|=L
mz*FjML*m<ZAGb};&l(P;Y_UMTX6*XJB|tDp4>a;;2X{r6KQtqinwZnqx7;3qM-Lhs<dmpb(lQ^2W`^+p|0cLShswEr~b6?WED59eUi(iIJ>p9)%;>=TVAu0
-tfmCYwepVnVuA89Q{jH0UJdx)D5AqY)u05Zj6k2eYDH!WV}ihP<0|uM3{Qjt41|ydhUld$Q1%`3Q=zo-`Y_sq9pXB`WIh=*z+T)Pu|o{G^~Ofn!hb4&agL1
j!oodn&})tAADcZ3IDlWAs_3y@+(wb2;s1LGR=Ln!#n#M2tv}Nym71h*BB;^vGN}|14ka6t+~C@%a9Y`$c+gk(C;yFb@lhO+Taus!pl1~umL3(wDZ*}a&2|e
YNDBNg8g+GcD|>S&sF!W#3#DK)biA~Iz6u_z%rY~UtBN2l-GfMlBXyl^E2$G-s;`_SR@;ME-(sz8Cq>nxUeG61A(32;>q-FI{Z1}gPw03VnLH9k@}SddVSwp
R=I9RaNZy{*vnsyc#w4-p<hKxO?XJ}9cj0xezFT6EU>|#mRu^8&EnO%2DNxNNMIZEFx++j;gG)-DYig-0?xxxHesZ?7E>icL8z9nknN=7&d{e;*c1$E!5p=F
^#FJH0`gzK{VcJFZSX-vFb0fdJu49zZ#O`#8f*9pfA%dz@|?eGnStQn?Gog6BcBlAH_wAr0gfcXj?o{`czBTrD}E*L8pA}EG4@bO!-+%)AScvK(w^}RigQdZ
2v84OBB~MCRA%*m9iC6o10x+9(K2|OQ5lu2$gsG6pYa3p({xKU{&wt9{V&21#KMSgv<Y{Ie6WekZ$YhMCyhQN!Tn)#(kuOoVZTT8kGw{q#ohqFBN{RpR(>$P
$vcM2ht@%-&NzjiNb2~(8flnZ5C(KFp1%?`htMw8n?+5Q&6WMQuU;Q`zVEz(?r1pelKIn_Im(0(X4})tBPSBng(~nsB99!}!x@{A4=dA4lxMbhX8%!b*{#I0
%~JX#d7v@uaeA=*PCC>FFNvLwt;dfGqGRi$3Rj>re-jg@?=gJnGILX>2_FOyDFg-rar&jH5vfru7kX^iuD1w(NCdo*o`5<D)$GKALhYClV)}x#>%&=pS`2U?
CG%Gbm%Ya`GnsU(?ey&}{u?gL5_z-N+e2tZOVT=LXVc9rqI^Jm1U)78I>o~VwmG=GV=)^xR=-wyb-u8$xeU2d`jv)QUK25!gYnD%UG+7;Re-Xzk!pdN0NWxD
T4*Ib9Sd&J_1MQSLkp~1G52LlmK6PaIvPbbF6v1KL$n1!s+CQJ076^VZwSCWX8#K?As%T9=^%SsTPQs@K&DnOG0PVeL$%JiJ%|Soa{OWej3B$%jDigSRiTJZ
pC2*4Y%|cE&8=j=O0tMTy1a;}qNTcRD6_H{Ak;8U{x{jWHiTkhd5Y5isJ+Nocu!49Y4XSB$8`O!GJRJTCv}abcT}FP)SgkO<9)vpP&Yh8H1!%x1HXtVU9)y!
tPT*0ta~Likg}el$*;k^JCB;Z+#jwt`CGE>(gEK_Z8$p+;C!6U$5ep?T&^6+jm*#?i}1{NNnt%kL}%=npl*pw@c3#WuB)Rh1km%5QD1Z!8}Z}_Thea0?}za<
L+0YlGCDgUQks<E1f0B|-;t&hGgZPz6rE35*()bh*joJu4=IEW=O#qqUYcKodI6H&nPZrRyf(l<=P7`O;o2rVHG|+4SlBT;%Rh4`h_thv3%fJ>8MX`99K5D4
qq&Z~@t7A4=!&Uw%zQip#5q6ry1wC_XB*bkr4+eRPV|9n7gDSx@}y8lI*~8=!Q(@tBvschP$)eM;SL9>k+uJql4mPuHYOp;z)+eu;!L&V`zvDb8dopA%(R1&
LgqS<NgAav+2t^17`?xo6E_3x(_?wE!D)R47awX8n@!l%^(wZeOwrL%;dUx1-e|R%L$>K7ZtOs6GqlsBJ^C=qRRKBakqfU4pDr1eZM!$<oYk+YYs^>a%XEbX
{IMC>>gW0l_D|MNc6|~GC8(3my<>}kLK{b^!`BSGxbT^mU=FsonLBp`&CzQ6c>;b#_#SR~6qbWVNh$t;C$~^J07e5YC>OT~>V*<{y?<NAi8O&vsDh)LN!N%A
hOFt+WBXNBkk#wlPra3m_d$Z?pfMG2H%?fu^#N)XR&Ac-i^I6JDLq2xq#1%so(t%`?zJ)4p~1abgJ?LH7Irri#f2<XlvT76<s{`Gw|s4=HhZ$oc~5m<D(4Ny
k2Fxp=i7xhye;g~42}R54n~}g<47!UHcFk{(T4gYC@EfZ3}G=vzbT+iEnVbZl_T`bVUCm9a2{sYom1mtP<BaP$Dx8P-M+#|+IO^%A2N0-e2mq)3)Qh2t%bwq
bw&q6m*R6`ejpej;*<W+A1XCg3NA`p>LHT0JJ@JjU`Lop8;k~Dr(4SAdqU3E1ErhDn!2y{xQ}ZsR5f*Qv6)%Whb3JWwV5R%T6n7PyD!o*bzJeb^a4b>0^;^w
sM5KWa5-g1O6hsN(fKx6zbuz8?^Oe7y47P-H#ODjW`NRIY*S67oRkfq30Q@GCl9Q-Ty%uJL{kk-dt-Wv457=i-p)6iV+#-BfA%DRO^K))kFJJgp%vgC=f0hW
&x|5<(j2Cfl!S`0mk92_GM15DmLI$>5vO4}%=lVidfnIRz6X7)`h?(XBhmp}`2vb>i1gIDZUPr4Gs&$x0?AmZqDkrCf-Q6gM=n8-w9Yi6F}dCvl@s8ZI4v^Q
uc31_yi&MJ($k0;_%eo8F<ivj*VLkA2JW){vhJ0cX_GMkmdyUDrO^Yxcu006SBZufwADsz*{>cCvFbc$XosE$D!uxw=y3-l|7o3WWw<9PATkd#uX_}ddABn*
(o20*^HZtKo704!Y2G~<^nHA?UKBb3ygg%a8<#9myM`BNcp$2e=8{RI7+t@}nrJ1ghg}0wYB0j<{c9srbZswp3t`#5n<Qo1dXX*&l6RPCApsf+n$8=_Jx5G$
+L3#xvS^6QpBLrdBb6`+5_$<ZX&2t_G5%CIs+K1k9nq$+Kyo&eJA2C>>!)a_Fb#lh67xGj<Z4*JVJclEp!r&tT8gpb8B_D`Mg4Zci-N$a>gx;UQ@dGfJYTc>
#I43%7J4I`E-$-tO2R_RFW12bzHBeKW;VZmV$N3(hoEv}^;`_2_u2Zzc1&6Hi4cLMAAc>Hmq_;{buaT0XI(I*zgg@xJ+E+=pd{MTx>Ea4@p!}2y-=cBr6DBO
Uo6S8hWAKr#HjO+b_hi~ZeAqxOPmuGQB!(2(^(zSFq4h7TCXrkjGW~+SMIoaLZ1(ex~hW}5?u-|-S#;hK|H)q{kWxNDia(%wp;6`niVJ3&!HH|@}hYS-TRFN
5&pxWCWavCtSw@5F5V`(e-koMk|FJO1~yzG#k3e2K^KbhpK$Pzy&poR3q^Wb!C;0XgX)@`!gKyh-Z-PZqTz4OX;J`ge>NSMuv2c232sl;jcObnKMA-E9&h>u
Zv}wSYxn_wMn7;N6w8;z9SQK#`j*x_-EtTPL-tP55+wnWRE<2X&^g7=oYE?A?E+Ilw7to>>YKp!KPtZz{KNL&KUbOk)G~dg!Y678)z8`tMr#fX_w`Dff)8xt
Uk~$z!s*5|aOq^wk$*VaC$t(vu5Jn{lbpjjJb*G{k=bczPKS-y_L<_$5a{^roE(CQ_^BAdetmcTWwUbKGz3Z#6{1pR9v+pm`Hux^$O!AN+-zri^AD~f{m3t?
rvF&i%-S49&Y0z`CC8w(e-C+_3}lv4wFHHqc`wP?|Imv&87NMkEw9$-hL(w|I76x6g%(qqhEK6**b{=R%EzVQ1-&+p^)k4+J18#pwnA_JUt8&7*c*UudU+!=
Vo-L^L}%u?&e9PX$@%44u!>KP_DA-5Vdt=Cy5I|UO50865-@mej~>!f6xDOO$Hr*x<#-O+yqRgf_sKpZY0WDCsBBUzh!l{B(HSs}I3D=KbY%Wibno&0WeZER
$O(>y*ZFAR!Z#zRW}0#^4x|fGLN=*Xa+=L&7G_Q3>qO~=aKq$yf!!`Q!Wv(~(QP9S?-|OIFKLWip%;VSXJ(}>2lyHR!}Ya(uzYj~7Jg;>hC^d>Bj;IQOfh95
Y$bQI;esJNm|#1aoke+Q#mr7D@OA+>I33ki<X1s~gqLq9bE5RZm0>GPh8qHc(GjrAwzqwa(z2-OR7%7=D;oFkTu2kOy1upZ_Aw|)xFy(9OHAHYML2AI%ZQUF
v&Fa^AbMWgqvUt%x1DzcHXLXK^><#PgD>v<ja(qf4x4@^9OYv3WGw1tLwKKIz}zwc6{=_QTv}D|)uB11=#bh6eBMDp8rc<=(K+ngVffd@9tGir;^F;W@R#U>
2#O8~ZsxjnTy+fVqb+msH8R1GE5Q7j=)9OGloMZfv^&ZPS>vC!o6Z``&c{k!$ngz8UMbcrU}?>juW(>5rQ8RZD3Lnf*h!V5`8-+PA&Xljl8FZ?&D<C)%H|^U
Jk*Zj`h_;aW;<-$Yt_YJ6GYIy8*ndUJv_kqFjp&I%Cx}h<k;DU17f2hkC~Uxm?nu;0ImO9qy^mrKBmDHYC9N6`60mAnM}M~OQu%WF}Z~bzm_AvQmAP&tJUWV
Qbc4u!|iQ7w)@}!dHz{xDxewhce{sFHX@6k)6pQ=r!MFmx+yVtP;x+&f}LF$v9D&h>WB5kg+<uK!18nWSJ!fUVx&HMB#v=h_E(a98vfc9v585D1xWDb(@Yh|
())VM%CKf@nwVX!V}qG%`D&zB`b~WE``Oj73i1QJiGoc?Y{Jd{m=?iEk9+`3WK;O)IN$g!;L9#M^<uV&2JpWN<1pFKq$NjPWty@BI3O#F?+9K#d9cMJwBIN?
B=Z1b0b~$Bj=ZsGJgn~-ShAy#L6k7+4HJn0WnftZmORrgC{I6n@71+g-vT}Syn#@ux3VlU76K-EVxv&hCR}q6aqBJl@(ia{TE*o5{%Z}AlJ<-8d+qdnLLWW2
f>#N$1;LSRH!>2?PGsZfrGSMDxZ9cYZvv{?PkV1WY09#08o*e3e-5xs?t?$zo_zM+k#{9++kusxG2c-iBHy%XOm!~Yg%t--U@lFr;eh<t@L`x>BlFIjI`dYT
>y?BoSHZx$Vn*#D<8N+jA*Oz?E!2{G;QS@oUlOJz&NUGy8AHu?n~qev89itx_E#WZCitU)07+GY71yp{;!2BLF~`YZ{lVcc!I|=`%)&0#hzwcU5}ixtW%}=E
n|Vh)*NnJHD0&Sx28HTCk-Bjvw*~9Qf&W_TU|BfJt63(S%i+b*;-P-?X3EeZZqY>jy70MwrTD-+Fso75G3I-3h*5(;`i_-$yz?a0EE92i;M<ifF*^rfdPngV
4kESN5i~o!_|h4>d}HN>xKW`U%{s!r0>TDhci&e7nHj{R7)LE|Ymp5H4h-6KQsP`DI{=)Z(#V3n=>0KAijVe3UE-LJ+<B7B{p3P1PPZk?jn#!YbLV-O_vyOU
OVd8!44QSBd|!{6BTUB|MPVYe)`rVwcz=1u>9$e^8jT*J)GD|#Z;$u@H3eA^eF8ocEyu8eq8(LT5gFTVEs_z7&a&87xu)1aoMgPucaJfxjTWWoFmlwYH0hJY
_xTeX*EXM%-Xbqyw&4CoiQ2AIzf(RgqBk?{w!;hO@$F#kCTP%$hdyk2@F|@T<oYzq+n9pF53KY4c|YG9V3weAR4~$&jMx7n8Jcg}Tmmf~9hG+bKY%n6ihXk|
hJawGLt+>M3SdZN7u$)?vXF19di4wY5$$~q@Irn(o*g!>A|)$bh-wCjI9xfb<8^Fd3l-_rHR}d?1NqzDVJ5J#Sp?Xl8z!MySOwOojZw<3L6CTnk<^O6u|uCj
WnA#uI7lUUG3{ir3WwVwu$19^omIQBEzfMnr8)gXcJT1-X9k6cm94l`)mPmC9#7{4qioPLaZ1f_Wijqjszy57FcPzKlR>cB<v3uol3~*4_54c0jcuBN+#7zC
X}F-N&>d`SA=822z4Z#y_2Pw-@_H&kJcbQdV~|j)Y@P49h*i|>bKDd`@+Hl@<yYot&5*>9sowsI5oI*t>e6bpRT|fb0Lm3wViua1-{d8})<DdzVw=Av6%Lwg
h5?%ms9BqU?$g1*KhkZP0mEU5S;gy8#71TNdY!GD);2Flq~yBNWeZRW<a(C#@uiGA;xBuOfT&0+UGdX{a<p+Ao8=5+5C)&n?Dvd7(I^y3jg}^q<TkbGdhfdR
OPPiAxOLoe_8yh<Ee+$(EWlLjgZX7|Gp)$v^%=`d)X(bYH{(#b9p%T7liyn(t9Ka)v9Uz_G+JR!u#Qu%8)!M2LBBeg2sQB4)AmHFN`QF@xG<>!)h!*mu}t0^
ed3Q{pC^@qd;|Xy2323Q=nLLx<S%lL{u*S43vhqqsv_A@@DP%Sk;Q{OsQDUk%Hn0XLPNQaj%0C38|CTDITX2OSWC%aQ)~@lR%0j;Fccatz=p_q#AjOG=^c}M
p!>*56t@JZoFmN%sJ@#NW`IoF!OHCIFX^XbcaY@x_sMN|SkbdgS~0$#0ZZ7}mzDZKds5}HAme8(Jj{>X0Z&?J+?^`&h@DcmZXkt6@}w|3EFv3a943eXSSPs&
3dr-B9jH)%zvPso@W3N4C@|KT;ois4-XOD-w|)f=)g!6!&~(`ECTh|x4B{nZJz8eVC|wX=g~G0To~!39yctVL<=7!j^%#U}wrbk`s;p<@qenUzJ#sT!Be8x@
mQ}7J9xyZXWfioIPNuZSHcPPh0`0uHvAgzX8c9N$xhLc$q6XwLxg{*Ct>Ja=1Lx4n(4~)r(q3D)am<%aMs9eYrxc*!sEkDS-Vslu6q<1hYayvNTD+8Q*^<fU
Q}9^cnHVi4_z;W))*mn;^$xMB7tzS=d3^Y;Bt9C+U7c@Ar;;vW$CbwG{imru+rs(Hk+>aW;wqX2I)}t<0B#`!&aclf5P<L>0&Sa-Ug*WJOwHmB*RR1&PMkT3
+QSEh8&3VV$%z*(Gi_@JtVP{^KfJnVvVLYVjo|}$(k6Aj0(m2>#K3s}7W@cNc%u078Ol<!LBGjIJ57;d0OuyTZYj5~-x*)^e$$}sY%WBa-k!(kDwMK!|4Ai^
AeemV!gn0E@q_}z(LU!<|5f&JzCZQT;LucqP|2draQo;uJ`X(Z5Fjx@6*R0ImnKCu#PR=pIe*NQcZW|;;$)ioCy$aWP^y0WlR;wduqoz_AKhypGQ#F?&Oyr2
{(!8t7XjU`S5h8L1rKg?bD&zJIX_UV#Enr*gm|!=%}y3)QA?zf@<6DM11&?!WWW$Vpx~r;!@$oJrrYwX@CVtU>OQNc)C<_5GKg9fK|&dk6b&Vqou|$ms6Fjh
i2Y>_#<%}Ayc!fwEA5K0_-q`}Z0_mE6dqrFKD7|AZHo!gPQ(3gP))`l02(Y-+ZlxyW(X&^O>{gv4ENH7NJqiFc{7Qk`KTq6-gl=1y5=W)ALl>~#Ih7NLu3Hn
ft(cU*MKV;UKYdot7z`5dF=)nrD4x=b1L0|_X%FzeK_(n%e)u-BfX;2H_Ts(ZJ%nlMGp)2GwtaQ7oNKW?#Lf$$wy39J-BM-6u*4a`1AG}ln5+(NV8hDO%Z;M
;?Gw;N;Lu0@mFFd*)odk73fiAwhgJ-)0s`i6&0X7<FmW2-xVQFfx4>T%bvKLqlbSq1yd^?OI+hvplh+D(MYQ`P^`K`S^k!WO-X=tB%djI|G)?Rx6Jo%$vFlm
^zbZJLMYS$UL*cxhXo2q>v)KJh@N6Alw&+wK>e6IeI?dpOc&wO|FC86U16bS$JdJ&A_thQU#!hG?<xFb#t*rmhm6+n6Fx(x8bwkv3?^65#QJ+Jl6IKd2k`1A
>_0=lBZO+JkyJpBo8r8BJUnz`qGhwGzsFk0A{Y?i0gLoTvX>I&6r?{YVlsqd0Qxzd8qCEJiG|DG%VX=4UVwp1;Sz&lC$&V%0K(1!FCy7IhAWuGD!S&_Hc(<u
9We^{r=Wdl4~9h|j_gWnnA$4VKvg@%nK(rnX&bwrE<KR&d>&@kl+unwPgjxn-|+DLwHfxbz@?hTDY-~J7s|ZQoyz*ec;cwZ!6V}S4#qr#gy}VBL~`;WO(gb~
1=;om+s&=LlTFTP>k0jNVnv3PYF$tuEHL^)Gu0kT->xvmEpp+3`!6oR-m_|0gdRAO#6Z6N4#+dU#_-+2jWIcKVq#OzOcsPrJ;!sHh*g=fh}F6jJMvL9Sxg9N
Un;}TO_KWjj}fabQ44)K;wBuPJ9p)8XV;5bx5aRc>x(6;y4_d%33lm$gs8wMNku;$wcE9m)C~Si7s=5oj`G%><@beDH6Gx`98i5MMV^9XCQklD9{fMyl-rpI
9{Cc5BaO$~f!2Ehdn&T6n}Mpj@o_C091nn32q#8SG6$pM%YX$wrd%G_cT3wXFG>D<B_+Z&<L=pje2T|8*l7D@3DjN|<-Zj!zTPQ$>|2)pHVncv#h?nmONvBp
?H@8?22~#d!*P(dX4)yr79=gg5Dj0z$)MlECM#Zl!%HTUkH?jL+6HIKbF2n->VVp1o2YQXredPT4V8Z1UZL~;p1(J%Kz82GZcB$6Co<ngg=(c#R8OH&ZX_bk
mT4u&D<YS*FdP@})FoG96R}5s#|&k-e^7cq(Fd_Hc$XlJbieKVjpR)E+o3;+I<s~kbT|fJ^Z#O3f&$Bjdn!C1tlYB$vu*9#6h=3d6|3dYJQvwZ21(E;QGy4i
#oh2|pXSFnB>f(C?yG!EH=WHOCPmR?r{#No1g94_v$o%Zi`aL1wZc7EzrF(ja+}s2>tiw1app_ZiJogj*ol0{&pY;rbcI3OgRGv-j3h{jlhx(xI|x=FQ?K-4
l50MW{&2K@bh+n1?b0^33hT@n)u)EtfRFO;Jr1{K&`T6JZ$q?^zN{Ha=@D|hSsP9oF!$E^N_P^It&8CKZGmT7ekO?qwrIrARp6niOB6KkcowafrLgv|A})>P
m5ADbvbe0i%rIg#Kg8nh;~J`hn|V3VrJi6nnt05a1X5q}3tujo_uYWeo;l~@;}0xB68dU6Rb=hw!b(mmZ=e`GJA?~Tx_zw_6bex#-@+FOw{%^MC>kNW30`u#
aWXW-FiA;{(T@Aa++Z0PbPX#R$jiw&r?Qr|<?}`61OaOX{8meYkda6L*a83^kfDAQi8f2w39b+<Px0HrL!v@8s0&T~H~n}(C*cNgg<)z(;lXP)$;J`b@Z&e$
eb(uUdkhwZ5aFeW1s}=o)`NLZl)wX&A(ajrG$_j?uhmg+Kzz*TE_yeaeIW7}<s%89v%3G7byreWg_rzj^o`I2*7Ik}4ZD9{exRRb!<XqOM}cozUyNpyGdae>
R1Ual=oM8$dWT)W$$;^Fm%zcA0<P7Q$nS}7bZvd{$A^mM?uZ4HT)mVPYb%$JaY)>-#vOgbZ<sNa4N5rh2sjp~bT}{I5Y~nx=7b8NJc&IQndGq?KU=sSTZwy^
{tAh%3?V#`#sJ>ud!YR+S-LD#a;bI!y<J<TLgT*qa2(Uo`Q*Nbxue4J7_WrbY32#eEWcI>ey)EoIre%2oTbE$=d&3e)HaEE5_t-7@K-H@0*eI03{-zEEb8-4
f;5*yy%dGQuVbm=&nHCJN;q&9%Q@4?c|XW=T3Vz5)OY^Cyt;Y|MnC_l-V;U`EVa)64B?Gg;#NLQ!&`5hHAsj&L(K355XvbSFw=U_rgUm4k%G)Rgb5J`@4v$J
U%Y-;oCKeM%Izzu(V}#b$P?BH*miN5m9iH7hTQ7d=q_J4i9YO>?_!)@y$(g$hcSHyDbe=w2apq9AeXeMoKKoTGS<}AsF*|}iH4%|Vt_3_{P9^s6Zh8;>!EeB
b)@yy_<msHur+91Fhdt2lnrzkwz+k&a4D)dRq)W%9tUw|;p8=47b3m%zeR9DOi<4rvL#g)`}b5q2z#5$Nv8q%-%&W`vQd}^TN`(ce1N)us*x56yG3D_8hKlx
ETUE?7qA`BrRhcCJ^k5lzfqqKXc>><zfKbDYzM~UnYid+mLuegfKtsrF12@&+)TQ!{}m6sSB9#$j{LVEG~=@9PbX^I#G{Z;GDZ*00xvo8FRNnlT8WDii0>s3
M#PPY{pj%WAX${f%9H@s(0~xO1#~e}6n12ATUV9jrb*Qw*x1NoQ`Dr~{c|7D+EHUwPXIvm{QTmXA$6P+vyB<ptlnkcQSHRiB+rGS$0u$!g2b6w;d@oE-d1Dy
UU9`Y_aNd=Et^^m4pNqI?DBDBRtqd>#<kNWQnUfd?4GqUzqW3HI9{S*H>Qbu=tUDYwEl{wva-aBoQs{&Sd5FG$m7+iBZKLW1OhaF0l1DXeSb!t@nX((_%~LA
?%oi5jgT3CsnZ@<u~sP7Ehu_CV6aUHFz`*Ury`j6UAW+Ae}}%i%X?aX6=TTlL#EIS;YWMREe+sp*Nq+@Y|1~`(6CpmYi{XUXW@?@>0&eun3U*x++2EvC8@Ot
(C_j0_KCr)k&Yf;Ox_HbD=!a|;*#EWF>@P6a%(dy&s(8?5=kr9tHGSO6!5H8D`yg>JxMy$SdyXoIbGWA^-3IO0*VaZdINzO3#t6xl)0MqXDRBa_nU#;7Il=k
7Dz3ACc8pX1w6Ivx?kdH{7FXfQdUzb|9kX_l6eGv4HpfSs8rM%ZQhAZ!}}v=gzQ9FoLroS=|3B&l|C8OXI7Lwbo^bCNmar@-#Lf6zQYRINq0+_f3?o9j!*_6
@xsr7RFY;jC=06pSH*Sl6Fxt9aRXlNe}j!nAG$@b4$j;2H$q|@repwqiuhocmHHEIjEa-g1JvrToj`m07$p&~fZF4}X!{>~I3Et}32rNpGH?*QiA~k;#mw@Y
uZv?@S<5+5nohe*fgZ|FuumrYE>wDTLu<ONhDOGZVO@rj*;fDNh}c#N6(9ws{Eo9yMudRmFN)V3r^l^gA|+$Hl_z!osDE^*z;g$oo(HX?Ry4KieZR|Y#9*)P
9bFsdpC1q0IItj57+7`I>Y~xEDXgnv`@B-velgOR1u32Ro^Z%oE$%0-_d7TO3+s<k1WJ2AY-qGZhAtJ8J(S@^&{XJNZ7yPF;ZInwH^fH_<Oauttb7QU*ZB@a
BO&3&;-2`-7NRc<bs?f&wDG}h<s66wbJpGRktvSdk~R{awC#(7<b8SF@u|}O;^1*E3M(=G{n<O?&&_rUnGE3F*7=MTpa{ed`z2b!+n=5)?|k?gRJ=B9&I`lF
pQl4)U_cFC6jlOp*5LemQBhP@aRB|btqTzwRjq*~H5%4tSfr=sES6(<(xnC}o~e|5R<Sezi>`1Hf)|?y2{<`EtHc_;@*(^!I`br{JolHfQm}`f!X|Wa#!Cy*
oMt5=5b<JN(Pev$Xu*M=M^mli`s4Tsy7mzhf_-O@tHZ4141z+YU&8jLX5K*Y!h$jQmqFSf?q&Jn>^a$V#7WnG#c8Z1?MI+SW9t6}CeF0jyssXl2<Pl7`kkqA
IWK?GslFFPqQtoIv1$~tj$v@rCO6&P4EYvXP-7z%(ztCDrX%k9r<-R}wFZq(LG4V*X1@ISiUT$^54^7~0%^xUR+G<;GF2nSw0r*^sDABVX$C?627(+1zPfMD
=Rt&ul&dk?P3?^0h{S3bIC4_BGcuIL$ANkD0JXn~7aJ;6C=(JQ`eCloV0;@W@H;e3aDMp_+*fRq**<UwDyR1eOp5;tXcjmzYFU1R>NO|gH1l%)&@A;e%yac%
t2JqHjy2;=Tq$MXn<;Tvx&H%(`wDHa^t}q-#TY>ZgNLEjG;V+$Me<?H7hts#{Y$gkap0Nu|ET4rB;hr}LuW<vi?`gw;Yoortz2>sBIIaph>6*~A-K;3(s@nm
NiN!ZaE+q%jhL{|rLPQ6+(U4EolKlT#vMj?`5)E%k!p`9_DVO1=zR}<Av+QO=OBy{nEpFAKb;^;EzR-yI+;@8(?bzZ?uEYFTyCw<RJ0B@#0WA+!9#{nii)b!
zs6)Hf6g-RLAQaB0Utea(x>=eSk9CX<Ho;`rov2-+QjzqSQ#W3xK?_{M#cVh9ktWU-H`W1B|~e~gAIl`Zx)=iNzgDgP<#j%jx^glbsMP4*S-ARo<ht21JF(V
d7ssvC~7UoJ8|do`j~d;U+`^lbV_8<k{SmVi*yL?K4q55&u<P+5ko4qd~wvM>8<_j10|OcXZy=1GW9L6b9IEXuerZVmYx`|vkTNiabRVgtpvQY+=7_dhXtSY
r(BoW&Tz6GaZTjoG6juyiVMa)P<gdhOs=TW(7_FmyxMa@t%_Mir<k#Oz$o1h)IT3iNq_UX2f=M`$qqtkPJX6trq^i?(b4}0l@U0pazNAMV$T7PfbAmmDI12x
_cI!G0SnZRdK|y<bfSQ#o8T?8o5(*09>sr6j!ovQ+8?Dta>~Gwtxdn<(?b~aUHz6Tgj6qZh%@SnZ+&2>qstpzqF-<1v^qsnJ|=oe&@0N386(aX(DhpsYPnw3
qe-$vN6&jgni$NPy(*2Tz@@`&e!+pmD2Hne89f0y8exdqsr7Nw|Mp{k@$~qe?)r8pojR9ZwCH8Scoq+1T0-AK{cts-WiE6!)3QuEhW(2C(D1B3rIQP__mAbN
_r6rBGB*%k?;|IArS!3lSkaxHj&s9;NxvEFs^)wlk!Ab2PTboaxZ3Du=!i_+FEn@#McK_Y_J%{ogp<fofp%h35K7JJclTQ(<wYN_{Uw)~M#w*DrIdAAU$%nY
J92PM6U@_TR0HPQU3P^#L}Hs$w4aL30|QXKj{F&c)4u7Mu#R-Wn>Y$#`t5*X<|oe95+V@FC4F4M)n7>1jqv;G7jO={mPqoaXSP!tf69louH0+TaS}W9L>(SJ
HS~N+(gT*ITns5Bo{U!=g{oEHH#;f8PCRVB%MQXlD$6e5R&nwOT#szAAo5$Mqd*k-n1MD$IZswzyk$F<^>mkV>wyWegEWWI91?QJ*KEFnA?;m)dX)3YW&xp}
>&W?yX7g`&7-}s%2am?xYD>k1MTeOPbqg&FZgXykKm?%f!0?%^XQV?p(lo}rvYp&O4A=njDp>JAojL0cJpIB2n4W26X;y&;rc$6VO8J#;@_$#%+sokU^7Mw5
`&D5)85$%0A5ON60a6=-zG!@10U3bsDOj(NOky|T>e4HXP#|n?i5}9>?J(HTLw>?_+BYCqAPGH6>KIK`01&S5S@4d2D!!pUH*AD=RlYxj>5<H3?10^<!}x0Y
uwtsgF~bnFxC<E9h_^W}%VHAZj7zKr3?HVsmexy@L{dVY;Ddsd3hFhWAQrr!BstFkVfrNoSFb^WCpa4fo1Cl<v(&+e{ylev<2p-{<}M<ZS$>Y+=|i>`^ugeN
L_ujfYV6FDZhp$Qq?v2*wk3Qz?*{lxO}DY|zq0l7%+R0yCcY`++p)D%;Y7Nrs-0Ob;bC|-@rlSWQcx;e@)EhAdV607n;PXVV4J=BU5y3tTwaxH6mel7m&Bl(
;~w(q3ED9&0G2yqrCsFch4on*8C(%PlaG_Td~mi&1D*A0w}|dJbD6Wj(E8<QgEfglfX>_xo*TOQMs7{!m#I_Gf#i_!j1<ta&~!VW*_}Us%T!;enH!y}O-LWe
Xk4mkRWAg6gyN$(_yd|O^u(PDpWBy&aO=Fti6bS05f?NACg*79mWF|rVD6Td2%E<DE7Uf+7MkBOU8aRm8c&GD`c6GmBoY4%#&0|j_vVPS+|e~59;Vo~nw%SB
E8}YuiTl@yAkhcvYzSF}A?+L!A2DwxC-hlW0JA!NyxYM>#3byDoR;Sg4)j`b)VUs->W7C&Fxs}KNqXVFcs<f!w=iP(F=__Yf8|DTjx{Zw&^?`dU?&Ye`LH~8
H_K?t-USvSVkdOrfCDJe{co@-K8^3!u|hKs<rqdZ{ac-ynXr6kz@QJQh#cO^+6$DEIfVvR-qd>bl|ihISRoJ&P%)O2HDLK4y)x-Zh{U<x#yb~s=%ZGVuCq|)
nV_ye>0Y+9P}@yx_uiABrq6M2-RuCanN#aeNejKMU7N|!6*l-ZI;V>|r^o+Xo*sB#fDVoLg||9eTmUV#!m#biNxU|M5Rk{7J}5`6z8#W+tIt{gD4CF8Jhcr;
?d`F6tIJ>^1-izyVU6W9!&`L*Rg#Am@Yb$^`fGPN%NZjM@I{>8yLz^zBFgBR(a&9nA-z4RF$@CDgN{mI>W0yDaEaiWAAieFpnU=f^hK}+*5?3<ks1_X`3#uX
NZxg8t*66Gq_@PDMF$_y^P)?1u&w#EE1H?`4lK*Ts-4=e#*tv(bwBh7b<^7(xR>b?V*wo^ns;;~x7Fo}q^jE+;|4x7uP<Wg%;-Y$!4lQkEsDHr^0$~8;_~f!
J0yG&{Az051Yq=jW~V4o0@xKk#c>;m;<qhu#}*O7npzY!%r7$};v`WQL*b$2;GOsFyN3<X7%~b`DAE!=OS;0F#juZ?ktKsKuXXpc$U#(@+&_pi9m}D0Q>)+1
K<Wb7W!Qzb&x=zl>7Uo-7qgwQ%vSpjNj<)rKwSGm7T~4MX3-v;jKpqF&hkJ1pmV7zj@N87Z0;fUAkh%#V$2WQl%p@2d(nxv5RWKi^MChw@{$XNE8?$I@~&QN
zvFjHtUQ1NoQ?nhXGwV=xL2tb00F1H1<#)sa|=B8vBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/treino_gpu.py')

In [ ]:
# deteccao_r5.py (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/deteccao_r5.py', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;HbYhzFhzjh(bq-BQwKy4>SE$m+hy=@aif<Hp(x6uSQedqEo0ysxn_N%4%ns$4kyqM%f{Q8hcU>P?Zi%CUFlOyx%q6
<KX*wM+ojI<~Y~xyJd>0y@RxmSTd(@rP|(<Z&AW9kOu<*`m$&O2jikfTq)qU_QHJ33#ExTKx0|5N}g)zl%2yPeM6db1rHNsI-mrdPrT-;d!y(=YrJcz+y*!w
Go?}tH3*U4f$lf5pLAQ&F~+woghcn(c!afk)qCLDb8w3g5a?TRrFTITtwYOmI7kOm^dyqD1h%wpB}ziG?iX+}!<z|ShRx$F9oG}Yu_U<*D&dLB3Ob{-qPIlX
kjKO*m<)%M4M+?iLhwhA39*{Vhe%n+K%W54*|oj}3Hbl}_5;D7PtG^<?QZ0q8UD5L;6}XUhq0QV1+6y6qpCr@)s}DBpi)F+C*RJo?xK~guj%O6aw{daAw%UF
1cjkme4KiN-~90;+?nuL5dYh~hCj1-h&5^GK6@#>lpOn_&(EF*yky2J7UG8!0ujcBaL4lm@|yCWaSY7?1!83Qsg`QdzH5#&A=qTw4C@!&7ZJ8wi~~-0m9PQ(
VaGx~*j!d<dn{Rby9i|DN|z7w`R<d$u!!y=EZys)#m{}<<olU!m89dixB$UdihD?XOLv`w6I#JYs;=n-`0IG7NiR~4Uxy)oGwkhx=(~@~@^BKf41M>hFYMBY
7#)^x3H>nVNl^4vFffg9WA_0vOCwz+o+|IZL@ckrM&GCW_VoR!b2x~;4fhYpF+~&-t{BS}c{QL?-|yUDZ#^ZS&X$mcI{A&paXwn@jNn5u{0h@7FVd7YK6GtN
u_?u^ikF)v>$&Xp<!+np(m`JQ-xb&uA-VU;;P_~3xUU|x*`BP!^VA3<6^gr7fhilp5112e^=FH#xicXTo@Ak~vX1d7;^3ry=b&`B^a5X@e1M6bo{G-)xE+h$
g_NH5S*ZxzkQOX6ZZ~&v1E*g!DEVuYKCEnP1Y(w2^vB77H!w)ga(fK}?`r?zl@X>5ARl>^@OUs9VqC(XbAW&I;j9yAji0MSE#M*<{Tlm^npSw=8W2WsEvElL
3y{EZj2!R1X+pmJ%xzbsC|8LUtHV!U_S(sd@z-{6Z!Gr!^Macl!GW6a_Hef+#vKU|UHGAxi3*{Edm}Ns%5;M2A=VFxnpQo?-GGZd*=-)I(dqHBPSGzq`FhAx
7_yXe9EAhZ`6JZj(c!rc9~{aH19TLCajB?l#lLnXO@yVK`u0<4Lz_H6GRR^lE4Wnx)kMVilUrE3kaBs=7Mo_v6a{Q&zT|nc<SZ~Usu^d~%S&9azTjrvcX+I;
E8klnQ$sMTNBR8yH>JS6DPc?~y|UzvO?WlAq;`TB2!myy9-W{aZuPUMU2o?`>C_GvAtWa|z6!<J{WU9|(Bzu&9Jpb}p=%ivlf0lsI}`G{dS~xqDYF!(IAH^)
V(>WOa58eB3G!B#HRC5XR~w0~JV;FhP=h=p!_+F|XC5ylV@e{xUx1y%K-OSzSW8$zzKLq$L3Roro~U{Z64BUuAL-dIA9_>g)>zvVTY|NkkUffJ0&BK1R$HFF
hRF?Tq0Ds%n}wDF`7ov2&uOR6uV$>QY&&U$aetI7&4{t-_5xXy0R^7;kt}1BmlmvUd~BBp;YX)PeV-_DWxaHL-?}PB%&kprWE4C7ari>d34cvjxuGoHxhBkd
OwfQ(8r;W(U}st=ZYwL#aU}*!h3ssnB2A7IRsx?lJ0BXLTQ07M58Irjizu;mWCe}h=^1q#Qetma-&smnzCx(D_Mg3<Dsf#eXn1-CeJi}v2}0&@+vO5GjL&RA
+Dik=sV%Cj5ObK<eC8t~{@I1;tYWrY=+{{vyUWZW%Y{n1ZgdPH=HfhpEX5j1KLY}2GaCx4{%QW8y?93GbCBe`^B2G>?_h}hZte=NDI5UT1WL8zr3tyyDfDB5
i*~0uS*)H4{}!?m_De-8VTO}h+X`nl<zknB8->nRa>0m7koOPFmG)iQGOR}K4I+T{0}~LYbTyvKb<nkhh_|DT5x8~L)~6W91xf=5uwP9C@dRiYhgp#$Q+RDq
*#U~<1pgIF#)SnGXu#8Ir0QK1ywfI!o3gm=DQ?fl8&X7cJ~CXuZn}NDg5?`kL2ncA>Ae8j^5z#ykKEs*XJD0&vE0kGa!|Sg1FDczah?EJ;q-XJGv~MrIdYV`
Kv@b`?!m#65!>AUiLbSGtZn}ML|=t2YD&__apg^fy^mco-^Ro*wtQ1>x`W$qxO7-<ig!`Yxy5=WIXbytVblGta?5O&AL41O>JBI<e|Gb>(@)l4c`!IrwfVk>
VoChtD#GNCKu|=3Kl<7NIhpvsFgwmaAv6@(4xD;iC-b@vdY7sxLsagY(G?Z1>b3Ba#ew4bh-a#a>VQw3HleG<F0l1$I??FlabFk23Qv7Edf^&?NZy45kM0&_
anSPMGayik^{nHF+@x0d_Fh+JG?Jon$lMCTLH_B6IY$pC<ZH9h>Lx`K=~Heg%;6RKj(V+l2k-I-^A-FlJV&vijo77+oM|BGeeFiwn!piur8is(*m7G{RexXf
Sz!t{?1B~h+XUvplgE1{Cv6*d%O8Swt_j+3HVDQeh*6p<q?btobbn=fE*l0v)8Y26S8kcT;x27HQJCcQhTs0sCR8;(htO=5^NNJjzr(2<0oa|C{SQNPAV|qm
n=ggA4_F5G5H3@e3=wnDI^koCt?2&EoRef|6lH8cb`c=aVF_Rn^e({4m%aB*3OP029Mk8VGkcn&6ouw-;x9<=%WdtLA5~#+Y})~4_O;A&m65oMsii4E$(W*9
HSPuNp2TeNB<QXDEVWAM&HXs6R=wVi{M9T)tiBN#y}3#oFw-gCq)u}4%t+*<_WVA6jowMh_3Q7?|B80rckD+t1$b!x(I@yFjEH6SwQJFxOmPYjF)6Hlti}-D
88FVEfQ6ar`b_QA@q5OxAn)clLjZ{d(6Z+sT3g>WIPlQcw2q3ReqRm|GulYPgV>P9aAWa`?mHbvv5S#K4xaQRJ#6X~2NUGeKS+<AGMODa!z@4&#$F0nkC`30
%qh~~hPodXn@Zmjk_)i}GT9%slw(Zg*pF{+&HilMgp@djeRAja<_Xb^Rt^=m&pxoFI@dy-Yxj-EW1k&|JJrpWxhcDtMw=Rmx=;G)mez4R6BAQL4V{|Qh2~@$
a>y&}Ghkk>NtYr7ws1K^XQj2q?-OF7s{HVy+EvLLgE^pJF(|YL(UC48#Ejg3HrA9D=1lmU;)$+QH%FQ%uX4dtjHhDuk;?@;EZ*uP!u1LP^4Yj6`<6$pJV&03
ty4h2G`^=95D6kM@&ClB$A$A~A~s09sM(ywroya0w59`X-l#lRc|@k9K&FIXXut*4{yPKMyR70p0_h@_X~?%%XAAzaR}_5BZ_L+p4zeyLECsGYSZhO49DC?(
`~j0X%{}B*crSw?N&{WQC*eBRUcSCJTyHTXWJ#W~s?Wh2yvP_20?kkXD8*eblU)h&$JnBO-7tbmY|1vJ%<EQ6;wf^$uW^KEMP)@)^FL?qF;<(aM34fJL+iO>
$3lpw`c+*jZ5-1%`D;3fBCZT({OL3SO5-z9u#tE5ut@p%p99(IC&ESsy3~;&i0fq*5Y*^c&5NqBPPPZh)<|xaAEv5us?9?2@>XM2P4nu;j0y3fNF-wS#-tBE
ZW7*5l%BVMW>qmhx~^kdJ(byJuU>W9XPk^18m2;*)&X59MF*S8cDM++(iZ3|Q-D7}F|=v^Ui6@SG!14Wp`^2J*?b}MptV#%=^U4qfB~I8C)yVpK=DGoWKx{H
`o$Q8=TviM8O{Oe9&!lp>k`5zZ!0uiPF!h2M=^BaxLyyBc^N7{6ph2v&!(t)IGp=(@ySUPw*!?oReurls`W1IPM!6LWNN!0(`vb0m!%I)H&-e5hv^S|4jhxx
)-vbaCduf&wN#NZEC@}G_^~?+ri!4!DzQI8&?#%&hgBZ@6bpDu)a)mu_+)5tKL=PReD)9{i{d?C$qwJm$&}S20G%xWh`<cTs<Oa7t9WugA7y5;$qn)@u+juU
mbZVC6KQ~8k`!>6_0Kl|5x_nEz>wD5OzgVvxnM^hwhY+G%jAr)UD!$}KP>{J(9(y(clHDFVsgrd*|`Om^&s6Ps!J?jeKE|!h!~rrfyox$@ybviAbt=j{^bA9
>$xfrlA=bu)453#raoO>bxG#0%JYsbiYncT7BR*;qIVy`hGyNJs_;PuMWZN4Hmr0Nb_#0rZ3)M0Y9m`?nFLvT0Q@qCk48v*H^GcpTXN|sLIT$NNj%9Nj~*#|
5Ui<|`eBV~6t<)IjBtz=bi6xKYyUv3T-0eC5Ymt<NcIg=jcBobD0NKKHi0j5GWuVa80{g)Worp|u>SF9i93n#w=uM+$^f>8Af{KbiPU>vL#tH|V`UOpo%fAn
$)>cFDE(Qc?X?sd4U&YG2I-VF2v2LtOlm7o#7Rm1$9+?5g_Nz_7+%(aj1o?u+@fe#6o9Qq1sX@beLO6?(Krx7J~)N)kFjkFlQ5%M1@6NPOzn`(PJ^AkFt+8H
VZ$C->w8s;*ZF@W%gPQYg@SA%rUsJ(nKnh}pQLx<cT^Jxoz4}<1P6&d;FDS`EzRYhqYn))Xtx(t!=qH@mq!eoHF%7oF+xW}XBrof4IFv0M%zFVmHRgfh7cB{
=;}MNI86Cb*ETu8xx44Ho3j*SehXt{#F%9>3MF(D=lF;_RA<=4G^mlI85O<tkKj%)Pz$c>+igX`bUSD&n7Hnq4WO|Z`gonNio)aHN69u3Az-fbZ+_&eI#7k-
n@~gMZ9HdI9ZZ8aoEfA#srBzH*)(r?YBlbMI*xY+WY%tLC8;yMg8ZFPd&siqo)a*@JL*^_pyP1;QDyB_K9j9u>U4zOcFj<Zq!OZbqFc6$TZ;53g*0my-GYOv
4=ck@v7^FL5oOVBa8P==*??r|Ol#dD8fp~$3;;~)n5}U}m$0q(hNo|?LbODw%wvsV(?C0JLFpM7fE}wp7<rBZ2`*KAEEG~hlpNTXFBx52JePnM=nAB+Fci~u
BWzf}ONe066^O*$)3CqK3t<?RhCIf?2!392AuzysrHGVdVpTQwn1WmAW^Hs?P{iZuv-HFrh5WGw<CtBeey5paD?lN@g~lGLf6`I+Cj_?H5=4^N5<-Fw3Az73
G=_~%F4WmhZaQ6IB~Z2J83$w-@~y~<wX@t^ad<GGG=E60rFaOy!BxZn2U$53_)t!}ZKatqYnMOG6@bOWONb|hZX4fFpJ<g6XRDeV<!v-n!{(f34U0t9s@1aY
>vD#xR54}N_}NAbaVnG%!Hh&cu8t2KXuY@1WziDb*Di|#O!2%PWKgyM9VTf;g-9w5?#jd<i|C-?Y(;pxp{~+#3Dxf%xdSycufNR9tj9c831G0{vyuy3oE|I>
as-ex49L^977f`9Wk&fT3hYntL~qSLW&WCzb&=_ccxrxt)00IJ7GumEmQt$e`FVu$(P%LwxE#~HE=#B1J2E@df(njBT%}YeHsd6gt!~9Kd2i@`JN91;>n7BX
Z=}ap4b{%_(jiwA%3qpz5n7}mTT#sO)-Wx9_L8F1-PF%&v=~m?$3nBo{MjSvm+DY3LPPz9I1_Gv%!v2m6e6n19HEeQB~UY4=M2Z%z}N9|)aVU4?h^52sCyDX
mWWNu6^1#IAoPE~%Gz6-aL-!F)x+TsM0pJg0cX)4l~#rzZe=kOpHK`#^an-1k7WXt;U)LQC1^`QQ(q?T3G97Yi+mEsIyr4#^2>Jt#KiMSOP1_#DhV``YUF=i
pazF&)pSODI@1zhjL&+@M!{&Eb;H-Rq>+XXOtkVM%q$|o<y4<0^nvO&<xHjDK|bx-6p3NbH{>o94%7{5fCrRtio_Zf5~4pOMTPcnMJ1-F(O05c77C>-%V>pH
fJO%qHKr<t(G<78g#SE?Zwb3jqmEvt3vYN>3yBH^I{?YMXIGDlw%6F?D`5&vJ12&T&Lt67#@@is(FJC{>FyJ#%UwNjVjRF&X|97A(|SqB6b*9VmUk1$ijmo4
ND8ud3KNOzxQMEX1B_CJ5RQSV8~V$HAHEMTIywTXoj(|1kX#?dY4K+7LL%KN`SSSD8I5k(WDG9AD_dqSC80-Bn5jRH#XQi+1gQRYlFAdBZ}=#t>p$R>4$Bma
-rNDM)*;@ym=Ed(cK$o{pYVt=pxGKHNlO5srE9Ai(g(4XV%&mPP{9cV6$I2(55a2+_*kYWfZ(8z8N*~DcTr1%fq>t^SmbtE{`Yhu)<q8=s5Z=4I(WhE#tqv=
p0~$5|M-WZB~GY)+~{%g;0g{Fi;rkvhFsI=(v`NwjMhS^cGp>~*3)M3Z`^JdM2%M;%T+lSps8TS`Sf_}iHV8_sLSVw3vChh>c!bjQzP|s>c9Sf#N~qGBQ&$)
7(S$TeAP{`bMNF`eryWC%l5;nmj*)&0r3~BgOtvymE4rmYxU>7`qmjZ=xeq=sW@ez5cHX{GGxnAvN2ix;_tcZ97@@S4G+UO#&J|Ydlf<sSk>gjg6xZmb}M>X
{QC%SGkwWWT;pT+6e076??BABet_HnXH85tPX1S6Y_@)`sFklm%9!}CUv`+bND?mz(B*`!@X-P}e{`b3QZhg2;UzqzOc;WYCUGbbwQ^%uzh#P?ZX~8hy{k!z
C^9KwWNu}tFOb%c6X+eNZ671gOJDz&-2Hil>a88LwVETm_yoGKi-9S*Cse<B6g;%5$$iUzH0_V0!sS}HLN(8SrfR2k)uZqWxl|rCPwZ{^beZ_TtzUe~KS1Ch
<^nSFxd6Ba_!gKrEOHeDTKWkoLkvJiS2VtZ{KL28(50>=Kc{Ofvn|ZoPVDUuwU_OB-iQGVyyM7WE1pFH<GiJOib{rkthWM94_jkd72kuDPRw~&{8e|#(AR$g
;HuaP_jE{=fPILB5~_>M9=$0TV5hU+Tzu6xCTzRws+RF7rZtH$GLAa<qSRk*7HO;^bfJjE6+@;gCWOw$XCJgdP?mQ_ZhYj2>&Eq<jO}=^braR4-rzJZ!bJkN
VKff|SrU`bBb(asStbw*x{5o@Z+3E`5t_{g&ylUB?3DgMiLW1$cC(P>d<xn8+W2u=H%_GF6B1vIU}87#vCnZY?h#2X!}BH%@THC5B!fo}Wd&z*jGu9<)(=$9
VwR>`46HZW+fcd9$@I!i+x>=}p*N3M`ldf2xrlguryq#N0fV(Fe|jTJ&V@vySDcU40l&a_J7jX1tTB%n<SQhu!Y4=$XAoRUi{Q;DNryKc0!>B3YwXLsYVRVP
K~D&&2Qd}UFs-J<gA}&^w?i(4R4p4;P{+%EWXvE-!XChR7o7B3Du;KJIi37`8}TN+=?1ZivFM|Fp4Rt{uGA#@S|F^j)9o7D&dCK5hU=f8BvDc_bi|{*m2)=G
ZRshoQ9r4|8F5wJo%~!UXnOVgk;sg}rEflZS1Kx^?^sQ<<}%sAsVh|D^t0V~^_Q1nZHd;(8Bd7q7dBz`F-o;lQzJ-O>30=r3qIO9J*+4dp-~IUPvbaOi4vCP
{a%e90B`!*zOAxX#G~RtWVw;1FmV7V!Gr#~Dds$@z+Ul5SwIQElE|aqHMoXvJXPrN%=jfpc13;y3<qUA{n~UOIBAr&Cm72Mw&8m;ivZ<q`mS)TC0KleH}Aob
5JiOr?PCl()n9SMI48tDyy%;al;$)x7?hc{_g|`*5l4u>vOC?J^y4D&nDMR$z=&RkXX@uky;7Nu{!HK_!^0W@wDG0<C?;?U{BIm~xTL8zF#KZ@;)kwP$bn`2
<F1(3c)_tp^NgC_l~|NH*Rq+m1#}tZuW9N@6qH4|APOIDXVC5Iw|kzUQwcM4?0~YjzVfw8Q8oBZGppOj+a+JerYGS<Z89E+4#Tg+RL`DJxHD-ZXSN7*FW0}Z
<Zqz?Yq%}7=_Xm=`szmZHh!oq>|bI#)@<-efpch0qY!rYyfI-oSPN5=SQhZiK;7c)8hdi}nI0*OwMYlR#RAWL<4D1#GNOza4;3H?x}ZsxIIaK<O9w#0?L8Rn
2smBSh&012jFy37K|~dgc~MlZ2_~Md9!!a%wz`({?&Y#@kbnD=9lK#bXQ(V$p2=Vbo9kGzb_2Rs1=bIjUJC{GnA<IDG1a`g__nvKqO=&!6aWn>e4*M5H0&Uv
&M$!577s@YvDui%q^<Y1fbJgnOMy8bL>7OARG~(SQH#iHZ6xD)qm#ZM?6@XCT^OfQJ{p__slu#{>J1-y`hnL*Dvxc=>;*`iS$6tA;cj;hx-FcV>XA>bVgU0$
l!z4vpxkuz@}B=h+C-+$kXUHfA&GvA&hjP`z?$jC|M5SYpQcQakJtTl`>DN5=#!3l#ef6X&>&H-4OZr7*>n<ZKsxYSFl?vHCE?J<gu~!kWi}W1FSI(t=_lW;
ov5+mAauB7Tp|J*uAJXnYhuCF?$JTECeF+2ctPS4X#1h$gyq!v?_C5|Avf?a3u3l6GWN`1Wq;iIJU<eZV0NUIF_F{~7tR;WjU5f){Vi<I0IEHADO06z`op~?
4{cj|_o2|I^Na2-XsQ!k(15}`WXV=wwdi*%)+OgVp6nukzb^RCLCR5bcTH=G+8iC-%pYEdPp-iEhgN$J5x>E%m6u@jbvLHsP_g*7_F@OnsaJw1kF|;okVQvd
j#-^h*4sfKL*bmv`fWJir9x0avRf-S#*M6{AAYLQGQp;wL)GEknPlAa4*yBQ^j~1U&aD0va$%L@ZwX=+AIyun)Eo%1KE3D$Z!*|sf6(`@V2Ki6LoBm8J1)B+
TLQLe9sv%>tsd>anc?y=$nI38JK8f499#w!$_$B_4vd6?BvG_A#%8$f(U;P`vJTgCc8uK+AI=%IraJTf%iUsg&QpN3UDi!1gm%js@;}eOgK|ns1u`nptGOV;
=c~NlJs|m%J#gko!#|bj2+=n8AM-g_fSoEEU&?lUsdf^7C|yD{`};%2>y(RCQ58KAH69S~+jT)EUCS87&lam?OnssagLU|Nb`{TEHB$|nS{{iWA=zubbwf3$
Vl(`@VTQM}&*e*NU8u`v0FJGkHd>jlWdG1A;-A64$c=JZY77TdLQZGT%s6Mk;3=|&T1s$IN;vS9meXuiJtXD%+v`@Q1|OdTpWM0bhM0k#!2d!g$g3sd)RQRC
QJ<3)()gH3vbZ}7{$um$nQ;Ewc;P}64J=8&GEet&Dz0h1jc|1PjO0YCd)7$yl|y#jmGB}wGa>D850X6|$9xX(FI58=CL?7qsCoq}9#F5N&fB5%7moRNaa5iL
eIg)DQ78jrz8O{ZH4Do@ucD>R?+{zRCR4(zkLC8Z)$>6gdh~WytWcfgno3w+*DaO`raW&sO|+<A^Rm`$xfqeICH%)4_+qKff_Yjh<-jUVkr0WlUhwOmD2bh!
4Kt69+xu!+N?t@<TZx#l=MZ^5-AxDxd2EdX$BEVPoK$H*a$^i{ev2o%yEX{*4$+?KuA0C&U~~;73DS1k@E1O-<dmH0epFF28`y(UTQ@;M+8!YjM+}^ha}iyC
eqTOLC}SRw$t7*&%~d)TkAJUEU^-h}zAOMhi_?6!9<-H^qFoZ!r4y5JoVplR#ll$ii(HG6$FbWiLwDkU%;_S*W~UIyL0=(EY@8R6IQzL}#c!Di(Rv3E9ilpn
Q$9)ZJ;JNC$4@EzSX-Ec7iY{3!mB^I?n~GixtJ0F<?dobYBaJiuiF=M<QqAM*c5#+*zX|3-lE_h{v8e2Xgl**9!4F*f=^Io-y{^j&_+gYurSK1<jnTDFwXCw
U!!F#=z)dT8?*sDpoiKVdLQ2nT%<iW6o>7Mv&JFnW89`vWy*yRRIp2@4E&@_V5f(VA(2p>t)auN;6@~V8GH>M*C|0Di-fkJ_O;oj%V3j;oOqbyf3&r4EYI7m
-9I8bqF0pxS2d72c&@Oi-P{R+G8ZxOUF8eCz}>t`jf;J{@swJNQMJ8Rd}{;gV2Gw%YV73`B0hBoEiH5iTk;$Ecu%BB49LclkcK+jizPR|_B0D+enSp1mF&=K
Mt+k@K4|y|TYWJ>J8==X^A9q*Y>~(>UbP7o4^(AG%{?af@4XGwInqGAve1@KMfIr{>riFedyr|eH%{)fisX*L_7|M;qMaGYt_z;<gFBXa)P!k*w*Txc_T9yj
-c|z5rt5*qziD?lDewIwK%eC?+94+J<gNloGOrInlr!V~tVPxn-D^iD`>R&CT-md?!4h*J+laZ8OA$94l2+U?Rra&cv<~;L8R$uqmKuLI1wWAymreQGe@qRH
DdS_sQl3q6Z0c53&fM<_6;s{7bBe^bHMsb+;C*~YLbT3Mc2=mgeQ^37wfOb_MC$|Ii3PwP#A$g9;A$ObD}8w%^um|lSkI>uA|F~$VM_oe3k;Y7;7y)me1ym-
%6`;jK?@BwDrnjTbNzdo)x6*5cUDGwJkQ1p`sXp*oLs+&tAn?4oIAr#C5mFV>ov`v6uW5zDhN?N*Cf;t(Nn1jSvBC1mCfE+waHYHpsbu~yMy@Ey-i8IRL&mO
XxR)Ic?*c6N_tP~x721xwx(Tt)EHpe&2{0fL@r)R8agJ%VEpg`-&Z6W-w`d{r3OklRi6GyQM+Znnx?df(hCl&lWo4==Qo^G2-%@vBEh(TE&^9}NQPeo8!3Bb
JmitD^7-a(TwozY<RCBvd0MAi*hIo6rN8ft(w}yeTZm(WM9{7R4_5=OF~hj^J=e4<W*Z^UqQ<>aFtcc#gjeI@<Ur4-_=THJV$q!FPCx8Uv38nJ9D?kI?_k5n
5x*LzaX{3yvjfiG3l_$m46DeglF7$zNUj1rI3>8fAxL14$<1X29H`Dn`B^@uiD>h#3qC4GEj*TVh~yWzE|QI2PWK<{>y`ag`W@wuIvk?nFkTZvMK%obX~!+P
?YP$|_#Fj$pvvIC>_yM^6{cH`e!s`$J&0f)=f%L?watrUr}h{thjdw(j7_8#lAjGM=c8`6%}>Jm5`=^Vw(siwtB$)X>uobagSCzA05c4_w&ubi2X*8_eOe>1
>r(w+^KFj1-l7@4D-x&@uZ01-ISPH5CQUyvxNUUALnlku@QTu##^+kJQa4FQ5{C@5P5xa_R0i?W-1NrgY(fn|A^yI<2qHDrS*tsw(P@*-gDGY7#3}D%r$J5d
RZsq&C$4EO@N*W4$lF0Eh`AW>tEmh<9Kb)b`MF7S6<v9O@Nq_S`9awu+b?}=%^m%skkV>a@SU-NaH|bHdxRc%pn1#B=9;#8Q44hEuF&fQ9ah_aP`Z8ga;-;m
=Rza)z2zL_`0?Jo7(IF1_{p(!pf&?x1G(4|H><CH8l1e&H`58BG9oFH0BEY@fsAqlwIA1}pBbIlb4xp77CHqR<eJ#3V&}F68q*n<$cO?K{_Sq@lhmCDk<zPJ
Wcj4%0D$DOkmW#Qkle!?Z>qp;J{=0?_Bk6oqQEXzUsVk1a&p;);m^3A>qWvS<ajaERK#E0#p5g^y+!qjDC`NplRJNvXopCkc3koLBvsmC<cIaJ{UjNPsw?9B
d(5^r_+(o7AJ%IvPinRVunvDCIodv1Y{_phz=SRr<kPCsvy<;z{10Gjz+04f7Aoz^R#|dHpEV+jui>VZ(trwroAySqXtpP9Q1Km*lxxsAY(%{a54&($m{)Vd
;}n7W_o5x7;x`E!x=c+U7NudP#r<tshs@-GHTqRTfYH@Bn(J^PqBaw@6`06+{1*2W{^!c}^r$kRi*S8(6MDM|d5iwNFG3Qau+X2TFP{47vDDlp0auT><7L&D
CK7N&Q0Jv>&IRQ!{>}mZ!RGy?PAcnLPxZ2ocNJWC!(@LmAfEQz3)a|wyAk?q=7hVUFDZVSlLh+=kngzIs^FxDWg5yQ&~|9pHw-`wLaU7lp}Axzt+c_wq4fD!
mfUr)f@vnCu&59H8-=S<_+_ct<ozk&6U%PiCef@GWd^kb>B0r<lBZy{ZALg>@nrCmkJK(R2z3h)3S{LCn4=!Ds8{u1h%TCRNC^cUDDYHwC64#=*)^_+TXw{i
J#F04I}qEsG?Kfct6l`@mpxvGfchvBKl8#8AoR{;qSDE`1&k}&fayr<o*-zTj>QH9-%kNK!rll4y<NBur`Si13l;kSpMxukNE~O}GJG9zaao;q>%%*QTW%-0
|2ObU_RA9SY+1Y&;G=B2A7uh6xH(o?O=}&&HUoaSo?EyH<L#y0rP&OI{R@_=q?hZxHtzye$0WtpxPFT^FHAhBSQ4Ph8*+jj&QrnFOwr1$z4sS#MYp9i;;6Mt
m`-ChFs1M=lHL4qtmoiaz3jSR2THB5MmZeb(}6gw`q^^TDp~sai0BKJk;ZJIHj0M@OJzDS?|lTvZNbw0z=o(F!jha1oDOe;`C@J_s1F{MqCPEp1E(5Q&-!CU
%&Y^6n2B<Nbr`%RtYJHzN~O{tljm!W?Yb;E2WIdPivLv4#{Z~!d%JEX6iBn^FGXDR&yWGNAyY9t7A1u)DnqPTkRUssr|o$jVbmJi&-$5is=J0T$a{6NIV%9t
=pzG@y#>!k^Sd=Wy=La_umgH&&&ci8La9BSmfwu<+Y<G(LFINoy+={VeIcetl0IjJ{piz)aGGVZg9B_~LSX)=nA3hlz%CD1-LWxyl%+1!6;s+*^4WI(tSwD~
)!lSaF`^dO{;=p0MgjD%ba)md5=(tGnCEO;Q?d`q`4Re0KC@xr_AS3;t&uk5_n;!Rd`YOa5H@Eg9C`~Y&C<x$dywmobeDONbH%}~FOU1oFMf*-BbIgQx8{x_
sp7ES=!0vujw*a%r5q;%tie9BN)V^ijs=2D1E)G|^AY1CLY8#2f?ShL1F7F=R@rGNe)@}Rl0FGo_dYzX&e~oHR&@{-Y?I`2iy<gnfC||3&X-Ip^96gjY>bYc
+D`^UWA>-lMO!*^GjnHx{rJE<5->NjeW@&6wrlxL8=v#ed0BkQzbMW91M_^i4=Jt_`mR&*+4ob<G~*RmUQ~2auY)(nd+J;8o5+Xgt+|(^89Qe!_0qMt9Gx@3
Yy`#PWJ)xy8xn6ECC{g!9Kbs?js)nn6>4jRsI#!fuFnW!iEs)?Qt_?`++Hq;ee(aPL2DL0>Z5_)5MuUIV3y@=KZ4*jF7VN|{5(9)^?%(*`7U-wJNrFnna;=u
EUK$DF;AO9doUeSD1|0KCGBOgW!E++fqvS?sxlCP8oC@+JwQrMGM7FY&|>UJy^$Pg3+DRI@N)R$Zj}JQI3HUt8`mlef)qByZgSin@F)QE#+4onnyG9d<ryHY
>~M*MvtmZ6EPvykIc0S*2>7K*s--7dvdI^7^}+o|yiG_v6h_H<dG4(Wo?gF`p4zTvY)|r1%OlnVL(mLuPDl(^+34(lJN{6MH4HKnz(6!d&^ny>CuO@o{Qd8e
Uh$hWiSl>p+&*?ab@XgGdnpF3y^gO)N54V+G<wv_`^0ntD@E4FM2dz;lL`Jp{?sbj%04niIMI)1Ul-T?fjeOj-1yp<q@zRCg_pTD8E|UUWyFNqxZ4wz9G`z-
dZ6t*1=TJ4#7fAECc)$d`~ev%ecYNDwAg<n{Fs``wKgT1g(dBNLzOSA6{R7TZZEYj?~AhS2kpu`tbS3{TdzrtvDMK=VU%j+5&8V&H0-v`k;l<-V1Cugeb%=|
#zNkOV)5c_s-pPLmtaLO&}m|9z6>M!T#bYzt%;+dnfc%I1gi}}kFwxW?M2dbAj=&HN8UNijW}+^lbimPu`<;4+Px2?&C&G)y37gX`LB($s2efB^Tl{~DS3}y
@2miejv(w^Q%kUChnuZV-+tqy#J$ho5aPJviPLvK+W252KS;Aak3+h<3r$@W8do9sZ%p3FtvMBNYrjAhRC=d0!CA*ofTyIa$?;_-<b{q)7!yf1wizTeAv0-N
=*tR>ZK!V*Nf<GFMMn+&FSm$qT}~LuQ<3OW_RE_;sIfFRaLiZ}n~2HjKZn)I&Q);L*Gc-VC&RBgSKK`1;T2=R{JXt&CL1ojiB;IOZFs<WZpNQD#JbL~nvs7H
g@s2Pu+&DAlF5gziO!rv{EfuE`rQ;pJjMV1M`QH!RRY9b=8dW$Rxoc5OSNoGl|BvRxs40q@99$58<3^tCRUF}3D5ZWyX^ay#>-11YfY(|a81O0PVuaBs>VmP
$zez=Glu^3A=sb=>AC&iEQ2~pBL_<@YAemcmA|Und0mh5I)$oOWT`dB^5fs#*gK{z*WqDjZP5h1f*p`RHWW}hNI*SS5D0tfPYz)vjyzG3wW3IV6?w&4@#Jr|
6-mfRc)%w9JM!_15+w5T$Y55&!wvqs3^Rg933DqjGVa%#OrIrxq${QejVV07`s9Krq&9Nx!_C0gIK)KU`e0(G><35qr5IAIpZi@Q`&ZZAqa29?Z2xYqUjvS5
N$wEdsuhXH4>0f}uQ6s^1@09d9J|&F!f)0zxYM06*r|H-g;If?VV1X)Eggk*e(Sb)KDh~ejvXE`V14_uv79k)pG#F|SApH51wN0TjJpPdUn5QyHSI2}Cntg^
QV6OUSIIKsQa7!ihdHATq7br}{|<>ti$P3{UJ_ipu(&>#`?=Vjy7NfE?nCa7>Ahp;jS66@!z?n-7k+<ZcUqD(I<By`rPEsB+#Wj#j%Z3FlV`tZs<UNN-v#;3
W)i=DNYwvn!B${C3>YUM0iG&dbJFj=@&jT7ZQ!?u`z%N2P$ae|A}b+?otTd+UvPCEi>)>Y>jk)!gFha1No&#u2Ac>O_jN@$RI}NJXzamyRrW1l9{yJkt1-ZO
V<%;PU{JV}0oJ;<Sdz!%46<+YYHBZikdDgA<xa>@Uu<s2aDC4M#lCC=m`2fiW+CZ)aKl4j!&!DNf12Fj-Rgw;a=_S~g>Jqf?+Ful$^%qQw@wYEu-56;1HUkw
V_ZFSr?rr8wPzS?3&Bn2WA7~*RZldvWsafdxH<E(?Mdr;^TA`>AII_v7ji)_C`v)@*NHGmaHj<U413}O#+WrlEC60W1Ui$*{M>Twuyn&TOa{~P5EpWhzZ8JB
88yAEC!TZ7^7zB={1RBz&%4h92*02%FuV_AEs0>0<8*5aI$)`sUnUylqnsmF>>I9$vF#~1fDvaal|F}>U#~2ITiEG!zwq!db^^s6(R(|=r>Ny=MzlMp5E@vj
myrNPac+}eI%0X#!@(ID*5flPjXLp6jwcEuqy1)^3w)?-)*uR9OiaRn@nB$+#l`SbXo%kS=n$OOAyCX9+mApt^p8Jr%heTlx(;9pcuG&&b@=6eAX$2wZs)Da
jI%s+Oeea+rg*gHW+KQyT5=g5(g~<p#0nwclhq~Mz=c~*DsEoyh&;QJqq9bf_Zm@;yhlyhDUw%%4fa0D)R}O`W~Am7%%bG@EnNt=kSisHl5EBYYX#?C|6TE1
d!4^CD{hrfVvF)*cSpU)oiKnA#EGxXH3A!bXvO#)iQL&lcH~ht$$^p4X~e`XgKn^3K)8L!4@nn{<3UYu4%>(TguUUZ9n?PW{<ix}b>&=M)#ril?ozrNtjNOA
IyD9u;GO{Uuz32I1sT})aG7{iYvqEH3F6pjamiOi8nXZk<KlS29^w;!k9_^-YOZ9gcAu)6NRTpRD|5>><s(FSX>J@V3sh3+&l56iYUT~p1bkJJeiR*xdjRG$
@@s5s=;=f6E(yW@JXFp(98gHT8qRVLBhKN8DG9bEyXO1en3X3`^1szMU|+`BvG?jKFSeZUfZ2`>?l|_i=4#HItM$zF%!*Ld(Nq8dGd6e-UbKtH#iN=(aYW;4
krln9j3KF%K?AQ^Bdyb={n8=^ferDRYPucU*r0-I6;WUJCt>?~an+$VCwVGw4WZixmZnU_Iq1upxYLW+MnphODME4TCxN(^NoQTTzxNuI1B;069&jP5jHnn(
9px{}O>yyx$i!Ov?QLaP`WT;~i><F*GKC3e)ToM6dAmcM<OiveS{XN?0BC{4hX6I$tgVCg^^20gJ@gZTyM=?nqVY@9n;-dCq+g8WiJM`Vb?RqYw+;@v9HA6Q
?4TYYH!~nkExV3~;R~f#haQy!V9YD?*^1D*n$(+#Pj^9P4|bht|C~|LmiHFzwpwfm?91=)ArZz+9yt8mQOHOq2lmPdl&jIDm+4a1lQ^GIA_gnjs`#|iEpc7t
Xv$lt;O@jFS(&$$toO(<L7GV-fFs8oZDYKgg^Qwz&xewpB-y4S*_$ly=7D7nyF+f14Cb@z+{0K7_gA{!V&uno^53%wE9Uh)(jvvGE9$<6I+f<V`v<o8k1yd$
9n!oc7SoRC>{W@#{sJ{rx*@)5)%K>;xP^9?8NBppjUbECQStaN|4fCvB#jNB#~w}V<X3a;LS#L*J557SbbXx<&$=2%r9p#>vXk>?*<iv1F-kx(tQBzRlAXXi
FaBB7DWxl*j_Bw2@YvZ;6UhXxYDe3PsO2q*V`a$Ewc?(6sAYjxerW~(({)8X3<#}6=FOd_?o^8{kofqo;MxoVWF1)uwY0_b+VE~t9=QYB<>E}KYWG#?-v@<k
R!*b=wte8`CQ3Tspt-Q*roB&kczCQFn%s>_w1>bNR0n<h(S!K@Yr{=&n8QS`FO>bT=Ev}^s2`(sAs2zMJ>asi>2WsqYIjI%)J$jiFZF5Itoq{*y{Lx1(@Kzk
Y(4IRGrj@u&4{~cB*iD)JA^&-(I!v`ONHJgvKVxyTta#&)T<@`<$!y^%`_mDF8zi&_PS&@Pt6^3EHZv_d25d%yysdaMcQ<SMjmsR*>g9u6g0@*s4BA`pjnNt
Ui5S1B7pLq(U7?9vJN`1+iv@sjSSWT1Bj0?<*g*KTwbQrh4*vfS)&v@<(eQan9pM}iM1Pw*10)D#s?2;Kq;&&sH;#X{-?1L^`v!~I^uFD-b<duETl`Vlel7D
>>$J0YDxOz<J?o|n~HjYdPrddGb^723!QqMrRcPsVA7f+I;3B}bXg7=u<R{pQCzC!gj__IN{kD>s`38*D#6+gdIrJJ$cp`fVwX{>pmiWMX8Wb0?8R;580@$A
ULgm|2Tx<?lz(8YA{FNy>fhspcrp-JQhpEf$K4tnuTn0FgI*z?;}t1wdsIX*kEbU%ek(ke)~OreS)bO5UsQf?oc{>9KGAKMV<=q=qren9-GKiH-)MMMrX~`p
$pzsVGAihZjs$jY>{a|G;+M}k+nch_wD9Evi#vTK;~brqFy4l;)P6tD6BMd!VM}f7ue#343%RyfV0{3&>_)H~Uy47X3Wz4e)D{&9d;_JV4U}VdC7g&;0;9G_
vg0s1AFhqxUA}VNyG1tg_LBH^HNy+Hn`S&ViyNvuQcNYZ{YnSVy|_r{VoS26t1d{kyeKG^E2mJUV0P;Rh6G?blH61y`HNdS@@)qJs5xD|`O-@6(h@uOJ9rOJ
iXPM80Qc-z<>(?{lu9L5h$o@Nmz38!B86;w_YVeYZ-S(#>}%q*38FJ~O2>>EF?w<NkyJ=OiocQ?@L!M#lsIFnqooqofappGb1~bxi#F8|22=hHTde_(`(wdL
t!ZIfr#bncl7HNTQ<%hMu#&1?NhK4DP2g?A#%OO1UzZ#yFWzpm7r~)YzlOY*C{QLyEq76JUdaWl-P5Z-B>8Y&4+|PUQ?w;i#A>XEQ3GL`3uIe|c@Bs0-zv$T
w+Yg8uXPxxPW5CNd+1|Liy#8*c)a+Z1c`ql1i-2Ks>HWl9&pcMp-OckLvs*6MMDMfn2Yelt9dgkd1GYA%gt_7J$u5yVLQGnNv`kh+(hqI`B!qy^}QBLNr^ps
1KgkmGkdM{{<vXRN)hlAGEGvYU_05q(}t@NM5yt{eU|L7k|<<ap7nFrP?S>>cikNQzxuhU6tg3cD`Mt&&Z0J!LlrA#pxgb|iP}H>yOVc0Go!HK9z3{yRtx`v
tB6v3L^&HI8`Ai=H;$eHl|PKWAZUKHsHF0|)Ft8H(z*0%<_EYYC1RL=F#;712km`#o47>xewG|F8VwqwvI!bxe{Jtk^0K;3)B~kA7q<hre2v~yPWycqp9Q6`
?}2b7it_XuDHm}nkYk9|oB3XzbOO>YJqzEyF<Mj6001VNImeD=B^?g)_>+lD#PUf)P%h%BTUaH6sLSQG3`=hr+h!I>+lwuTlCGOO6$^ejHYAxAHMlKqr=(<9
Dv5}I690gwJsMc7WS2P<SZmYNrdbhP>F@{GRO}&OjH&r@4ve)v;kj@_#yhwCxZ>}Q)h|!}?cl2^NsE%a0NKCeJzOyZ{Vsv={&f78yPk3AWF#-g+{ssaTgO&#
L0xv<V@-6jItnWXWv!s%Ps4QHzI2kbtj%FLd3iN$i#bZ4B=p~q4^6nUzlju16Un~6Vu=_>X4RDKjAF~<hZE?E10|UZocYG2R-FdpZrSw!{|~ce-)qL?T_h92
^*EyB5v9kJJ$Tcp3;8d(?8LW&nX!EPI!5o8C$9|FMio~YtM~esiY~<blSL;}E{tII4A9MIq()W=Opc))W5GM9y@{EvzwM2Y#xHI2Ae*1f-p1YYci5$?umAu6
ZFs%EoZTv300G)>z|jH#i`b&0vBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/deteccao_r5.py')

In [ ]:
# listas/conferencia_r5.json (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/conferencia_r5.json', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5-~!G8h+P0Zf`9=dMrM9yQU)ur3Ekve`7t=II+0Y5_<jf6z7xUw6At9e6Jb$7d(tQ-TtZoYcEf~iawj%bNb2cUPrAqA
7&sKD!rUsxEb^f#PPJ}I?2z7Zo35%(yuqGHjS!kCgh5~QaIMws{uco{`c~CSxyyNBJGl{YvEM6~D3P-9e5!V|R41`GQ<mJEA*U`4Re-v78^$XwzWr_=K|3(0
Gn(H2IbO#+<CfmJ4MFJ#t?Sef@%e$L!UL%CX=;B`t5IUb7~>oqACHyn|7|j*^=fXuX3pY(*vQ29$D~$jy23;l*djp@@!>L!y^9LPZM%)AE1Bk)=g3j(CI1M_
gb1?P{Da`QQnHWUYJ3cdW6gX8)fD%@9K0I|YkgvGAVUmqrHGV9eFeW-BSkJ9v+sm3s{rFT9;8*;>No+e>g6nhJ=G}+8`P1Q(XQL&EUw%J3&fn*s1uSzN0=x>
Fn`$c_G^9vqFUBbSJ{ej`_cKcK2CsJf8kt_Jk*t+DNan4JPAMmJ~@T-CtVAL00E=}&jkPgPQjfnvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/conferencia_r5.json')

In [ ]:
# listas/divisao_r4_flavoria.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/divisao_r4_flavoria.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;!SNenOy)f8cNL}xSP$f#0`*|<eKar^QU6leZ?$b3vsNR0$c=dg|TTJze1t>-L(C2!;vfPK3uSTaI!ie@*Hz{`DDo2
4x9(I!n&3mcTetINjmm^dM75R9~)J7v-DGDY8X~*_v0-{=i^@umI~GHRNPa@`5dWMqjsoLnzp;lUd7Ji_mA)I0;G(WU`yfBq{}ZaufwqIWAMf7Qh>!WAa=OY
+>LTd77ID0ipvQ2TQZ4Xg|G~xnjstD*T&Y(KBc<;S@<Qs9U7X@p5cDxn>~aJ<5*U+=Fmd<`SQL*sRepR9+$*n*W8psfRPrkCKdNIrhQ?HzMoA8xQ+5wl0@;2
?a9e{Pk{c=5n=P=Rnd-yNNouIVpA{xrho;Y_Cvi+FrdpgrblY;QH_I!Fj@khlcwf)S;bumsVm;TNmjwTB2w@g<G=RZL4xkI?vq$JsSO`oN2rhW7`QW9--?&f
qoYFghxI~O^%eM*;4v8fUxwjwl_s={Hxyh3<SsPjT|kNg+`kZisyYxF($x41=E$gzbcg!uRESu&#r|PNsqWhV6&X5KC^+0#bPK>poZc$y$ImBdRvV2HQs@@j
vH~>Js_-Iq-gVLoGDj8MSLkdlSQU7x@Te#bn;qjqaX2qN97**TirM%M-m`isj4JE%f1!N`HrS$lJj4>hb^{oRCuxSU?o(!Lo3nnj&abd)Q*MJzV6CyYbiA2*
6aC7g**#P)O*LXZV56cjBLVQk_-|<M;FZl(f<<5sxnI#7_&KN4T@-Gv<xJ=IA1@_F-5Ewfx!@<tw7768EX8bic0^14+D@o#qe2g=WM3liL@cwO=Kot=#(_X?
K;jWG?muZaj<0)WrP@}TWd|`a_&9okoi?6fJILF|VV|!PRKx-jke(@04542%HgIYm6NT<_oJ@E6VOF|8RnSOj{c3cAIfNQGjL_j^W-!Cm<vKpsv^(kn-N?zZ
0g&%#gZo0xK){6?yEfw|1OU+M3VoCD@}<g>=Vim}&Uh~E+jfE_4!JhYvP#es%>)n0iOg3To+3u@N>qhTS=d6>N_tD905GXAki7oWM6*2rbc!_xyTu)k*?72e
uJC(FF{K_1&c9C>!+7qEb0x?D)9d4S@Rd{yeN(5`*}sy4jf~z7q`pIb5~jFq2k)XPthqBHd5E5EJMu@{H=OvI2)2d!@gX&^VQTV{W%J8?uyw?$j-lS>kNqeZ
5z~~TJ_^#-%GEbSE>|6e7=#`i&2QOQB-$-kVC-bR*3rY{%_#9`0PwI7?LDWfhB;ucm$!zZi8L{o6Vus|RL_$4)SkuEiuGYVo{rVA*^9Gt+{J@8x}@ir4@u<J
YVO?<d+SMFx6;(n{M4jnIoZY_fbT1dsn;g`zVhf2xBJgW-{F9*^ZXCA4Rfr+Ax{9}FBUT_cRuNM=EwKs)C~+zM}vUz?J6FHh}A+k6o`i?6VYb6y$MyB=x#nq
^(#FA(BJkQcjHvIR&Rb8-*7{HgdtRiGj4Cvl2tc@4VdWj+9Q{X5zyI=6|LWgg%vepws!pH>Bi53gII+P1cE+A&lLL&-EPx^7xLNWF5hKw*pbs>EW4<2=G4{U
0;cefFv}mfRV)|{sx2S3s6ZK|EkwO0iyY9boi(HUbpXS7XgF(Nf8mZDugM_`S$V!`i}SUvUmC?gfNY-fpv4P0YEa)I*xaEg_XfRYTWQc0LTKII3o{<yLW=)W
p+hA^Bf{k!qv=^!H=+AAk3fW3xTvPZoL7M&e;K>o3!?O(P2!t+lZ}%+8T&1PvZ5+M#V^{wWse@eV2sKa_4zU`-AHhrK<#KJ+Ro_<qGMVx+dfVX=#}XeIQy_(
#;vK!xPqOL&5@UYcYPFSjT9{81_2{N=9bK#qU_81uB6KS8!xwBn}JkD&UgZ&@p3-tN6f6VN!IC&uKDSD{vtrWb%6$c#|58-kA1fSmuOQkF;)0gby~<alA2)b
mHlJF4*Da=KN&I*QhB;Iz>i!VzfMVR<QCrb$@WqJ7S8m94u*N4*!9Iajew)kEQ%XxWeDyw>j93!%XVti^-3Bm=@-y~dW08i`jJcDU{qo=@g=;|&HD~T1PjWX
w-8Nb9vuWCjHGiMU$!e5W!n>J+5A5?3zl2^6}mD>2VMTyCU86=I1b)c>e+#?@O>A_d{Sm}b-Ip^o0#JHEk}#cF49zS_#?vMkOUQv=Toc@NAs?I!)!Cc<tMV+
pC079HkrQ<+N*dlLr(|#2=|T;%4GIq_vVB-lPwj5Jgt4P!5dD{4Wogfkj7|R@-sv=as_AQjAWs$tMePRNt4c7U|M!p8ij#^5UP3EY?X<fGfsV^@_S3Wdohzg
$WL3Qe~-ySbU0R0$o0wFtzRkQnXH}N|1A6}Rz{t;^p%CJR7nEc6CuFzO%BEJcOT}}xTNKO0MP=8m#pqq{hK<rctxfctj*iBeRoCcb)M7=9#XHep9$<4Puh>`
l*eTK%q}*sHgpg{cc}KI)C$;Ji8>w}3pTM79szg4QIh)61A+cmZ4W_|%ElV3+0DZwV_hmP1?7(ce!)!;#(V|(_`{k8Pn{hmpIj_i1JX}qWehQZVD!_1z#brB
FKBqS^{HCmcui7#o5h&U=FdmyOlinPV|&|O#Rb#|1G1~x|98Yxj}5cg={4f;GS}}$P|1qfJlh0aXl>q&Qls<XR0P7kS9jgE0ag_lP*(O|M|5#PD$&3zPx76#
07wyADPTE7A!QpSep4dT=XkN+xO8&HeCQ4gmj~WeC6e@jNoaeWM~&<frj$Agr@*fsI*$J$#c2tlG@ddTDh)CcK<;KTwpQ10Q;@xP3CEo$(nBXghkG76fCis^
tvoYl;Fd0f5z+D6KK^znR_o%6_ktK`?zzX?-TAt`4I_Y)hVmRYA|!nZ8&)-F7%tx;8yKxl6aL{%*0LGPxWoTw+8Rg)RqKi6n5{;TFBOA0!O!afI0<!?pTEbG
v`+adqsi@pxE~c;|M*r*nvi_i(DqJZ$_4J1t%WO!EG>&`DTMUI)>E|W`xEMQREE5F`gg#uTpkH}=otb|mG)KAc>XvnW((i$kI5uv35e&p0f3sQ=M01Rh)y>X
glwOkj>vfn;*S@4apc<QM+E`v*b9d1X|?g*^LZT8-i^0lkcCvn{XBL5QJnpfjv+rX+iPAeyY)p>-&K{ZU@*u-TCk{7(X-In&`4a@avEC97Uwk(qH%E$RsQqx
(Zrr_8KxiJ?NDZg;66Fxkznka^6IQfg)B`RkKIfPsP-6Sj9<&{4|ggm*qtND%TJ|SV0%=<P{66XRKVuU0>)42L#OPcW5dYrmGKOqTVrP){52j@pbL=?f~Bl;
jnEd?kU$y}TJ!GZV11{GA_^UQ&0tUH?A=f(yh!>&;Jf;%?lX@cEQ}h>o9_EO<Y4Kv{qdf!Fu+^BS*G_9{eHSaRbP>jID{paUIeJGH~;6Lts&dnvYc5ZJ1i>+
kHsq`Mf7&-q$FU06#@RnK7H>NXwxSq`ww5j>r^pqrptn{z3cR=T@e~(YJSfA>Y2Wd+5D!j`?~bELYJegM?HX6cd@W+@ljmFTa+D=yfrb?@p;P-hd>~-vz!Lx
=sCz(+H>gsm*{yF2Tgg)%KKCN(jA;bzs}I7wwEdO`hPkrTmE)gV=s=G$xzKi?P4nvSXG%stz8r<IA+8Jd~CvU+F1^12_HE%Pw}&aK#zuzCU=B~D1&@-EZk_&
915@n+3$+~iuOQ0e-R*kAiyzX)DPYI(QNva8LTC88t$nuX{T!5XRXhygQlrD7=9}N!grkcvxJE`<GZ<dI3t!AP{v=VRMZwb(d_;1a)RfW1{8TzZ24R{UOOn7
{))a$w?=|xLr0Dcvw5&eFhu;Ge{Y80Op4mUlgXFT(Ez=`PyG)(DysaNCOm+Dj3~H&z^z2JYx~UzNyEb9J@pat$Wx+9&f&W_YE{gVvfL4fXgPGDij~oB{-^RO
dz}cv)h!Ear3gSZjV<q&or+<ys8zDNNUXHEx*v|zy=`4%vC=`epkAY#oAd38#@-UJ5OuTNEzxtPUc!J>TJ8&p+#$&!c8L2Zp3MBb&#fidt|&%}Z819(*3`ex
E1OdioPQGZWV!6(_o|lrWvv!e7IKqN0za06_I(kkPq!(v<{<%p-%Z`-c$`u2)ob;iwzqHtm=dc`xwaX~t%4K`XO|A77z=<J#zLniX4JR;=_WtLPLJK0))|ma
?u(1ynFo)=R=QX1uSs5h4MFppperx;_rXOKMUm*cTn#|WV%naHBt_obcU%bQ<C>^3Ic3LQLkbZvJEziMJi_#5madBKLOTAP`?}=-S7`W#cPhox>`9(k6czYf
-i75R8K?0QWf$ETywGKyII;~a*MdoucF#y4rP1R>g6o)@l!oOK%_OM<Msa6&Z_bS)hO9jwmfVEWrr-HVkIUbolu&Yf>We=?%M%3>!9H-MfEJ%BALJ%h-D?0q
yoy>V{j$pE+JxQ&#)-e}%V`sPgoKN19BE?k4ZVEm8xLxjm@fAqa77HJTPHy%8_IpJi95K}W<@$gQjQzyL8}?G5Q49H$LtI_O^$?-s&q|h1}Nfc95t9!sQ|GR
Jq>V=lhbA&$Tyc+@^qdxrXOpxsGW0+r({3N%%f<ZuZpk2|5lk&_*%0zfj}}2wnDtld2FGORJ!1Ir8-XK^|KLi<J;dVE|(<%5d>t5`-AK(Tgw1a0|~A%?Mvzo
Y1Q+HX<73M!s6jJ)lSYjz`ui64X_(bvgstcy2f<C1>Wn$^?8k&J8g?%^mwCy<vfNH*Zw~(1WGIFdZR<$T2L?TTtE19@kluRLG_}Z6bgEZ2C&JuXUN1%sUc>n
_wGmpcf(7O7pO&k=DnS-#WHx0?nwU*4QC_)BiL#s7ENlj&;Vr2h~&#4RK1Zs{PtAx%#7bOPS#{VQ-a7$E6DU?Z!myVv$<b6saJt*L-%4{!KKI8=Y}p_=n?pO
twq+F56>Y8TwQLi(&(~sJ3@cucJ;nwqhz8bSUI@iABi|-xaTh@uI40+N$W-FpL^}P?%~=91-~(W0BJKH;p9L(xGgNOTE0RxLCE)PTi|8;Y%?W<Lo>C>!%<6g
3M1HJk<Qy>`WZYBUmmMbH-igXn9W|`Y*!T8XnloZ4e>-?im!(+Tx2_HnX9X|(O*y_Ts+@J#q2cX#-93V)S2A97M!d*<oMXRD92|r3RAOn92b+Jb-sKWFSdtI
?UT&2AFfV;@BBqyL=-)jj+rO*Orj07Sl|66&M~{+-bH%eA8TFZ=xNHkJhx|Mw)ZA!aFVXRf{s<bvavbiqVPCP)Cvgrpv)@s_UG^3a|0+#uf6l|h%SqLkBg+f
*>tUEIz0Jad>iDc7NTJN=DhJ870!jpUqP`$A+saLoY8_rw?=NaH%ptjTY3RBH=dl4T3oP%tC7+^Ci$IguvrX<TC^M<X|znt=UzzU`kgp_8mkQ)0jR*!av<SL
w|8vuyEn%N@gKwKt}qO5dBg*B!kw*&ZYw7gQ2y^Da0%VZlG!?!TZH$AI_@q{bqy3p{YpMQz!@+%4kD%pD1+%ScWCt{YCz^=z*82xkGRXLK;o#UOkDxY+FD+O
;3R7X*wZIEzu-oPe?laq|FTzFH#n%CHvJ<F*6{&P(f>ljHoL3-@h{L@ZByUr*d0TC7Pb~O?9h*hH$KC4Ba%0usK@a1i9*(QdB4mo%UtFx<kQI(wGwXHgUbl!
6t4T@y#E3?XJ@;6u^x)vsWXO9aAAzy)%t(DthGTtRzPbrD&;G<e)@$r1vOx-fHa|__8vJ(q{^tqM2as}s)a1B`<?xzDK|zH2It2H>gdOPGF{|$P6Ai*A;dh@
glDcHg<<yy+zOLdYCfScOb4i|#+<`jR}`ii7I@<wA?T6Awuo*^=Mg6>`vHxx52t?2_z?kr+Vys&icDZ5=05L;s1V63$U39PwTp<S=tA1t4F3ysq|Bv+oeF9X
wuy9yk{|;Zhn_SBo^#j(7vM(tnzi>e`7j&XZ6#Vrh-YhfPA{_rU0|c(OK&jswL$kyEC9Nx=mLj&`C2&b#BL6`|NQE3J~M0BK1CPh$gL4#3a*E%)AyPX)&pmz
qsyRLynE)=1v&Fp5LDh%ZC`wz7{=1>ix;be-Lmu}T()O~>w-3ehcvj$!ad<;z+Vq5A=YwKV4x1Z&M{evIg`6Y<^185p3?Aq^rW<3T$Ydkr}t{fsX4^9#GKi>
<gEElgQM?NsOK7jYcOtkGI-&Wn%7hllNXnKNCcJ%>3$sZ9M#n2QOQ%2TXooZ4L_uMjenUNd>)Qof@tcmwz4ok{-;&-El4n^P0}8^L)wZaf%F*uIqS7>r6zL<
l@G{fR`t7UxVH$AoI=+q8y$oa^1$owCWu1xbdadZqhrMOJfksUnxNKV0HX7w7f&T-@!wi6Z&a2C<u`rI5#Xp}sLK_J8d~xlFtGWRY|x$M@gGy&_KK>evWo^B
HVA};vx|=^diqH=B@v0-W9-?uEZBh5i61+AFJFCr{i2~f?}prp+n2B77qbfoEtezbFVx!Dz~d2SAX)8LqQa||25)ji1vQc>@zz$n6vsi&l%#E?TgQy;N3lWL
@ipBf=)}#z_$51X>Ev_#2(kcTNnwjJ0N&M=l+Ifb{?=`V0J5JwER3}V?Jlo?-{_}SQI<8i&oW)OC{(cqUi_c1hF@L$<p6jNa)ThyM`;&I@-*M5IHJ%rMw<zJ
ZvOM?nB(Ul`%Yag?NkN*_#6=1)Mju%uOH2Z`#vUx>A?Gk5jQ~ZUt=F2PB!o7+<=&G3OO}--knxN;u36EV^HdC-U!B#j;kQIig!@i@WukUgYZUwxcl?wm`u84
UU(;k@_br0^F2UFt|hxPZv;C$YPCS4jod$cddv9+EdZ&a<f*6twl#alxXg3dE*Tqe5KG5APaBF6E*4nUu6qniH!t#;92pRdH5}Ua+O0P8sTV%a>-^Wy)v03e
$f^j?m~W8xtdD&7Erwwnk_erla43nK@gIqa8(GQrqN76j6wYW^#^CGP2jW&YRB>%{iwz9No?U9GP<<8-(!J>gpataJ@W7u#nvY~gFirFZ_}r(()+-q(QxG!$
O`W=t&(ToA6@h)3Id0DGY!77C)^UuezDwUl*hPngHzs0$BjtRJmBvISQyQjb4Hgq}=<)Qgqh~Sg8JSbQ=aTIqv>Gh32&HfED$HVSNDh$AB4mN1A;Ov%N4?Ln
I+JRd7=cVeX{=!>Gd4lVUF`Sm|FZ@%-`v+;?lK?sAt5eiLmOG0pvKQ?9X68GFXe>fq9vH(Kc1Uxb+>!xTZUcyz;^MsimXA}0^DNX49>VG=ng1$SkM`y;e^uD
xBU@X;cR#T<L-T+-EEJj6V}_sO_TG-k1zFSF>$dP*nR57Erq`Q@sgbY7~$(@IACE(8UEW7?HyUf02Uz@P*uaQG}|AsZGUg0N(I2UsI;B>;I6GpniVGIzWGJk
5=LZVLTS#Eb4Kz7c88ac`L=F%<DD-UT+IFAt@Z1vvEWRzjg;^Pfh{R#+a1N`bp0qhwbfoc?K0E*gBHJtPLaHUcgzH-OL{df!MKqSyfh{PYCM5&jxuYS*(sN1
u3fauj|pNrnyG4bde@3wk3~y$bCcda9P;emk5O~ih<h~XJrP9aCqZ;tq}XsIfhu=W-ZY7sD7eCo<Z!q6iBVS?${+2EYT=yYx4LJV_(>x$ik7Oy{iQ!q-2JoV
GhR%=$Ay|D`=4U&(OyHP1Cl#zmQgIC1y>69Uu#D`Z1yq{C)*BF3v#$J%CONpAb>!{ddHLOnY@*e0n^1>!Vx(XEna+G`k8n9r?rd&I2BnSq^ctf^DD=R?$qso
5*azFQyC?2-XQ+di{dBIuYs${y;3014{M6NKQ?<7Kkqc#OkbN~wB#q)3~Zn+DXbo{G5j~H7Yu9>FjB$#WUIZ!WP{p^#PEV&&?PE4AB%1+dyJY@`={`&=vB);
;P|e{*?LHMdAb!@Xg`4QMizC92eeyRIeJpQbOrTPU-vB@P>DPt6pn~Qcq+9{zg?I2m(_i_a>W{Au-v>j^X%R~A=FswdD)-T*Ll67m90}o1&J!v0+T9^n7J$J
PI;dy7jF_WVc6y%khVdGd-EtRv}`MA+38g7J*ukd<&LDf>gEmouk&tc4%X+@A*sqOs&Ek^JM`zTu}|z@@W1A041Es2<fX>ciAbq}$vIANh!{DNKh}5Z!D$#l
heYijl4dmtKH&+0=!FRC&Wvr}yZZ1CgY{~wU^+j+D6Z`JcV2s55!i-4QoP?#5EbjcwBwv^pic=v-T+VHqjxSkz6e=rxZ2Z2=681+CwlxuLcS1{@Ri@yyTh0i
0h{TmiotG3heI#bYv`8{B&8;y5l_!O#Y!ZK#OGgwAkDcc)Mq9ictt&-kG>(3zp88$Np?BfU3{Gxrczwn3s`b&+uH#D{g#Q`L7Izkvu30$Gh1q_6FWi1{VBj0
I)9F$0pop;E*w$UnwGs0W~=4Ea2?TH)X0>;#_aM<N}m{qU?9|;s_YJ;YeIXhiW6+0`97K0oF!*%m)%C6j6Wi#*{-?e2v^Y`sBnY(><<lAy7LhrPNM3Dek_ve
<D}U)^oi0{lf*Q!fd&|OD3Rih+Cr$LlBorJ4WajI!9#Cd*!jS$bPEWuh|;CFEheCD?94RQm#fOra61Di28gp)NG3!m4L;7cwvIPjaV`GeM6)%9ArdL;wU{<`
fTtbk^p3y^X;gb&`=CKN&6}1nF%oGhQJe!e@S0AtFV&ow`~#^=BI$74MY;tu)w^kXTJF8wkk*@q&Gxk+#}&xibw6#ek`bv1dkf<fB(cUuus}mawDmsa#RR;>
?mE&xm~vgKS1UVzu4==Wfv|>$O72Qkq-(?&!L>=RQzG+@5e@Q;^R3$fP_gsomLvC&OhxUge$PQ%WEFQbC76cXubhG@KhRr<S1(pA535nl)>~&yP7~ZrK!S>^
Xg+=vN6Rz1gy5dLeY7#kX(XAM4i{hM$s~VlRjRazH?FDTo3@3v+TZbU(o4Kjb&O&%mnE+ewf)>OVh%6IGP`z=;TCV|$UNm#=%2?=Nhi6jkZw}T)H1+lVP2Jd
UVz{$IlLf@JITyb$8s@O5fgsl`viWOrD(mH^L#PCp9<!TAYT;(iLB4<ESy_O$&h*0(9M^Uq*ZH#@CKDV)CC^<>Hio@cRtO<+IlU^sRW_`7RLIu0ca9qixAby
^k5;JbZ_@ys509TXg%bxNC?LM{h5!tNk?$uP+$%$6r$*_o?UKn1-Vt-659I<{-7r2ar-kRJ2(SRKqNOe*HCU9>hT1|FAdr96N_dOzXaZaxF;JSKu~7t?$xwh
(7a;p7q$Gkx;+H2>t&2D-`(izIkgn*T?>EkoWG|qQm-|Nvcy(7B5ek@Fp~OzgNV1ARy-c{8jZLXGF*+}hD%Hot2``%4Ccea_#)dW5sQr{!Ha-Zzs?^7KK6;l
dWehol470LM0j@RrXbjwPk3bVSvolXih)G9En89yRQD%NC__Wu?-lvLY~9?08RO)(C`LOdLGfF*E!d>|7D`%|vC`wq_Y^dni|_$Vr)YNs(=QojOuG#uX69B+
P;ye|wy1^uJ$V+0<=aS;6@d<CRGHxHUj&6#m=n65T4OCPwQjInY-1cTELk1%uEb@+VCSV2)sM?n4?u^@Z?I@mX4$4SmQA(2dm^AO0t5Kv^g8UaZ~{k%`RhaN
^f5tgh>f@SW(C3YbI?8`TXG<ku-TX`qx1}(RvUXkzSF-vPY)7dm$RigCLVbwsCXAFi_uzt{4!LVBv9i;)gi*<9eo61v+@uhm`?0GSV9Z-GKM)~2&#kWuH#XU
7UhXKEj|2bH(GZ5iP8i)jCg)wIwT^-u2v%NYu`hg8&9U(boa>EzfMSxTEFw_5(!Dhcg_0`kU(I)0qxU|uwS5ps`LF|7G4b^nI3<qIFqf3Bs-hY7^&CI=2e03
?4Q*P0&>I{huQd-W(LN;@n&>71Ci%p0cZj?bVhtgk`|R7i>cdmZS5k~UGV^}s#GF2>*njxA^2JD35zX*;0T@7nO@D&pkC0GuQA`cNF65SwOA5>0s|oP(mX1F
LEkb!l>MKmZKHF9{_Xo{WZQPw40t@X;9W5t$3!}vC_IAoBEd~OJnKQc#7F+DF--;*J-HuVJ##7hq>7Y~vd(rH{SyWf&+q@WtrA=kT#5|T$rT$U33GrO+H{^~
06}#H<&Qyh>CFoROlBwj%qX(1X4EdQ5r^Aq8h|Jyaj6}F$YU%dLCUvcT|ve_R%T3_rcQ}Tz=A+m`>xsa6)2+JpQHy{TjpaR8U(~)WFBc0I*o=A!;XQi$uYad
x5iBkLT&(f#)f|{`Cf7Mp-k>$W$od(V<H=EL<!%OlX3bw!n@HupsWxpOikkgp5ku3TiojNP_R%;GM7sO{&3+BOD=H3@1aEOygrL0<XKUSO5xXPypo){06Hol
f_zToXiKgE)g!)9&?cKtwre>?(#wawP}*cs(wwE|E;!?ODOZJ-wc=lPwy&i*Y^sgO28txCz625};!$)7Qr>#lk5I6m6tJXwj-!gz;a+M3k-cal-6I1*xjL9X
C@jnqSC9pGwURAU0m(fFd)2Y6sg;fEnW3>a#$%Lganw}LSB<dSUxogCYOidptrdn+8S1l18;<`o8{O-Ox#e<An{(gmhk<TQCsle)hJ+jacL(UeqZ)E+`(--$
md|(DDskx+=(Vnqdc9aGIjj5VYHUt---&KL*^zCPAqH;W88jbx5wn|N1Cq!?miv%@U8Tbe!i0MYG^e5Mt33gHKuPKclw=S_BjqSAxnTiwzM|{!pY`n|lzZ?G
8MCl7^PXIxdXeXac}z^9@Fzq2R<rLMPEkJwK)6u1mkAE|);K+vs)8GXCbZX0;s8sa?b<`MLTi_-VNCUNd3HS<&9JN-SY29`Wf3z5SM8{90;z-+Z;ec+ad9}0
;bk*{nx}J`GiW3NiSK59I~4CqNxt}>P*J?HDRsXX7ca$Ib2Sfw%{Bnmcd^XLNKRJe7B*f`a*-uky0P`^AwIL+y<BC4b~JK$P9nr=%vcw!zJRTEq<e%ri0r#_
V5X54OE_OL1LH9yqCk0dsP}3JSLh>tWe<L#Msy3vSED?JV`kzs&;(epyS27mwGb{be_Kgef9RNu_<Il6QiC>Tk2x|mesI1?c~v%+9?Zs<snNHLKLU)M`1oLt
zFJocZqeQ%A!AYVRj+)_4@HK)ImLZ}cdmck_rTvDnaP>B$Y4_idsn8WBkCo%m5>_^W{^*=+o6&xx4LHD8#)xa>!IsN5IhoxgvqmlqR~6VQp`U#g;i7*QPPqu
{>Gu08Ds8gQK!EPlY07B;Ss*N^SLABov0_hMOm_1XDDosQJ>>)o}oAiN>k)>^ChAVbVch2M4KRB6D^)`@(Fr@u!E?r$ma4HyxR*^HBbH}%?KEQcS%g2MM<Ck
BMe8J7K7`Uy!qz-^LAg6;KMy^pv7hCisY+$p?g>a%ICnZxr;U7Lrvtzur#&5zt{LjMIcNOs#H{;3@a3;&vm8Tf#^<YgH3{9kPem&!FshJL4Huon-W#tltyAJ
=HJ{b=w!V@Tg}C9bMzb2>5(%yC|IWvoad;T9M>V8FZhRNBy{h&?b3lVm&1<$QOr+3=#{z`LDjA9P$RfxP#iqsc)jSYFzo1a{~3|)xJkq$%|4jF2=p8e{=#oK
8ba=vx-*-^#W0GU@Vi>6Xp#X&`$C2(<y&4k-kL~5U=vGLlbG^>%iz3x*P+cAByjyD+8xHid7Suw2g=Bh7~;KJ8OvI`iq408&=Vc{4qQM<&|rBI3l(&F9^m%;
oH&#da)LoDEQHhampvd>rV%)Ja+aplXj?#|dT&emv#~xs<7WidSt_doe?pbLrr|VUm}GORs{5GPni&1DS?T)=c~ah$7k87JveII6(7&j{KttG7iBYqpZ;Y(m
Go*&B1bReru14u26x+wrOOGLoOF_F|b3T<sFT)kGkSh9~S0wkwES=7A%?1UOu`}UL#S(`S-khh=<Ji>BAyYnvQT44|r|8{h>0IWQA_L->Z+HyGPBMTB)o!K#
{1DeRfQ8Eat28cINQSjs#vJ(*l+zV?2I|o}qB(|fRw4>zFmXMm8%B{#BSQ>B!HDp`A4ggsL&)d7HtUM-q4=gwq;j(95pRF#ku|I3=jehcnBU<V&T040q@yt^
?AxqpO_<?g0i(?aDDP+ID_z36pCNi}v8Q=DExYL2gM)OG_$}Wmg`>P&dhlQSDs_Ozt-<ZdE{ZnfwP30Yo#$Pu*9NK|=w?vJ81B8)@ZO`LK&;G4$N`X*Ru{kR
ZB{=>%}yf?-6>#C)=_WV03-N}vFC~;_<lEJxda@hV9Sh$^9uR3BhP{8txb(XdsoLiz&@Yin>V^5I_(8ih$mWC5R)Lxrl^Y1ZQ2acL35oeuVIKjk@HH3`pv?A
=AH5mmOj1qj}6xzWmO4J|IJ?XDs)}r%Zo(}Y6XVIm`<oIIlC6ota+q}U>_AesLd}_fSw^XQku!0FS8ch!n^^&M|@XEE?$=1)>Fn(34PZCU6pr}(^4R+|4~fb
LYWtA75sVMcX;vh6qsND_0O*7SO*F+xSgOjMy6W|C<O~w_N_uRi3mAPr=kA4=>lg;rZ|>jt~7#@NT!*XNvBB6sDn}ktJ=Uul-ecNL=^9cZIw7(RITEt%*=A0
c)$oolouuA40Sia7p*dMjG06ivR&cj(W*#-skdgs8}^nM?23Ew$nIGKt#=b*-psASV#&{y$!;#z{`Sl?Kgv@lzTZuRD6Dt4yP0nW_M!EcYNy8uj{*BeQlm8@
!Vf=>M$63P&dX@VS(!MMjia4ei$)Mu>l>;ku!(zH!D&dIJ9H^(1~sr28yt2YBo=kfnL7QuL2G_t`?Gqe$GkSi0Ct6QWaucu4UUx%dfv1&v`=APrW?lWBB`5d
D|=X<L!lWeR{O=K6yf@nr<of?raatU<n8df<^$Cc^95W3X-t+}mv0Gab911BkXO4zmn4U%pP{ZqEQAAFaj|H1V|I9-(-0+oEsgNqStj^SB9L}dkVNDu&fI90
m~RS{{UzWD*VnA&Q0pc1eXF|XOhX%}@mq!gEOs3OR<%<WLcB4|GJ+v^v7eVC);*X(5KEqRLhyHSUOz88bXwiU$p~nCAQ3f=UEV?KRm*xN4ilLmn*z58Gi#Kh
=cL#HdO#j~K9{h*IxjD%^-B>HJ4^4aUwOk;Ae&!tFG_eZU<NQ|_w&nqK%}|B%P2;rdNU1<(b_b*{d6qGdlPjwm^F>Hasu4y`(<`><&rbD$Xjn`p+8MX+;D#+
jL%kcSF{`2QnL;mGx<AGx1kcnJZ}A*%IR!D@)XeR3qSm%xfEgB=slLmXWP3Bpk)*F;(e9>kV6n~jx{V-5o<Wj_ApFJI5+E{?X7m~QY&H@Q$%!XJCu<3DYo?W
jSp2HT7w@~;b+SO7j>qd$ir%TT^ues!TDlohui1bHaze*f%a9F(I6PvbZ|ex@$;h~0#T(#FfVV-hMo7z1r2O8yH-Mgya(pZANX^hy&Qq~EC`p*^grYxnI;^)
BZ`axwzI^Mf5ebVQQ_tIrtZ1AseHm#mb#fh<tdD;wM}(PG3n%cDA7-sW?VY>uG633+JhLWChC&CI527LhFpXzVS|@RfI~q#M!yNDkJP%NZ0@xgs%t!h*hu<(
F1;srf{v#>PC8Biuep&-WCNmVxWv!us4XoTDNaEEmc7d&-Nzg2`U!$rxR5bj8X|c@h<azmFuxBiNpJh2<ZP<v+|V_!jP3W@2~G5e3%1QYJIryXpa$d2vigTK
(rFk=j~UySE0G5l?hqjn|4j!-NcZ!ohpV3}C4!5MkvBLc9!byMcG%5sAXibNbPN5G){x*^u>oCk%vdK25*ZCRQJhT`ATHPMKR>|l`l2#^Kp;<)!K7QpyqKBp
#dJZ;-)VxJKUK}<f@0Y6iFoPi#41!1{Fwt`gZ@DGJk7FMG=)~RQ*m0t@^Y>w2}%5uCXi`jF+Z3L@<v;47>Hzn`Z|_(a`+IjMg!vXvTJLtm0EEtyV8FmEs`fa
c^VV*&hnn|>;twP#zn~6ml9JAwObC;(&1J!MPdqN{npvF?_H1H<J|#=?7q4cvxys>oX%cSN8Q6G=1}58oJ%GQb-#8vDAFqjUjkv8**#J!8Rztk3t5UAF$jAh
b1_j&T_*k>fw0CfoxK+BPd`Is^#}i^+ZgK?zV!iDk*4=E$^&%`u0f0=7xxTqWz}MZ!|??tH5*fJ&YF2Hs7ab@qsq={AlDQ{2KG^D<7IOq=q!iWs;QHwJRJx*
Sf~(lxfROtW4~-+Lr^|F=op`~%f(>XJS8W{YB-gQ(3W@fMTLKlt`xu0THRf%{P?y2F&p=Hkp5LGsDpIKWQwI6G7Ixazzd^K{xA1eO*OWymc3q>Ol>TZ-rmJW
Q{~b4%|X=}%jY#4Y$gDq1N0C`Lu^v;RN{CCQgUz}Ogi-W1pf*$a*bS~SE17kAE@FRu8pOfT&lk*o;c9)kC2?fFmvdCLdtHv9O6aVvemap<0BU*bKh>qfJJRf
{L8BPOt|D@_s9J1zBbO15G<LM&`l5K7TtxXlwK5E{z|<k0Q2N{91}py#Hk}E{TLMXy&5F?R^GMg4A@b2<>#`i0S;kh<ttB$SedY&Y?yccGTdMX>x_q=W@#XQ
QyuSk6-g?anm1*`Qvo(~*Q8x87!ffzxYCC=g#syOTtWqXzqTi2^Rz)L><aG52x5g$_R>qGz}Xms=w#P*;gTeMf?s~pA&_I%Vh9>t6N|{fo&JG$jhHp)L=y{=
+FSrlHA`qr+6g*h`{I;9;jtzx=sBiE$Z;OT#W!(|QY&@Q4U_TJskNL#5{Udpe&UA;hlc-Kp$}P4(IyI>uD_e+otJOZ+tuu6_D1Q_-o8ybY+8k{2-tA4hct9V
!{w!B3~nnE&OdprQuXa)$ntNOS<l?>{8u4PR>H%!LREK;sN3bNYpTX!Hzk+qc$6W-N*`$Wn)3=>%q22yO4Hyp3|Plt*cTQsm(fybn5$?j0vj<!k8PuXTc#I$
p_d-1t_d&|sMzqV&#-#p;&p5_G23rX&HuZjc<N)Cf;5l7xi8e{K_D(tx%R7lLOJjTkrHez)WgKtfPh|V+O1LkyBHc!@OVM`6$f~w1^DrBdw)LX`9vT!Iy(NC
@RguhYMeWcP%iB=d#415Mn%WwEm++7t_=*3@2@3Ff7Xvmvy<b{Zb;;dQ&&$^x#y5xCr5kSjXM@nV~0WG<8^bw1uV1bH(sXSU|~0GDw~^aG6%#y8{F4Dxn*tM
FsBU~g!VT=h+Kv<2>wYT)iym^!jYuXboWE{Jp1POSF!Q%CvL3a)d=2}vs$7n?X7Mf!DaezE<rj>GE36p254ia;}xjtJcJaVr??JTz7Uq?|KR=TkALbcSV*C#
`w~1>WCZ(Yve%v;+nZ3FcZJSmx|AavlvFgrxe(0hxLL{w<p5r~)oeBTMha2Awl-)iyLaCgz^t)4De*iI4m5m68Mf1}rdOcMSgmrsYLJ>GI`@cyU2y887T34|
Ah<PR;Ut0w$F4HJ@AEky4qyps>yq(v2d7(^IaMRNJ><7hj8n$KI`OV>f0@Fq$hdEHZ7dm~)cyA;oR{zpzS3-<5~q0RQss#n^3COojz)9fpypdiO?C>(ZYHft
=A@*V)rj8*pBUOrqG)-faVIcfC07f8-jDLE+zl3;sZ$WGc53Hf{WLe7fdaS+b;i1O0gQjtEG<vG!4EC=bE?$oL@HTP>nIEIaky~q!r*&eR$av{qJw|e9;6?0
Y5Xa-cyWPWZ_ZNs+`3Udl<HGI6~WE2Is^_>XN_(u3?W+|Tzadu?CYb*tix497@D%HuBA}}jX7@WiMV;0&e!z4<jBNfA2AGaPP6^nu3`OLK(G8zAQEk5lQG%;
Hhz_V9*G71)RP!bRKLa)Z9|VRi$5^uv1iL0`>l!EfCw{X#IY?0R8<MfP`<Qtn$@auiM^EE_0YTh+7Eu=dDb|pzLRIf-XH-?mYFBh;M|yeFWZ(&nh!>bj%4<s
B(cGMCa@-wsxBf3uA$2&#V7tWTFOk^%?F=gjYf^}QGzNi@w0#2K)foE{2<;~TCJ`r8gga7*#g*c8&IGXz<-^#r8c}wgq)VBF%;2a%-K1g(3222x+BiK#~Ck6
DED4tKS}+`%Ea4yd#Fv^FKlWC7!qKyF~+`#PTdw}ZPL3D>#7Z*dYv@a+VD9NYXod{0q=-@)gi&fD;y9z1#Ue76B~$-%0+6`IqAd)zbn;!isWC^;Y{f3Nc0H<
)IzG6Ua{-A>)X}Gt=j>@l>xsbWZfVg->bkP&&d0jGbUHDQw6F0!kr+RAPR8n-uPqlT|;4|na|y_loL*aBm*Ih#QC3nH_QiXi=ijrxc$xAi1{!;%pq@IQrmV!
3inLZeh&S)M;amVv0;B-+ref^Ff&%G_e4FYn@EO}gT?@Q+-NW_s}$b~M_mfAT^T$@z)q-e3dm*sxEV?kc}RqCcJCrmek7}_e?Y!(s|o4(@O@--^uRsVGsSzx
(1H{TVIxZ`Q5o`yL1&%TwS0Uc^BnjA=nxvg{I0K5VN>ecbRQQ%VHYPegYwB;*%$jl+HBwthgpD^1;eSF&imqp+D!R+BmeMee3!2W;+k0sA{pfL>Qx8yx#W7o
5;K96A-Mu<T@W7ft#zc0<V^JOukICAcP+i@wc3PLI4b6s73}KtRpA9jNuEa6e7S!f`0M=(fZSuDj8ntNEg@XCzN(-Jp`OeJanify@CNTaVzSHu5iDO<Y5$3K
6p*g=hq10mo3USju6F<Ev2^v;eE#>3y=B~&Gt&XmAzs3YNRf8Hxlx*f^|b3pfDu7);yHLA^3euKc(F~6_o@jeLyk=Omctqr7_!M;ml;bd)=3P@z~6>lt!M;C
4O!MB&Aoc*j+wj&FMuohagtQJBE=pjrPG9yz2<1#A6pM^jz8*3W$lbZlQ2y@9N)U~zA7?Np%jiKIiaoBC@DTuRqWb-`trqVfbKLn@JSrJ<}t$n!@6Ihn-~%b
uo#<DB5mxw3d%6>v<FmSm%3gLx>&H28AX=<uL5LEQSN(e{kM<m03a~BKgS<vs$}Cey_$on_;0?;qx!REvu+Q?0$&(C7Prz60eN%-#Uh;}&m*aAN_-O{LY9`Q
B>lXif_2F-_qq>Y0V3e-da3KTR&F0Ai;F!7FByHKNYTSQ!t@uLpFV$&*V}t6p266$oP<`n^mo{l2!{ZsEKJt~rr_zDh5s>6n5~TyTHW~bB^ql*v5)O@P(pRv
E!0ct^iI;53%>tXd7mAg6Q-5T`^jrvjF6GfPnZ`SLjnNSSE8N=uM;f*#f2~aO?Vcd&FgNx0S*h>f$YsVgYw4+c5KBM2du#$)S9CrKLNRW#`E8BS%6sNZ$rS5
&-oy^QAer?KkKI29FYlQg>;B)9x3(G^?k8IsR}p=gASEsvSoH>nx@7p-3RMQMboQvxb1<Td$=qXZraWJqM_(ofztg3{dp^4&&J7&4XFzVLMI~eLp*y&^ECkc
eh8};n|@ErBov&X;ow_PG$1!#K3vY=qI8d)Z5NMUr(BMme#Q)ee?{98Mt;Of)-G@9<{xmCZGB21tn>zI7XI5pZS{3vOW?3Fg}qCT(ej;1n?VsqKFc^xL?5jZ
?dylRhv6r0caUseKn;K^M#6L8s-XTXxc$4(ji5=7nNI-_32g~}()<XN!Npw!?ir6piX<63b7)FEi8<sS?>1MC5d=(0hILCw0?TUN1!;=X<nD+?ooKp~KT`_6
h9cr6_Nm*adaACHQ2}sAv9Lf}dA1S&00000&(l&U%<+_Q00FgY?wSbzBcN7rvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/divisao_r4_flavoria.csv')

In [ ]:
# listas/divisao_r4_lefood.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/divisao_r4_lefood.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;GF3VkX-;W8cNL}xSP$f#0`*|<eKar^QU6leZ?$b3vsNR0$c=dg|TTJze1t>-L(C2!;vfPK3uSTaI!ie@*Hz{`DDo2
4x9(I!n&3mcTetINjmm^dM75R9~)J7wR^Z9<JW#Oex0-F69)mXKZ=wHqQ@7FPl+n8r+<lOj6-HXU%@EHSZlR1OGgQ7u=ipFZ-N)3U5&haau$1*Tl>2pRLDAw
{;Bw9$?h^~2gF>EaCZD83@v(sEYd2RfM2)p26HSA>GCIV>TOZu1sCwkpN(l#wCg{gVCV|*Zq&<L78E6u*rpwFO*!P;KYJ&4JoxH%z<!ouRIA1zQj$cgwm2$}
!^axxSZnvF1vLG!#wl2ELCD39fq1<6D+PPA$5q^D1q3(1K#^gMaY?Ij(eJY6R|mViVStAu#Dp%Z2||T#n*H(<|3+9T)?{uJoOEjq!Z*Nl@7HMG@21;+9={nZ
;_Vv1nz%Ps>8$+*qUg0}0(0{i_)qC=J1!2mQ<i&NNHy#<5qnSz4t$7L{@jaQ`xWSZ*1h6r6~yxD><SfWk!?&1DbyUTz~H!U3YBDqH&)^=4PkBV4F(6m8n|l*
@|hsWTu5I2vF6-ROVhNiyFStv#C3P~lPW3~9VxBK%|>{u`xBp<9PNSuEk4bqQGy@9+hA^C$P{7e+@$p9hsS-3UB-YJ%YhTPO>7~$M?T~+8u~rqWRWCJc>#IL
8>)_y-nEVpdm^#7(*w>zNcvG+c(-YN<KI?URvjbT+9tJ{ij~(LSG9nN)66%gLgp+db4*l|zlfJjRJsltwj4&@&;uZjgV6OpX%(%R^({r7Icy|*Q)M*Es&ZLp
;1Z|)xfpJGsS#PuAL<(oq7O5kobD(~Y}+o+40UKw&qS`Q8QkldX>J_sY(w}aWas_g)%+e{A}D--mgjC%NvJi!aP^4y65{}84ndQxS?)ng_xL|wbT;+Bo;{}a
OjQ%@wD`SOTk=&7^Mi`9ytXF|kW<;VS5-8#CuxSne_D*!p0R^`oQwudB|>G=e_*MH#MCj8H+P?@X17f<Q#+CK9=mLVl>wQb9@62D(4|#eBpyc6`CtiQ7S|^m
Zt)^-41l_vD+2bEIv3;b{$XrrN2y-`iE@pP@W{h7=y#yJ4rJrb@u1;r6Y-DAMnvSyQ|s;xOXXicis)2kx1cK93WM)0wXyhVbq9|dLdF3N{F(;1mq1>}tcxyh
Nf>;lZHQNi7yLfMh$Hairf%-O+MNpLU6LceUpSaUCFi@N;2GIM<~}86NkH6DC$LtX{2%}JQJ17t#zHYlyd>#9qbhhby2lPP7SU$_E5LC2L}ADvI@5X8b}p}w
0yh6~u-=9rr$5foaZFW7*yo@ESJ@=j=s`MRhh!1B!J<iqgPzG|<eU~BLZBLCi2nLm;Q$3Va3(nGTP1yH+=}fYK#xC;lan2O*l#%o=UfPs4a#|X!$zbPS427l
EYL?yOq};KiygtXu4oO|m(be5>f9dtCHcpop8N(>eavOi_5$~N(thF&v|GtNn;iT7w;7}`AF`PamiO$mJa-rO-$_Y09U_TRA+V|nn|?@G9O89W*e%Cr;6J|~
K^`=<7R~NB>I~%RC#_N11Ru-4@zHbV7*jjNHw=~s0KMCN)M$p|)9f@B`{<`xs|PTO?H9i&OzJ2qp$nXxh_kKc8fmT09_sb0$^y4(&a3k2#Pk!`MCF{R<P|G0
hecIunLORT@}G_jGTd_@NdRXuNSLy7e`!jWB3E#p$H#_+XznN$s~|>_hZrPjqy5uja-Z$NYBD(n24-cu8yohTdaWaP!e$>``HQ@(!6ab)%kZ~f{3AWy*8&wg
DPn!w0G;C{wX|gZj?GGSK|5%o>}#&W{8G`+1<-yX%0{ZhT$CBL6G<r*rcGGIaP&lyS`c~8{1y=IpZR}iU(xS6M`x~2&Jxa{-CJQQ%V6ZV1bEZnw{adI?1sN%
IrbeB!q%fQ^br*wEwY8=#h5pc<-vOWFfGzJUonQP9OB101xfGZ_nlm2Zg46}l(zhXKp;jYB7jE~ZcU;oe5yEg!1<LtkWEx9-S&`*nW1JS69axEkaz2?q=F|E
{v)grB1%{x<bg@3Do+H1F~L}LatrHHH{bJ3xSW0t<vw%cKO;+>!DK43l7D@LG4vvFSV82+kxt<_J_GycC^nTDt*&h`aby2D0koS<13v$)UVZ{|XbGc|p5$zO
`{v)?^Fl7;y<=Es*RuQx<cEKZD{UFA*vT_dIN;vR8FJVnoB+Ie4m04y6F+OW*TRrqf$M_xs?#Soix9jPywBnjP}|CkeUwzoE6HQ2!x<gFnzEnODWu{WWukW$
3&vu3eL#}jbDWmunuP-#cdMwWu&r4Bv)}?eR8rglT#BLAYMG_{N<V4XFGxWRZUng*C1Q?5Yt*)7>y->7AzrpKF^_p~J%6V^IpnmB1ZPKMk`!exREJ`qkaW7L
uUg`x7mikNo(#?PZ_dl~fIuqQAPjh)h(}+KzQdfFR^`?E+&<kM7EX4K7eqghZ5kTO23-Vc3K%LK?xjo_!vd8s#es80!_omIbs5{Aa8-)be(8zo=b4`rslCZN
%dmfVZ$N1H6OiYhpD9Yt?fn(b#yX{R{C^7-VC!P*8n5RI%u*0(DUHKFTApYtg+LqR7+*(dv3iz~C`Sc!OV5nucaVS=rq4mg#REHde^plnwy{&amMQZ8M581j
a0<=xINesWo_g!uFy0JjX=ph%hgAR`Z_J%3b*z=J!j<LghBp)v!Yu`ih4=nsdLcSa=`cXg0>N<`Jr(`X1QyKX`a~{U83UcOX+1(bp17apHvVYBI9BL6d!KoY
7khr6$*>l!5yN0G9CF=|*28ti_4PGHcqA+XPUi}cM2LF3pP<5!%!%cF7Hc<0QozoG<d@n=4DZiK>6#07K*!jvpX<%}uVb)gUSO8=r>!V_i&=F{AH?{9MUDK|
pV3)o)kI^?WSSq6kg%i}uj;}?FCPnL=w98IgQakYD#tNk%!nSjH-_qVe}^%^R_pW<HmeKfD~2J|O_5`CqO_@%+H{hqQ5o*}%tbUf#R@fHUlF=eoEHGodg`&_
4&uhPk(|Rsp=S%O6x(ml>7cJv+j$vHVf!T{``yR<{)v`*OHzuPyrMKW>rje32E&I5jSJc&CQZ-WZh>+dV76~Q%K5tb9kew+U==$X=#0OQLw}?$UrGdZUc-+>
`$rWt|M8&2L~hB=tN2_&QxM;(Z#`Z|_YG>p7(WpE1b?g&YK3v9l;k*CXXo?uy&VsFp@vvt#Owwtc*0T2A=<k+Gu5Qz_OU<I(_TD(a}3s`DmD$*d8U6k6TYgg
`BeKKI5<C!25C6_Dy~E1U58tRBDhnlgpNOY_11xqK;fQQdpDSIJ>FZ0EW`6oPQ8hljl(Jr^+{aoz`;<w!kgGPV8$V8ML(z$_EWPPgW1=gK*#x~u+&f4*>Zkb
6wSVeyJG)ZT*ipe^FqJ#IW&B~W7gUmWQ%@Oz@wSyarv%o$#pix5*J9O6E~Vk7L?{lw<IFNO=KiKB6PBYPM`<Ma}1l7!40P-hcC+VHph<p=El+5YJ!SOHJmbI
lZESpPs&-f<PksSHr#ftCurZlB%uQ5g~~gJgLaP3X#xESW<eWc5|Rb!dYAa(BK)nw_oN-X0Mvuh4<Aj+73#p_itkDFhAu<$L!)Pqw7w7}U0eg$D@SZs2e>v;
{q$2LZ*|O1ELXTcm~|sLh?vTf2iEd0jaF3sGAS&JbOMx=sZ(2-y1Po`&<-ZElCa5Y+~%JNOeCJRYw1j=(l_+r56@c1Q~nASK_!n}MaZeDJ959A=Iu`&m6r+h
qhQS4jU-tiz1!o>5&eZ0zG5E2CD4U*NYEHjA82o1FU$!R@4H1EZ@Jc`NMgCFCz&)l#}!4Wz|%`D{yq(2k0CIDV*9@GPEC7LJuT-@9ekRWWLqL9O^Zm4xmTw^
bOcB_TmXqg6c>X~O+7fO%&v>!%uW3Cz<3-;D;j89<;R621Et3zWRK7;WnNS_)XA!ty^Zwn&x==<2)8%mr(!#7dts*7^$aqGtCPOOaWm;ptHs!zcrpQqC@1v^
#G>RDrd4T}xT?|>r4bs_cuL#^yIN?9_$%5*;)WD`EKOuX0RWEQj({hRb!nU+zkYHUp`IGWn7OGWHx$#jrd@dGpu;-5@72g`Ex+$!W4Ct*U*NJ*zo=xCS9o$U
Q5)8=mp0OKtiqUp^$)M^wY;DWu2T6#;V8ZP@QNxVB=+E1TyJd9m*9Jvk<wY)cw@1jr|g3}*BQ7Og9Fz#ywu_i!3<YLyS?()Ns=O>Ci0iM4M-ts3yeU(oTUuz
K4CE6oo_f3Cv~0&bNx$WH^5G7&U0MyaKG4&T@#$E5Tq)P9U+9tf`m*OJT8$y^82yaO3^8&+Y2f&pbel={_q|xw{Aa&zdPh*%vp?A3AF_<=KLaqJW&XeN|*Ud
<wbJCF4;ocRa(iJe+4F9txrTA8W(%t%ML$bK}Y)gFDx)e4i;L8`#adtN!jKA4L4t4S2-I#00FEU>bU{{nx(+|vBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/divisao_r4_lefood.csv')

In [ ]:
# listas/divisao_r4_mensa.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/divisao_r4_mensa.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;4Fp$&|LsB8cNL}xSP$f#0`*|<eKar^QU6leZ?$b3vsNR0$c=dg|TTJze1t>-L(C2!;vfPK3uSTaI!ie@*Hz{`DDo2
4x9(I!n&3mcTetINjmm^dM75R9~)J7wY>kRM9dNEoAvshJ9cQ8otbPUZ&_o%o&?Z>B{wbQgN{wP;Bsxs3myQ1bEj~v8<}<0zJzr9YohB6&doFyX{6r&knfRV
VdP7k?cDR1Bj-}IvNfkoVFwZ&=hLc|o_Cq>@~;jhD|Sl^1C~RA`R%bl0dgRTv{L!K_Y{HoAXJO;y-H}fZw=i!OlRO1F%qX1d{;lZ9$PmL$q{aoz)+<;=y(k0
)kc9L0u-;$U(Iy;2IoLPC%i)0nvaT%6QW)ve|!KOb<}=<pJ{;)<HwUao29_)shEX|5nn3iV*Fsmh|~mfFh3nYvYCAxVZDHg?>1>>Md@3~(`P}6_Ohj`<<Eb?
y<oka>@C%0cJj-NEYV_g4R_H&EP2}WERPpGwfss;U-*cLz{==^UR^*|-*;th7ip4l<_U-B;xSJQ(Ne!2d=r_WD47PLnLKUHtPCnAfJ5#gtNN#2ek*2We3}kE
!=glfmxoB0_B<ibMs3SyFefaLapi6fb$#-AEwe{*XOpIhaf;P+l8%F5#q;(>frMG5Ie6L|XMy48)`lMn;yl^IVAoaaIr}*QSE@yV78-9EP5qsvvdnGK8M%C(
c@P}&WpdCIm09X?lWV4UauFs^Jww}bKF3QgKiOWmosR3k{M@1m-~82K@S+r_CbcmjC05NIC0NDDhnbJ6$ibLh?vqYxCHRN}=h#=r{O~HEQ$qB?+C)|Bhq&L)
+vzfQG>wcFBtdWsbdhRNI6jbpP?6OYmveKjLPK0Pxp{=yO4VY6zJ$58hlBcNpn4c}6ons<H;oSJ$i}idb60>#2FX-VD9J9Tyj?B;sYw*nP5>Dt00Ha;hgkps
=^M~_vBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/divisao_r4_mensa.csv')

In [ ]:
# listas/externo_flavoria.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/externo_flavoria.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5=d{0^)m;EG8cNL}xSP$f!ssd@4s5WWOrlr3gU;hVGuN%4>_a%;*@>!T!hp&~KS2RcVXXifz|pU?X<ej=RYsNHgC9gZ
^Yyj_@w3l^xQPPIw;tS7j7!DT;i*-LD=CgW>TmISg+LT`At;aA;E16tHJvpXoQElAO~A0P6>#q#QpshLvAfmNV?M4YwQp)!i9($3hrjK1d1i?CP^}K<x?@~Q
_U<}k|FCG>1i?@oUNb!>Z=HSr2pGCZ^&s1&j_XWHr(1>F>$hcl2?XsUDVg;L7w8rvMTlHukOURZ3Nef+WAJgTA@Xc$ppbIM9w`%CkyKXG=WZ3duqfbQ1z_g`
N2W}#Ui%oQe%&YTzlOL=Ph%`|Zq2Dm3%W2wqX%~!;@OW`NchuDTF`BJCUr*XV$#Qcu;Z?5h8Ah#NLWAV<TMBJ1(15_b2}_{&uDS+uP@I%DZe9%%k*?sd9Mcc
?qGL`Xt@s42wKQO5(W(R2o0gr&g%~-{E|Gg#r;haP)P@^Iyc+UEJI2T*SKP~a6ZO~#(VQSLBe6(Og}Z;2iQW)IbJl~9aDhwjADehrDV!{3*Io`co;w8PY{{p
p6Q9Z6@DYfuP@xEihMHfFTwm~@-qDgyPlg$GjJ^Jz6*dmGM{zu7-gfQHs{J&`jbx2V6L!HCu(&YH8$UqsPoavKkF1B;^irZ<uF;4Mj<lFYB3eAbB~<9JbU#Z
5#MkEB~+p@rn(y})~uOVUJ9`0B0n-Hi&nVvyGT!sI)5l-AcuEEuLJn{OE><CFPd%k=*ApwXx}Qt;wgc1{pM;!*I5I!kr+8CoOBm;Er1?;RbACx4kW54sVU)S
ekbdESzz*sSOk{Q0DsQCQe{LF!&)?e&1hKr|2N-Adm%%I@sy3+z%utA;aU8X4KHs3GjR0Y$jPSHCM|Mcwmw2vg?Bu@LDj))(W6}z5lCQPE$Iap|0qfy!D0jt
1B02O{4b`6Oy6*49UtnWekNUa^TRc+F)R{pWUA$#qk)M*Arj4rUFJMH%RTXNL!0M`B`kW2r-fiMLgk%ewh3<DyDrtj6uYi@Z=@A7?4$wZ_`Dp!@?})RNB1ek
fV|da_BLQ%`l1O>KJlG**Tkn&+?3&{o$ZE*i{y!EO7cEN0U7mF_OTYuXssL9toqTIv~Nc$j2;HU%OG3xDEuCEu|v8n+uHX3^d)TGCCc_M_4IR~=R9t(;jSJe
9ODMlp6B8TlFy};{<VtrB<njKZE$R#fVqUzb1LMLbf>9Qap{UBqbF`*GXkRqKO;I9Q>`2L0%BR99w+!eUoM47+l?l;_@qFM!>~(Q=<*xfMgof0y40D)X=me$
KpmpY5EtpMt6v%s5XfEVv~S;FE)4OBMr4~(_@Z2ASjiUjsrjW^kNexn17Km@4O{<jn;$t&oUsoUtTTgSaK2sN_tsDbH1>&5Rq+9JjRpAO#f75^GJ|L6ta<=f
2}LoPw0!A<>jp6odc?~=wE$q-6a1{Ms9JDnbbOBfct)#qe$azl_@K^-CO5xGt{X40L-c4=s>_i$IZx|L$k1r)A+J*84}8%g8d>z&(a$(-gyO`rhSJ3v5DE_C
ne8Y)EfR5AS(Zn_t%?RsO?(Y0<OSbC-po4ak?_0y8ON$m4>Ax)=2bVp&UeBnu~7~?2h&;L1s`*SxJ=NZ1(4<*wE5e{DGGdLZ)U<L>}M?EWhhfIep{!|YTj2M
g>Y?jj16)+^ZB*CMi{C1R~W}t^a2s5ITwE3g^Z|2FwZP8-Er;kj;Ab=J-bpR!jD4%l6=m24gH0(qiY1=4P$9_Y6q2N)ro3RCD)dk7jzI+*On)kI4&PYQ%vTo
llgMAg)5nhRX})mIe7D$OWqyK_b(9Fq{X8xr)V!y2ptkY3i{j_@aWu>fJ3f!DZRiZUd;{|#M;pzG1FGdq4se)%Cj{>GVFZw+TCjD&pFYLok2syP`tSp#6#9^
__o1MD_ugIY4KNFY-Koxi-_T~Oj=g{OsE6G#!_owerrJqp1rCw@e0$)LH9r8COtHn9fq^ciRIu(Ru*8%zgHtIJNT@xf1+ihz+uZea5`Y;-omSe85A%QF5$xd
l>~BbrxPdMi7bJiZTp5F28&<N_C@>-ii*bw$#HK#XoSwnyWyz1F>b>L6jkmGvZ(7s);wmcpUJ1UL-%~;6&<o*aJ-O4&Xksa%!=9wW_d3VRKPVaqg(pk+7!J)
tz6dH7|MV?@vZYm?>!-8bPsv^S;{H%l)`Etrp8J~Uoqo2ad8VJ-(jan;OW%Pateoeq@qJb%%=onH0Ubd$Ji$mknJ;4PyNC>oSc{A7IA&ZXQnzK7@ut&jaZxP
0!rdl8Hey%a94aad;3Qo?ER|)4HZTwpVyiZvqz4^b|W;%#wy+W5&`Xn0=sr<P39(FgIs`-(ZZ9F@pvLr<=a@3e@SZ}&uX}16c5v}o5*_*QUTo6(Z@HhVv)^S
9e?shQbvrFf+}^{XP7B!B_OJBJUfjcp+=4Z%;HzDx2!ik({UT!L(;|etj!Q6!SM;KzU&qkaEQQaO);u^mrRs*dfj1OPAK1T&ht?eL{N4$*>5zSuUO+n0ekZI
5WH;}7+Wg#?TxjUeplWVcac3sCN=tZKFXy!e;=@xB=`L!M;IWiu`ETwMbg}w0&s?HGdl}kChNF#x7O?<(T3q7Rg_rJkiq9eI(f^%BdP@foM==H+bn#S-GdcO
{|F@_sSP@%ZDe1oSL8$H<Ey}DSqV|kY+rloC8w3(*#4k>&4h#Kht>h$2qe<?@lKWbCoX>;Cm}}5VG@G7veAKg`{A}uSW`xwNww){Nwf(5!1=H%@vpN+wee1>
BGu|3!UV-aTz2f#xg%}&Y5J17$_FRSIo!<kN`ufu;+vB5?FW&o!gIC1-4ts7D+b^I$;^>V75BpG3_vD6hf%LxQQ8Y%hHQ3_bRIgIrU<S<_S9Qk@CHvI(FiAQ
Duv3x^KR5$_VPLya=U>giagbnqk^e<^i;^5!ihVYBhQm*;7@NmxlBlY1(skvaZUuU!JX==)2C1f>A604wj)8oF`4*p^XelB6sIVV*1MhNyF_9af%rx-qZY=&
<X4MptlWynHfn(t<2MZbCmkvDQQbR$<>}KJzqly<@X|(cgy)$_nbgibIVM3^P^t*1(nHgpOoZ)i3sj3f5#md5h$6bHRdnH1bhVae0tul2t|D>#ppB&3gS&co
Ba!jJM^J}HniS}z&uQdP0+1ntu&p@nvbXpu`mKHRCS}4k(rgCx^NIh5K`fc?*acVlIp<W$Ov{FQVM)Frh`}NGpWg&L`Jty-N_I6bKQZg{eq7}K&X_N^BczOF
Znq793Twr<iF-)e*+lK^Gr!})ZffdAwcEOmMhnd{P)Cr~nf&)I@1&j|!XG|iYSj4p5f=HhTCXkI%g<^yD++I~iu3~41zX52@23Rby`Z?C&pWeB{f}B<YJe}T
Y}B0$-Aaz4=%FEYwusO#fw$cG+?7Jf${hZ$c+6n7SfxLs^B%UCcaIkHl(TAcCU?jUfFU?0?qLn?3E-=&^)N9pHnugrB1<q8_qw)2@{f4=HRI7pzB3d!knOnD
9Af_HMN>$|SJ@m3>|w~BR|`jVV!h*+SsUfDQCs3gDY9f(&8-Fv|M$+=uJatHB>w7ze9Aol;h9RV?f_W5qpfRf$o|yF+s&lnJ;yn{R5jIk>eeh0uBUT8(iSV{
Ahh^9MhrYXPm{Z>{%^wwtD=lxn3GC6FiRu6x|uQ`8GyVjOsGs;m2zqqC->!R%p2~%TF~WgHONPP>!+|zQE?@;wSwYq1fT4j3R)T|^O^ib;?jgA3^M_VV3s>f
8bqqep2P8k@=4H;Xu!4?6W?@c@Nv*Tuxx@Sqw>sy7vc(}2LC2Q35&~sbf@l7Hq{Hkyf3FzrHO<N&!?oZs4__n?6UZ;a$-sl1)U8m_k>3+A4Wf!DGJ37^?{&K
l1&~Zm){>spD)SlK|ZH^rPT1(61o&i3QoI&Yq0fk4|6=!lg{|HMIf8_aVuachpfQSqx{&Z38gEH>wEaHmE+ORQ<QDTR!(rqx_ekMDF08X<lUBqfVmiTklx4r
wPL;Ylrqp<{-YX~kTFoi<TO_k>#Or=LK>3fo?Ah=THpCAyfwp`nsK1!>~bWet67&J$PkpmWSq0AF?PV}!b8A;?uEm~b9T-TF`{z3V6J$nbvBJlJ}_~lF#1va
(oj}ZaGo2)6w}!M9Z&kF4%aMN;+}y0)RRzw%@i^t32hzSQFD{~G$t!<aCta!DIE3w-iu8d)_y~5q?S)=Ix}a==RH?j%LrKUX-Xg7ax~m&%Iw8dd2DI<eLXc>
*ahdE5YCIQ)hXM|<~b>P&Ca+ld_M+5rPpO{l2KqLl0a3r<id@Kw=V<<VQdW%2sL7T`Fk9+cn-H=ARDOUzk5&n=WZWNl;(3qMWw3~ejsqM7&hSZ8_HPSunJ*b
$`YWe&Yr2}1otO&obtyoQP!$y>czXW(`(k{ktQ)m0yELQop*{<RZv?%Qd&s%ZG~vq!mz^qA3i=4u<C(9pKTL1G0vK*+6?HeNDE0B_vIb!{o}_VVkrTLA19m}
DN0ye%sw7MIT04n9R9FMvO`t;P{F!MSctO31>d;B5llbk+=UWjs6d^n;CEzmJL|9+!QgMfw_1=$oQcl@E1qvab~4${tXP<;ku9X}VTMI?LrI>x{cUG1Rx|q6
PRc9fg_RxCk7ynOa~xm2w^4P}8+(_4G0VBjp{Mn>ypO!Xd`;4DTkd~Np!G~noZ2~HscO;PM3W*&i+=t)4n9L<>+Y3Wi)pJ%!vU8h&mdO|m&B)k1PM=z8Um3~
TE_qz?!A16L7@Jxv>szZa)M9BKQO2$6I<5%4f1t5^`c=~Zqs&vh`!;x*Kr;YMJmEd!b@gMET1mP&8+nW^KuW{mZL7yqZ<}DVS+Io?lPS7?usFOzWn!O=h6Za
R2N*ag{|WBiVPeBy9F&M8^qW-6&t)jdem=Km&$L7P46S<$9hXs;r5WIpiJJgm}KdwgKUnQjJI9U5Y4zejw-6|c<RFG4*fSm&q9=j9z0xVH)~_{h};>*JOQgD
D~f9v4Ok$rD53oSn5PdoZ)2gp=PPQp@HPx{mK)o=!dBf_8+5&A`8Z89UH~>s2SGFrSbScTt}S38A?+ts9tmEGWRqh|QRDY1e7MU7(EX?MVl-IV*zd1}FM#RS
Q?9x;ur`jPy~5#Fpi&j@gQu!<u+yA<j0AzMS?@Z@gRMcj7x+cjo!4-81i>m!B))EzX@{3n(P`bz;4!cgy4fU8mrZip=HXTQ1VS$B4Ep;~lm??-X>w@mBZfw9
-94p5EHFqJrjkJJ1I0W)mMOt(B`oY9+J815l+W6dt|y*il4RnkEr(O}x2U$aF{#eyxmfX8Cw7G}&mbj8su#EQq_S@^JXdY@Lsftl9B(1<Mfv>k+0{iC#9g`M
O11;zyvNHTqK1Y?A=orB^iD!qm#*|v+G65u>HQ%<cTDM*6#xmtBzoLCrtw>B;&s;f3%;dJLX3Och`M<>6_%pl8JOobOLB`vOfjJ=tws8~>gp|@COYj1arAV%
=tmfVRBc|od#fNov{RP8MWQh8yu%HWmRl{!u<8Ik9&YnWEIdC{o!3qiI_qIwb2;5BLRkS6-ZJBY4F|pya2|e6B8>Sfw12z=r|K1>Rm<q*C3Yrq&mw83<-=Wx
F)^Y~4WoWLXh>|_Jwy0erjLZlbEfKyF(tJgD6*e1b1br}QjIm$C!SLhXcvRu^FtIFDo5i@$U%;SZy+-n$N6BxNL=8@9=GAD=&*0OpkF84xy3{Xq4DCx<*6r3
2V+#l0tGcQBkC0Egtp%6OXa8s&}O&v9t=f5H($z*{=g;KX6+n)yb)9YJ{V;HAti9sHoW=8FGY0~bhrnU`-o?cB(9^!t4Gjp5C2luf<l}g_s@PG1I%DBnmwob
sqE462`ky8Eo4e{5t$#vR*1fn?z&WGOP_76gvsJ0N|OO3l|Z_FNyB&${E}l=CqY8LNfpgofXxu$2wxm&vfS1)TO0qwPoB`+$#~1;l72`inx2v?%Uj$ohRh<{
mFVQN<p%GaQ~c;!B|KMfw)JCl;=te!86kwj^!9$#|5$0Lm|*5x2sYn^f<tTE2Q4D#QyU2j_R!nXpE-tXa-sF>yD&fElZa!{x&+-CyiHQsryiMx++(sus#ZVS
x!5dl({q*J#bRc^-Dm7$om3AFqmcyWpv!?nTF(1Z9vz%ofy3Z9TLmTkHHz$2)urnTc3ISzuP#qn+U65O-pVukVf6=5h8z=p)#*yYe%;H@I@w4qoGjwpU>`BQ
OjX{ldX~ecd%+&Y#7UT#I42D^5s&~dFqQrbNhHS2w-Qwut0`T5UhztQdfElyAHj9<5Yk`mt%jy=)rnJ;pDVht8c>iyQyeSh4H+x*3C%>I3wZxgFR3w0Ke6r(
40NoS`wX4O9dz6I^3T}1g69E*EImBH)R^Vd+6|L#Qxe)cj1HDWzJzIO(|XDc<jN!KV8=9nnCRR&<Dt&(psD-6IV+sy7@&#)uw~93x-dE1q_E~9E!Y&sXe+X_
UQ`u|woMdI;Ymb%j2PjUxfsB%)drnKiS=tETU^hgGd&5$qEA#)+f}OcyD;1Mq4$h$v)QuB!nxc?W;Q%0LCPu>t|!V!9+j^(j?JPP`+|LAd--|1YH<GadV4qF
i+9ZRcdE2xk)<922a#5d%jS3H%R!J}@IiF?VBnP9?jVrj6Um2U5`zBa^eIR-H|mr2+0yhEqqCqNmva1{ajIKpI5kihvdf4R<o>G3YR$_B;-H};TXoRb3eHE_
$QMqeKF~Z*f9sjs;n4%A*;Nl7`ZkQtc9=YS>dgWJ9wk#tDCTPlSOJe5aw*&%DBZD6Ld#c#uTV(J|6pH~+yI}s3|kmj5I5%jdGp`RN9zU8h=Ygf6&WQxI|&i(
-feGN6W}AIgD(UE@wl{sbZOUpLF=cxa{CnIwip1UFT+1kX3#rM1Ubd8&D~hWmM7?0k6OrK*-|WD=w~iuPmrR@<uow@Q#*2MF3_9&@cz<?@@?qcG%UXSaL|c~
*WGcB@VQ=K^-Sc%7s#>*{z4MH3K;i_x?R?fqU@f=QH7CH(;1yeQ1!50R1EenwS0GC0J&t?LS>E3k#Td}eZrH<tKQ6T&}s-?4wXRAj5BdQ6a<}IP&-zAD(iXu
Lz)6o&?B(tYbJtTYDi!&>r1yJK?}#!!0+}cK~Jia)lHge0}K=NwNZ{nW*ZHC+@l3lB`t{QE9a3VN|9Yk;3q%+4r`Ahd7!N~2N0o+qqH*)5PbG|*%}A3H)h}r
i7yIf(h<i7dUlpmx;YSyMfLSHhBPezf7Hc=jKsf(2wF==NIX*Atkpp+UnY24s8Dr<$`$^_J}%&-dqmYw+jJso07`+5Xd1TCX(5%{wfA|8;g+~0bD_mJ4_gCP
7D2W1+(zBkCF0i4!AT%hqh_gs2d7m5(IK=ukaYbvv7zDf$G<&AcVn5oGu4GGo%C%1p5%-q%%HlAW#9ugVviNZTY=YouOg9`x3*UD5UFp~a}@=jxop?@njpnk
4~QrmClC+===mdVV;~9K%gal8cT)l~CwWlM*UI_zIWl=WgCx{5(_(N;r)TZUm!;fT6nb0fjEMs-p3#EDjFT|;8dHxq$KPq1JHN@P{K!%or-@jtLOF2)3=<$!
kbC;?1|W+6SRy}s<4$d5X_=2DGY*OX@&x|qbt$v&MD8uEF=-(A>>~3~VIZtFawZRJM~g%Ur@G@eD3El_BpbXjJIbobz=s_THt5R>W)}WcwdR;0FUBTK2X{rp
|MG^twUH`qwy_qjw1ith92f>1AagKH&;3rL=1Jf<J3;W`#wDHyVL7#4x*pKRKIZ6p2jk(h-t&;iYdAr*FvjF|Jf;58aj}Quk4X-fDo2*BLnCzY9ZjK*b4a3D
-Mb3NuX|joQ(#Xo;yGGCQ_Y5ZxUB=)RxU>5JOHS^3=gOE6(CJ~3DCQ=IH)qKn;`yuh=G-UWIg?Qp+E<pNT&L68ghpm_7FP%i_X(^q8x{6$R~=FXRfs_On1{4
3HzTA@o0`%wVA7pEa|1?l4pm)-uG}8X0H?FqXS5jrQ-WV#fecWE+HNe*jVIGcji;hJvt@OacA9TZgm-dbzgrjE4g$JEAhZ9X{i5m!iZ!8Jz7?*fhuKcdV1L!
HDRh;*tomlY^2@4Vd^+N%Ee%&Zgzk^>7Vq-TBhZGj9C3ocfD^wkwp6%ELmU^&ndaeoq`VO*VLH=@qZSBs~Z6hG*JRP+oc4-m=TuD_@};R%MUdphfSY;(Kdfy
<tTc~xu*oKS>5KIi~5hfObz0rU7J-9)A0{-@;nrwu89D<=AoXppGtRhL#i%-Hxej8XI<YvV>^ZYg)6rrhKBg~@|rwqRSf`twlrg0+QqyPgmZj}^POkmJC8DK
M@7sPz2@-hpthCidIK5+!c14IKvY327;~sQ=g`}Vf!sc8w+G0MS{pT!fC}5jkcZ`3Hs9$il?)h9stVF3#;I)!S^(t`mnl}OavV5=FCxD%WQ>Zv(v}r@(hg3}
5ZIjfFGWLJk?>;X7WX3}oc-+aL9I}w+!l|@ziMtrV3k5)e<&0911Z$k<Z4mbfGYEB^1A}rzStL~hTYU2G_Oyk<&nDO+S6-OEEMj!ms2o!;AEkr6AO-Gu<2#v
OG;7$Fty=W_X_x<yod=NY6xA&HwI~S04zN)_wG^b2C#Xu{E9jx4xEyNM@g>X-n3*M2{Z5A5^DH}w@1tQGHVFX-jq`%)5%-k?+KX1!k03YeHWm^LeAydFH_G9
xXSmKWHVJ}_mDjHck+{WsmedEg5xI?CaD-PX<yTURZLAkYl7PD%-5(I-Kzp0-hmeyJqz%w36mnQbaxZZTOf+1lmyy4ooCg)tC4{)2G)kTnaUhK*dY<1W>llu
rSPQmD8WOS*C70OJ7N5D0NqOs9HX>zo%QoCTR)zMH$){6F-{Uhgb_yWK@-S%)j<xjlui|HTdr?$m|JgM$8PlrBcuNu+R`|A*yogzB57_OF`6*M5XKvQX-WtW
I(8cPW&~$r{qg?qraC$0sA76xmCsd0J#vzK=ESRQ@3we$1uID=@9*@Zz$gj;hI@~inA5Q2fUu!qx7)aL2@ulZ;!?d(MqZlFY!>((R4d5xth+pplalxer|UST
pNY~C{n(L&r!hU*$mM02NEUQo6u?^#q2^dS{1c2ru<u`UsA*6?r+&sY5;l{$r(2zbV+(<dX6QceitnYyimRQ93>>h;b$E7>xjYmB6NcbSS?SZ)fTIt8fdAdR
8od^ewFvX1rRSx)xtmoJ;4#2tALS>;d3x6fRlUmlLvltat@%NeT_vgJ@XS^VD|*7Eq1}BC`G*MeFxA)~-Vd@yxF2vK%?)<<2KX4yM1FFNh0Am=sX5j@mdbWM
;h8eL8Ps6ZN%~W&<(&MKffj4beFRs~!PuGAJmQwVse^;yu7w9LLl-|2t1eqjL+`eRrG%;60xDonVd3r97Omcxe#_jbZr5`ifRZRhfQKiVZ-%|ik%l!M|J11e
wIQYj$@mRh7`GvwHv=sk)O1Xi5Ilq)*mF5+Qe2|Jom$${u8B;BVCl4oeTTI;G3pc{jP2dLP0c-(I$JZf$R_xO5&_B@2Lb=(^yh+SlcR5vR6*%nkPrCo!x$?V
l5?hri?}qg$ou3?mjPDJ%K#Og^+DNSoO%TD5BDn57CYHXK;sdqFSGmFjWGqeeZeVt4RCS+N7i{!%K_$(A7$rrv7I&nBV4pr-eBp7>;YEeCa<J#CShhIo0%*x
ZhBLYom*dboIy73g-49@7_uB@8j>Qq8^}f_T{pItK6o<ffkj>_q#o56s#J(xK?5Jp*)F8jDPzc*?XdRV=X)S(9krX`+^Gy+X){EK3f>-w_6X+TZ~9j$bbTWY
$P8?MtMg|pxh10h^#_z>M+ZJUYyU)Go^$uIbo&zA)M=0$_qgU_A})dux;$<J+q|~s8oyx5p^-(sNoHybRCo3}WD?MmUzEvTmKeemLozzCER@a(1~25b;rL(6
@YF}2kjG8zKxaOlTQ<i|WkX<KP6o}RQzZiBXckjL=vz!U{MV<v?@%zF({w9DKw9Nt{`zM|xDpC=ZC_lz>nB4l@Z9XWYwDrs7cf`6B64|>%Dh}Aq|ayBejQa%
k-W;d4p`+n{xS#S-cCM06}C=L8&)+tvFM%BPIUQ5<e{CWNL?i{3Q<OlCW|!%%v<-rmY)@WH^z2nNHvjXlxEi-k8G6i#X$b=BogsZpBx<Ks8$P)FaocC$0F5?
8(a_&)O|zy+#MpIyd|uj1DT#yoK`NqqXV&Fk{Db3I4FlT3NR2wo<a@$C}lG-YQ>o-ucUJT5kh%2sNQE}C*MyDS3Dj-LHzSczjfjf^U}&8(q~t5OJR&=Gn6Kg
%sPOCEIs^+Xc>u$QFP1jZF{bV{!kSU{I<7Vr60A}G0p@JiBkKI8A%ac(O_)3(qa-!pLfWe$6Fy){G83Csd&wCrW6v`QL%^_Vwnk!FMgV%-e!qk0EiqxdGhe7
2Hi4ska^21ONG_|q5tc64`;BqH(SwR7=0=lE5A7a00rk{|C$}sh`20zx9&gp+&AEHx$YAVPR?_IP;$O9>91tCLCL1mfca@`;&g7vW7&f6oNsQJ;zD<b%rleY
rmF1e0;-6)Rm+1HR)$XzaduKBgBe7hz0e9@8vwUd-z7ZBYeS<u0;ZfEpyU?tD~WygFALmZUWO4^<#Q!ZLbBJxEtlAQjIIA$#laz`sd8qgtF`Ch@qEgjYn&TW
$16l(p?<dSeLpm2{L=W=<>9A^vTqgHvSgZ=FT#W>GLP;pH%cUwljs*Cik`?L0$D1pLfm1j-dCe(9kr84r|!murKf6OL>uyX0~J}c*z^qyzE*mY5&s=U)TNT9
FTr3~MeMoQR?M8BKrZe#^sW8}#kELruV0t!Ou$_cY;OQcUx0%^j9zeyF6Mx3hooysP-;jLufl!-6F*)*kcvK{Gy|%c5BrsHU6@4P@3qufIwj&FKKgt1qno_D
?FUl}-tFLbFhWjbfqB_3P3l$zL9&0q?+;E%a8uRCXybFSA+0b~Npk^m(qj`gUt6UU+9_w$FR&jpya+-U*&Pg4ZCL7<s7M_>AtQQ>FU;sz01bD#K(3fwuZ#^E
#<KocOZRo|NrLKBeQV(~L$1Z4w~%L5F5;GIoHWwVJNwTAFpQE38X$&uH>~SJ1UAtCxi$iTWKYZyl^fB}gKfGx@1rsZ$vbISu|voE6U$yeWn%1#DA2dP7t$Ux
Ewl1@q%0H4G|@0@LpcLhqFS(g24ZvARum3Jf(A@-M#?@p3CY?=FYRyriJ}4gVPb<t_{89)1@$R{)WxS`S<+f@Ar9`etP(hc>MXMnn;>&lj6`0y&H&Ct0W*X%
TDsYS+8F8pAY(JxwVZcAXnLfac%i^$WZ`wAy;<`isg!RiB#Im#NB53=({|RHSPeh)7XltPiVe3FL`FG=f=1P0&7$h^qe~%5VOnG2<OM9T!DgECR-i=0$IoiD
ld1$*l6I@K$0-t}+NvDb*MnuphnZ|J9-d0+oLJ_+nj6mog}uq}Q0=ZH1C6GvF>|YxUoGx_2%A(_7S7E7B$!z4h$>Hv6<r^sh_*qd?z0Il*`A%~p4ISbq7+T*
(~ZZqLbF;OkSYQkkd(j6Afsg237i_Tm+)EdLZ8aJK{z|3c;u(oS|h6V_wF&oL>xsMQS8(oZ9G%9>SkNjWsuzeDAu1`W*t64UxG@u_&V)Vvk64aN}cOI%X?_O
4h&z6)^+3>MSh1CKH<h1lh1@o^KI<tuEdQyA62kF-;F7H_t3}@?77Jrc|tG?PH`6fBx)cRt{pFr4gmJWNW-?gI>|)LSZ06te3H%0>^IReN_Jz0mPDKYV7G2n
jc=sR$=QWU2%Bff!v$vbtvP~+iRlyvfFxV_%=nrC3k<RK0b;VVoQ349C}JyCwoxnD-4$~-bZqTPq_2!UcqS2bAFwDoB&I)uT-?fBo@Z9;03ks|ZBW&-+3Lq&
SY&;D6SZY{^SsvT*})vbl~GO#Dnp>WzJ|j|P-H&UjG(QlHMxtMYMVks?lU3?G)ghNS<A%>C}5x_UCHgddtJZ-#u!(|d7+sx))p^b-MnXk&|}XwlG-aMw-k%g
G!~iF{2Y{MqmZmC{}uX3c|u8WLu<PT)rv$8CO@}He(gQY8SgS{ch(Ufg;3Kmhf^km{|)Dne$fW{iU0@pm>JI}lR+i}p`!iF3F$_28k5|fik9xsuA5MloxfDI
if=X7V~k+RBI9x)=*EQNXwd^oKhypAuE@c4qWqR3S`fZAB%sd@k2Vm5(rwi%!7p}VH|(@|MQD2k-N)bC5Jtawcdgs0k)kfy+19&jOhQgr+|$R06Ue%yN(MeR
-fFD{RO0kBSu|IaCRn<%tjO=X3ePmW2cTMcuRmqzo^v1=o(;d$+T?MCKCy}2^u1qhKa`TR->fbtK+z|YNpDnEv)yNIP?3&r3fGT9cSVk_&_T=Tf`C=%7y)Yn
vqi8v<Y@L=1scPM*(I)aGmT<~kfGSVymu$WsYmX@10#6?;7un{1rwaBr{pj;5GLb`MA?QmpPXMw)QsN@)c=mP8xBY@I_>F%P7HW1+;yVBNyeh12l|0K)p+?*
j=TdhOYYpp|NG5}f@wM+fGZsFIpG!E0s>2GSMSfr)@KGh%+e%ZLJC|s$x+ecNCXv00#+R8A3L=%fXb~@2pmz(^d`0N4x8CjrBT<Wb~yRl%FW%_RlSD6+GX(a
IyEf`g&Y-Q6U6A<^4D0NHM+u1IiixTdjG7TORU$k?M3r--BU}3(G<XDVmT@K3qlupdSI7^u4JNpL55G<zW7N6o+b2Gm7{MiXS5H?o;l@0b-?4YJwu*AmDO)O
VXW8Jk%ildqh1rMCPIEX_Y$GW@;xM6TMrd3zT7<^egg4jA!s60Rnv@J^}Em^8k}XqI?fKp;>x#NuIhMw-YI2NG1;}~#MBwW(<kA-aAILkwQQ1d?xI<4NMElb
PBy*FbxhZ=7*;Hhqquc)RBd|)C=50U`SyWefEiI31f6XQZd>uAk5juQAKOshmk+2bKF?gN9}}nFuRD12C6;6m?2rLKj=Im^R3ND^Ys57CsmT`5Xl0{UJ{vdZ
adTfrO7gf>aAc(i@X->2Ldjn^Fumoe9TBi1kn%y*2t&hr+>aF}7DAoRT(|dsn{xlnlD|X7D8U`H4{d+q>=3|MKXiu8HF%t*XBw;T4VHspJ^eOh&ruF-(~}mb
5{QOowCy-84(^Q?qo!mWSX6|qV&rs)`l%{K2HOnQ@`C{pD~kb7K(VfO+<4L|WGeQoYFL2tyAxhcaUxsI&){gkcL$}qI2!|*tR9H<Q?2a_fjI_=y{G&W7KqhM
QVI!6{GJb!!GJ#P8OJbQze$?p=eg12xx?xGXXeDfK_Sk9F2o=T-z6vq7W<7<+9|{bE6|;DZtILc8%RJw_|{<f$80H%D1fq&epiNG)M;?JxL&MpCMu*NSg*gF
qex+YGu=@;5)78E1HkX0Ks_=vPt^8AD`9ulv;&0(kfY^SM*hZ{!hg~gL~!L8gC=`G)af%5ZNUPWAg}V0)bIfxO83d`RUXVd=Y5tJxS2gv@Ycu&p;!e{9~z(S
ujTsIFy#%Y;xYldmlaIic?JV;IC;rsdQ*D^lv@&2zdmu<s52s$p{(eefl&GNe5=LC99Rg~W56R5_MW)1J-sB6Fyg(chOt%$3xKPMsTkbTyW(=MaI)naG{e0~
o}pG6(&l*6W^(KCsimL~nyeJZ828WR^{2FG&z!}N?cg6X-RKY9phueNKzh{~7^kCA?z|^MEOHBct{CY!J_6qyZ?a{&v**UU1PqXRU!iv-G(Of*up`!F(%{~H
40j@852BX+g|Dx^C=ob>{`osnccFi09|Z_+-yE~Z6aD!d6<WK7p%OlAw9Sce^92K}J^41^pL7XbC;l3dD<6Nh#!vhFz>S&XB>GH>#I@m{z8biOE*%??LI*P^
ZgzqQfPq`<AJ$+dj$<d!PWn3RdFN#0cOOnY`Lq=Ar5p}!OE>X(&gDjk*;!oc5#pXPjIl6y#jSRqD{|fDUBojP9xBPQN*8gdR*aJd-tA8BnqGxM$27i*)0)vd
m^{hB$(@)QWUxAm_J^m~MSF)CcZ0W)qvQ3LstfHoHajJj8sRm|x;wMx3w9&PX(d&|I}+ku^}g8%EVhXL+F)Z<X39~%Ums`}V+G;93r+~VoC^Ua4>7)1{B?W%
^sryl_8D0h)F-EL3M|5-I<*w+`cIIzYB6txJSPt6nm3Ozsuvw7+}^xoJ{pV_-H=ldZW((0f^H`&6e;UKO2?g>bY#<e0zghr>rb;uSBHYey8~gM@3+D-G56X9
C^H@q%rwI#A!$$R7#CoV0GA+rO1$WzUL1t0j5{E~yqUZWMHPi}V{YIApla#qzXfayN+5XHg<y_V+0GwZb^d-|g^IhtjrbyKI0|{Y;NI;<yUN@aE~hf1u0elT
p5i_=1zY11wIDg>72<0JT&l|X%@;}$n?wUo%5sB|u;Xe{hqP<{Da&#xjRGwmnCJdS6o}JuTZc~>3Iff9VZ+YzyP@*dR~%$DCaOG<blTzxME!Lhj>4BtC+yKO
;4zgY&(`$wi=Tvx(HquVqF89kDZ}A-t!UsB?*y8L+HQOP-qxiEHa0!sW*73O9Yi5OMYwcD1qR@lf9eY}+sH`Qr9MpPpRpTl*3@6Pfx*2rSP(9#f-#1{;NbjV
N9D$Ms`|{_gF)_>yR%Clcr(-=+0O&z@@XE8YyrSLhd6v$Qam<wQE_+sXv*&GOlyGwkjF-;Rf3DpCj1Q8f!(l{$iwh$;b5ZhCD=5T(3)Cvqb(A(kK+tlcJ^RN
=R6ButKa3(ycz(Ut=REYdFe;`uFa{AmyjdCe3tGy!A2KIMdcxplEYGFkB1cqPXqz9m|t2%1-RwiBlXX1@>Q0F{3$A+JgALJy&jVaKo%%4k`?PAOsY`nrs2RB
+~oZUgI!oE_;m;qrd*a(L(#}p#=<fyLoTYd*+j{n5z6uy(g^BFVqI$k*iutf;WW__t;B)uhE4JUL|-arXuMi*iGbl4;NHSGs{F!J|Jfy_n@{<Z-VZzb8l?s<
yz1xi6&YR^2&uCJ+MX-=N%GVTNb}g%<}qu_Kz5-ryZ`{<FHAE%`hGAFJg}Qrl}9*o&Sg2Qm#0iS`vS*u5V6rMnmplb|ES<U7IO%`i#d#{_U0tvpuaRh#?#H)
gikWqt2a}tzL8x*x{Tvd7YTCpgvd|9f5!}QUq%?uiQB4@>XBqt)<Pv~kM5aSdG552S*yA2jnm3i)-14a-KHxFR2U&WX9V+wQog#ftwO|K!yx#*w74$(WR(b`
26V$0t7xK?wJP2h*-<i!ScRn&2Tnn$uo@K2abA#$Gh$h}MB0J<SZT2k#^lzTux5f44n7wvel}3bNCk%k$J>u(tInCYzL{MM<905}j3fyInctQiC5mC`<GLFI
COC1Hy7J1WwQ##qK6l<gf$f+WM{Vqsa>aY?Od*u<H)+J#>9Vt9z<Lq_`~S|RZkNJ!VS!?Yu1qzzaS>X$tNC8((&nJIo@+eGL~Y&$hMNM!9{LzcSWXsq73Q@1
9yu{{6({9s9lsO6H7lo5Wh7O#jEQZJ$FU^p|7@^OoM(HU$;ZvluHRTODkZ$rw>%hiu_s6sguXFyCBp0s+bk*GfF2fvg1mx*3-Ca+VMi$YRQ&fI+fT)CnLqi)
lvs@R8H`SB$D)>AUQLH;6EYnkzxD>XNVM@eZ|Ht!za9lms~b9oEjR5YzaMKOb+qo!BAcWp?=K4l>44Aj9CrB1Bgcx*l>I&i+`EMGA-eq_cZVJ*PN(W1^pYkO
Z;H*2mFihNPQlP&(xg+QANVrF%_Mi97p+DFX>nGA_L8meR@6L)Z1mCYq^xS~p?B$=7)*!$_1@|9Tb8%2+B&G~@V`jd1h5I9UvXQ*b(5O4BH4{?Bz<_tBo>!7
e-WZB%G$m)nE;E2(L+suQ(P2D1Fc_azs-1U8h#NofOV@RG#yV8wp_CCFwCW0w3@)L(9RQpTfS+aL@FYvKANn-2-F3E5Xzg3V1u@+p|C+JU$&)M-zz02Xk5~v
dP{gapR(c8(t671>yv_HfIT=pQN<^F_VqPB5aAR$j!mx9|6*wBA#TS`e(K&@a1yLKy}(iD%6!MDuVMzm^M%f6wT9WyAI0`uJm85uZbJ5FB}RzEMvYGwB>LRE
HJxZd(Hn4lmJpHfsJa1sY?{zIwO5@vgwP!Z`)T>oK2xnaBb5sQnX*}tTa)Br(acsS&uxi>tYd`$V{wm4Xwdck|9Dn-a^U07L>x#O12GW0|LvJ?-Cf%!#sms#
=Iwl#pw|vwUzGn2Bp;<UEh_FNtd2XB67_h$uO#D+#qDFZPa$P4Wy>_n!9s+Hcv4W)i0Au49V$-z)WuxVVh@z#Kl#Q>AMtJ$FvECkt#N^==35vk`?mP5??WJR
ib*Q=N6@bU)hZ_aPOL$j?xSXmMNQNP{;51w&{%1ce4i1fnHxR&BiyhHPV*p<B*OVK5>_}C4%-e?E0qo|n)t^4;j$H@4e4|3By3hubTnuZjWGC%s}#y|(w}+l
w;Wna!>Io6`a9@32TxRIO?5|tv!+g+9fWoiJgV^_(*p6^XfeTTM`+!FsuqHz9GdLM^8B5#)z;X6UdhUQ**Y3}Ha*C1pUV{vxT6)FIEDoH`8uzs5fJIKn2@4y
EffCLMxn|p;18z0ngNWSXDlhQs;Sa>!1xYP5ycU~_b*4Bprvz#3)jFlGk1wxzdX-Vn_tWNpW6B>t?|H#Ut-3<#qYtYjrS~z{nats$Smq&gV07*{H}Ubt>r?^
A<?&Wau+7cITzj;65-tB0ZDOR3ppN*`}U9a*W{6NrP4YvxG$j-8mp5C6<$gcbEeRorLX&MDKkXup(Bm#E5DLO%)Ar2dn7?)*aY%>XWIK?L5>iA!E1U#Cvyk_
DTakVEA+zBtDkM3taoYGXfxEm3G)*TMW=yipWsCc5almqEx8N2#R$)0B}h7ONx6Q&F`;8mB%3kSdEro$Ltl44n13%CP2|x(d6wfG{sQ06NB;w?w#EV>B>3U%
Kd=64-I<PL@bHH^+y~DXq#Y0Eg5*Nxap#b0S2yfF$&mW;+f>7*Rw5?AwcEzDjVnqa6p4+gWD%^zKqth?m&kLW4-c?8Enz5NNQa}Lf~3*l!aiv8eU6O-g}9!t
D2BTf>e11l1nr^Mm{>*0`k}=X`dGn+s!tg%jI!fx<C}Q@);EcD5i^%@bVn`enhgQ$MwZF0Af?Dn&iI*?F+(h!gfVepK%F~A58t4uoc<)EIkfk^i?fz~f{?e2
Au?VFrsiJECz=%fvDYeTAuuq6FqE`H4>8N+aFB3l^W!m`Q%Xv?zMfi#S*T!KAQDU7O`~qS;8z$<uDk>lajgr^pt}d}<gZ0sJJb(mUimebZ%H0MQrHS$4HJDi
Q}H2n%oJN89#O1UuY0I<UZp6{-gt)kW?lzIIN2D;z5W0XnY?vbc>Mp~t;~tRfu&7d#kn;_DJv!<U#LgqB4o!V%qk2nL4U&;qY)~;#==zdEK!F5<sE5&dd{#|
<LF8tK1BLb`t|5uOHes45zUC=pz%uv_*O4W;-vx=SSYuL$MtR!^xq2Hy?925fTYy=@#cjcTzAeMwlNx(-|Vw{XSj?KPiq-r>=?GMpoT5<JLg4vX{(7+oFW=e
pN05vY}zKzVUN@&qCE_7c?a-V`uc2F_bvBtX+$o1-u68*(bHiP$dit6jdK6N0=&JwTOwFgH~a`4YZwkGxEV*)jF8$r<+oQR2?yiGnT!IFuRPbe$Nhz>8$0E<
X6rYK`afFSo~G)yY>%%AG7>qZts6d<a!m4hWm|00!o%7hMrQ3p^;uGJ_{J$Ll=bsn`Z9X~O9j=n{#v3J!eN#pRPE@g;N%)5aOT5A4a{J$2?pP)^&c9^wk#t0
RWspn+F#$Q>&XO=0x6c^;--sLK7ZzOc#`#W%bQu>+OR2cb$g-ijQNVNMDHoIw)u*xjP*Q?!4(!kYk69T?r`-2U|3%!As}NS2d8t%xl$w~K5JaAHyk3Nv`xW{
%o{jTO$)w@Xv8nu=8XsE`3pxi$ZVzZf#QzNVDqy1R{ZgRjJ_RYKwO3lfSANgUI9nObq`zzRZP6DOIcB?OanZY!huP8kr3QlG4Sg)wBV8j4;|$Owes-bz|7@m
5IG4GR}c6D@$T8V4wOMkP9L56D${bFXz7BaqHAL}r&86#RpQqP>t<+$iV<}{HF}t-*GxE<&^He+U)xGEeS5fxPs}>VqeIgQG54{nk5e8W=kRDhQ9FBuLNW%J
XpJ-Sk#`V9p?`ojRLBF*L)IkSpaxZYm9Z5+Y<x3?jpXJ?s@;UZNBdh_xFT*dkem)4&_gP3Ya=k2>3JZb=`{x-#lhyAqjQfxuT3)x>6yhoj5jw%)ugyHbubv&
>Q*9U+%3^@8TPl2G*BBI0VhN&j_^#n23$}Tj``E`7v^Y`{TR{h#L4XlnD1_@(BHqTiM2Vd_XwL{q@k5ORyKED@|jMsi0mDMz+#;sc{pRXtb;N~(vHoWT{(#m
Okf`JrsQSoa-HHk=Ge5xW?)N&0>c1k8-1viFvA_9`mP;WIGkLcv(2qz?dWFYI+W11lZ1%@(Eg=!_EK?w030ol8=VPPQHfiM@>Kf3H+9&@7DnP5lAg~=j*)3q
M!i`bms|PveV^xZw!XUTS$Pr(!}*G0aVzi;-CGlY3}2mwo>Ddj6*REeOwR0TtveinfvHf)Yz@p#8xEY6lGn4>o}g9!P(i*Y$8Er1BItS?%G;v%##7pWHY1#_
$fJ3tndw>A2eKHV8xrtY{q(2$fcyO|9cqfU$Ih{0vf>ICpzLHFzb^u~@7d(^2Yh?Gf&{J5Y4M;q`Ft>A50kh<Gr1KG{g6igNz%}cJ!ISxYOz?vQ24PU{pw%0
U1mPH(gMuG&5j=7u7l-q5P0H@ya2r`Xpf3vtW@i&vu~cvbszVtt9`AXaJPnG2CE=*PDvju{?AOx`m!`BBtxPaT&zr!*8ov2Ve%7%OL>|E{rUGBL5gdE(n7b#
n0=o2PQ?mcA>sVH%8fSdfLzv8%FF?^l9(6AxHSU}Vb?_5aFV@0%wGgAejx}=CjeWdcN=Wj*h8-$@>kn@A2b;TvGh;C0s_EptZA1f_y&-N(yZ2pex5;UcAREi
A>kOeE5zE8F+{0=YFQXU*(dNYz@v;=agwl2({%-W)Xji3^{6cv;bP>@+kAbfK=^OgL#vCWyR~TMA)tn(JBQ1slz)B<{h_aY_Hi}VF-_F=MLZ3$=o)BTv}Qgf
YIk&^^*P(y%i~-?VFQ?)ek57M{+{Q(7g7SI1o8{t6tcv96;~EwuP$#q-%Eai0hnux*E|wXnP)^uNUWVDBy`%?J!)#K9g7dkImE2Vc2c)u3$s2WJX)whcYq%C
P{Ae>GI;pIP7xT3_A;`hxJTONFtzc|*xN{4=dv~AdNu1xgmP;O>T2`~JF(NJ|J}IRe|<Wu_C-DsSUDvZ;OCfry)XEhRmw-=g)}1l3fAZ49DL%@Gg-d?sNv;R
S5Caal}Yl27zs#qKl2<HtD>AMstOGnQ)=mve*eXLsE>bW|CuOMbIJXp^l&RwZ5P{zLL>gZF(iD-SBvWhm|iL1+$CkBR&O9@CnZO{Izq%i^q3<ej5P6F;2QJ0
wFa?U8C(HqU?D}QVa7o(`T%k?*Tf6St8F)JOBtP>QVrkTUnB&`&(qt$jBz6vyBW?&{Lf@^<0GUSu{Hv5(eY8Bw}F{A_K={g(L-_Y(^D`h7tZ1|+g4<Bh7|dN
b<kOXfhlB+nDYZ%)3znkL3Nqx2*LL8S>6LQ2fC!#?_r9)`iS)uj}SCVhs*HE3Q@B&q*1YgE>jHYufe3>L9R4?#9fKXcexLknEH4`GIpoN>QGl6-e$(JSk$){
eZ%!f1VtS*8hK$LdqNxi(tyvpeSw5;F9R(g{B708JAyM_5S6{+w7RZOKR=!##g<Xc`yh}CfcP%O=3|Kx*tkCk^!H09(h7#qEqG$Mu$5f3e13RZDRL(>m#4(x
w<1aSG}OO$R>0%RRCF9i^XkGBv!d`PS~j0IgNb<e7|~xYAejAMv3$VD=u_bOR_GUU)C{-3f+{aQw@j9@FtZb{#prAivQDBpdgYL3R`vy8Xy=_RHv3i_u$mAX
ON-#X9@UlNJ~s+MTkm`8JhxQSk@R6T+&!Wzw`}g7t`~SCa|+LVh6ZJ~&H6Tg4%EmZC=2^6|9+PUFeY@UTw>Ci+F3A?r#V^iaWG@yGxxx2L_To@tF7XV1)fa2
I74~0k9fC=U2VCwv-nDk*zbahiYXn>_k%~!`$mhM`!d^%*W~if#v7R~&+p&OX7WUxp;)}uHIBLvI+T*Y{Zky<d4|EUgJsPLMMw<)M;=z1hDq<f)BMMyDSz%Q
2x@8=w4(nlnR0FXYGjKaFN-av_Bm@pIy5Me27MSm1aM9k0<AJ1<mskpH&$i}*IdqbtSO6=R-;aii*~yh>iHJIgiZnPu~|`-hWUN!$7xH0gsrA0EH81nrC;W;
FJ6)gce0Rg-jZwUfX@Jb)e>*Scib>wVLQu@$^aa2$~`A<P^JiV1HplHAoFjCF;6Z1Z|=R`a>!uL;B<<@KOn2?@NqwkYB+b^$=NzQ*PDA3WDU|WCzteGhjnU5
!z5{dh)6J5T8c1YISy;jm{82^X+OcM3Z-dVaUgVBSIKFvdOJQsxInt+8Dwr#u-u3P1s49_zpb`t9CT3B1!w6$lK+Ah;A$e^?-BWTQ?U&(1Ur|0;zWxirLgTx
HRDLPmfeXO6BW(k!_5)*tz6O5ml+VvykE(SfD4n^Ax^Lk;tX{QC7z2`p-hS+X6sVRHTq29T2iCcc<AAPHr`wO(v8bYxV^?IA}oYIYt^>%FJ3Ulrdd8}Z09z-
1A~Oiv&`4cU&tJD+bUf-T8JAKzg}1+*7rWTpD|4z+vZU8N1~>uub}r4pSRL-NfCsPF4o};{$P%0+ZAz(UTZ_=+Uf>o=V`u(|5!9?sKU0b8`gG<6j0YS5C-m*
5@zT1E5;7L)*99O-!n32qZ26rs)}aSdSB>6;1>Lb=s`Emx-7|bt(ha$gXj87#u#MGs|Zfk?Gq77yIocr>h=hqSOZnBz|@au;Xi~ofi5L5rCl!b15c`1WQ}kw
(*4zyD%1OoFhc6xX)aFw636~o>J6Qwo3aWSaGcxXX%20;ITN`%DVbTP+-O*AAS@~(WXwT{G@k|=BlcG`X>;-rXUg{5)Ha=^9w-G=PI&2lgB^i~E(zV|Iwsed
l=S=o6jp1;ePyv3_oSu0X!fE~#`n_GhlJ9+p=6&oWTC`w{z;RL*gEhzF>?|iY1wVNgwj_RxGH8_iL&L?8mYRUw;iFW%;iZ6_hh?LQk^|76iw{rHc-XDULl*U
ep5O=1X7X~cY@y9Osj@HZsD-%&oA9f^oG#_g#09)_qz$zi&$@+IWpJszDCT9TSHyc{h!84NZuV9ynsO+sz>okssnV$hX)L3N-q2V{*~^>`1=k2>Bg}OQ|f5t
H>;esp^2$-N$oDE)(H@TP+k)!2p^#bgHwp%{xILJ41H?r8Xv3ut9j}*)ezco>F#|7$16PTbqr0!!&Q2;s(g>`R4K~aah`1*9>hXj{}p%iYYv-gIjAoTQJ*nO
GOpt940AIl{O*r{4dNOKVB5dFG2c<`R@l8o4!y2!itOxaG$Visv)@37%qDd6>6!Vv!$T5l5AyGB47)<%se}mDlF>ghfOcSn;gezu`S}6Vir@Ih{Q65WJn<v-
-u~lgHwZ_wV0^w7L`1%&epTSZP{0u&x}tSLN!$nhWzSZU#o$k+s@xazQ<&2-ONn4}`corBfCD+ePOs4*wi`f=3MG1kbUJF>Yr%F2ws!1r)5-61DQL%}@eCg5
KOU~;x`kzJLr>-%CX8FC!dkc#^W_VYQyUzW%$YD&IZKR$1~8ZW1$2G*n2Unh+gI`VH5jO-7Q@0^ySHp|>g%)zzvB0%elwD{r(&M|@npv=a#DYw=9i?|G!ucq
GCe#@1+EQL!59QgG~4b@PFnr*WXP2VMzGo^mms9-qI?&P?GkxLCUWQ#776f+(xlCfe-RhTNJoznT^;_|M<p~=M_oSdTPLbbvtJ`JOwbxFe2}=~EDhNN)9ahQ
OzmOW;_vH%0(XTG$@LZNEaO$$@^yXLl%onK?kDXlh(5=Na>#J){O+@>@hTT$Wz0-CkJ`J`@?v^XkLx1ZHo{20b6vP;M6y8*X8u?I|8|XgQTC{7>;E#^KWZ>l
lktwxUQt<)F`l70!~77PKe6)xBE?O!C%BM9bt8cy?fYTnoLM<E{m%kmMIr(4w|*fpG83LE!D3kJRZ~8=(KKNx_cO?XcfUpO7~f#h_D-QmzZKpLwwgTZ9`R*<
=#m>N-%d`we*@kq10Kiq#YL?9O1m4_5#K2PjNER1vQO0Y>N@7vxvyjzlMq&aQo;QvDv24%smY~Z{T{tGrw3vHkFva1`6jo=+X>TxOTl&^Q0+Ebj%$&tiT5Uv
8)&7*gko&$2&`)b@GoxJI0noyQV^Py@=!^2F31@#X~6*mH`MoPzg4WbPi<JBI`NB{*a3)v<=*}AAmXxr__aYcV@k}cPYCx$Er)4MJAAH|vh9UsmjO}d=R4m(
?lUuVgY^HzgTD)mYP>+&S9C0lH5+iQ+gKQtExX{2T2t4YwF-n5D-DDD5}1s$8Cv=2<$R@=41WCx1yM7&2ncp&7?*R)pz0~V2U}SIoC(fczkM#N%K7r2`rqbw
neJWP?8v*+2ZIMS%;Ug@-r{|V`k1@;18c9co$nA+Sq<w>$c%qn`d9j<F-BjEcvWb7B2_Bp1yAcCNvl(NO-AaoT*ug;Pb}cb=I(TJ>~Uqcyd2;>I0sWhRe=Iu
6pKJ_!7HrxZAeTN88pP}=A^P6{z&fnN6%QQk&;M&Wl_-V!>HLmj`-mbJTzQl_Q43QnvAul^3X8Gf`7ak;m+`<a_(rI#TiV1fLph=R<Xo<U0@gWR0!0UuXyp*
^Yp4doi;=8@&tmoT2SWL)K4EnBXJyTA$@NiklI7Ax@#$UG+e9*&jW;Zq<I{#J*a^QI+L7oU%WW;q7}qxpnLYON}NiZB{^`lTnCo*{OG2edX--9*vpw`499lr
o@g%<owhr+oO(_qDX<r{w)NsYL_s8vy#NZV9FJ5uIaT-pbT2K&lI_HGfS;zQOROE!Ewn14>?)cz^gnd0Nb8rG)07L_7t|UQ=X?%8H%@|im6P?hKMM?lj-B{4
m9B=cM=<z~IzEQL)e!sFV>v-SI##UpdXWok3MvR3B*WC@XPy0?Jn#Q|lwxrCzR<K{Bc)aJ0}fU7Y)^=Du*3MuQIUn$&Jq?}B6zURPF8EL@-F`xwk?6pqVP8T
FlDq<ot1v|0Xm2bcP;2t?4`e;V7KqYNVj?WHH(>n73JB203uVFgs96Y)<K%kwm$Q*5Qb=mvohC7xLtdH)bd{$UiB$`qTK-8{ED*Cnmo6<&RE-3KiCE#bH&TX
e){{nw#=B&x(}5;J6+v{VR!kqF&(QT1XY-b1VjqX7s?l?s{T)8;0Zs0w94rt_Vn5f9>*g+lp@gFa9xU+ss-ws2A@voMI7f;xgpy^Z3m-yW<6&?PQiPW1-DMA
(P=e^E(RSp_BQK@aurExN000UESiuE@0@GvGGRi5;Qe)7IDzyU$!KexK8IkQ?Lw-<Hf{}U#N({ZcaukNHPn7xaQv`P%FR&FB7VV_m=VW`q!*WPsa43kFCWg_
W(C|Sr8c{aEW?*0v5wFN?m{IvRn++Vi_1@f@pA_=&n}EV!g@%ewW$U|W2N6)oE8bwsHI~<I~!^Ad)+wy^HYs6Wz}T=7c^>ViU_uGuSZWoTm>X%Y?k(DC?-%d
bRFH!Wo}Z-$iE%B6=Qqa`Qh@%HuMy53w5?7=imhP+Op~}F3{N$>`TNYP<2hOfV-^lIxFr?f%aRR0XTBM(*ibhzfnp|tgWC-Q#dBI;_w3cMV>7Eyt->aR{MZY
#LKj4cDif<l+1Rhtl?HE3ba$^LlmWqC#MMcwT2_r+`v)+;t(W=Tcz3Gx?qpDWu(X5gI7*M7-3!#=I5PlUI!HpKD1Z`&iwQtMfp$`azRId{fyG(`*PT})xDD6
+6v&B1?qUJT!SyMb(Yge;rISFXC?B~NMn|I(LB-viKa~>8yTfUXTB=l4Pq+`R9TJqA2us$UUUrxv7u1{MMH7wm&{e?eQ5Mk%q&+N_6K(-PC)PliG%FoQ8OVf
8LEp;DybyEH4yQQ!|+DOLyEq6s0P?i?O@sKxI|>GOMwE+U?abda%#xz7Q?aUrt7Mem>BWO!tw>7(Ebq^>s-_geS~O5IaI8bnWOvhs{ob1AiER~u)I-LBuTdM
%D7!|Q(G%lkmv6f+b%azTVAjB;d6<?zrw+TqUHTtr|>(l2VJJ`evj$NX_*zxV2_)|f8aC8uOcx?Oy_^YKq@v@{1c-mN!2K}1w}xWvAGP*_1fe7CJQk@q;Fff
n}Q}4kE+I7O<vYztZ_~ei|ZRu+ibU$i-;*!N@hW|kKU&$+1La_&E4Kjh)OEpg>;0<IC;^gMHY#Am8!0muRi(WpQH@XYdc&1wv?v0n+>hsSpBJ4X=2yq$VZsw
A8bJMAe0dpTujbA0=sbM-6lvmlW9#(AF6JMkW!6k2OXhobJLG={drMSz{V^k_5VK;8fI`-Q2gfvvu(7$qK498^m02x>VZNL72ft5=&0HKPu%f{m-ehBa#ZV7
zM!wN{S{eGue_w}UdoPDt;|S+##EE6Oev5O=o&v(B1uZxS7)(!58P_);lDr<vQqEy{E4zYL_K`eYNipli`3oX-%vClwRK1Gw**QK?QdpcJl?VV9ArbE3H>Zr
Lb88x-7zUR+s>|%@|5Lp%J506wbY$(3*qu@xe{+!#8!v81jnmC+i6nDBt&wQ7_VgkwG<r3`Yf=y1+jVr3#kl)2~_lWdR+yCquOhAaQ_|bM#3X`$Z+-u0|l+g
7Pt*lZ<oSuG*5O0+jywEp~^>(QN^K`!?0X<A7uCa^1Z9}X@FJdWN?GN38i3Kg`yFN?eSp^O6QSb0yMEmGH%i2jh;%@u_X*KsYt`>>GUH#l8;*I%DxFu{ah<e
Hynd4zS=^|dIc6jzYzl=OV#*J;(5Nm?M0h4z!;;Q9vRWK8JdL9|07u$G<#T*Hra;%`^uX^AS2&7IY$@dhlW>+l#CHkBLVV}*}Cx`SXJy=O@yR$20&&e8o)0g
4}Vzq?J^@hd$P#e0ZUu2cv@CI0omJdgkoxWi`r2)$FPe5BYyhtvq1Q|oChSL*8f7(h%kJI_cocW-FYre*08SX&8f~7Ipo?Y>%n((%1zGTKYi2~<zOa|fC$V_
(q;J0Cs2rjOiuCPX$owMo4Z@$FEZ&lw;Ri*_`J>1QeNi0Hbp0j(eUUr#O}jiEh<?GtO&~Gwy<))Gh=4u$I}+0vDrS_7j>vLHc~Q$!&o=sn2^sENKq1IEy9|)
&}GA7wC`ztTI&sQv+ZMm9M#HB*@2g!%!y&y6S$$CRd6{C)`|t$b>M@il3P~vv^e?gRBVb@e|*9o)qgd=JvGY_UJlqpypM@_%zig?jao#yh=nxlSdgto#UGDZ
Dz#n2A2a4o1rZo{Lje(RQ*4$9aML_>;M9)D{=Sa;#LB9oo08;bJZrpaLVmh^0aXithzy7HrJ>Jwbj29LG2${#PzTO;aG4&jXHu2gvF~%mZb{Ch06v$eY8cJ{
_xnaV(OJEZEb=7Ne~zwIiU^>IFdJV&CHPJfbFXDuA&0&6afi++a5@<Y9utFT-)tov0g;6MChP4%sCI|3uCpU(t&Y?}G7>Jp%{c9H`&Y7$+jG73@KV}Mp}0%I
(#^5!Rw$O=oHq6?l4dE8&}$i~1^J|m>%P`ncbV|9Knc3**K#Q_m>iQF?>LhhqAKA-9*4nJU0<79&OE_(_F3ZP=LVz=tL#a*GzMWj$kBTyydR)11;#BB+8foH
qFH9Kf)MFM!G=i;eD-o?wjQ3#vuJEcUpyqvjCEe&G9Yy0gNuxh(tDRyh2`m>5mZDViNqbuYWHnLC^2g<nW!M+UWeWO7GF!%#brKV63_>s{=(S2b0~$+mWBWL
CfSzPGZ;w^Mw%E@&KCSW@9Th*uyu7^?>u@<;fc}=?)1Sx))$v8rnR}#^)rYKigspy|JMT?VN-^@SZ1tGDm7+GtDc$|XZW8eX@?TM&z=RPW-(*v&yG@!agO2_
kVPq@#b9?NC1oI=2e`O<P_zD$MGo?=hM1)#!?5h%L6*!;5F~My_hX%rlc!!y8OT!@Qr$uXbexR#E5YmPAUPQaPAoB4GS1KX+Dp4K#~p!+Y$-k1u5dR9-8bfA
dR@jnTZOqJ&1mA!q{n6B`-cYghH8t_aTLFvOE&GPTaUmerYIyK65-j{v=u+srqoe4HV|Jbt(h!!S0BMup?HpRqImMS^v@id2A2LjhWQtC0aa-K+y6MjJX%%i
pVc}cke%xWCSEyKmBq!m4#^_HF2yfc#Xaot>KLxd^yI*7ZL}t)wP#g{AI11G&mb}$(n|c^`KYKOM{B5(DLzp2k=6C?B+zJl<!&9iKDT}r`$d0i3Z$&$$y4ia
A{$I4IFVrbJ<ot#_HB8&LY_2n(`wTkxM-cfngM{uVdvDfDoauhF`U8Y1?KRchQm3xCsXDT@6_a*r~0BGV5P^U@8C2B?+v%tKL9lmlss<9s*n2{C)+l#gj~*M
7mE@Hgv|F-P4{pt6BVU~^;xSJ;!RM)G-(aVbi#oMTv8!Z1>gcsDuqK>9i{}HN!eL0GzWA<$16gIBBQoB#TcDTouGmHZS}b?uck&4nPUqrN53kRp^s3bWUZRH
9eDqVT8XLy5L;%Jgv!q+b1j;+V8Jbc)Pn9XXR8D6|1)80$s}^TJ(wSnk~y;S)26nl1fO~Q`dKW{XV^~WPV=1S0S;7zY$68?vPuz5pR};@i#l|UQP~3-QotaW
wet$~ZZ+1EoMzbIx}%tUTQxD=5tNHZP<b9cnDC(xq^~%*JHscC5a;kFVf!8R87#rn1Bs{rMX^*r95g1p1DUP3^Y^q9ikKy!+s5Z|BGIrGYV?Uu#hx2u#Xow9
f>U{L1Gb;Z;{J=LwaE8ns%&=U$5b>30BRK$oP0KbYv{Mcpu}&yP^|zHs)_XzGRI5Ky3LP8rXG_@a6=b9&5Y-=N=D$vr=+qfNx~hlmR%L<(fek}eT1ODVclE?
M)lf~QjCR(3;ha{h)6AM`QL|3T1;*JW8N-tXV{2xwnJzav<abDcN#&Hd2d5yslYW`BS$O4v|7OH`IuC&t(DPnsZ4)c^+>GLIuF|9yXX3^LPP1U`?7!fIv4ah
*P>}LnJ^gwD=vj-d%uu(a6)Sgw=gS&Gjr|puLw&qTO|ih&F(m7X>y|g8JZa+Ty!famV4O5S0Kx~t!tr}g6GwC+O{x+7(udq5@InFEXdoUWTO|`%wwwxuygKJ
hf7u58VOy@+$mktw>0w35X^q$GjRxAtLEUTLoe0l!_N*7vn}Vs)hKOD#86~p|9hMk{-@dqUbgEN^Cw1AOM}41$k`bY&trM?mu5j!p3~|MEvDhMbT}q|!u=lx
hM&zc!gAA)aT7dAu+C^j<9cg;i1sRBYIx60-j(Tu&1D~z{YLVS^jvV~`g{GufR*PkZcy6CtX<Oa3n$)`Z#^J*cw}-fuyr)+=Xo)ED*MCPyd3mZ<;hK<aGiy8
9H52x0}S933QJwOer6CcMAWqkD#TC?*mAUISz}E<ly`hhSjmdTAxg+VD%V}>5fX1(pr3R`N~Z>uD<2?8rmhIBq%C2G@z%KKQrt?zy2D+6g`{&XP+uNrFpka#
g4Ur5C^a?pTgup-EA%@rYpT&G@=P{%SdC#xuP3lRiJ}U(!3-1xjZQWUPL6O(Bl919lYvy7b4{#P;G?Uw4wEZa4|1q`r-ie2KQ{|s)6;XxOb$UQS!UC_K2@;d
A&#;(%i}58Nh;0GKqK8!zlF-wXeFx+t?BwxiEU%zp0DKNs7e_Fq|o8W@|H0c7(qF?r@qG_MFBW6Ep{o$mz5``Fyuo$xMF4s?TLT3_{}CEhkrAs&j4Jfv{=zW
Tyzq7%2kj{u~En&YRGaHBsxoWvgg$xiSLq2g8Vy|&1uPX6iox-K;hHkysiCs(j{ZY02J_6*neOogVIdQV=wp0!i$Kw2eLW`X&XsnE%$Ml!e=~aRH=6@#yp1=
5XxIDLq?Z!I~}sC_QnunYd<x+CvX7N3lMQnI{UAW(5(S8`L9rtVXRYz+V8n)kXkDhV<if+-kOb@u+#~mbXGWef`fBUPM&)i6@Jd)h{Z^qtvO6@d@r4`zhlr7
HSe6-$?_zcm~pWr>p-DS?EvzVBQ=-bWU353FK#JIKy?l45-D(Vv{g2Wg!otiO!##m>eVjdtry@T#SJ>cP=G{=>T!GlD0De<Pg;!x!Ff#!s_6IjIF?0|P97dZ
;n6M6q$eoWwhFSVkEBv(`$Trpm@&zU^1?);z2P1bmQhm?oH#Gr10p{W<?GR`u0BO`f29%@F=mSY%|y+e?K@Fa6%oe{QzXS~L|8yXK^kX+q_Pe!zd4hL5dYJ*
gNcNnqL}ska~33`?a=NbtwaRt0!mXG{}7#{i*l6X40u749(xj|R`dq(s;T>oO+bntX#L!-+!~2vG|~jH{^Q1C+=g86Q(op=jY4g_L-qA+eUa4?^fu4ReeT9K
4VVW9LXSJrGhz$uh1DvhH4SA>HYtT;Ab{^JJ=Mbni8RQeiy)BaarmDj?4Z|$-cbP}?BO6?MbBAsRiGu$07?`yag?8%&**lSyoBso(7cTOe@aDgmyqBY>ubeH
q}U|_sqY<TBS6sQ5{VT-Y-$iUG|$_sKQZ<}mNrY=U}V<l<=Tz#oJ_N{wVY5BjkWBR;}LbmE)~X5-J@eW(q)&1iCj4g@guC+Xq0ZG?%h&OM9N*o@qAhjq-|%b
LM(2{Z(5Y{5Rv(+d#~6GCxz*_3VLy`wCbK!)HrbuC_es;DLasZKZoX2>Dv%W9T+fhJ{V{OuiuYDIYd^dKQO;UHBmnv&7P{VIc(DIS4zl}p;!;0V)1b(iPZa)
g5R*L>~GcyBx5L#m^M|4QJyOV708G5G{LSfw)$r}>916OU9;RJ(p!U$U>%j+rz9`)ordS)_pC3u6;kcENGA@J)epJDaE&)}>@b~MFuKEjEUjtWo!SmdqBEO9
q#P1z8S4j*hGdJ!r7G46Vm9e(y7VDbF{ux!5lQ|rpjhxG@<CWu0Aj$XrZ%OwI!d)Xmc>(ls<_iUzte08=?3|SG?hVtYexhz`WL52U%D5%@^=kcq*MNX`DxU(
OH)@}?^3qVx-s&nNOONE+-dbf84y<e<b3L(<1E@zg2<=8NC;)XYHR-wlCUamsC}{-*xR7oF!QaAyvpmG-pT|fG1k6{@o&W?r}_oH6U?hZw7wBE3=V)dkk;J~
K=9M_W_V3h|Ji2P)mu>M8q!q1)4~PN5?FZH4OGJb!)b=FNKadOU8lh>MR;}1eTvn#xg;7GL0%QAU$>xf`VR0JG*{}r6#G`Bnz6r8d{~!KVcUx`i2yRKA~HiS
-VWs(D@*mwU66x6yW@DHY0!*x*nbYc>J9-~4VERJ9?KKOuGMsnQhKYIUK(m+;J(G<O4?G5%o!)lU5j00CH$acHyFIWjMpO<uQ&$f7M;UH(_%g!O_)M}f>}2e
877KdRqcijva44;T9iJosZyv%8+>&TU7%qz=MI!hl|VI~r#0NxPO;!{KJ?j(1K({$By$PaVm{x4D2SK37p#}xlj-i8YRRFv!ho(+e^mQ~vLETn3L(6peRUYD
5CTLXBikJ9iwKPdMxM)Q1Juqv1E9k6F6}ik25*4V0LeoYE{uTVe%w|BBh7FzS_9&(@K?wZW%O&r(qhkT!6z7rZZX6T5D1lLpf$n%@o^Q}lI9CIHm-y4_s`K2
^W(PWNRv6~4Vn3SpP^dB!2OE(EkML6>`vJaOS+Wm;Ehl}b{z@BNZmtIF3AZ$apE9xN?jdlK+a9`p%rSU&oNL{_9DdD%;#%(tk=7<;$s|zBT0rg@ruBJ0p}Ey
>gVjhpKIpg<2qdKTDgzT5=8;+tdOYf!{m<<I$E{5Agbu>vQE2x-2NnKy?ugl95&KLphe5LpXEZ9BpcJ3*9xK1(6HflK{h($>B@JfTcWlGOtHZ}0iXv{NKAb0
b`nRcv6t5Ef4t{sx?|vEo7^X+^bbtY`TOnboe`GBasGbu0&|ZNowU!|L4UewkidUlAc1j{<Pi|LDIU^AxwdrWaS4+3>Wt}{PEskAs^)Ur-6gZAjq)^u*^&du
5e9N$th0F61WNp&$4QV9*l1Sa#b~hHt*g*dYx_lfR)<jU#gU+gvrlGmTzbVTY0g094VCHi1!%IX4amH;66cjKeuy=<`caez-lh9kuz}to4^|0NhDE;SL@gv+
m3|ApBBC3Um!OM!=ljI|cVJF?(%koPLW7I0xIC~FV-8wmp-|c!k>#~tWj=6aLtKIkKCC3u7owwm93qnMuA{+fx4Ks3@t1}{5;Ms>u;y|*f{50|M8F-8!x<KG
Fx-W^%u?h0&COt_eiFV7Fmxc}r4=h7-g0Rxf^+U+<k%^JI>w?cTQ}FGum|fV6{Yqe#P43sS2|t>2&eUYO6B54%j8`DE<pZ~n6)x~;h==&oD(!-pVN9y$Js>j
E=2k-O`T{6K8CM)&L(m1HIUy@KhduR9p3TZK+*xxN35h8;MF!$(P3MZ(70A>*OxV3aAiSyRUd;$;z04Fy@t~E&1lq!j;Tx8ZfZ7+Y&RR#Tlq@4?aF>ic|_I#
+iRqQZ6fAW+5t-C&fe|oD0yaR=DACeG08ks_A;;{$_h_Zay%BA79~ZS#KY@@+)<dQGlzYLT7RaXwQ;;cUmZX3dSe3*dG!!%)cRVWsdOv>$x2+)y<~v_l{f{J
HPy^Ol$IlcEASP5hD4iCeX<3mE(t4WrzOCL;%5-}9n0(4DNf=&zXq5FXZNxo#X2Ncw1nrr%&t`Lwz!f+XG$Txv!Sv7#X-2ZNb@M<Zqqr=F1ZTvUI~y*o@O)Z
a}X>ThtxKemlB6-MJRc6AF%&xurER>ZCOLnJ^}H6e8yMFVr7%YtTNOFFr#{Dn1iBAWSIwJ*E9Izm*gt{wW(LR1qYQ_pSp8+CgBtlVG@imwXR9-<SaO{(^09j
G)`BkHNF1r8jD;E+H`s-P{9HtgZc+X>9d#h^3qiPz=hW2h$qV{SvM@}@;^V!h3=aQ1Z0?Tg_mu@6jvuM6Y`V?eJ4<2INyivEsSCyUwh_giLDzK5Xj)1KsB&;
5PZodAU1e~vi9&PA+f#`@<k3i(stt<a}&^WkDCsK!VYTRE*x^g>KVgHxZ++<KM16(mx64>Au?S0+E%<n^5-&zKNe||C<wYUtf>!Azxsa^Nc$WdLaUs<@X?66
DK=jTu&B-z!(XmbP8YkL6SU~95_#Ub6jHr1AVg?krNfB)ZZv|i6B}9^5`(OV)oW$RqbYw!GkAh)g9Vy}f26MSbjgg&WsLOL6yNt$9ZBv|HW|9kUER^4*epCk
P4IeO8fjM0r4FSL-?iuOF`yW4=lgZ4gc#HA^e#mrj~~yLWMsT*hSyl&b(FQ~MP4U&>sUvafpJ)h%|{&bO|g5g_j@?;;Z6K*=rVza)dupYGN$e1v?%iUmu4}e
Doz#)V=qs3J7d5Usq=|EBLf3>VJL;gxP(A_3)&K`vutAFG1M1fzl+V(g3CbbBSc7o_qt2%>TGF4$v$U?<a@XXdsq_SlWGU${q3uDcLA6-6<Ej7J-eOmqVA%9
!W8eD^BoWO=oXAl96Tc*(dN#S<T|3z$&`|qZB^Me!cN+#9GVs6@CThk?f>~c_CmAuGY+N5{eh~*`mC)E(wrkeDy4W;M}Kj#Is6y{vIj?N!7#RRbthTK5SGiB
ZU-^^sW>TLI<+kpp$+!!n${%tvQJ&Opvr0L9Br@`j3*G_A-dp=GPP^_x9sA8X}QQ`;9fo2VJ`{tW@_nnneaht8M>1-VcvY|pQm+x=YHngLx#;4#iWjn=R1<N
ihW$q!X%m)j2UDEMRFIsu>?(=`%KqL3D!VC!Fy<Pcb>~Zi_xaQ$h;8I%fZi==VKvIal5;o33?;4X8(#psqG(DQ!%vN)hPaJrT!<!E0@$R%s#{<dY)TT4@iB4
sePn0jIk)=_v#C3TgX_(wlHI#?d#*j-=s-<dGSdHqw!QS_T#(YH80%4V3KW#r=4;=C1qQd@DZdbPtLHI#>e|eh61@m_2ylN9aavnlFo@nupn@y9(A5~PpX%~
00u})lPe-;{LZ~aWM^#>6rHaQZX#SMi--~mqG}(fu@TzUsp3`kg}Fq8a6oD2UaeX07|(?9^N&P85R;$kgpg006_;owsF$=&tUk5IPi9Vvbroq*g3t*o_!76-
g_}^{LIj!b<xc;0dtuFX!uoS9QrIyX%8uEHnVUy0{R)pd7@=(OAY!B_)K7Ykt0g}_k@<&X+Wrl()2F3X-)T<MlU6r(;oaaW>4P#k`H!h*-hP?(Z>qq0v>WR=
yiHjPs~w*}^Xbfo$-St><v9EjSo1neQF_4v_^V2nGfZ;*)jq8dMqe`zK>;oCcJ>NZ%lQ>0mVbWqhEZYM0Gd!IaU(E9zGg=Y;)K7e(Vmv0RvdQqy2q^Ht5!O-
=lI?`t6hS??B4n`Wu3{~s8anKYjKEYT0{t)&9wNh8F<0dvr@jx*Nu}LGQJ7EvM@~Qh)bN-)-a>S?y$KPZ)K@@lfqND%24HYgF>7n@JPI-_L%EZc}+!{$Ch55
0gmWZ#-!&>r#}-559X90nKW89dsN6q(9$MGyk%|&YF06;$2p=X>gygkX(VnDLk~eDCy-Lj;u9X*(nfH?Fw1p}SutVQ%ar%J0>5Y2Y|{P7rnOtiU2Z#%e@EwD
=?$syar9V}UCSH@5%K~f<m5SbMM188;s4;V{MNk<XmV*|wHbklcY<xD#x^GRGVrSr06pE1leZ6;@+Wm*R%}+5DZ~ijwO~YGxelrnV8Q~8ERY?y5P)^(rI618
WJv4UH|s1t(VNMiM8t?kpeoWr+Ew6Pj{$Vf{FC`W?lr(@Gk<p|?%+P*Vh@S_qMImPZTn{0(Xc}JM7esJ5W+vujhv+aSkhuvkiw-}j3o!Tkmr9S;q)wqbL5Q>
o3rsFxrtqEys?7Y0s4iilycr#H81iFdH;lk3>+eV=ujYib^xlBGRx>ybiALKe!q?C013fw$T;?>*7_bI%Kth>v4&qP11MjB-raxB*ui)oV3hc*t~^R?zflcr
V0MrL<U?y3mVWsZ&Z1(IS$<{%%e#q~nTY=U?#w3nHB-noXEx9aHH3OkCGejJ!)*U-`0I6q#4L6xdi`VND$CmvKv!8Eel<13eiKPj9((AIas%SM!Oh7n4Np4d
_pdyR`kvu>Q{=C6AmaGXZ``{S7W)O73D8)*i1#$85c@jn$9bOZB33oWhuYQ(m&`)AC!I;69=;B;?@E#swzpa`=7+!_t~Uo<9@?92B=4+IAWBR!K>Cl60%Qd{
BY<q1`!-{!^6Ckf`oa7zKm_Chij&(6TdxUKD|Q5ZVnJ}QB`F+k|Hev?;QGnZo2fV8S#=pTZ{CsRzymmZJz~dw&<_KE77#YeD;7(dsrr!EpT|)BbK}hYXxq~m
RwFIN3@R%6i}P~<&a1(yV;H48u`=EB1GXuHrsE>grs$14uwRXL*O=Q7#$Bt!8*AnT@rRj^+($?a9m}sr29GCO#!frCLl35?mEp+7$8mC(m}*|g34X0s7F#fc
jb3LY%>TD?X8+<5zN>xGJhw}2#1ZPe_fd>RTX~WrhZTY9J%{NuT!XdafJSukwa`BmZ|rOEW1#=G8)bR6-}@-s`3Yztu%JtBMu6^;v{3tjvly5F4GXh?d7xiR
L~?``eoMsaqeHrhC>uTlnPNLmIC|fQuP~@T7HjkcKtd0?N@Wv-wTKT4<3FzeeZBxMxgKgB!M2La+LKsV-em2O$@V+ltA)%<V{XD$5iJF9V0QWyeh|b(8k<XV
KCCHmMTx0v7zkd>ES{VT?e`@2XDgt&66B_I4Dqn%QG#%V)jOT9R;e?0U~?hv{$srjH)q0oT(AGA;3m{`goO7Qr{AG^vO0c}6uUR&zt&!76+fSM-99;Yvi_<G
8OubQmH6~xS{dsA+kuWM44V9#$!J7xfe$TLd}psL&#tsTxd;KpEIQ@U^i~khGI^xi$)cn|0wx%edFJH1d{qS;BcM*O_kp#y1hHDbyW;l{)MtL@rBk;?9!j+N
AbIpQ8+J8OMz{K$w^ku_rCd8mjyFM?fYkC25_6O^Yf&9;-rLc<x-T3QNnsuYirx(k_&v|YF!g2KVHksl_2qX|6o=&-_Q;>rZ5P)*2gk?;Hg<gJ8BpM~*WlTV
8jW_Ne|N~&ZIm)&`8Im^6Tuw<d*(b<YUx4R9>S1pmeGYNB<G@qf6JXSjNAMJkLGgE-r(m$fNpv`O329YnPJh9xs)AEM*}@f-@Fpl{npO26$DpCaD(V;+}P>@
P>FA}oj1~tQJf>|GSUFk-TYs^MWM<!up7=`3k{x5Md}<O3kt#h=N+(=ky8&V%+tL(BR3;iU?_NO?R^3)X-<sB)O-6o$bR|NaY34aFjfgle-tU8hqYXW;^??x
wn>&+$%_#ok7aztMSz-`7cM_R(3`sE2TMc?*pMQmY=w|E=O%g4)KOn{*P02#msDnj15p;$5`2#uH@rj(f8_lBnQR0S3hGEqrJCgp^~?l1dYl)4E&gxq9vlM%
g+QABi_ktj{ZT<bp1WN_1pj&rNdV89Q?r3i;cApPLmP<Zx7}f!;cR9;X%o>581Y03wxRv2dy#et!XzL&>bSms9=WK0)W9#q1jGq~Vc!!k(O!(96{7@GU0~O<
he3lK&0zdMq@mdbno-AJz}J4Kq*{WT?xOluniJyj74vyOtU}UBlp^t-ab-pa-9!Y)wrK|P;`}BCAmkY5sA>>tSgUCr@Kyh*zpaG8<jdMwj=J;)84TX7y&(S2
BRHVoxV~tV`ZmxKwyPFOZ54B0Mjijt<JDD+s8Qv}&)5<uSI5cdhq9M)GjfZj59cMXSWZ`sB?w?K1&#)LQgV;mwm*}94!}~s;N|=q{25UFg3fmW%;?F3O+^Z5
{|jld4(qtEX<855Hf4rgUu}Z?*MdV_Omer~Q~L5`%mdo%u{T&Aet>^2YbtcYKY{fO(Y|x-U{fl^+>vc|X3*R{+sSbK)#Zau2^t2Rv+Prck5q^1;XM!&nao2d
-L{7#D)!<TZ==@7Mws=WlOm1&)VoM3*78bp(fVOZ07IHine_{!Y(i|RseXf4Y$+GsZ@WbD!&xgj7T8AU0!2NEsZ+8GTZ>Om_@i!UcD-BS@X;F6EA%eYZE2pE
GpQqv+GKIyR;)H#@1?;7X~ufo*b4x|D4sHC4VX94<=d`AmYQxJGpVbY(wDT&lez~%^bX=E><@irLZULwxB83C&02h<(#`gJq)4)`)0eqc$L|fcH(}c3<z+UW
2r8_>*UOPP6I>d#qEiICt|CyH9i2!7si|Bw31C+lq_2w!Hz}RPFrFx%N~b<?Rs}e|2D%%!)?DR$x;gr+IjCxezsn9?7le-Fo@BwTF!d#0>W<{OJSviIR7qYY
Dg9`NA3pT#*ppF!*OIOdoBdFq&k&%_J9jOeQh$YzEqk{Ah9AOFRu&ODbh%3t3k8if&#<%1nrdL*H5NYwNfEfg2Etf)=-&FuC=O3fm&zbTx1S7^{Iou>-g$m0
`8*>+)LrXN-1fD(ykxd+bepe1lf1H~_`8-fRCeuFyKEcn3bhwW=Q?26QCK57=H@NdKr~(W#9lT$!Le!#KmsUS6%qAOGD_?kD>hl_;+wW^(o#imc#>EEAJ6KY
*fsE4D5^XtRk^fep+km--bwaefEbywDsPYWliQd6aGxDwacrHI4ukIM4%xqn#K8f(DJtWzh)!zC^@4sdK~x%Sh2-XFZKy_3$Vs&mD><g4w;Uj|tL2OI1QVpD
VI$RZQ-}n#-+Cb~b@_6kc^3P7g?ri7rCAt`pdG`|PvWnp=C60%CH?uVYu&2L+nZ~ECX2rz(U~<Pk@!>_)$4jXh0uCtrrHdwnCL(d`Z0e(rWm?Tu{We=AM5^u
f}(EGJgA}P$oIgkG1}7fC-Ew3)-NFaVc9U~bbY^KD@f<xVoCnO38}V>>z{gyzyEVA>N&+@X@yrAm!P1Gke&74jV5vQz7pk29aZ?+T>!zV{4Q6yq%+OX*fj(j
!+ny5D9r8{oG^7CxdgG_bK#V6y<M)Z6kNDV$HWhRzcp@Cg-IVUfPy^=wMQWBMXD|qRH0+#=2qAsF4NOS5#9=lpw5P(9e>zrOTcK5Urd=y&|OZ~tz{4vajZgs
Vk~)!x3j?QQy>5T21dTo)^85vpJ_q*uT}BCAhQhz0+fV)k`gDnHEq1au=RFt@sN#Nlz8@wL*Ka-C}$5(Z;tkwU}=MVz1PquL*Dk?IY*1$zH_jxyHX_+<^b8R
=2|?ai`~(Ua?xIKfM<})+Zf&c@q(=DNT995yq0!)9H4n!st)>l8LqOgXMeDFaRls{ogdS#g7an601qSGvWc3*Tzmj6iS8^)e(|c2>>(k9{aoZ7sIUN67+nMv
v3X`|(h63WCRI}guS#>%mlSYS&q{=tI1tQb8n&V#cs0!`K`etDi#^i6POB}*HznW01mZ3)9-XzmyP56Mm2c7QJemFMgZRg?{Ogb*FP1h?f^yD-pwNrcjnL|M
)tKgG)zT_P4I4+sdm@dds-B9`U{ME{N;OMX8uU9D%39zN5?sO!3g@zxsyjL(cx<96brFL4k5Oxd0MtLg;`o1Xt2)XF;~L-(fFiiKfOy1sV^1NT8>^iN33P$M
OV&9Uj*X><5HKZ3M(Lu<59u+ZmI5#O7=O)F)2yn-_LG+4w6zeNe;Cw;Bh3$5#^O(r)R{346{)W1BJ}%}QN~fMh|L+=6`w`A`y@*x!+G#!SaDNq1^q8K+ViVa
rqp)&`UL+-h?Eo=%6Gg=3|{-7LeR)yO))*Jz;h%ASbYU$jTi#-c6}@~*{Uu8-=sq^7E}Haq~%h=sX{8iJZgJQ%u4c*Kk0|Rx}vDG5OLIyZlI=3U92Bs#{prI
k_Dh-#S)5u<cvVofcrdp{6`P!t+0n@XmqRMo5+VUI7=FO7pKe>p}(R@CG5Dov#vE~GPQ@X)m5CPJ8(HWfFU%5IWRyV(xmhKtdv`~nop)e)**-T70f*#K%Fur
`QI+d0)LzSV$~59o^$~%d4y@2%7;b|EqFl)n7}T<gk5Ku$W2WaTW8{|U12@LOX(53pUirH?|U8e<;y>2st~^)ZX@QV+Zw0S{u88!C)k#BR@X9VXbgelX&Imn
s&DC!Ti#tZ+CmOJOVsn<SsZmnRid5DrqCxc3H=8x2g~pC%U3NrBFZ;bBq-f&ldm^gHX8rX<4jX=B{mkN(kWN01KR?PMs|-=Bpm&(HBS0nuhW-#Yugs7bAh{X
;T>-e(D|Y%@EkgPOmY9F3j4(01rzWm1t}s#LF!f=(@d2pl?RI@0#JTk6!Cm%U%}e2tLr$VJ$enV75dID%=WUZ9mrPtZ<dDFX`vRDyhcx91j1|+fZ><FdOYal
gh>&I^(X}v<w9ktA#++*)~MTg3$ls0T*>13D}00hh%sApV2jG&9D1~NJy#Km+2g~36ljGe`5#R#8UP+DzOYK#QuJX*Zzw-3nTx$7nhf9wfJ9r8V5QwwBqj%z
jhtyjCB%KQxSb{Z*0=CU?@Vn<_R-$Edmj<M$SYbfoM#}=D<LcS{3E9-U}Is`F$O78ix(XF1jXyB>f*)#wGVn`<k^B6(e#6)@}EQRCYp5XcDuA_3neCxZENja
bG5%{H2yqsHf%2cbJMH8mi%dstlC0Cj`SYu<w9Xx%6yV>tS(YLnGja%WNs1s<AdPU87yblhCwrC0DQ5QJO#Ks8cy=mYHwhp5p4AS{SK)~=CxY5Yctp<E!k^!
WHNP|J^9bLH~K$MTrSL<wPIYSZ{(M)ldLF33{(sNy;mmIj`Jw}=oi0KdG}B}wz^uhAWwQ1eUdarx1Z7^UVw0$F;^S4vXs2bmb;U0uKXDj!*GDntdWY(EOu61
uh6fNII@I!S-hEhq7l5x)?RjT&`nxTYlTu2;CzuL;n-Z2s0>Ug5DxH6G#>PG)79`1yg9RSc8=$6RrpqA0>&!WH)C}IZc{t|A)5}(ghV@v&Fu)KWxBIq7vx@o
-?PqjKodZALa7~M^TJ+^woSWfl=dCUhl;ue3Hm)!&p(8uv8DD5-bhF!1t6|h>&YnS4C@6q71*HoAhR$Clbah(iudrkk?@Yvr<b|mCu9*0k_{2diJ4Pc4tL|q
W`ZSUPbh_1M$AU6qDIR|F5!?C8{BaGc|1tm<WP+`mwI|8lDIN_X`gGaIb!a=c$~VOo-NN|xK&YQLqmnw45nW|8htJOZy9qu5Zs4#`H3;i=AQ9)5J$$z8s`XU
ojVn5?5R`7W6SH`l@xm$fl}4(vq#-e$D7ZNDj+E;4zrUj50MNO$duUawuq%SUx67@28=!R*bbUFAM*u1YgO!H6CxtW)Cn(SWyGW9(pO_btn-K~BHsWp=oUzR
*lK8LL{59SpBrN?L$hP$3uo#kPL%~qhUr48;yFK=f*#gQ8Nx$FR+-lb+*W}(M6HzL@!Z4w;fw58t^kbA_!k@&-~9UmMVTMIOAB)q2%hiFzk=MQ&=8+2epG%)
tkb3*b}J+?p^9$jT3;FHtXW$8nt;RE(O>?MXF#H3%svq)Cfr+?*rz><pCw|9LJjes&{c^P%~YBK=_G?EHP+6bg^FaGrCTa5PI63kLQdR-E7FMgfoyo^{dp10
jgV-ulz;8Y(3>laZFjlME*kYjc;@@(nTxIr#4WTr6v<-fbHl{MV||D{Sv7_IOkcD+e#~A2l|@El(uzEfj^w|1AjNrv8YjVL0@$({0OoU#^+JLfx<LbXv)sJ8
;7a+k>f52oz$&lj#TW|c)(VXhA)6{^;^&4=8z1z(Xf=GSqZ;O2hF~p9XMNaE+3_~C1=En((px6X9}ul>kwHwIVPXd|e-CMf`Ab01vJi-Gl>c{gn%P<DMcsqC
G0r9y`0xXIVeT!v&5#de@}=#yGBqO6<^NML$lJxWn%_toDx_EM-dPaD8xukN@4OC)A%vC~DIGaKFHJvQ9pLm3`T%^o+hm;nCRi^6;Z%c05JNGUjtd^9Pm1&L
a*JnES)Pk6XW|7gdM+(3JNch{-^!eoe#|<PRd^q>(~&lFv+XGb2=1}s3NcUX0gqQ-siugR<rzOtkd=)sifF#S(CYunO{;6p9fM=EkFAvsiwYSF{e3&>YpC&Q
y!F$lhQJH;w~}O0p)HB0<<W3Dl1W5_v&H=HA~(@J%)grOGtM5Gq7L%PI<shmmm7SV-W!RL&0xL~m^^ufp|poy`}{{9n=4o>7HPSCN`KeU>Ibv{;84#2&~QT4
;JBRnn{rg49M<t??I%9$Cnikip1fl2I}aW`ZO)97T_G7u*#~GFaY=Xx>@E~uI?$;R${beI-7;7syKD*9&*OGfGL%EQ=D$xm+tAf(3)6NB4T2QxEoNuWg#N)@
5nGbc@`TP_gDukrhj&g&P7zf8C{J$r4{CgX?beW~Ab#Dc^sB7}{GDDfE!gc<<58e}aAI5X;R{Poo)Ry3S@3$qg{v+qJZSfx8*fnU7O(-ECi>zU-OyRVgm8R2
tLMcjD{Nh@bL`;<vKb<0o)39c&Fae}@x;jh3-FpJUIoqXJK)+E&&~%yEC;aS<0}*a|MYKvA_GIK&-y`#6PoC4Ln-wM_=WGWkF)_NmS<c1iEtyC9EMjiw`gfq
>{4}1tw4&cP0#6TV=3^RfvV~mM(wKu3Tka4^4g}>&j|(`Tfj|+A`6%I&rbc1KF}afR1xEr%yA?`Gd7uNRp#F&<*XfHyB`_o*BA`y?bPWUG685)gYu@k0veB%
+k$G0E+QqKn*QdMF&GNKd&}mOmMaihZeIi`mXd<w;u4AKNLJKn62p_RN)rJi5Bba|-sp82J~6qVe`5gpK{<g`Um8At5(_qMg=N;9+HmvH8k$uY8Z5IC9R=Fu
aSYGp3S-$mcp3xW-bb3*xQ2f%%l7G+T^zr%r;U1q_WClBx_ko<VE2bJrb*Q-p415q!~%I!#eWN64B)!*z!a(#wuyUnwI4{xEL)%Mv~5;cKcCAfsE-~Vi}1%$
DypbaV6OCal_eg}4C}k8Q|e2;E9l6%6eGOZK{IU!td{@=ES%Sw_{gE%@niD_&#Xa=YTuM;S~64|MME9zBN=NlC@@CPV}IHS1O}!4jLY%DHLDk&FJW02_*YU)
*Jv6CfS$`GICWLx_=8P|hRKb1)OxyoW}GMYLK|Nq*9M_pc_UN9==hYs{3E{a`NPoqDKacNH*j|GERq>Ql!MKs6kBPjol+Lq^~G=dPezrH@AsCZI1sn>$3~5H
f<LBn7`eu?i`xZ)SSf%cNECm(D;S8MWd<|sfg>HYVJQudC9v4GoL8I|xgyOuuJ93P<lHyUEZaq8!LLM9SG)p;BK{-!9$!<*{KlENG|#dba1~WdHven+J3jO{
J|^?V!s0JobGOTDVDBUjR2dnIU=$P^dampv9>;;tx$I2O-h3=0{m-krrSh}_loxQe-n15WEDv-IGj{Ijz-hdb*2)W55O1t8kQ$Uq;m#m%lK{xp8m6$(=F$Zm
!%5b|zE-9RTOD;$fhut-d}7wCex^hpUo~9BO@6~7@Oq}-Y~2?wwdmW|acgud+;P0PI1a7IZ_jPNOkmW)gPYtgKt5{NO`Xkgjr4VXdU|wyj<T9#2AZb0jfGUW
04WPp0^IGEMu*g8mR<NILpUEroq&qVs;?G9mi)5wg~Yv?Wm_7*TrrwLr{o;kc72w;j!;`3*{y5#4ZQfGiOWtNQe>j$Xyr~=L8u_hR}@vWkUac1tRCV;tkM4W
lh5H&g@mO7H=sjyY^jq5PUT{F=mO}HopVp!!i<@O(ckWeUbE&9f|f5?Tasum3^K;my=53_I$2Eu#fw}#K*F4ggVK?C?uZ+{6Efp(W=1J$YbMX8UlJN#!N<lO
9!$P{$coTe!~Kg2HRAed;T!_uL|tztN2pAczyf&nMaD(S{*_$<i%+0~ml`f0b14W_R|Di=vQ?rEZ0Cd<3_Q>wVKkD6yhfB9qD6oQ7fz*=&oq+l;E-%bvj7r3
kAQ(-9>#8clsCN(>AQf^CeE2aqp(R<Rg*d4txFFGhwFUEhE1H~9!PdsPr&jufyO-);GAIn+TdN+x1+8-uL`sAh&H<Xz=5~G&+1Mda6#|>^N@dne45d*gMx6f
PEjKWR9_oBQgB&j;$d`OEZP-|Nf!`8#*zPD3~@HCU_Dt5h*ex$_nB8uBlDtRsNC|Y&d^<*s_gaPvk->t3Z;zJ;;Wce*QED$n4D0%V}iZWlIF4dbllzm>OU#0
!<a7DxPs$V$2Z>222x%IhdrT{OK$x$2>zvb1N^-FB52BiGbkng!^>owIk!4VTdNhbf?3Z-qnxO|fdu1s0=EQ^Z9nXawMAPiWA<snVq_6wNag9d%(t@ILq!G_
;C(d1Hioz5mnMYt5mjBr!8PC;m$Z@kUt1C(p!=6Anvg9I(cfe;rPb{>kgb18>{a(?Bg7hXBrQHjCLs1)esgT`joA<0$SkvrKe0O8jPKwD0N7cN1B>jU_HYzm
@ev6j6}aBNxAxJw<`fm{MutmTrq{`Az_v00Vl}Kz3CEt$k~n%)>S#vkN~(6V8U1N42s1QP98CM0ZKN`P^Xn}Lc^!Ok6p;BfEo4j}ReSP<n4)+D-30{{e+GZ=
qg>pQs&f+@lO*e(JNA&M(-UZw7LXOMZu-UvrAXt)G(V%SX^{hN{>;?;<>zCkL@~Dur8>j?9Gv6I-EA?^bp8Q8C)I5v)A2ngK`}%1-lY9DqYXg=hu#nJ>GSG4
?EK&r?1u?n;U3HmT*RVn5^=r|+~6WAx(*yNmaMxaz`J2|+xvnv+5mP<{9wi+erxLjIV3reW@b<xf7TO&=lr)_l==yOwBxat1V0E%Pj|XC90()IzN@Q@u804c
KTgO^12p)vSw6#S%Z}}b<OI=cSu5U;0p{IYgCQNMI*Yxyac4T3{Cf9t^GuVoI87XYMpz=_M_Y>hh&VcN)#ma}Z4u6Bu9W_hov|~BsH9lJB$-3l6<>6!_G+(U
=Y@CgS|%FY-!Zk`qNQ_*Pnh!b=R}rB+1IW`A<ZPajH+=IsL5(U-e;xG)NrW2W1mS}5YwYkf3UjU##7+u(3!nzH^cW^;GK|U+NY!mI<TMt2GjutAD59_l#QhI
L2<v6VI9C0b6}PJA*Aq_aS)KXSs45OJNU{F72avkn!l|TIPvlC64enhP(*>fI9PG~YX}i-1}mrN*}m8Rd(^`sWpJQCUQU_EX|&wbq9I;Dh;XoS(tcJb1jfml
Y2x~2#4)E$x~uBh^kAFB&2fwdHId^5L-XU{%j=Uduoh4qJ-DAeUCVY4`C<7nMMnzQKcbat?E8RrFVC#Trhl{E!J(cUTCC5|Y%|uC*Zz><WFm-FxG=iDk$Q(@
`Kv#fT=;}%J9+i(1Ts+L&Ons6^Ucu^vhc3*&owKa7Ixe?Ey#n(AL}@L842*_$)B8*0*9*{Vv+~w?^Wp4G}^;T<xG<91^522IpKzxY&uKk(8$RA8I7pgABhJ>
!f#+n$|18G>$&wERVXkrDSH2@SY%p+-rVJhF0Mc9*8gYq2RfUU*L5~!v}_$;=%E<E)#_F9>q!#kgAyPXq{Z1S-K+~2vOF>-&mo_hyp;IJ-aG)rv)*_TR)nX!
_~a=ZPTvUxfFVhJlkVuLRj~fORJw`J2jLy~T)!nP_%X89!xoKeAnH`qrHo*Ps1AhtGIP4q>34#my<^MSw86m?QEqlr4+GdD_lY-<tu=IhX}?3zoK%BMDjj=~
T!cr?Jnr-fT)NNBp1|!+1NR$BS&7lxssH@O30d^Wb?dc4!QIL7xx2?DpXHvr@xK18fS{D{DO3@B`=>xgfuZSKfT)#GZy{aWnv*ewbH|3JZ1LfScrDL?8~tQZ
$U4Vj33jt{w^UJ=SZ0+a+JSzL&|hk?nmU(QX<p)TxXZPWMf{NH^{*s1ErwaZ1g(|s;GL~Y(Qd8YNYSyOj2GPqXQ3DU<(^CrUCm+%BbNN`u|d~T!<PrCV@DL%
2V0DihozhDU-C-RI7{fX7R;shQTqF=e32W!cAZLSTwF7=V7OEsW{bq37LgHdoB)-hr*FF_)S=3Jqsk!U#5zZ19V!%ro)HqFh~F)WZG^~*8&1mK^r-nZDo5YQ
KN(r8GfvdOLxr1A;q6Tf5MH~W6!s5+G2KyAH$l30r;20EL_hh3)i(0Ef)Kk?47$kKV#c5)7S5GREj9&H>@7!@cT&2rL}kDjP^&v~R<d&==(5j|L&Z7Pk3QHk
%F`2u#7AVf9y@mEs~zmQE|;*$g?W{X{Wkz1RedH(F71GUBc7g>@{!QV5h@PBU)D-41Md>`dD55%3UXXp#iOqDv$AvQ$>tEzcG0)00C?tYOL<&gHa||s%;K?l
{%Qs^$XsapqL?ni+d!in=asI$-1XNpPLW7D;jGx)+rvRKx~e;0L-{rq2F^3yHW{ZzK*h`{tAOLL4dLmlZW|O0k0HM{`^6ayB^dUJ*75<X?(xiuZ!}pEz*9`X
S((ec`+1`fBzUKcI%k=>9OlwQ@OLQM6@2mSu9qT)>|VH{$XGa^)$bTZIn<xFJ$rfTQO!OOcw!Q5$K#~0y3q$>i{U20^ZPxKdK?x8FTN1?%rU+{T?=@drGHCz
k{&M=-I#({WCv_&C<LFl$}XmctMttebH_lpWOdj0sZATGGw@$n;0c!VIhLCxtLjq3VgBbtA3)!Lh;erHCAL?KCLd`%#MXF4i!Yd@C)0!6mguw30K{Roe`4uT
X2-*^P*{|UVwzKVFOQ7os-!z3KN41Sq8nIRI&UORK-n_4s)I(g(98|KI@GXu%+~HRzT+M)hKa6x`7fno!~mz>^;+gY(8>4LIz7lwGG6C+Ff(_v><@*O;Zr<l
6E6i0Dn89r0rX-^v@e;LSrrH*p7Ta$3Wt4X9M|OBkr;LVH*`&Zk*0oTYp2F8$Jw~Fzv1hd&@m(WHFHyOAqH9s#^*c2F*y&B<^}X1K%e&D*rE!FMh*hA&^yM}
)EW<c>ZlZFoXmJ|1d6XGM74q_mJ)&iTRA$&UGd-|yo4JutzAOT@gcDSgj1O#gkHXsj9@V@2RhtiGZSq5&ZZNO>tQH~Fn@SQr)B4WoP-c-WxYr<Gy=<F1K~YR
DmY2d(iSb@{D(nY5&EosbFzBJBBSA7u_c?m7|(i4A7r>UI}u@sVemyQTno0#e)_~LKez6Owsxn5mOy#W6f*6HsJjFY>gS3R=nvw8v>8_x+rVl-z#2`{*{n8@
ZjrQl5;lMH3a|qr&+3;VC-#Qf;-S!rxyRu3sM5fw_CsE>N1-K5p$Q6tay$r#Pr!W$bbRsI;nadLC{T&yziN!sY&8Jx!-AyFMA<sbM&bkt+xY@rCN@(k<N}sH
2p!%)Kvf29;iz7(3Y%ju_7w{ODlEIX7(EJ2IBuP~h>3pqsQA&2(Apa~np{G`cZO7<XhiAA`9GHEJQN{*Y9MS7sN%=y%yC)1-n|UYvfXk*SI!m&Uk!uh;PAK2
XrUd1|Gw`B4sjx3B}JMHhBP%JD#k0@1jft<y$!o7C?m{dzA{SsF|pQqLC4$Ywp~Da7y(nHxCJVU0MAreH@TuTv@-~tmn*lrsQEyW(00q~$dDkS=FOe{Uw0VG
3Gpdq(}zMF(S;zp%xV3Td)ZP5+N-L~AvhMgsMjp%?EK%dPIifCd-ZoKloR)3Rwr&p2npO>ISmFf3tZEABsHb*XGbzPAO~fQ%S&)eQa67c!d)}~mMUD=cp9EI
0A$bU7M4{=(vuw{v8K;WF%vAcJLm!E#xrIq56yhwW%#Gi<e?&3E5;DG@f1Bh<+$@Zp+ewLu0yLwfQoRp(64Td@Zse|h)xKmr;B`t<(sAxP8hTdoI#-T8Qh?@
%<gM7{A-h0Bq2(;`UYMaG?7ONRFcKqmr+7;O59KVZzAwvzcn_>?ImQRuEV^l%WAOZSIDh4t<2%{B&4};r;8&(etZ2x8o$ngXfF=%Gdt8HIc}r(z*s(LdcA1v
xyRj^ECJ;9mQ~l{JkpQiGb}U{r4=kMvn;V(y@Ys`Ow4Uzi?E2v(FEi)3bVWkRZ^NjURQ2<#ND{8Ft<Dq@gtLJhHSG)w`R!(8E~VD-4Vjgw=4YVe5I1goYmR?
xAe!l+(9J=37=p?08D-pZBV^heiW{9n?H=x2-h)CfQY(T)|w>PNe&tWfadUuupmj^@zEpzUK$emTmf$H6G_=49R`AdoK=Lf#07qpLFMbJmp#3<)-p%}ohO!}
9$0L#N+QPOhsH$%c5m&hoWr>1@WGWJE`iN-oKEIz9nXK&EEbyU%u-m6<!Qz`l<SRH;|>F^ZJo`-t~G;vJ-H7|#s2LwnRKP7&SXjK?S7_7iPLmKE?FDlC?U72
dgeCYQoNUf%*1JP6?C105xclr4`^f<s_Xs0mF|qc5`!iD>(`g&2earl^ejAe^x|-#g)7yvQY!3Pa386PEF0ClQJRb{jXQ=AE<4*pPSo|e$y#>N${<Q~0qbX|
IZoqOZ7=|kjD0Nh`#c{}JtS{YG)KP7+q*A*d1USjsJITsn)T?A5#A1Py>ZzTkIpW9J@$~cz$!JBESK+3K1N2A=|dd8MNJRKQ1-mrH#?Kbh?rkQ|MC>*Zb+Y0
^(&f8_kjt3G{313T{U@(h}I_3C3N?XqwC~1W1NhlSXbP6z$3De(*XFhNZ`J~7PS;3E3e?k-hZsT!5NFx{=&Hzy`;gs*7_brP5k$8+}}%Usw*GUJ66|8uHQ~5
y*y5GPlS-LA&=V~XI0~PE6m106-)~~_WbdKd#;+}l&UGLiqLq{Av&|eIq%FuvGV+ZFI3nm!t=*fDjoGqvemUBkeduxd1;Erx=X7Tf3YN~z-5`#rpvP%1hEL(
?x}{Uf{ZY(OgHW>cLbyN)BvO4vou`?WEx5?;0%zcB;;`1=F&bXw{-xHoHRSN6m-Ll?2Q5QrUQ^L@+p`?A&~wDcn$@XPVv!}M{9?A97-wG^F>OFhUu_*badTT
7C|0!kxwKb@jE>etg<n`ULtUj)oqhtvLBpZsXhTU00=r|<drmhAYDRFZxm-kfx|;F14hy~n1iz7hfc+gj<*f7q&P>e&m}Sq^GQ`QR8kVzNWk-^vv#|$RN)ot
=PJsz62!1Pi6~ywu%FUuMX~@&`HP^vLM%&SwUgO|tQrF0lg4C#{_G>`ySnu`GvCqD36e9E@RlZBU4|t@aPC?&VbH0pj*1h3B%_*S(g-OooTjPLOPdP+ZL|(4
y%j1_lCD*z2d&(jUu7ar7>c5pgL5?qHEF+^TiW{vZQCO<8Z2bc+3OsB@GtnUJQQv)G$V6FAsV*_(sdirnvta3R(bq(E+m_s6u-rTZUCRS!**n#ltc0_FH1Wl
@NwBhN8AjyCwmvvUjEFr<r{uP{PYeSz1QIj;0H1M7A|Yh`cxxmqcW@7-#Qy@j&AE4@h@XrEx{VS$tlG#sgDne6(O)ys~bjcf>15aXy}U1dNzCh7vD#fb2LjH
GM)oo{-?1A3(rY(KXK2U2&Ge}TL9Ydw(dJb2ezO;o?V{ROq~yr+0{lMnf7*Kz^o1O)&aJn(9=p2I%PeiK@QIPde%b^We;As&#hv3H)U{B$Z9%dG+GuSoLM>T
eH|<^(zXyv?Yrhb4}yh$>kiKAnaHPW9a=6w^Z+oS@|~5d7OSv>Ub_=!26W_+6eSp|FyjLggG8gq#@%ntxlx_rnHZ<5L`kUS9R(uh0qwvg*)X+;u#YsqXp80p
V~IN26JqV1xhjur9kp7ovya{ZaIDTW0o$a@?o0VJ)Mf)D1oonJL&Jx|^WHVy-eG4zw!av;2(iu)EyPmMi8ea1&a(JUw=VDTKq6Qq34qfI;$R*Z>1IB^q?Wxt
b8LW=Z}y^)fdarij>tT9QWf^?Ro8;QUmr0UA$*1bm7M`WF|l^GEQ1N8nu>AEol*s6Zm)f|;#sADILNxT`|8NZ5wJ4^&ehcm#J)EpT|12*uFc$~`))cM*XrH6
(=U8WusikP(oEO{HR4KiNcIAkn78qEA?zG1jpCC0P-RTdOysnk=>v8NjP7hMh247%=`{hJfwVIzJ%}f_=dB53!T!-*wnA^VBKM^DRrn5=Bo~YG1I8oQ^e^?X
)Yq<o?8jaj3Bo-^?I2X^*MPRo5Vt-WX$D#-eeP6|PD`2J<NP67YE!D6KF3Ry?db6WuC!t<x#gu#j&hwhDSl1Yre|45S#jiwN9W+dPXRI2%<VP4p5+YG^F!n~
h-iEC%hY<EqNN#-f>vBBBZcaMn)Z+>aj9}Zb(}^iUUC=T(rXaD^-vUsKNwaX8KhIx^r`)_+DUD3RNsSBfI!`dgG|b0<c%#a$w@t*N9$MgA-rYo9e)biL&?dQ
^rSj6%l`mFDjE>3a-W}$aR-+US|iQ%-^IJy;HC2@1s=dWK8ft2TQqE;)99#aG|}DKEH5a#Ocd<JBJhSZg9&UnoF^oK4{5_*lcy(i`^~|u0sKbG00B8b&SmNE
Wj&QvSzP~EkB+!36m#d~0q`MERut0hAPMPyPX}tlD9ZCALIBP9=MKlrmUnsu6i=XaK=U;{m2jGzs*tXtgQ24B9k*csgg)FWB5d8gDkZ!?_Fo8|p0&e|bKpp*
|L>};hQzn%l6Yl>!MyzcTbijcOMKIEA_==#A@CX`oDU}L2oJ(7CczuJ&2+XpF#za$W|0rYGzC{L2frZe%LfH(Y4H8oHQHU*K$`c3%hkusBAEpR><|H}bG;F5
ZbuC=?>blZA=RnP>1S|I;{zO~-MZHv-~xK{4I3ovFe-9l`1$59R+5-s#$NCzgZNFO(e=u>@lu(AOMW-vv2~FW#I+w}^_mdiC14+bM&AXHAx0!01k1`ahkzZ5
`iOSJ2_F;k>87%L7@MHA?mkich>H!qv6_K)Zp{auFm-1A=uEHY1{6mb*A|IwFaD10&Eglj{llrmCK$<_P9=d|(0jU$8=4vuOvu8N+Rqh1CHuc)ml!20uZ2K3
GcS28tdS=@bkF%`M;5ZVpZ$gk#$<aRm}Bof6X?A^i;uAgIyqNCZQ7lHE4l9<sVFfe4N~z@Q+MGEZ(Zr@T|r<1boD!uO0?(oO_=|Ko%%G;(-^m8zu<I)oR*3h
_|y$}>*w$)yUbEx7SK+M25JIu|DW>tG-#C5JZNYHMB!~+wCwOx3Dy6q4ooF~nDz*M$lqJIiBj0;6%P}xh_xZdjaF4|m5xgO6khz8$2q>E*(PYoK0cOnmidz|
Jn6v11-&4@-z@gvv7NMfFHMyaCRAS~ho9xc>EWr^RM4P~QUWVd)<?w&0><6SRoF($^uA^Ax0A?$t;jgFUCOc1Xxpe-p4?J#lc!^|FG06m)Qx}Breg}5RM2F!
!T#Kib*bk)Y|7Q2eJShCBZ9H%#$~5xBIjCuZGhRexn^R}S7BtNp1t&lTQUpLjMYw`adv~~L-UJ{Sq(I;b*GD&iFv~=2SB5_ay-<AfA|m0Mqc^V_QOgV1dV19
pkcp)G(DW$m06Y8u9~CIR?FBm_@{q#cC0=y{|m}Y&?cBT8ZS0H4jxo~FFi}KEEzzVc<J6IEyxKTLX_MDqQEHKv=T+NMTII~sn;@=v0jcJGIWFw{8B^*G=#t@
d~G-|&j`FM%c0DMME^yqXvq#&vD<tuLtoDO7GBW~XTXhDtu{3NNM5g7p05_2&mKO{KC32Y9lifo-TCq~fpmbmyw<Gl!H5g_O)%+Do*x7*0@{jGMhg&E+<LKS
#^D5ww3T*pOs#&q@9g(KDkHy}9?@MeAv*>@ya!q`!oI(#aVTzesrC>6-W_`g=KMnu{EO|clFh6x1*Bk<+q4%~JIE5*CF3koRE~(9t31v+n(Q8QHx-V{8@UZ?
d&l*smAe(h=8jbG(<9-gjzS&26l4J;o~e#g0Ca+{Hmaqm*B?Q>anXsmvK~aHL!rO33~-(dt!|Tvz|pR#vc}@tR>>T9#KAf*PXll6ihaZiV!~}8d+JOrx@i8a
UZOTEBc4e`yAauXwUmvRI^4mz;o4v(e=$5X{cqJ0KVJXXfDuAq-8>nAK0lrs+#C1_W5Kx5CaP7eqS|#AuZk_9bYgo8Itax|FW&o*iUyb>D5KO`5d}$*>f<7%
aiIm>Z^qrUx1GEg7xN$|Z}7l68dMInR01_)lntR}N}+<Py=niqQ+=Dd#YK<!<Ns}-EvE`)cWVA?d2}PyWD`m%d5<zeaCiP~ONH)R$B3&7P(p4RU#<Fi7ja3o
0VpKPy=QS%|Hzu+F*(rqCz<=q0!Opi0$5O0mp<)LYq0S|rpGEa|Ls9Se-A~#;fe!ARRjp_Q77=-W^vF@Aj2*h{YEG&^WCFkvXwUlu&y%f#SIND`^Y;RQhVSx
4DfC+3_}R!y6mu{r<Q_xRS*AGz`;9+gJIzk$JA<C9G~Dz3pwsE46?(ZNF8K>?o^L}#$C3d)`QDUeP^&Yt6^1nt9yEDn*`%mzeH#~r7b;r|AmYv)V?^`v<t%W
b+{2Jh%%ZmmUZ*AJXXxUBY92-a4oE0$+OXYoW3+LPBA)kk8L%FarBOz0Y9imm*ERfdr<|`;u85cHZtM=NmZr0doRfyanb8y?;f0l50YhFwG_j8&#|d7#D(8W
W55gx<{6$Ug#2TBd!(*?ZUAzJv3_2unTm6=t(Ur6#y^uc=jLhk!wys*fBF1a($4S3JuR_*qFb-M%39L6K{iwrZ%XD@)NHv0f|BsOOsq_2ev-larVVXsVuDEr
)&ik`SsH1Tmht<wa>ByY`Q~{JSxWegn{ZcQGVflbq5po^<Xk>NojIBwd3KTJIE<@iwku_W(WWs?H`dWHKF4?GODH_3DQSPS<Z1SHO|>54l7?}{xi=JL$7=*z
(e6*$YTw4~vUx#p+W=|Q9BVZbqqIZIG0Q=Pkh*m79hmH-QsA%|3^&qAHMRS;;uF&e8JUpP-2-8OM16A82Ovi8C^A={A1;%;YpxUt95$?kCTR@?5zyIYaTH-Q
kev|yPC#zKA>&V^hNHV8L{JT*nS8cyW2C@dz~?$pbfk6r$$!>$HOScq7lF}(>#PCD(S}%+V9z!DLnmP43vs^qQ`&T@{z5v(V|Ho~<X6RbblV*D9U_KoUJcFX
Jc^6((L6xoy@45WWh@0SB!&%U;!zB)E0(bE#31-(4{8o&U<8w1AP>9G5q2Q1&yQf1F&%?^j0KcH=(XvM+j#6!tmmdRTfokb-w~!EGpLf@A-Eb|<c2ZC`0YE&
qA@jQUM6IWym$q{7VE@P?D40xbH}QG3P#Nb=YyV4=-CYJMYun@i*7=*1LuO8enTnY98utPudp&_OAcZZ9AP|5GtZnPl#x{B9qmcc4R;;!mfxgdI%`8opn<dm
Q>ivLMQ1P<iCs$E4DlVhe6t4do38(qIm9SF68Wz8CjS0g?sBaoYOKbZjy_lqSm+XlSIVxc@B-l+rAN~UE4J`!YacBibgp|2<-J*r)Wkzj%j6&LKaIUsQS2p4
_u8z_MS(a?Zimj~L12^Orf6Sx-$ir{|6!*=S{Xdt+d(nZtjR}P0j|H`UtGr}U)-%5W<Z;}!GW?BO;v%?<3EO;X9)d>EM6gfLD<Wnm8_k{cwXyaifKgevu(2P
%E&FdX;EUmVT>RT6F@MP)13}qw^H*+k%I2zpKqY%-NFvdxZWc{Rw1dgs~t+43@8Bapv!<a=L5ep%5tt2cvp$=`Gmr(_Hm41qN6@sJ8oLw20|)KojOMF!YNAr
75zI(=DYvfXZy;$lNM4}$uF}m*<|3;x*B33dP84XvxsH+H6=NG|Jlyzr0Ro3ZcLfJbXMo5w2hu?=Lg1>q=d<>zXfc5s}Jq_BEqfkG_qP)8!W$+rM41x3Q}3z
R51n3$^HJH*1F^9;#kQ=AIf1r+BJE4_C6j4JJck3@Tr8dH?ck*$)yA62!G|d5VgKV6!Ezi=lOu$V)Uc2cBf-r9#)k0yqb?0a(=a%DHz{GZ6K*g;L|JvYn&qI
q$?VI0EQWoNa!V6n;K-}=tKe(SIox-svM;OlO^(ZWW)QAjg1+rEM)CXeFNm+m7sNLqt^X5g=U&mC94l2iC}d~&U(OV)FlSIkX<9OUbQWa#N>gMt9G3=qLRfB
|Izm;7J#^Jac@jhUHm4XZoDQGi#<NX-Uum52{mTPk*RH>$w6w}3Cl{Ru6x^~$DBH&-wn^!_L*cgq~=G4|9BFnmqQ<vrmBBAp34j>Q3+PwHN%o|RX2T-u^-as
Tmi%0B!BC{x-dB6lDK<cRhPn%!K#|Hh$+O0SjC#DR?FQGJNR*y=07xRsA^JHgUNqp%{ky1i}DDNW)zeVt_s~F;&h+Dv?+^k{67^!{iHmqtMjlf1QQAUzT+yG
e(^(+UdyA2wfZ)0RQg8RrBrn8IcIV}zD277DOL2Img~Mk;8zrk!hiTe4^f*6VGVXC0=SE#?e|z8p4Nok=ZC};anC09Eq@w|sWq)vG)pDNYABaP8t$uN)&kev
sb$B4IA_ltXahX&qUzoH>*5}Hs#{+6hDiyxceqpHXv%(3?4p$*X9$q@$~?HiP6S$x?xh50@Z;kB;uXE^olY*65=U_r)Et}Sg8_9wlyIPgyms%*gjq986))+j
#+;pAxZ1b03Iy;sj0fai9hirSr<|_(Z24}xx8Nb7y3la4!^yhtj!B`hrndB1@--o600000edQ?f{6*|(00Hs20>J4W313-GvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/externo_flavoria.csv')

In [ ]:
# listas/externo_lefood.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/externo_lefood.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5<JXN`bzJ~58cNL}xSP$f!ssd@4s5WWOrlr3gU;hVGuN%4>_a%;*@>!T!hp&~KS2RcVXXifz|pU?X<ej=RYsNHgC9gZ
^Yyj_@w3l^xQPPIw;tS7j7!DT;i*-LD=CgW>TmISg+LT`At;aA;E16tHJvpXoQElAO~A0P6>#q#QpshLvAfmNV?M4YwQp)!i9($3hrjK1d1i?CP^}K<x?@~Q
_U<}k|FCG>1i?@oUNb!>Z=HSr2pGCZ^&s1&j_XWHr(1>F>$hcl2?XsUDVg;L7w8rvMTlHukOURZ3Nef+WAJgTA@Xc$ppcnnU{7bdih4?B9z!pNF=Kdc1PQ{j
khJT|E4KRKjFNUmimR`l^{%Nzi4CIj3Bjhg_*Tv2VeY<iML9W`cFNRMLkE<>-IR9vIAMD3jO(u34+3mEFoodfu6C14HvnZSzk+>!CDj!t2ML*@Xb3giH&9~m
QoV0RcX^R)4=3q(+3DD5k!dT@q?J2MM>>8k!Lh)J0LHf8;&3!@Z>LVT95$Lv?tYU#t9ZaC;hC@F%i;S&K9VDE+ffB-b`ohyc+)XrUb&zc^u;O?)yJs3H9oo{
4O$IJAuvi4|Dm`U|MBtlStsl@HCa4Eo~$yrN$W@UX--%mAW`jj9huojooOA#*g4Il&o=+?^LB^42}?M$21KtL)+5^ynxR*-yzLFg8rSZ0$BUs1-y6t&dfOB;
r-`QjV*qKirPpI!1paV$6Rdx@xIXRtn3??3GDvj53tHF+&D$hMGg7&C-npAgC}36Mlwz=_g4Byq%6~Q^b~BxGf6qb5Qh6e*YgSvOABn*ENP%{lveOz|6NLuR
CR^o)`pi*<%RIwf{c#;H!UOASD3_f_Zn0~lEz*+l=u9z}%|-T0x6SYZ$p$=<8n_)kW`&N_P!GY%81owTLlHt@Kjrh<kSP^7vUUaToC+<F!^Y^SZd35RoJ4D#
@@8t+3V5sCO0|y}pNQQ%8DgHA7T9=4(%7RbgCW9%R#6k6kPEC`h9+ZIgG$q2kJaP5(YzU7vz!nf7xHLd1CwyVJE98BaXJ6-airQz0&x_Nn+bYc10=<@MIuNq
to39|2!CY$ub*Xk@kDokNqE*8LbcU*HV4u8C7W=pS`s0!!If3TvMhHSQ=RrUn$9}Ku5|2sQze9($X6%<EV=6LGOQiLKJqGR{yFKj{6uZcvk5R;b4z8jOSaUh
7C73^Q5^Q_wc9Gk_4_mrAZpVEd-|N(;<s7lHa`xIO4?gRs~5gyEjYj}GFdrv**0|l5gNFE?Sc`=F*YtbAkPRB<)$lZpk5v@SLMx&iL*#TARKGh+bgAE!6c)$
AN=IOh5}w+-(}c=OsuwCiVoKHfX0_gB|syeAWm90LRXY9jhfaT$tQDU@^#3Umt<zwcxopeB0*tw>f6*dNGNt^_)KDpy*1w}ejSwEh!fytKw;a^GOgV-Bg-Kd
@4H`p_m#yhsuAaDPflpv!};@`dPb3fA$fF#`J_^6SD2H^kCA7a@VmNFW3t0Uwz#Hm4$iYIFZK179QZDS`>;B9I2k}qE)H5v(4b2O-g%E%<>V8lboGobDl_=b
YN24o0+xne^OfB6Tv*$UZ7$8;scSi0O*s{*8`cxyvu_a%ya;KuhwX@lq5P(L@6A^?k^))(;Tc6;v{<K?KZ5vw3fQHC%n{uK(IR|wR_rap<oh?Q>$$2uFpLr~
WxIN0<YF4FOzdX{;9?)Y>E?QpbMY)0qvrbmY3KWWKFi;|ylglW);UeCL1<3Dah=ylYH!=u@fo72^9wYWt$L-vzv!q<P+HQ|nkaD%kb6FAxpQLrW^oMX&nP01
J+<5(vm!cDUJPxxV-?lIoEmls0KaL-UqTv6#QLmpwbTSx_M|TUQ<;x#|Hsv^Vtv)GN`3pG@{C$5uYQJmo@+&1fS>gk+M%i~(In6713BqR3i-aE@tE<u2XQa?
q^!wuusQahgj`Z!@vej{bz4ovGnF3f>Juvp(B}x}K5LE#CD}#+?%$<4eR}J2*1RwzJPl-*+E`1Rp1{&~5<&hgnOZ+_{>Jk(0V6N{g^b?cu%&RSzUwPazXcU}
S@W4CaG%PEint)r&}Lpw^eYnSSo6@KUi(n2($k{hHQQtDI>J|r5|*S*peiry{B=A8?ht8qAx+rM9RQI`ukYQdnJsO4a}Wqa6oNqdHeq#|l_Xx~P^MtRO5-@=
*QxLbfdB4=&@gWqN(KHZo!`YQXVhUecSA}<&q?cufthj_U=0zkFquzmOyO`wc;~ZXy<(Hew^CRR67dR@k&}gf26jc>cR7D$c*|ki3{7FK+dnGO@=Dg@wUS~J
m*f&GH7EEANE-@%uInodA35}2eQeshn=)p6j}(6tCMdSPV7(*vHLPVIcK#m4ryDj31fWHd)U~qAxs9le-zS^ZZmhH)kFr9+7_}=u((bE4XJ;G15C3wS6}6@g
yIjBb+*Z?e{Iia$wU3ACH0Al(L-%;)tZ|a>cAE9kYvLW_3PkG-TM|c<GWy5~V2RmNe<qGvYFgaa{PvWh-lDCMeDvb8kmfsrDr4P7yGs3f?x7ubxGIgqR2`YP
zXkC&Pcs$<(Kj4i;FcgsOtm(5MVh6-XZo2f(*;L1$XWb-tg?U|5lIEoL-`D8bSoJNf=A;Q)@ia8eKR4)t&KQ2$3$}SSDetuJZC+3q$LrcHr;qD{`x#-PwkER
l#uXed<j)aKMpX-o5c?7p7@>*6erT?@s-giZR7_vP_87<Yo6rxB9vx#^Bm)w+F~f}E!@J<d`s4QuVm%O7&rE5eCF9;5Ym-j>33h1x$g8WC_p6I{CUhD%ZU&e
NGGbhS7{k$C`{rru3;sv`!=8_RmfAdL~lk6ga&iygg`b1yO2=0qL`|FkrSNtwxId!P`)g!aMgC`!JprJjVha*EF|@PLscO!JXBVw;j1F$<u;ajsAvuYfa|+n
@&~}t82EDV#R=YWo$|kt9Jc+OWhT5H4Gk9kjAMA5Kf&FEO=2?HQz)Q!ORl#A#ub3>654H!6eCpoq`5v$GqX3LrP8zNd5!k<5Z0FghiGPd0C)D~z<VxYIu3F{
qQ2dZG*ABZdt9NHOGw8kC4jg*Yf;sUU2qUQ$|Bc8wc<KvuIGidlE&N(b7=d)WlzmpEB_<BuXxKD0L`OMemOf&PgAVDQQw)acyJvU#HGu@vWLA%BU~w^BN}}=
&xbaH0-g#SCS=>pWP?l^pY##16B&MYzv>o;Sl7=O8}h9Z0e?L2!4yah`IjVVtGdg2kdfM)y?k0n>O=eHbOx8rlnNeDk5y;rj*DD=_}W+_8hB#ZqFqWQ5^*+`
;0CG>%6AntaEK8%6*UyL@9BQw_-dMA(8VaU?~E|o)~3lX9P*7^7Y>;><HY&-%henmc4bJ=DCpsuKq3!NovqKo_W`(L8;^yC*YbF@CWFSmJX$L(?V##N(`9kt
tFKv-)9W|?zo!OF<BUNPi2GEGG&hdYARhQGp5o}(W07(!3H%X0b8Q~~0T6mkHq}2kz%7x`w;?*djehvB@g=styuEC~i&)*nJs2HuYGJB(3)71rcbarS8{Z;w
h<SdhPiMEgvnA5n3JzFv56k#(d^c?&REC{KOZv^vA=-x9B&Pq>i!>~}d0QH_Dlu?s^Rf0`(bzJ(pRqHnw~*k5$i3{QEx>%VuVP{-EBwM}D??q+LBT|xfC=O|
LQC8hBJkAiJT&m{iS&9TCLBDtg-M9G_Ux<R7k<Ib@n1A0C_qr3zMTtE0)OabTj}@LStEJG-gWw1q^{{D%E2tzR+}`06$yB2XX@dS686LAiX@=jKBNc%eg#1T
NdeRrl_`3ZmkAPu76N@`qVito)rSs9p<{}9N_9&H2YmID7t*I@9?O{BsPTttu}YgO#<!mPluh1w$&$j9%gp7oa_Xq#cs+hI2x$&zGQ;6mYv-e4#ghkK=$&3&
Dx_a|Ey;Rw`V!EGH=l|jl}j7+%RFqN51U!4^&I_593S3*<Z{c_#)D~>HeJOs3Bh5K&yD(lXbVN3rQ)&vr?sK3a&r7Z-)EMV_G5JM(x|))u@A4FZy@{A8)6m5
g*VTZ`v)0)*KTTuLB8x+{O_y;SjZ_vlo~m^jkKCDl#aQ&6x@Fg)qUcJ>Osx8)ZLCfYr6{jX~!97NtA4Sj5N0fm9g;F+p}#Ml>5h3y}9K+G3NG^Hm}=9raDm1
`8q^nz#lk#H(S83Hnhof_cI<SAZo!%a>yqj&e4#D)MMpMSMzi27V33&sBRC|Q+_b78%zR%IuXWtSblMJuBt~WD@LHkP(50Ci+Dr2D_!$!0i^Jy3UKeBC{A(B
Qe^cGI=70eY0eePQ>Ze@29LXgpfPW2Mcnn)+mgn5zO~<|X?$aq<~4v7DHxpR_aj?mM@}N8g$DNq%~xLVDRv6FLie`}G)mW&u>2ud!vd7T47QKnqG5a2f5f1H
E;Fda>MK!CC`J-c;(&RUxs}Vb%S4!Uhyu*!Yqx`{K~91se71Cv<Yi^KY$2!kt(AI&Dbim)$!QC1rI#z|gc{+f#aesOtXZU5J;+!%yFeA;he6IahBfZMvM^Sy
KTZH&3w*iTBY=1mAQ_jMFu}Gw4-NeFjXFBtH@E~g&T=N7*EH+*f`-=gX_EgLUq*ZE`hCPIZZ&<76;ICxFYXL-l@EBjgTFEbttBBsw{SuA#{y^f^x>b$Un7W*
xH?G$FSE&JXuLq81hb~aJ*)5h+1HK0o!;#GH=GVlBw=&a%D{IjG-&TYjE9fk@*dJO27`%ryV^6|qhYx?tC-U=d<`b%lbBzHi2W7(-?Ks%oZt@TS&u%y2i4VU
f2jzz_*3lgp!nE$<%qs;gm7qXJ?X+I##du;8-AK$rt{2{rNcElpT^p0aBT^unp#|(;ZJ6{NfM@F3(YWNS-NGl?2!gSU#0b|M{#1yo%qO`$4&BrZxA`0Z6|X2
KB1j)?yWDuNE8@oqi(q!p5e_;=kqwa79(3{m<UUAo~6*M!o=n^j|U9Z`X$=lY`Y%ls-ikdV<P55d9f_l&N(8-p==jRb0kIm-VQT%?5Q{@4QUBivmiy;*X%@}
_SyCOF8qRMQ|_qGRRHuNaTxhWJM|s#z7{F}9Pw+B{-zJJ9a)_)l5o}t$>L8qL~ALk6pOBpyWYgLP)Bh2^*zXb#svkz;qjz08O}L*;TW^LSx5<Nz({R)OBoAV
#Ti!A)+jGcgX~OLu9NOE{*uUCS1M2US}=2<{62QbiDj6tQdIA$!OrTX^==j9`|g$#{_HhKx32rRxGwVGCNN&67-Z5MPV5(|<pWM4;8#K)XoSb9xYbk$48I$*
&$E|v?A{Q}fpZ|DYB<49EVIvhtrcpMB|0uF)WY=`7pSU4mP>@45?{rt`b>O;_|*Kmm5YI}g%okbsH=jW|H1pbQ1n~@1R&mT9qFQ`L8lS7!>HVVii|+@b>I@&
fjAwEeN3sh3ZwAoun;e_iq|H)E$w9?pSu}NPeV|^N)q_U;>H8urx1<sW1b(QbWNpCRl3kgLGER#tRY`+%P|H&9t}BXA!YcAt`cE5kX)q_rpX;GZ!upB5>Q$c
c3g&d>&%lThatY~TyoFFg!!XCzkG-{F!GMvqxWz4W0Et?eb6h}dS_w^>#G|dBuInBb0_9ZMW@0<PrzFlu+ZEU%p<xS8^kRWyc`Gvq{QxVA9-XS-Va>)4Y&fm
<~obrcVkk!3Ct&iURu*Q3JR;&ns}$)IB`}`2X{K1TLqO?Ay1=bkto!1*7l*GQp0zXcVz634#C4}AHe41sw?;)ySzu<^rf^evRaYk($sGJ#g$-K^kxn$tc@Zv
aVZ!`Z;eV&RVu-JxL;=Ut5BR2c}6+=aja|4CVXPKd+N<tHR9mI=>g~^NpXOmo1+P#Y+YyvZi()kzA@NCMYW<0v6%!191lOeyk}uNRT;QHte<YtrxLDBV>>k1
jjdQw?v?D*MuSEh$4%YLcj2AO*L#Ob1O8igSRSQiLL?`Zq>LldZeAbQ@EextE;%sfJw!?B?+mr2b1g}Y6bWwcNxjS88_5Yv7BssT@yoSXm+X==Lc7^XbVWxA
priYyzT>RC4R9}b4^cjj$}OcM{hbZ~wx2i<!Xr$wuZo*d=YbBR4i-t!s#fHi(}GyXBM-D1`o_DC;~MKvp0@((RWjY_FB#u9Ud0{JfiGQYw?9vXOPzAcxNQF~
CbKWjf!s{1Nb^5?u2l1$yM0+xTBsUo3obw?vfh+=vrhu_GN~$pq9oUY_)DtFxy1cuiiPvClh<ub>)`>zsY$dhzRIhu<3_N~;e;}plN~MeAtPCccxqH2w)g8+
uCycP`VL$~O<>!b2^N^oho*>As6-nolv=5ii5PB!5d6LI)1E6hqTUaM1^oApw5H7Pk8xgp_?wz|7-Ppe(ddj^-$^Wr(W<1%X3=ZQrm*fWTTp=@A~pndQ+#n1
MoL9=L;*L3UtoHys<B8-1lvTM1ry-Vin_^hw<3o-xA%pHq+&3VXugvu+TrPDM?L_S;0R+BFgJSsbaLFAr57i;x)V8RJC%sH-K0}l2*z1dlZ3&1Llvm8Td}#{
E+}gOm!MWx2&h<HHz86rkd^D%@2#A<d{|Z2w%k29nPBIZcU|^skr0c{+BuxXm(tDp(4^KP%DNfj`;<;XIF^8zUt5C_)A}QxbvS=U@;Y?gXkrl=*Zm6x6l!9U
)<K1Cjj`(CBXm*u@(p-eqo!+eD)kQoDs3R`e-+F~UV9T$=N}3w(Npe`9{pel%mj2sk6JESRQLego~D8qpeMGK0sUPqP0tdErfFSaQKkvnO4q45GpVUcf3TYT
b|)!Alvsl?FO-<_8vUiXLrOWhhr3+#NO!a?$Q~8nC@9Z3qyJz@lUHDVauu}e5$2yiY6$C3ht5|<t9@S?KfD{sTzFTq_{n_DUhPT0GS7c!NVbd&ID@7i8Y!i3
WH!yRiYa>=G4WH;Db;Uc?Gg(-dQ(@}76L0APBuN-N>t;ljuK@k0ZdA~KkPZIc`r(uDR!_lCP76TyAdU&ou$glw&aJ#Ko0jvA?+A{Qf6tBY;ab+JM|{kSu<Y6
aUi9cnI)QwrdQ%Js2zmgT{T*7k)_)q;#nXhE|Pb)#wh(}T~l?~MA=TMtrk307}7Hz(bO9xH8}z)mbA1yk8?=78@^^)RevwFUG4H$UOO*;1>^nB`M*r<VPJDo
d$6C#bh2^*q=ea=9@tQH^C0Ab8PtjC&(kWnfv}k5#b>Ph_tdotdI%0i-b1>;Sbhm7FL}nA@DPcF)7qHJ&w0!D%IP3~6w~z&@Xwc3Wj_Nu0P4C3E}TA9R~jNS
&<|&))kKp+@TOxHDNXBlN;>I(Gfw|tzs=>RdN;?Yc{=-4v9O1ZFF*B+z*8<G_M#Exvo)~X)5t0pI5#PKIu<BU`WSzdX^^V$taV}u+Pq9?=8839U$u*kBr2#x
+Bvv>z5N=3j%%FFZeVa9)G=m{x{WrDZH0f1%IR(k5!a#a-iU{yJ%9Jc(qV^KOMj~&=H5Cc3iFAK6AC~{aPCg2hzaRaG`;SlD>A)q8%UhIVWwOm9AQtx=NFrg
+(z~zP~4uk+3-r0`T~j8(cmA-NVC%gUAXINd$~vR<&YPy>k#`7{jPP^s(~b^#sy7i0dnlTlu@14gk{akp;2p?NupbJ;Qa3O{YZ%edYPR!b>LH?#<@&RxpxH@
ZN=;&uc#T!AwcpQy{6g2gRa{pjvCqw*0*yS>{iY)lId$sH&mB|%a$Z-A)P;RR=n{g2!_TL#}DYTLLRFDw17nlCW>D?caqdP+SSGrxHBfB@qzQ6IM-XY6Q?`i
cG`-*NiSV6NSrVWwEaMTPV3tv<?CN4LI2lMro2}P{BMnG0<40?@FaL1vIhq3Ro9}|Fe{qzT>i>Ex);m$UtRVR0-l@-jS|7m0#UkiFxh|vT!R}&NDEH`UsIOl
kh2wr5&1j-{3ug&Rl;$2a&M|dZ!_wCP)*p!r7W%aKYDpPUG@7u;Y4j~au}b;5H^qanCUZFsKv$p<ZH-NL6}crTMn&?P-LCY;O|FUVoJ5>uWo>U?XMz2?tLCh
T%uhb@fY^qV%@b!ro$k^EO~Zb?~7^Q?9FuFsE*yk+<fU(3-kg2BBngHxTA)o+qNO(Q`7}ZKsXIb$)4+X7cG&FpjU2*NaB$7j$mbsTQTcH<q7_G7+xzWTvWlr
@uLa7r4<nU9|mGR0y7Xyc+AP$tKS(7+kOE53q7_#lC@A`iz_IM#O68MG8<(==XQ?A=adcFWU#UmsQil;4EE3Du(}5O;3v+&m^_-!aV<$gXP|ZC6sAZ_!0t@X
eQOI8sT<H0D5wAZ_5*b=a-=UHXW@K%a190e(BtDino2i_wo@6(2r`auZ8nE$u5T{XH?2~%YmJO%l>K%0HQs~Y?a};;DfUGRZZ#G$e;L0Sl2H1CGN`md-nu?h
p8KLgi4+I-1m(*7fV)g0uz|h)2Y`nasWxL>^z~#%ien}Sx|=DzELsQ7>yVxXkmS29AscqBxtO){2g>oAG14JKQ*H{<_~fp;hM9N@RQf2|lgX{NRo`#aftSAT
pC(uEWcJ_gk6XU>*Z57mf#}CdxI(0da{a}zNj#Kbs<x#wbA2h=#mS`vN9=m({jRP@hGv#PWUaMWS8VL#a=MY_PV~DNJ6-@*by?WJm{PsvLR61%0l4|4yB2qY
5<aA_=3XbiaAfEhl`(pwhDV%Fh~9cs@#TX-L1?kSv!2vZjRuqIWX`*I+sTq!aebm>Uhtz~;A%2lbEih4rcl>JivSe_Sx)Yvo3Q<t2ZVH0H-oBR5K6=CgQ$=l
1hr|{U+URi-@Z(}b+ZHdXU2cvK*3a33`6BsSq9}kS@qFCO-G2U8OP8B)C&!piQ?Glra|CO9y3>LPNjq_J|)SrExx7vDS#;58zoXTPL@FqMxCQaeBMmbe$G1J
e7gJV%~LibgKv&y{`a7^#*8r}aiV>J!{B_I!I^}_WU390$|8qOHfAM7PEl9eTRe;097@-*f(FH26PO!K)a?aXx#ftz=&34mJGKZ??S7Luuo9uHy?(ci=-;dj
0aGplvQe+P_NWjqG3<nWr2y9T7Cdz_F4gphY_=b*KAeUKvCLOVX@%S_iKF)a_B!>v)0BAfMFa%0fb%-?7kxOt4_`2oo@4o@7Hw+>dy_1|VNw_hv-z*zP*&>_
LWuAj@SYIQT~~BlVLyO56Akm7ce<QQ^%ph!ED89U4T}>b5+DEYAq+dlpxOexi@mOG>M-kX!>xdEqg3TAsufTB^(%d&*DCGE0nc`DjKluIm*LjJ3CX9ab`^AR
{h7Z#BDtC3D7v-Elq&uy^w8_mN#L^(oo69vV_B<-k@f+g&<7oF!UQUltAv_yQ;V&sdZfsW{e;(enfRJCV;G8s8T#f7i$|H-l>y1EoCxK~n~Qm1jx|dGBUt+C
=)c@#E-|Ha23lzHUI6^T1@MLo!aQVeoB8gMY;zKGG{$JD6onmVZtQU{qEO-zk5%3WD*1_(g$HudGs|)va$3kE2@QXaiOT>ON}hN4amS~}zi~_#?NQI}?#ce0
MY4w31O8k{*mrj7cR&p<btpO&0%=6421YS4l!%_t{Js!askF9k`dgTXi*8Q7fHH5MAEcT`Ed3bykre8T#M2bk{tqyfhxUH-4E8kQoe|f1X0EYmm^=@cDxYmL
gYEI{QV1u)9m<28VNl*QWm~m8{i888v14FNMVk;<<PDhxxFdfZv53&=Y5$dK4)KV+0o>bQ%D_C8HbnAF9U_(jM!f56%H<Ql&Vd`pAUMom$nWFrJ*Pp`*G?)a
RqY;ZV6R{Q0fxA)k{zrCdZA}vtt21lLqRoH0k&(T1Pi*WcQ(d@$-4Z(SYaIlW=Wt8Vwf2-+8^RxKpOS%=vMKl@Z&_S$`Lq%9GC0Ls%$)=d>rrblN1qBkHr;o
Kya85wFyU?+p#87p(Na%g7gVGfJT^^LqSU@SUmUC8p!N=RXfo%nlvukyy#w)Zj-xn6*K6Y8YMBm$>|7zCOG|ntpx}=S0O-*7kBtRn%f$!UC6q8L>goAQPuLh
!HFc`n}xjvn0|S@zt9oW9eowARoF_9VZ8k79IW{#{lhxD`b3ASvS6qBfIm^Hr1FPYNUU=-=7yCL_f4%)x^oViN`M%EmB?fQJ!adqNC^%_BAk`_Fhl*__4fuu
X)jk8hq#Ob@b&v%zjPXN0`T#LwMosmi=xUE8iM*aa}lwzk7Ddq+N&W!yIfB7(!a&EFTXEC-tlo9eZ-Wl%iKhO@co4+M8p>VIQ7R8!LVtVLehxU`@G5sHwfE9
?ocU~>h3WAg29oc-xIK_dQZ=ea)Q(LzPCaMPUsj2E_UVk{LiCU%E~WjPS|GKU?J8zo;;oKjsB#jn^bR2&13$Ug7V*<M6SzQr_N-Yhvm`up)7aFYgW0X0F6wt
bInp;_h+m53^WxA7<-3o6g8YzfMdd=_USLqG##XZ#ZEJ^Q)WMdSEnT#{Oh3n$xQk+|39~KsY-5(%+*Tcg5t`1?|*8-Be5|4arYY4@U-PHWPS;dd;4f_v0LYN
4<5K@_&Q8PZ>M(nb*gjwXzkTBgXoJ;PgDO<#hJJ`tZfuht+>b(5xi(l*Gg@QZE;li2#uwmTEWF++gm~2)jy>(!6;UD=feyJif<xsS8+lKWZmqP3dvB?BmA?z
)^)~_;{HAgoU15|Fb^vacEWs#8n%B$WPKVo&MO0u9dE8&^L0=IcsJe+m*XCZ-G=6MP4;$L8UobPuJO9v5!U9dFuJz|bW^!RD=&Bf6+^hYY3h`<U;i9ah($|*
DRD<E>@|f6PW9_eY0T24?KHKP=DR7*F&X6NJJG5`2fPY%&#<tKk0PPdNFSStC(>d5wo5bs%Xm4`FSQYzp3FvSvha}3o5V8Yqsj#GC5(G<ckN+ercNgI0JT8I
7Y@MK!w0*+CjN*`7gV&OqYNl!_Xs$wn!dQlz7#4!BwWj6+BralrJDuCQ}3o3Ny6UHepDvpW!2-&`F*=R;3V6MZ^1|oAH(7-SRcF?f8zkmjL(0YmjA34-q1o{
OrUcx+qD?<$k@Mm*|-t63JI2pvahP?E+rwn!#7ill@o6%yw+}HAS!&zb?+NYv>o4fl4*`IL#Rbj;+CrhVtMXbiZzc*>Cal0KStwr)*Jr;(WBGzDfVNXniUjr
WD*_=paSwH7`o3bpWCq=qe7xo;<_PM)%Jk!DRxECNS_&}{1ykU<qx0Tzh6aJ$e6z!@5x8(%OvsQ=ys7UjCiVxD_j}v4NnrGz7`EBjlR=**<4xz3ucuv$Ns^?
LhByvGslFa$H%HtO9~^UD^5;m&T<4uVrd##Q9QAjQC2Prnc+x%DfRbzP$c^%%dQ@vyNPD?N?WX+qd~-j*ByEmr!R~cV~(Kl1Ae{upj=G?6)PU9e8#0aZ(34!
X8qNdG!)%)soeoz20!m~!Fma(W#pLdt8E|L26~!VHs*}$LVmvChF+8dRsR)oSj!kg6F-%!$aLn_O<%(qBivtE`<n0`(teh>D(tFQRqdJg4wIwE7tD&e_4;_y
4BQX+tQE1*I?m+oFEsit+5;+4Ufjl%W##k7f=IEq3O;`O^x5;vtpf}~pGCQ=J`T|?x*oVxjmyLETs~%|iDy}+0g0I<AqPFiGG!O&{Lz@7Iw)YGHIg;$)gjWh
066OLgCdQvXQu5R?W9d#OS^CkA!dIb8)MmtEL*D{;^%5{(IFZJo3oSd#daAvpCs#589|}!9;JZkBN=))vJJ_Xd!>-FrM|l9!57anB`2qf3bMq0K2*Pc-vO<a
W$7(1>sv#e^#Gl){sn(XmRY-MsL;H<u4BaA7}Z+WA@||zgY(UwpHhFh|3LItXWakPc$#Nm-na0@Py&RI854*F7j_f~vDgG8`-MAH=oD<1=SSRz?RM>xiZtx2
Eq`H(tib+0V1&!psXztVK%;N3#cS&4W`|B-!_IqwXZa#P4#yQDPvGU21k5eoSf`D$6g%Z?bBk8|h4Z-=(UrC2J|cwlq|Oj{&#{`Ql@3+BZY#3n<x1gxcL`nV
`|(#TQMBwKD+qZcP=F}W8!lEA5~q1_0~=93$DHP}z+KVzU3Az2$f8<l4@xeTZAUvGBRV<}x7(SeqJ^p!AoRn^1f_Y^$3vu5<8*PUC*(HentWkE0%1x0|2=}^
e1|z})z~j9@)9bpB5e#<IU&zDG=kACo?S8ivDGG$H(V)D{1<+kJchBiOZTztbVfA(Sz6f#$5=ERTb3kDG_jm9pl6<F*Jy4!F|xfQRb`4^doZRqC#5*-{WCEV
(%yqDr%7A{W&km;8Kc1@7FMRH(pvguaZtb_Rc+G#Gs+b}Lz?m-6kFscKEB4ZY+0+~({MSj1zury=Ey$@+0Cq}0rXF{n3T2~(uDeY1C<nj)y0;Z>`Vc4`J}U4
rse*ay0$Sm+^*Z~*SfA}HgX@VHX9V9ZB-1Zc0)oX9qg0wyH};KfiA-*p^g_Qy5}o}5DStwd!i#S*5vOuX$Zy*yE$Mj%+Omydvvb~xuj$;uZq#LFn=X4F=TJf
{PDUH4+wkUOX7{K8I6)i=X@8m{u(cWOOBZINFBbe^r%yQ<sF9Sl^*TC^ELEN*);`n6`u8nfyV9$p+d`u`}8T4V+QE%(0;9VsH*9CJ=IUV;%ssGh-yQZjK@$|
MlmC{%(UI#IgRe9-|tv`qIqyW8)0whTza)nGMP7&Vk$U97=y@qGEuu~9khO|Xitoc_PPY~sGDqxv>x!(DoZae07aJ>(*WlTc+I4XYbLOM@se7Alp((9zeQzl
C?B!}EXF~j<-OdDqx{ET40bPrrKZj?6>Hp*^<Z<0?hj#JT^j%so8gJ$84Eh6c*ic%T-KR;Hrre+LtFB0EJGOuaF7xR0Red+SG!d0$2`*FYZ5N^>4Js%&CGm!
$qhjUG~br<H5cFa6bTP`k1kJAzq(0yM2cITq?-`N0^^hB8*DX)7)KokQUhyBDI5S)&V<WE+6FyZr0+mN@91=7(^Q6n1=-P5D92FrLsSY2w@FDjG<xVQYC6GD
=KV&`)^Of^{T(Sy5)t140Yg@C{Wm))dqJ|j#IK>pDLs+2PhysqNOMzLcpTFAZRq+r+#6LVUnLhV@lz@uM{97Ddj93ppfa4$<%o$|k~Z59!~oYpXq-zsV)B|o
)lEcOfN6a(ZD&-;l;tlEjErYlbJ!F$z<o|)XX6!T5BT!4+8HX(xWHWGpq_%9ZALiJ0O{Lb`3$C%fndSRP8L)_4uY4)Go?Qp@(d}AJ@;oPJ3fOc{wc^QN6qd|
k)<~1SV9aPh^Z=oXaJAr@BzLiIUAu5<9s{5^eT-6v&n9?RL^{OR*p9Eax8x5cq>HT54<UG;G(Oq-Ez1ZBAqNB)>+>_*6@EDt_;SfJ~E49ZtOtUlPK5_=Mz@~
n6Zft6okgbYL=h?F<_OZcvk}*NB$0GVmJxb3icf&Ic0hTXx{X4(FkgchZaVtoyNw|rKY;VSPBKpvAWeX>YtEidPD2xJ0@Kp=aoHjrf1Rx0^pA*flX-pV84(9
XnSa^{oCo_BQTqYb#rJY9|_^RjDcD}#0$}F>cC))5p&?LPHu4Y5+<8v2QkeO>b%WpX8q0|-G9rtt-Hzn7jj*I=ZiXr^@u^jN(;K=JqPL*aJIar1=WKTq26Ir
ko4%$$eva8lTl)9@k>d?BQ6CmGYd!{HRm?$v?+3s@d>KMx?Xdw(?6u5aT>j<(68tk<h1cwO~fKHbJ?aMR?FtCkpT*<<-P-v?NZgY?=4but90Fr^k(B`AzP3w
_0W;vLFwJx7X@Xv6ssv@78a##GPJ$@E|F5O0yBQ&0Svr$(4%5BeHDl+82Hdw6jI`5-|q6v5zYVes0y|=v2X-gcJ6aWlf~>;GedK3R^dwcgXW6>;Q{!EMX}vp
$O}rW>RBIVIN*l0=#ukxN2fV7R6dK4IeB6DT@0a+E1fNNGx6jRgIosQFPf7n(K``XV1Itn1Wppzj?(!Ko2!iu9Nggo8@{pt;a7=C+F*29yZP>$P4@HCi6%=v
wE8f6E?<tH_JmiWXofr#cxkR!UFkP@O<**tgJuQ&kA+yCwGF?sv6togTdZBMAaf~S&$(rMk5yLtYISs|rF2*=vGuHx0<F4SjmiS?d40+Rd%?an3g+R3u*%Au
0B|@HDY;W3CH@Vs0m%2Yf5z04On1D7=H?|Nm(WtBA6r1yWjH^jhjl_mVlw4v5e7<X|7ws^!Cq9C$H5x`F>dPmIKpyw!P1Nv1F`E%+fAnbDS<R6A8TZA>oiX&
L{YrGX%dj#RAkrBowRf%ylS6|iT}bTaajiPpTrg7N!QF!BR&6CtuCmR*%{IQK!lYMYJ0dqZFe}XOsWbd8Q0^SuF*tw4B$^_bK1tkEtcHdpnF}_0PQ-XdPJ|e
ac_<4c$t9z$BSXUZ#JHWFt2r+xR2}yq3Ns=K)}?|w4Iizw%S7As{N>iU?`^acL>O*WWGSePs?E9m^qgGQ3@k4z14nCV72K%YBYN2#YtKs8`_Cwb2OOnCyrr;
{(0WyNuQ`rO+S05`b<*I@rst<d)^0CV_zmzi<%rk2fNUMj|0%(5V~qMtB;3}A>fCS3st|0m(td$OGs7qGQuE~=d0?7<hRZAVQ^=>GVSXNV;bsd`se9XFtn?l
*-RPMH@uv(1np6fu6Q0#9J#Xc&ZU0jPt@ph8kvBPPYMlDoqy1+Sb#GNB$+Btkd#4B$C~yVHtFtpXid4KMo9LrGo99-n<wd6=(Fw0cRcI{I2fC``W6Q!K>4s;
<$c7VOrQ`EjD~X<(9T?wsoDWu0~yQ@OmR%q>oVs_44!rcdt%2W2dPP!JX=>zJZ6$u%4$Gk><~gjuy5=bZNCvaO{wHU2!v8$F#DeudpvfpHl-S;KrS|SWV6Gp
Sv!em$0f#|Nt1;N!uI*~O>gH>a^<g?f`~}e)Lqx0DJRg^RaFy>6;3r*1<M9R*?v9<Nb4Ww7wkF83gK4STApu4^YBH#n+csXyX|>v7p{t<t*3He2?gJC9PoEe
i$(;=<QB^uZ5eJ6z{oh(@+gMcP!~TufahuXTN3$5uH^NAHo-!1I)BD5kP4mp;Fjv#Bx@wjDHC3t^D$RpeTJOPWOs3E%3pvr7J)g*POI?T25tsg)daXxVyEOf
`O_?pb`Jsj0iJd>O=$PJ%G2wU3pBr<iG}4ZN4PzI!3r{-q7)1J#?NFrS37-qB3N6jys{RF0ft}(=XZpOX1yY8{bj-bmI>u}&bny(DbS>3d5T)NUBy+S$}o}b
*Tin;^+U-OKnKutyl!_FPpWzo#tIbEr00DGOE=@{BPiq$C_Xc&=2=nu;DHfk>J)?DXuF-Zw4$bBK(X~xclm<h1FW}imj|Mpz@};G#L^$D4qK_@*fyp)J{y;x
kPGb9&ez6&uM}>|7-A=*OXlNJ9EU2YzzXZfj0=;N_5T$`p*~tUt<<z=>lnoSzWZp5kw-n%8h?zdoMOF0>2e9d?B~8N;J9N?%+_#^;8Xps>nj~G0FXmu_h6Gb
mUof1yE!oSV_LDu%SP-PAs+cCWQR>Pu(2m1;qF#R^IUZ)M%JrF{!f@38U!)d3M6yY&biDEN1I-n0Qp%3;Yn@BO)ebP@$z2;=S*4JzyQ2cjIy^cKheF{s1~8|
0Mm8IHubTQ<jj=F=-z&2;7UOBM}am<7T<D!&a8`Bz>qF2ujRqwj4^E6t>G!2b|bp*K+1lZv2&p*e<n250xo<Z3`I>nZ}qYP_RqZs1rrqnI;S{`5|+wNDq5{E
Zsh4L;1VxGQJh^b0A6m50sc{L)my{-In(F;*sN?bqJFg79KS6xeV?H=9uR<^mg|7KXla*qI#m(dRE|!2b11gjAI8~@N1}tCNCS7>qv5qOnu?!d8L&-Q0M^L+
8qz+uj^#u>_%w<9m5-+XR{pFIN^b0h?$@c(JdMGD*HPcGAd28WM98qn2QF(tHF{3BNK5+)S*%N7-!gFQO00-e8i=>o96jiMg2g=J!D#qJ9s<(B`0kxPmU==M
mhhutOE|lB)oUd?_R*z7<R>%DE_gb&3?s?F01^&PAOEpiY-AyK?!0o9>+z*{(k9V)@o(=hF}5xkjV{S#Dj26LkTa-YV5blF`=^hv(r|3xO&D|4G$3Xf7z66+
dbHGv?M>{+)+5F+6|(5xwwLqSvSh(vyl9xwgF(hz#9#FkSQA&`-sr-CQ)mqneJ;(N>A8ev`q3xR-P{<;_IyhYaC6DRV6SR*d!LHqy`7tv!gg|-s~Ir)`_B^-
R45G006tZGKXO)HjcVKj?$B;8Nje&QNulgXg*gr-rzuF62&1srYjd9jT{)2a2wVO=360cQT5CtezP|^1OpF6TPR$Vl5Yb$|S1sf<qO|RS3}N3yL60fhm0G%?
jM9RyPc8@UbssCBWOBlJvFMKlp8UxgnB`{jL+jqe>YXTa<_gT4BV9Vqp`I6zsrXh&Fi^Qw&v}#Le<2)Z7hrkg33RjzpEG;*HhYmiIj7EwZ(mLqJ81~Bc`r~;
@|EppBBmeCjaIENJOwn(Ky=gvWF)4PIu^)Ni%NrO_;%URH3p=htO6!~!5%rwrN_nH-UvfHKT8O59b;YLrz0w{TTX3-TS-2ZMe+dNsFx?C94V5+;)I?YK7d+2
ft9`#t>@|5TeOSGZ(7^=kLa*i=UX!_1ylY==D}gPI9`CBTW1QQ%4h#b&EEuKoFkQ$LEUhfs4h_20xd{*o>|H1qz;)T<5{0ltHtO?@((21ZMq7hRab%Rbe~5I
WgZglOY%GdLTVK`pED7HbrAxu#?f`b_isPkp5mtbKKblDDe1=P|5)}*?I+LhJPN>~o?h&BHc%%~4V1R}wl^YHJ(|^U`JH?~ycSUD>h0A4)LTuf<OhY6akKsF
1kog6-hqtjN!G&1T0Uv915!&GnmXO2nz|XD%=@{6)`_GtoA=xpj_4E<u*r_x_^9AO9e;A1unCbfP@vBTtA7QK@zvSsj#jkd3D#4|ok}Bho1W*KW!%=;Zu?rD
cU?vyyh4o6C5@QXcamGW-<uYU&yPr;xC8H{>J3bFUSLoo^X>?p{=&v6Ct##l<$#Wv?BENN<@D$@2~taELqlhn0V~)vhAA?GfLzA=f&k7S))}17tC0`>Sa)}>
mfYl4u^N$rW)@1FHvlOQ`^!~N8IS6LB7{^I`Y9dX`{!Mohf@Z4A~oRntV1IS5436ZifVM{5$U_80@n4Q0(;dfn~!aN98q7VFvs)U4zHVIz_)GBi<#H?^PA=6
OgN1UbS?z{D320lO6Nm^3E+a%uSIv^dfvYxk4|ycEi3RO(C2=#j%Tl{*tsmLYCqF>c0-zu_nIF4IU&sH)o&%V9RnXy(7AC|D=MiD={&zJ@Z%g0vJ$QJ)gObe
_aKZc0{?%+t`TgfP$&B|^)$~gkQ*HO_eM^UFF{BacnStVSjGT9p(C!&LN9J#o2djcT?<nXMw}%eT86-UQM1JjcGp@4=$-sow5_^e?%6(z_pWdd*^~63B9E<j
P11Py{m{7Nj>~tR@7~s>R!C@m``*mq5;WJau_4K$ivT^WT_qMY58;|%7|y%102Mu$(XM!9<I+*O`Y?dylhl?-T?l_>1%&{jrZvLRFNqdL_02k`665`rmdau7
nKG3%QKWM^HA_?rn5-?TFe2)2<eU5iHZvQSBaz#Z@C1`!jshD$E{cjnK8p*JC7@|pBlI&DupE@eg@B%OUE7TtKkMCy*0hKMBhBl0JVkz<QxXvdaHDy#CNIx~
bs*h&DJyQIn1zvmDM*m2{!Uz=TlRu;@ctaKKysI%-n<hf$N($q?#U><@-%zt>g`=I+AZGqKPz~_hpR3xv&ww%uG|b|<xYo<!2*mY@g#5pzxbRluf@<oaujZO
z7q{qFey2j?nQ%y(@N15D6nP2XgFnF^w7SeC<kGt(JIHCW-FN@gJnqm*2W|wxXhoG529*7K=bGtmj{roFxg~hTU#6gTB+xC^xhLTV=7R^IJQen<!pJW<rA`a
av~@xlRnizNQf0?vDUjcPo1}73RcR4w(2DO`^sf1d)-mozoQ)q2B+Fb@;yAJx-d7m-z?evVO%dd+w_lXY#xde-e8oZ9?kw5eF`f3<6H0}c)<k#ptNY&ruEa+
AIkdY7;u&$rRU;F>1=<WssPv0&Pv8*Q<FZhmaKDLrT0)+*FT6A`M#x@1hpq!;gRh-6OPRUEM29P8P^}h*n7V{2<8SbT}X|pt!sul5y$V=E>%l!sbbC4l-S^Z
C3l`fK$kDE6^kp*`R;x=4dq51bHK<NW#Cp+gKIB(Wi)aP&odNbn#OiUm$CLP0X7oADw37Vx|TX4g#W4~_KT7KXJv(9+(t<cgYHyfRDBh}x}}>oCbn|cp^~Wa
BSSApnSy!#%xf*^!SZK#$~wAP=8}@9G9qIiA~snOXuwbu9K31lwK_q(c%J7IQ;jEPiYON>ct6gfE5Ol_?)nFrda9xo=eC9&Ahh>7(v(G@B8ezv5y=bkcNafd
5#pssfM@K}w%gzxog7wfzj%!W*PS%>Rq$@AUYL`uD{qV9Y#d}2JDhIKb_%?-T@Dw$PNtDog*wBRR{FQ8ByTnI+aCvvx3-;Qq*vFM@&{;Y`p!W>N!HY$hJaB=
K@#G#i6Gvj*C5(fswAm~A8h(KWE5a2scLv1RFHTJOeKqj2L6r<vuO@Bvc*~Z1=EBL8UPW0d3m<j9lRbKoJO>C(DM<*+OA?MC&b8vjys@1Mt&P!w!a)qhTuIy
vw5^KK)j~h)o9TX8)5QYKcfZ5Mf%2>%**B*CTJ;D2Kx2tQ7HR@S_6JhzK2PGBvXyh&prytX=@HKcE4E%3SM5cA`x`}cNxw=axD1Hl+vXp92nGaVBX?y;_nXS
Nqk|kIE`&2%zY^(4^Le-1&Z<BEPj`kyun0t*ktjT-1k@zOV<RQK>9IL#_5}N$N?OjS6B}ZuZ+)Cv0B$zuWkf_!DV^J#xC3cR{8UpEH5^RhnJoR;Wi$p#ljX1
=~g$@(vF1E&&S+7*u|XgH*JZbKAU_0)FpBjG~sc@qym#aHQ9zfy*yCDYTzQ~LZ311weQa=;I9Xwr1G*__>5?YYis$nNDkzc{V5X37@4ZRPIO0Y@67yXkR$+y
%K*6P%)XTPNV;~(xhFK5xf(t6dRnsv-~b(`3e}OQ_AQhVt(xJlr1$-zKcN_f;x(LJ>br`(pmn-ZEIy%muW7e`tBmBiq(D`pmx{?UA1;_))*OF@wTWd(g<nqQ
jWRkfL>_5T+_c0S++%T(IXFOAR830LX^k`$usOzMy9G;so}I4ce_nXHYi^){i4m@ADPM3(J#tqaC}+l>aaX{a7o#xj#Uth!>O*->iEg`57gbf9Pc*#u;CI5?
n{sKcz2lpCqeOb;zsF=SRmr>Jp1N_4x+`_l1u`D0MXd?dy~X+%m7FYJF53p|fMaS0lVj`yX5AT|pyiarxh(ErFZdoj=mBqe%e9_#ZJv~dadQy3brY-c!9)8p
X{h*F1QXvZv-OAosx2^%%hlw`B%;**JFVXwJFI%w{!?)>_vbP_uUf6~QX^T&5cmt&a5*NEx2j;ro&E%Xyd2TF(nN^M-WaKVKPH6>XPkQ$?tXG{HVT@Nb)5wt
z}Ypxgwt^i3G}HQ&iqh_x2CYP_1t6!u@4S46_YQ1Cn{}_&}+ZUJG~G?Y77sG?ft5j0{3_Bgg^mdWW}6a32)dKuV%Y8ca9*SR^F20hIuFSfpV=XA`?0*7qz^3
TJI=DU87Po=bN{`_5kI>)o^(w*(p_=)bebSxDwgis-mA9z3S}0h=NM`kV$V#h<%FbszJmKTq*~H(X>gF{Su$>0KPV}ADTf-K3RkTvo+95%NY{CMJc`9jnmad
O`4wlEUr%>Xu;1N0<z)n822>VXI+%eVlPmn!R?Da0^<JwVWS0Hb1`A^=`!%c`-!(4G@&dGNw;$4ErFd0_Ke<8ivoofY`1Gkj%Qo7ey`M`O*Pxs`1}RJF;^M}
sE(Jrt>Bs<N%T;5&H3Y4a%Awz10*<ZG!M^qK2E`;M$5<{)?(E|*Nl{@tfa*ZCG?wL$T_~hwyH`*aw383rpVVlI*~dy$L1^li{NryV=FCQySI!ZR?&4gbs;m|
ICv{lzsC~-y^xbRCF1-el7f(5VTvyN8;$tunOLy@%k(UUTP{Kdy%G@>Q{99jfF}v^LjS(tjH-ASE~t#~QO58Phzg2MhX81Q65o)gk;~8|M6SWQ%ijG|(kZNt
F^(rHyp4oF4DuV;qdC;2z-B8rVV8AMWJI9z!j)^&E6@~fasXrVjq4h^J3Ua_m5;(Me%hKgGqhDyf@2gq)SN#mPr!ss87)DR6Bk{Y?NsctpghRXB~aTni(DBi
U<vaQGXq(-n_C8_&a174e>XY{+S~hV)8H}0p(|Py_r}A51ZGMoP+<6ep<lAPTTIovR8%<)mxuhq!K4Nk`Hj<U>DrCQlkd&a*(OZx#A>FK$x3Iyvq#i`j{Sm-
-91$JG~S{;v3}*qOQB!%9Fy{zo|R3D%%O`yMc+r)LBNt>fVfSVYmD?HN9q^q&<1#R0s)?Kh78Oq)$KvY@4W!}Y?HWr-y;4__^Gt2%s)FY2u8iTdz$}^;4-?r
6UwNZ$7#^^4R%9+)B0t+fHO}!5*p)vAG?CM&WSwy!K|Kj`H1tL$rcRA%fe6b87FWUAhu<BqmlcOK08<QgE>(G%qtlo1TmL(?<xUlVn3cFj;0$vjhnm@@g}`Y
%v*#yENRsyggIni)(Csz4JA-y>O(mdYQCgG0aAc!=fW$fHGAaiJNU!&DDo4gq`i`yAW*LMMINE+@{75^#5S3_lpC^DC$kR@BPTu;f};?-#rT4FQcGmmi%meu
bO)-%ToG*eblDH~p$?Or+09frntcS%IIlWxKUyGxFYw&N5g8mXA5bmHP(iueZDbdi0)RC{G5;DNP{L_gk&K{jV}>~llVkD*L^HeSkg1wkY}3t8PJu4(GX|bF
E~ed9{?b?Fxp$T6FYRA;D_>-#GgbNT+|$rT8_sWc4X(Kz1`Lqjqm}^Ms5BqiPHf^Q<%o`rJf;?FdG7^Gx`2b*)38L8K_<)Hj+);kFE;FII<8<8rBP{Z3mWGX
J9ivE2?n}aR6`t!8w>U3?@8xVdOAj5!QGN?wL<6yRR+$RstathD}f|`1@T~Zn-^$K9qCM4h)x3IC)s9~c;&t)Tukp0kE%U+U$d?RS8RdAi;+Z#uGXdGV?MN<
!dJS3ugoq_Q2px_bV5NP6<(CSGvzop8_bV}%VDnB$fxm%T1lT}EUs8wb>D2@m#w~+@BcaR4Ai-^e{N<PLh|1q2zq^cY#u3r)DemEbRWHm2mA|4s2_N6nOWuK
xTO5vsfzM^6xh8g730{u0>+WVO@e0Wp@`c0I@6L<H%!iL>ccKyoB#(tFg<o9x%#em`?s(x&*hpYyCjEw&#s~WugziO_Tt-xo*?<&=Lr`T`94!6w-gw<J=S06
8ws0C!!93E^mCgqf9+g;ytlQU|1JXsPm53xxYEHh{fo4+S3|7$`6&<x<IUzpEj+a&QGudt;`Ni8JMXJmrQxEwDP<d~*T`u^T62C172_IuofYA2x^iYQR<LcX
w36B9wuTG<^>`ODpbt6H<_q;uJ%a?%rdpwzY61(Y*ASajEzFMaBvQM`JGUSucPm7@68M*<(<sC@+|(Hi7bj4|mq{X`xVzgvm}ljOJ58`3GVLs;irN)C9|c58
4e6L0K--2jh#ui6k#Ny!b>V`GAc}N&^-9F5OB?9RZi>nQ6vHEiZyU7{y$pecS^@YOPU$4y3;S752z@J3kai``vnd=8%V)@SH*O~2AF-H~ZaMJn8v~8;j3k+Y
UKZC~-J6v#Q*O}USarVgm4#7U8`_q&pjYbb6m37Wmlxv<imV;1pC7E6E{I0cNpvn8$R5Df(Ek^Zjy<e55`h^*TrN|LzT+Uot+HD)FIv$z%H7wA5PTT$kXfRH
^aLVmS0cZOn*c7KjPYxi%8LJcOoDwdpXu;`E*Prh&$zWm<ph&~E#?VCC$*-h!fEPja1v_pMS26!&#?1K?<xHw%7E-Xt8V^vZ!m5spsCBK&p3Ej7cfn>!STRp
vZPC`g#6nK-}A)fOXBat^96F*mXxmv|NJ<}%vV8^#Zbb}Mrqh=GZTj?%))(7*I>wQYc-i5HJ?LX$w#VN%KEngMUa$cANTPT=iE#NR<MxKg*KAfInX2C`-pmJ
Iu-L)$!`3A@zK{`tI!YC7i?CX+^YXJ4HXL&@21<gL+A1fOE1Qj>r`Nd(8#QFOtJrH*!>=KQ;S)F;x0AVb>}Z-_eM6AWxVT7Hh8{qC%*ZevhcI;9vERJ2wm`B
Ms`{6%Nk7lRy{xPx(hboffnsFG$@u0=@QOY$fxb;R*@amN`=807bRQ$s*W_j+AL#JUh4bGtQ=N0x}}z7>&?@AFpAoi^LO0jS#F1_a@GymKMb6ioAu`WCq7$B
y(>hA?yZ$Fp9)CF9G4joXRpwTN1AY&vuZd=$5MdP?OnHcYhGQ)GdDUX__d5eYo%nMcT)4n9t?5YT#F_#g~~5Vlj4VUADk6;=IGhfeK1xVj|vDK7N6r(^AkhR
xCARAODxSpbDV-z@wlsj{x9{3qt=jAn-s}l;C}k5xU5cNV-GRkvz?(tjZ=vDeBlWw*8E1L(nXLZ!xUJgC^2nMqIZVT;C%g0%=+Eyo`{;_gPUB?YrLB@oLM1*
5)oG5Rd>aJz_V?!?!1BnB4rjGLm{QwpcLx{*~@&B%RS$0Mlt?Rv+=mr5W_F*NJ>2W**5MOMO!!NKzN}?#dzDn3*!71ROj+qKch^^JeN@f19+`vKb=t?NeTxo
+SJb86E8^FXpE&t?+C*P6~Khzg)Rwme(;8_T&-ltbO(#}17LQ{eW%F+af<k+&maT!u17fCC|p7*o&@dMJz9uJ$QNy*F~ODVB&XzZcI4Nxd)9b4I9zu3tc$lm
QZ;%-HRV)uM6~pMtmClbjiXjD3tF5!Ov9C>haI!e(RzwMyYpq)C(Wips7&tsCk1U7m5&J9$%j^d{zTDh5ktkAV&b{1%=h|Jw|5s+Gd`No75W&-@JLj&v{cAM
vF@Y;y}-e@Rt}(;6A;zaX$CSErySYSLHcK7DNO>bcIAZ30JfaR61+~fsJxs5Q|Bg0j7JWUXksW)W<0x!<EgK{yN?~1r}DT85q0b&);2l#`#>YcR?xa?y<dM7
ZH+q2Z9HCc9%3+KggB(DN*h6><r2}vINiV%*7+)$ji~$RF7<HG9qB<9(Xm&tBod27u!3_|21*&*Q>No+1@AHzixx~~7-S&j#F+QhRha^@Qzjkr>yI+ksO{;V
U-|+VB=60yOYAHIkbRV}@O3-1lCLeZm@93lHs(*DsQ5(*lLuh-FB|qpz)WtySS>Fq1C(8iAw-Uv5z~}GRY&4r8>nR>72yH7wL!-n5&yxdYBuFwec!j?3sy>0
4)12QJsp;Y=1PaZAHMJIG0-K&Xt;1P0ztPV7posMx*j{tU`JcPSrIa!o0svy1gO2uF1W!XwS(&f5nk1G^#+q%kb|57n(G&|j(N*b$&QyaS?gaVXKzvUy>+jL
BKl$e%7R;I_ic5H8=tXViL8Ygu;TkPky%2$$iuos>hB78aX;VzSbkdh9>8<$0q5p?YbJM{I<AW7LrQR5C!5RZ<neM35T1S?d(tyQ5;WHbDgDGK!rNXhAW3Gg
aLe`}PKO`mzZkDMCY^e53H?;xbYhfyCUAcqeWnK&Jh;8cn4*&f#+dD;)#Jm-xLy&<p?d*Y4}#(Hcen-a0*zeh60j((2y+n|E5w_3u=m0`7h<upBHcAnM$7{q
-@+Z5zQ50?$uSw*@SIJet;UvN<P7h|dn+=r`93EY%wD@wmc11Pl4Ixgh8hlql$754ctC#)4*55|+4TqJd>>^9lTQVLWg5<>RPM5pce_1~1`^L%=qnqwNf98%
5mqyQnYH$Iumum8_8*m`k;Srrg+xIrgRKE&r2`c(I+Slu@a{DMyf3D;ET5E^^<Q&eefwX(!Hc`>4|92YB_3{rrcN3|u(uiGkgfzCV}yGXi59~)4y-$$MwXwW
x)2>ltje0q(|--yUOZk_`NQu%WOuY&*HYr@P8CM2&kjaj)qW9Pb@k*DZL(teFT9cc((Eb7<-JN8xg+e|5VS=8NZ4pxBjbGmDse7wFI+3;_r#etkF(?}4w^nT
K2F2fThmGuOtd*3OcX;u_y-r-#PAv^2RUjY)Nvr{-ZA|eV(p!u2S1Vdb0I^^I|rmlSO)}<gp(h~DDpaKmJA}vUvElm$9q4^g>>mDlC4Tu1M2o>8}98g(J`IB
2VME*ZlD$n@RyNpZun~<n_zz-zlE(JW(oG?z7g7WI>EfTPo6(7jlM)YS2u)73l~s%YlafB+DYH$!-EY^KA*q7vxbIu%os2nJgTWGv^Uqv>YVDBy{G2av;CE0
{ya6ic;iITP-<<pnk<YNfzGBpr}2e^PWI)-+5hJ~w?QEtoPBX!40)PEp|X4O!QY<OOMC--vRrtQ1RjMl!9zrR8E~SdU4&UdToJ%yjm~h&(M&N7F?9a<gd01Y
jy^X0oqBX)*7K4s9kh!vu-BzYqVB~vX(Od5ak<7gMn8~aFZ5Qe@=dVT$f1T#{c0u<KtT#=X^4QPs`(l##73|Y0vjFcLKHZm(DC4ae^2xBI;4+aY4+ZCs7k`2
12|Bm=(ulJUtaSay+=Av8mp&V8%dUb<N9_pp53%scZswDni8}B0g=scIf+H;O>!#+OGT%b4k4dUmwGZ`B$wU(@c^2$h}nM@hEzGPb;uF{AaQri<j`$eQ3fAO
c7ECC%p)}gzlk#;;Br0uMYk18tz(Hbq-~Z5Db-4Qr->Zk@uV3)w+Uj0TMMGB(<n?ezZGwBYu)d+&eH=l!1SZ8sI7pz)szebBfqWaNO(b4IM)UJhkchWIkC|p
W>$yTHetBucyVNY&ydg86fmpwnQHq*ppo@@8wh<im`6QPBBFtzqWFOn>-y2DzZ0mK1K{oyd;rj^<oxd(pfUI51ZuQujDA-A>CUnHsa`ix3uDkIS(F<vQ3)r2
IYAGv47ml(Mj!3ANK?<htee68iyej|st;C)VQ7117JW1-4?MQ;r7`R#5T(di$zVR4=xz;|?QYuAZMeV&OOz9-<Q+gF+NAph*p6-6=bhv(v!`W62GuIbc92Hc
Gda;w^7i9NW8eE<XK206npiEE8%pLOy$nKJznJ7J<%ttbj5=;)SJGOsh+~1fp0S!{&Iu_iE!KD|bHHN{S$kbe<!hz^A)qx4DD1;FPf`HMScr&K9(SgorDT0S
t7(T%FA$?;o(R9wf6awSG<Sv)Kx`vG!H=UWepxk9`H6{%2HU(Xu#&g21`LM(Ju5=!7leK8IRthR<Xb~v5dxle;UlcM5f_PNRX6?1ryW<EFvBvBV8RhebGY+p
hV;9SI;XwiEuSNUtfd`4Mb~|+rfEz3_?Z^XxJYw>?92K~|7#<bX%a*8c6W*K|H@{RnnVn1$_B!3UA8qclmW!l;3=QJ5@~<ok&_Wwu|k$%Lc%1hp8NT1y#W7l
#M0uHR$)^fd!Y|Orx}vHag<V3nx#XYvuYz8OJv>+2D<L&Fcaf+&eaCtm-pgB-Vyi!vsq@IX}lc-IrV1CUr$`vLR}TkOy25P&J|eYuR(Db(dA@%+{tfH^|K^~
p&eFva4;^Qtlx?Q=0Gnu@pKVaExRD;9tcFtDq~iq^qVA*KP63mFl!>Sc85-^Y~SY*8)6;6`y`H`5rB9pR%e<`w*(<ufe!CNfn}!4C%167^v5I0AdfME^%Ox@
Yivxyb|aAG`X4s2cPp*%>D;m6#FOP!)_}@N-t^iu+F!kUeJ`Clfw9h&vNBw{_3k>x!)33=Hlgec@B<%p3*R|W$$ly=Nw4Ol|1h*0;6eXCb7*Zi8`rQYX~9|!
wpHYnw}^==`l9j=^M1{CM@chU9g^VC;V7{H$e$Aa_C+3%2vuGglDv~CgZF~85^t9-j+>ZA#mfZmpNDGL7i<^Qf_B)RmCyv!;Mrc8;||bX1fqJ>5vjxR&GQ4Q
v3I;25gfGW9E}(lWhpK)u?(rk+f1WMD4_M;P8^zFG7m#Pg~%{S#f;tnQ&9Ex&3ip!xD(d1E*jztL}I?MoLuab_jC-EagSsOvspa`VJ9{(u}v!4;_iM?n_!MD
{;+LHUrDl%_xc2qVbram*A<PEFrsXfJ+8@7KsY2~?U+gsv0&eU5hsX5rdx?5?q0n^#BhprK}^A`WenCUsKg~ct9XJ-5p`DrltFjCrNct>Gfr-hr9vjBmvi@{
JVN$O`q@w3&@)CN>srTpB{MJ-hGQkl^991JpKRv^{*vNJRzP|Wr}ak<3D9DIVrd7eWLR@#O_PBb!cX*I%-YM>rLM-;`^Id>>rgs$X?j4&S~4xkpY1V%`*wUC
A*1tecxBWCrO72`E8GdJmmStK3M<T1u=N0I2YXa8Jc?PI)IlsDXfcj{Ox54Ln}@Bn(bKCrx-?_wjCR^qw_-P!jyyqj^oQ+s#q-B2Bg?M}b-}D&=q#=B5ToF1
Oz45XTlXnnLUb3pk_%$QdtW@eeQW_bnd#@aNUS(kfTw#98rQ2_Zd6zk`!(+FSI|mhvD-ROLRu|vrTKsYB%3wmQ98@kFP_x?%12N2()H`{nOC>tUM3^1wMGQs
*9($q0{R9;U{=o$@bo0s+@dJ_(3J{NL*(ViNTS^%v6LUo%Q>#g<T4B`{_5$1R0PL%fCtxXp<nQ4!QM=E<lv!7tTR(*0KPqB5se*o?r;`B+g%5}6ELhljymzL
J<f9={xg^#s^;u7pa%++ilc{G#5{Tq=tE8PxJOQ@(64#_Z{LJEdg;|gdv5?~oD%QT2J;S5AgxtZ>L^P>`GS!m@W8)=_`Vmfb9c>F^v4*qZ2;OinK@}Q5G?5&
G8!ci<@+dsPokG)BDCZUB|oZ8sNM(j3I1c=EFSiQey@1vb=s6)+#{Duxexa}Yi4g=(Sk3368Jzi?(!Qu%2OV@f=4tPhc8~&zdYn}j-c#i!Atlc%96htDHQ#$
)Bq@n?$ZUO$pmy~9&JLCU!bG@d>Lun;0xrd#WEnmgJClQ0luria}osA%Y}oW7Xz%jx{Ft0jr5m4Y!?8FY6rqTP$OLG0X$7%xvAKY@c@#tS?YK0Us~uobxmV0
aO+TQeWm^jw|3a{5l;ihQxfMO;?RdPqVzVPwUoF^(R}yg=S=(K?;JUwPbzC0U)#~8rBG<e?xU#Q=Sn+S`w}t+k{nhr&`=w-m{h@bAUNMuFaao2j%3B>MC?R0
rPUWj5_6xBAw?2@7ueqE(YGWp#EV%RQq~=LtgD!|Y^VtxQ_)U?1ocM6JQa^LLAR^_HP~fIRNuT5xFf+<?2&A}(Kpu#6bXoJeQ=%dI7IMp8d7jr2iZC(RLr3B
x)%fgB`{C~Oy9!@C%ujz^EFtnj{1Fi=fVA-hZxe-ZHu<3;xhVf7$%S97ICl|LN@7IIu6_FYWr+IOYG^lLS8u$#yX1pQ4b&m2dhqcfb*4T*w*7-)PxiLw-p=w
@0e$>9wJm>;75N(ihquD?~dkt<-54YiH)}K+Yq7#Xf&SF?6ze+wWor<*|7Z#XrjYl&9h8+v&sT}3C$5X2an(EWJ+@xnhBN1$i+ft*RaTMcibKPcmm;%tk5%v
z<bl1!34#BDsUJl0+XAy!~m%1@>wx1Jf;lFF34l;u&{436sS)O@f4=S@4TL_H?E1i*dT!UU7nN#*o|lq>u~b>E#OPIhkLVTBjbZEF=}LssK2*E6?uyjxznr*
!mU71Z%Pxfa@|&Am*2|$f5l?w-_4F1t4UbuKD_an=Z#@5Hu+P#DKc(D7tnNOE%IoUw=d7eq#u4nkt{`87j84n(jx-M$`1{dFeWiSZt0RJD+1T0-u#2*OOG3-
*GIaWc|2n2Icc6E2m@9%{ol9k_4JNKm*AC@Y!7U5$4Xj~74(}aCfTR3gxq7G5JH#kRl%=`{4Ge$*^$eOI})S`Q~G672^>Rf0Kyqa#P$;<KR_|Nz(b33faF%+
adFwt=^@=04OA7)!j&}e0<cI}{&$}4jhNcC^lU|&i&7#2I778S@~X2L=Xq_xPMvL@h3KY`Jw3Z{WJ<!3DUkw?BEF2ibsNO^k9VX0ctRgLX^adyS-Hpvmi=Lg
1V)3MifC5X_>yR_ATugL{MER7WdScw%W|q~+k~z-lRehX>e>nsx&wR$0I4hL8FFyJ&%Gj)jQvSweU0z?I{i}m8TzZ*poTB#c-|*op5dcn3n$v{$O+)a!b@;c
1e3Kp?@IZm_(r$Hzb!&8h$INV`7lNjO-2`UpMd)_`qp~tCLXP-T*t4%zcVRsvuGp3BYs?4Qno87UAJN=f*j22SNHj=Y{!1bt(%cA?GM$)p$y~a-}fY<&DLJj
jr#|+_HLcZb4Nk4lpBDQjXB`9(FoR`&aG1|k#?V?#9u^(sjt=w5_1$%<UZ!>|5^&RTCsA)NpDfNZp#aE@s|1XB;(lFcTDcPivho7xXJ{0XF6%d)t24GpIk^y
cHcXNm^C7ZDWbdw->EFp#)kaUDg6`-^j7kc?e*Iml$YnHp1a@BCM9ISS-Z~CFgf~vcaYc>P)JEei}jO!L>^ToYmYVBkakP1G_k);@`8+@p@Qtqzlk)M>^SBo
^QBZr?w!a?_#EF!v|cD7S|@lw&P)_K?ptmvcDT-~w>JX#mj%MQ)7;+wgo}J$WnXSyG6q-`>aT$3eQ$bSV~p3yfLwxIo8{f|;&P1`<sfdEYrNkn(VB<NB{wiS
pHXAiT3xewtr>+s%p19cKDhGBlOC9vc<A9Y@V0|BHQ&#-0c;+?TCPyaQ9JNV5B9vvD-9XcL#Q9Mnq+Xpx99r<IRUMhQ8_GPiWVc|MgN+}`50k~ZA{P+Zio+W
U_yOI1`>9J$en2=MEER+>mBeh_cA2@;am-uM`9ESv<bZK9Bc+W_xdF-TY3eyvxb{XukNBPmq04mu44xiQpW8@#h2(BzC!NZ&nRoWlwfv#QPcVils~3Kf&0Bf
w8*c7eaZAyGS*qVVIOrWZ!BP+B*Dfj&|Rzpa93>(by$1dCZoIwkQ6`^Y&yJv!s`_;H-dqh-~IJL%3Ho*U<T1@7CTeKaT>^Y5qQwMEN~=M-T9|{*p*$Ac1Opj
;Ud>>B2Kk#PoO=mN|TZd*IacJi`lWEeP+0rw#Dfv#Hh8XXdn8Y=6N}ShOn&NzfVA4qFH^gASi0j#@pKg?<2sjD4@c;03q{(FR*1LGi_!(lSit=CW+T)t<#{K
lo>R7!-P{(->0*#exuL6%NF-V?ckbPtdVuXucT9c`Vn(CuIW7856lH9x8Ox=XJAJO6zM-p+<qm_p#fzf^PemU$ADSv%Cwp4BDxa~Qv-z08HCeA{gngWtQ^Hb
uBX*9u?Cv}hUf@3M_`=VXV;b>jms7$xWnBDvXY&6!jXv5_P87@q1%>3(jK6UVMBwMaT*2AwOUJDni2)JGftuJ9Lha2P`ES#1AS)~pTe4Y1R|#`@PthtJ^yF%
bM7x`pP}iIY|NVIQRJG%!p-NDnyX4mwUd2={&nzlNbg;J&jUPMf~pV!xX%^);8F9xU9F2rI8o{#ldmyl5n+=5MiW=m@iLz_ac^q7iKV%QoY0EHrpV>0+iINh
0-aN%A!?MpI;a<{_t)x34!{z03*L7#kQGr+gGQmQSu3bRLI<n_`*d)GuMXVHl~dLB?XxhBq;GJfW7KA&>TG?d!H^6a2m+B-tqwUm`j@sp1G%Y!-jImjsO%ql
8*hc;^{|lZZJj-y#+PKd8z+ql<Gz-b048F2dK*Cs2GUWq8m0mlDtMvqv+ND&^OQ^ax3>a^pzP!YE7;{P#1NEu$wt0k7bnG@uR*?9`U>lVM==PsEJT8^fRCxZ
i4v%u%B@~apQXFe)+UEpTfT~gsr|o$GK~k-<6&B4%f*W1iE0L6QPAY*k4>!KL<a<SV(Mh3jX~ipt`bz6XDihyqbDybcpHZ$b@;M#=1l%lN(89}_-7=jKH@V-
@7eoaeT1Uo*xBE*DAh5@@-=u3gep^#xv_zymTjUvtPnkok;qL3RLes~@Kvhwb(ywZVpyV4yeH-0ffjKUoZn`m!CpRGI{u8;cVPp!r?Si5yXBnb;F?L~t#LYz
`sBw#GzZ6y+3S(dkjOs_jTPp*Z%y4bO`n-Ri0gd^5?qV?W0qkGack|45#hL7VFY6`{Na%@hF#4(>w=KT(cH+)fMQ4pY~|DQYk`|kWVNs;C+K1u`l?7*g|HTZ
jjY=-e)*4bnj})1BX7E{m0O+{hkP&LiJe{%g?>;|d)Ub?)3UXP4LD5!Im0|tX%FMK4E8er5oyf8d6HyFrh1<}K)xBYlLR|%$d_stI=>6}&x);`?TzUU%51X&
CJo*hT^(4s<kS3yh`TBb<!&S1mGFrRViSV@6+`a~lXM7_26)9iReLyXIsE-1-C|`K&47r~$Tr@tbLfqj13#)1Zg#nxN}}n8DCKTlcKoYXIr7vJj8G<sN^IsG
kRMzaG-1P%IbN@C?NDeh8XStSTNsKWES{ZvxckQvlMxHYunOc9t#vReD_d@UGlDNWXcA^Fuc^9MWdhM4!kU9#1nl!S9((#@WhjqBG+Sr@00000^oY%;v5SW4
00EJ?0gkT^2A7ZGvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/externo_lefood.csv')

In [ ]:
# listas/externo_mensa.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/externo_mensa.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;Skdi9$f%38cNL}xSP$f!ssd@4s5WWOrlr3gU;hVGuN%4>_a%;*@>!T!hp&~KS2RcVXXifz|pU?X<ej=RYsNHgC9gZ
^Yyj_@w3l^xQPPIw;tS7j7!DT;i*-LD=CgW>TmISg+LT`At;aA;E16tHJvpXoQElAO~A0P6>#q#QpshLvAfmNV?M4YwQp)!i9($3hrjK1d1i?CP^}K<x?@~Q
_U<}k|FCG>1i?@oUNb!>Z=HSr2pGCZ^&s1&j_XWHr(1>F>$hcl2?XsUDVg;L7w8rvMTlHukOURZ3Nef+WAJgTA@Xc$ppcnnU{7bdih4?B9z!pNF=Kdc1PQ|U
QJ#v&wd0IhSXHo?ycwa@(Ot2zx7U?<NkXM5n9#DM+4D4$p&uYl1ZgtI>*ILLUdZ&}+!n*kJ6Nso+Gnx73$TxhAy?mVijvJp?=iXBe|JZH=@;2sB$~B3FBRCz
LN}+vs;NJb(%ip%S$_;G7Yzp#Zm%HtN_l>GkbI+RmNcC!+DGNl9dTX50#MrR#YW=bqpX5L(#C(DKe~`ugB>%)(Z_s2ZoAmlMwPDqHWHM#$ol>=f+FnbcwNJ7
3R_B=)oZdRZTyRt%Ti9mJP^|LGmBBJOjbM>pduOEa(qaeLaG=LN?}~XE;CFGD0*R6^!j|ORSBD9r@m{dq+8(zrg_!fmZnM&KG}iQFE=kk>~AJZ_?#0VKbrq=
MRK%VC%rE_tgIYC|EZ#A*J-K#wTjlke4FRv`954dfX_L43mGwNngRq>u~c>vu>e|Qe;rSM!XR7P3)mvx6Bd_b<<*%e0$61A?&3Bzt%@)nwXb^+J*El8mydR3
B(Tfh01MHc>JJ2xYuMb5Tm{}i)fzD?kBh+$9dl2Maeyiji6o(tH&UYbINnUZp=RoX8Bs9IoTuB#$<CNVhFmerzf=7g4aWd&+Pw&dPg5Ag*g!zq+cS*U1*NWF
_}nN#6+l8ETVRHuaE?aaXcr=;Vc7_<nAW6rL>oT1_&TUb5~l$RUz!Kjv-40}ncaJJ*Ln61JRa(dhcm^eI--J~;ZERcF(NBvd-}N@6f90#bb)MdW1;?sjSEF3
IRxu6FiZ-;isfGvIPDw|T@JHA2bjd&2M5XgN*2MAzgF1~KHd&F;UP`_6J&_RJ1yW^Eoh!E&q7vT_Eo=`e^>)m;d!RJ84LE&a|!3$G#IHzY_<vo4y8#oz-(>g
@*La~TAhK5FqeZ(ag?Z*wA_5hB(0(>i+8+>pMOJN3#7py-(%=-!qfZH^#w-C)wd>|3W5ZQX+L4Hv*gpc$}n;?<>6%8j|sZIJq_DPyBd4#HiRPk!MNa8qnuY@
&6wb$Rk`UFmyFIff->#4L>HSwNdw}TDR5`;?O`b++|ujkXs#m_pzXK1(n^{fKW!&%FS0V+u2B)bcM-$pYQlc%&$S<uytTtOdIS}f=cf%y)oQ-b;W$7>=?={4
flry&%vun00~MHwaFMG8{~McjfX7~18aR<bs-~DthOl4Vdosc-C@X*?fmp0gA4ADILi>Vk*+8&zxV_Qkbs3ctM(f++HQ9cC6;65;fC#Do<BNh%@0QFk<~1fi
M~z2oN3s4KInS;C%&H|ydx}->$wZN=ld0c!*&HAu|BI0^2!<(87V+bGT2rc1eLfV4M)p-`?`Ul@0s}6^S{Xsr4rW$osM=Ji!}aXy$`ki;*WBC9sHvmQ%`UgF
DaMblxDZ>*h_ZQ7q`;g?^^o=8x4%J5?^gm_Q_1g38<P$9mMM+!fviVVWndJL<DC062xll{(J7$LIu=+5b`fZozgJNZrVn;BMzT;=UWkOB5wIk`R+0l0uC#`(
sL=B{>j);k%M0?a?nm%2k~6+<j+UB<)SBhOw{^&_C7b9fYdH1uwAF?x*OS&2S+#X9{Oyz3>z^h+(b@XQplU?#3ycF5s}=am+TA6%Zq^h;ODq*-(k&JN0zA4Z
xD_|)lazvmS4?on8<rf|p9=KO<F=aUzZ$R@!SUQaiZlWQ3B@v|gvr`nCyR-p5s5!Uu;ll@;@odR#zh!-Px`ANxA08UX}JKRo6@~!v#tO_o50uNBug4I%m=0(
P?$u1y9)`jW7}s)Y;a6)#fR0UMHJw1ODmqap@$mEDm0fUWr&@L5u$I>RZ4ytf?&@q(buPv_g+Hq7u!RLP2$mn>vu^nPjo<m(sjNSUaR8L&!|6=Ts4(WsZ>&h
+0)^Z^+E`wbC>^T!odx##RJ?bo9>|&!AEdM>Q?SA!4S{OdZo0t0;AmRf~v_Zk;afn3ujHL=wjv^!AM8JmBIYOC<#@%93+8_zh{2qaq<r3p>gVYUUXHhk+_W#
q)GuzQxU}WJn;T*2Ld#oKlk!gBtd;DwX&U_ss5L?U%=;E4QiUJSMy-EPXT`$*{LYo^_}-uj4e&-6XEJR0bkfSyBXEv-Hc5!R*CQ*;$AS4K<#mHnOSW9XFW`<
ia8jdJ>3q}Kb1FW?EAkL(yfP{Bd*CK-H3PB0o=TUv<={yEnO%B7jvF*Ljz-B>xi&#sqLvUhXVQjDp!?@zvouHYVH3kX1ez?I)6OxGcF+Dkq|Apm02yvRSWM6
uKbEren2M;%u(#s^No8|tx2(6yI+a$^%=U?o}-M6UgS%|a^{}5%ibR^h2l+~3?Ea?(9(qeFjK=Z%(VM=`HkM3EOj@^R3wtCMD2!;K9*4*I(ouBmelRSpuvY-
6?&E%@dw#7+}oaE>V-ypnTiy_gNCObq13N>T|^ujZZu^*Q0TW7W)4o;_?*#@hFp!qs&m4Cd2mK-2KB;Tkbp>NU1H)1cCV6m0L`8K$#)sf{wBhNZ)>%Oqhjl#
zFmlUT9DuoKTj?)<s!vM_~YY&VUl}gGN`UidUuf2yQmx_8y@Y}I|;MJBN=|eQ5hC*3?5}#tV-E{MgQBt2QAAVnR&1XaG=W_K*ZABKWsoRnk!~TNDOqpce14I
%AG^m>&IVx3#Em|Ttdtmu{o^p+ldzj2=)J@I2l`qa5IL$wMHq|PxZt)N4d`cpVC6Md92)(`SbU37v9{EF0Sl?V*_6tBWcW;Pm}4XNg3cCu{IlhY*3t%9W){;
4IPL3WZ}X|ilYf45Rw3WN6CGRMadqZBPoO^Gi;#54?s8;f>$6Jcha-ZCJm3$eYK&rMr@d*VJ{hZ(;7K{QkmQzH$-uF16VUv#Gye@dW|9usw4jZG7|ZQqL?X$
qe5RLJ3&^+pa;0Tr;3&EHDF!2@u@WeN4R7>df+gK(Ru+@3VPT*ks}A(2p6l8jy#1*qndUwpjDWu|E@8Ds%rYZ%v$H`ebcX!d<~pIop$LY<%O~HL!Xxx1W*a_
fM)1~{*P<?I^#1|ZABLg=?b6CNhWu>43Dp?p4&_*gcq33POo3y)h6X4JbPu|W~Yu(d}cG8c8G>;u~&nU#W3A_?h;fGM_f2MtJzAJK3Yj8YO(f#*%473{H^=t
r5woN6bw@?<Aa42AJ2JwKUT))CQM7@ZizcO`+-DB&&W5jPiU@FNZ!Be@S53NMTQ-JlL151vcYf)$XoFCKd6iGJP*d!zK4b&n%YRub&ru>^g|k-R>NA~(Q*Bc
m&^kYRxC93)%<K!I{an}!usxy>8;JW%5h);ILK6wEnTLGUD2Al!HghL?I6nRwrcF<ooG%qR+9m}wR}OXAgRnM{?+baJH*bc83+)m8C^m6FJ|c$0tIWGC?Oc4
yKS`4lUIjAu|SmwKPtOw;a_=gvunFqk&6tJZo%uhmg`rhK5cnn;p7(+^7+7Z{S=V+QLMKL&JGaejh(cPQ3A=}@rz$R2RN8<d8c1@iCQUd9l=N-;gQ=)T<RR_
go_s$SAn(-{g)Yp2)m!Ih2t|fEM#Z`>2pc>Bm6`*5k$?9p3V>U_OZv{V*rWv*oMJyQ56z{0Po)Q7xB^%E>Mzabz{{iO`x@^w3Z_t_<%!}!^x_R8=(8iTOC`6
YI{sd8>O9~zNDpPo#t<=pvmNRm)kqtb9*F^U5%>?+k@~FZ}6NMCJasNeZoPK-?0F9fXJu{lhS$%J!8pC*?#VqL)Nl~m0!-F^*CFRoKvdq6au}t!2Nc%PwhA(
REUHhGvAAd*lnVaso_F>27oD{>ytDkScyEZBwrXMSv;M}P)^2i6aIdhhk1WJKg%xdmW{8sBZJ<39$yI~MO4piyZ)a^VT0A(CC*-(OF!Rvopz!?`wmIf^jx|e
q&bB1UCg$_!&Uqs8+?0?2`-FbI4Isb`9eT}h7*}@ay&gPRe7|+;v$DR6(7?nC@|V_xhDyc;Ol|QA94&HE>uC}ODOYWR(ggNfGBQ@SQlV+9HsrwIWivXF(1*(
d9~h(e~%Eq8x-rH45&xc(t6gG<Cu7%l%#FhAzl_CiEQV*MC)uIgL-NCYlU(hO|v0IKTvOUqbGQjJH#oBtujL@%6&ZMYI1A52C}J!6YRQmvK+N5z#4g{AM$2S
Y16E#*pXyV#k4{o4ecag&#W>`i*Wuw&dy7q3!!!>nr;1*DSB8keIaRKr>`(*rOwC>n3|>TT`Q`P_3%H3X&`2ksv#hWn%JFMMK0H@M(#a#Jqp-B;n5#>k}+l;
dfaifBo>5$7Ba<;vLQoOnNCRqg0iZlN7JaZ6|5-?+89k3+MySHC;Tj~`hj9dTuo)U`h!y;x52IbtTtK*i)wZj4$G-mQbvzMCg{lo%<x7HzOdZ<m;}v$##Cid
=6m@1qq(rn<4|cjzE?CwqP9T?0n@<JY_YnS`JVK^@o<u@nGfN-@Xv+&URs4z7~8+sTa#s=oT9<%Qdj_pc*XHjpdFg3oXLR;P<svoth1@A^sx?^CKD{$B^9$N
doI+Q89cXBR=E)jb`G~{3k-j(<*lCEqH5&X0=dU4-H*Sh5ja<f8;e%3@Qw>%B#0puFdukPKD-h3aLEyD#>oMk0uaG^i=o88hse;SRz#9<a^apeaA~0QVyY?)
a%US!m$riw`Qm7&bs#6l8I$ES=>nlqgx|XfJ<Qanq<nTGTan&){4@MdoSELRW>|abagVr}^Q*U}Az|4A8_)~`ArwKhfqE)u$ql(!@8qp@!Jzf}2eqyk#{;ap
D#u4ft<D(&5CTWrN!}G5gvbI~7#GNhMsTV3Xq9EHXhP=T;23hR`~JQfML|NO;KO;!u%a-y-yo6#<SR#3OQTTCW{2!F%<6sBiXhzex25|^%5HhcXNfomUO(^1
6wI2VliIEGWrEwjf>UN(eY`xha_C=6X)`dBnTl1$e>(j4dEUV{`Zj}%tjmCpBC3dStls2WDWFk*a6z$;2ZjXCe0gRVj>w)ISM2sNHg>Mu94K>+!(7ceP2GYR
n(Q1KRfF;157C=ZZa!*_G99-4qufgn)aL`=cJpj-)kvC@_iP>8@JnBdjX=#}fav*~7K~U?lI6${yB<Iqdg3;&&M&r=9bi_RB@GgjeL<KjjdPRq2Ya6{QOsMz
oI}3ATNEl|r92M$T^sQ=GSN$J;ei#y`}mH>e*O7(<=kWVPJJ3k-0#cxnLrLtYlYipX)i+WlHo15$v#;C{6Cg^ik;CY+?<%$<dcwMx#z}vTM}%TeGDajQiMsi
z`_#xzAhUCn4%3NC)3ZpI7~0X^wDO-(}j>6{&b#>q8HQS6aCFnF0<SM|K&S}AZMh3hXdAOy1XgKpC{|oz3>7NfhAFb8Q_m!dqW}aY(ZA4?xfJ@Bln{h5dZ)H
YP>1$o|liW00Fup)S(0bH$>&uvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/externo_mensa.csv')

In [ ]:
# listas/procedencia_r4.json (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/procedencia_r4.json', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5-~)^Sxm^G~f`9=c<Fp)p>rCx{$66ocxA48)4nzoP9DCX6`0NDs?Tl6E^p9u1wQHc;BXPHKLy7}Jp4u&1v%#^Kb0k%6
iO)dymlR5q(EsYY=0)wNVX~Vb@n?nW_<@N64aEOUwp2o&e+v=6+06BaN6_2tvludfaq-ou^Ed`I<)DW&4MGymJ-_tGt1maNv<L&y_3Bq$NbS<nG&})9M4RlJ
%fYTsg=wAs+xxi!Tx7Sgcc3s&i$^#Ed8*7+Qu#D$Zj|Vl0&Y2o^Cj#!71p+F<t^&HN0U~MpbuobjBA_td5^CBEQXa3CQgr07nXB~%uQ>m@<;5(Ajlm+y?Pac
o60f6=M!P6#iQXRxgmFv|0k2`nBma|h9#J3K~q#gM$_G@3}iLveme_aI6~{R1=tPH0z8A-g=3{j9jiEh8+#56LBc?V2eIswGx|Ibj*6)xL|iZ5G`||r=n=LM
bGHo@)54WlP<P-%bXAm6^G$fox?XQ$kD={t{)*FV0@?<%wBX;uDEle^b5oM7^>F*FSUjc#T3n1hcjPSq%xx40__i4aXLjXK$c)MM>eUJcAzFV|00000<dRtn
=3KVW00GqljRyb#@>_b`vBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/procedencia_r4.json')

In [ ]:
# listas/treino_v2.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/treino_v2.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5<twuA0bKwx8cNL}xSP$f!ssd@4s5WWOrlr3gU;hVGuN%4>_a%;*@>!T!hp&~KS2RcVXXifz|pU?X<ej=RYsNHgC9gZ
^Yyj_@w3l^xQPPIw;tS7j7!DT;i*-LD=CgW>TmISg+LT`At;aA;E16tHJvpXoQElAO~A0P6>#q#QpshLvAfmNV?M4YwQp)!i9($3hrjK1d1i?CP^}K<x?@~Q
_U<}k|FCG>1i?@oUNb!>Z=HSr2pGCZ^&s1&j_XWHr(1>F>$hcl2?XsUDVg;L7w8rvMTl9(JVVGirv%q4sY9B44hig?`BE%r#o^Zq29?%ejt>DGIwE3W{!9Ii
^T8pa9s<-_nLk_t<B_+dejJAjh(+J+N8U?`f2xn<Go0Y8oPTpiTaK?%UG$Sjy(y^*#+tYyjUVk;J>~er0Lq#aW|rHOsotdtZQnaJZQK;e*L~nvL)e~YHZgq%
y|t{-ugR7Dcw@u4+RaA>@wR_)+(1(IvWUB%z5xJsyi@GBgq(^zep`iXdO5Te_WgU(g`i{+aPb>`3Kw-Pg`9|tJMZRWSk!W-!3HvP6$Ci@hZaj>UmZ{C1<}s<
)Ly&9Z13IhYP8cLI@P#*j_IgP${=Sh;pikjg<k4cBq6&X$GVF_^D|*MX<BDM)h=C=H9d!6;tm)sCXTboA$7A?zjnLNiJIEA3P7TGg|;U2bs0wGE+Uz;X9IH!
z#=ArZLUb*-j6G3B)i01JvZ4BkS;O|$sEE2SZah?`|w~RW$Z_}XumaOhKS~2h+4%)6CS1S?>%30l%EgXd)kR#fk@aDKlYnECfq>5pcqHj1xUcnweASV3pvbs
V_4;m!Z{ku(YLn3?*c!?MzW4^ms)>!>1Z)*1|a!?>ZYyi7|-e<5nj}PdxEP#Zk}QYrppJ=L>nijHKG{J0XMrqwF|kEAL-CbpFvkBmv!um%ihllIi6d>y~V<V
lcBE>!N=VU^b2cbk%pDGw1lBU?^(CZ(2o7QaD_VbF~`yoe8em}0kc8)@!GCcWxT+^x(TzSyg2W%ytCvV#}`Irb-G=hDQ<dGzf?LGr;!;iCQEMAX&bXvTh61$
X6nj?R&JLrzm<+e@;)a~-39}3-$K~i4<MhzJ>k@@JMg+GxHM;c9iN30DZ%XBFl({LhYno_)4I5=jz5Rz5X3hRwJ}azCik9{2hUNZkNws?t&}YscR*;$iwL~-
a>(d3AuR~T*Paq~M>fv~b^q8TuZ?M$?ncqB#{s{805GY5RG+G=tD|do66(z<e^F6s$yJEx!QaRzRny<;q_6af)m@!me_x=cK_75!%%l=3i;0$m_g^XX$XA2+
K!=A#1gL>$PKtCQ^>w~TR~o#M$szRRs|fk+$hDY`AswT4vYH;cKej<dW#=_Y<5#p0cxgN>tRC5}Qm*iHOcJv!E9_=|lfV6R!mk1+J{DLKc2uD9uE8GphIqf2
-~H2k^;}CaHT6%3#P3{5@|E}FK(M7ML+0#I&>OF3fzyWT)(-n7=rB>@Y<G54e}WIpV(Bu37Pw2#D&}Zt{vhS20IICED}Ljh083RWl&I$(^}Vh9_w4uV>yr6f
Dvr&*UT&(#;A~@Q?F5I`p!01E4~+%cLs0O{*cMDIy^_q-Ka3=3=&U}NuM6tO3}}+kaXy;Oio0M6=~4nJQi4%{Yo)DJ4xLjN%>~|thmqNY)8*@R_&UGhsI2aC
>g2)d(KxuFwI};W=n9ZO$`Y(+bCBCdu4!JYP`m}P_5~<QP)hA5govSa;^T(nQ@lK#yhB>CwiTiB^Gg=(YBQomCkmTFe|XkEqLc#y53K#u=LF~#7jhBx!@^^C
(1bWyi@gyVo+rXOpQX_-57*~(*q%7f91M~{-*g3_fq*I_njldYTe!t-kwrp<GDTVqjCw)yEtHhOD-0-$(Z^nwL#71~5%i<qzLc^I?KE$<y+3Jqe$iXE)qT?`
o#hLy<#!%$rv&i1E*Z^(`vdP_tWPO>sGp>lqQwreqOPdh_8`vz!R5L1FeqbRj|6uoR+4f#D&!*i!ZW`?fW`XyM=uSfTBdvBPL=7clnx)L))ekjY<$lC1=>g|
Ed`_*1~<W2u*+kh0c2>d4AjrKEb!V?>Xr5XQf(C{RJsyM<pL`5pzlhHi{~8WjFs2O)7b3stNagtbxvW38}XS+^q2U<SS6MTHWF3nGlW%>aW_Sde&q=C+z$;j
z3p+udJkf)`8?0;#k5UYn(z)4yU+LrRL3M^6CgFWizp}XabLn8Z@?Z0qBzaF`b#)fYt;g$qf-II<twwO^?r<{MbM66MKFdyOUmsXD-()WS8C=!)vr0zTilyN
m#ofrqoSo8G01Q`z}Q?{Y?pykTl$z$D3AvHVE2*7%DcwVgVrN+f6g<T4f~lW`l<H8Ema>$ezPa?{-)wJ>F13W@iU>ous&=lQc7U8djcC;r|V35u7$NB9+-@U
`eOllO(`8rs1Nv5d^vP4+!w`AF6S~_oTWlkW)_6*09a$0DG>YHPbL{W6m%s6!y`q6_(QfZq*V%R?N|t2{SrsV0KcVupym5?iIg9G-{lK)tR_lBg2|992cam>
1k*@+S|2iA<Vg!zIP5|&yG?7KVcT5NQE(&08)CY5<TT-tVbTyi5qXKQ&xx7DJ#3*N*z}&7OIs7so7gYY-8T9N)wB4QD<mQH#05=%=PI~Laz<jefi`nD3Vk|%
#M@9-ZY9SP#An4XqQVUo=!Ua@Ul56(%}AfISfQ7yEML}qEe4(Pfoj#|Nq(Hq$zliN%~Z}oFW9@?Q`rvvTFsUkTEj?R(kVFg2&&b5UX?dhzhrq~4q}KrNn}6m
Y6(f$Wa~rCmGX>5ys3}_{hm^kbj!$eh8g%(r3<easce7Io4AL7pX!_^uZ1%)>qSo4d7l^y*QfJ<e&H5P2<mpPm<3dPh?`+XuOHx)zBRP!em=g@)uqX_4Y%iV
s1yz+h-Y+&SsNF4#>?c9BcV(v>lLRdGv1_&7kc)^AW*%vP?ZBfb5sEp8cvmM4^)D>7;7{9Fgj~7fCu1Ve=1^2itK*(k<sj(!m!jY$m9W}c@OxV6`GVB&R0^0
gYuGDu@q*_l(nV#VIORsUzO|V{;5&1&y3I+g_H<H>Kb=<{g|*9g#K2%_S{3}|4VF6Ead^k0GkjPM&xADaw~x2LyHgE&pLzzOX=-jU!h2~o$08Co;cWQHS<k4
m?$&sysl%q`M`E%F8+-V!UHec|B5Czuj*C50MnX7gxs2;3-i`8+|Xi0{J7A3y<C|NPM!8b;-~fMOr#y(4}tEVfQbCKaiM(5G;}?NsNb`vRCz3Owsqk>#@0O6
#LKCgI5(JyODpXZDOj%k-x4hnM7ijF8VZkRX)FW~#SV(&*FZ|{Xyvzuly+mN>hBo|PM&)n?eiU&DHC*dPXT#<OI)(qR_rn=;@@z6EdAvzbce3!y6ba!LUqn{
qN?SOvUEB_P#IynlvZmVmJdG}XXXrR`V_L|H;qKT<d*%%&`vvnGc;GJ&#l<XC~D{Llj{>6Ya&<`oU!OQ1}s6vKVy7TF%C^P#*AAM8qg>_m&p71*f_OSgV~J%
jI|LL6tQX@jQ2b3(C;qwwb(!^13LLg=QS3*+34$+8Ke;t<(zt`)6ud29Gz%yOi#1f<$g9GBgWmg_ul8pnq{LXkM>;B_EMa!oG?5O6q86!)Q-wP@#CsE+o-II
=k4>((<}FO4tFY?z%|Be{yIpB%>!qM$YFei&$9-$DJ~|~WbbZiu(cJw4iZqxW0<1ksf`R&zgrdJ(okFlsOY+gAhRhk9k+~(Sb~~NyPBT$-MG$(fwz$8K5Vs_
*tVZSMqa+dcy>08XQ;VInZ!AbXlI#w<04e=Q$h|jc-FYFG#GOKt*%gLIziC@mXSprW1TbOM}!Fy#_)ziveKn3$xI~;yfp33d#E<VAO&i|kdh7W7(+!Axb_o`
-s&Sh8f4|w<*M6_&T#QhkJ*o&i@EIG7JZ&#EItN4)7X@x4B}N8c%OL49~XClys9ljPe!S3?xtuqU@<}2Dv@cPTDl1ivS2qbaZp7ZFT;u#78x`rIkCI!Hk7wO
;M3I{w}E%}knRHwEgAKQ1D<zid2NMZkgnVuFt+Sx?ZJry6L&vZ#W2yb=O;9LXp&G2W5WoKjGEJ!KF@gmp!>pMQ-o?!HF%Mu&&uPr^5vm?G;+QMh!rZZr(cP-
keG31)=*XX+oIKV-xa5(dxleKD$$o#ST}%^8cxAsPkAvFA_&jc(*gPB2!QTCzfWRdV7%<&ga*lgB}U$xqI=%QsS4T9!5&|lH}ni6E$*Q73@%v*zfdMtWg2jt
f%F27UdS|XBJDmcBXPrEd=?4km%B6^vC|sU%W<xAz1kdFT4Fb;B*!XDsrH(O4?lKR7ke9JOSZ%)V*;*g+bsbW;k`;?m+4Ht>>iuw`jtu`c(pUSLf>QNi6OJl
hOch_vfp`7^7MD?qvh$ED@`;}6zybmCoTt8*24$N*UYaVe1kR&==IPTYYsDdLsAl=_Il2qU9xsX#S(tHh~f(%Xy1153y~+o^O2qB_+n%8K_`?|R`YlMBwoPM
TGoI{)w9RHap?PQus9;Qp2}5$!hcWp&mPY&tR)?1zLN7K<!xSXEYa>1U8nBX#PrWmn(@vr{$1v~UhI!xyvK8rDpk7^VT<Fzwtg_HE`x-|hch;sRVnxAcrR5_
vJUdZi}a*#LDFd>-<us<k97HQtju2FU!Tm@!VlwkyJNgdt~p#KBgES$*(P>ok)R^*jqBDjg@LJUAK?#F&gOy!9Zik)8t$EwXI|Nxv4_81HzqBC;(+{L+|1cV
fn^CfWd>gB4bm`MY84Mh{NsFuyd+gyMg~IajJ$$HR?3ye5qs=I^x{4goPt@12zy;ydYs=cVq*Q<oY>@U)Zz`u`DrgHGo2}QN8F!_cq^|cGZA9JS`{jkcSd*9
V%Ta@UlvXv)U%f!yYo~<*1VH?=cj=^EzonbXo*y*C`N_7Uh_?)R|Jh(tN~1Y;)POWt%Y8G>YAL%@FmhHo8XIa-<P*8;6u~rLTVb0F`(zqn&AC1rv2j7;bxCJ
6wXd;*;48|de8PJbbah3uPd+r(pfJsG6?7d9q9Sq)|8w}@<i0ewkFB3lfdn@zg6dPvyKG2V`)0K42DlPHz`n_vozw&9=BFf^o~tP5%5&^ZWfxQlEX;SJ_-+o
(3KAC%pRz)CZHvu@)TQNm>M2C8|LM~P*%}#RO)OA4Q#LMLm+>JCNK>C#1t!IQ=1%X@Nw@CQpiX7@XHhke#=S+w&D|`7hJ%n(s`&$hj+u7nC5XV(Hq^Eowhrm
V1<sMz>)s6#QDXS7RAVm{5FlFoyx#E5OS#E7nzI=u=-`)a4$`MyiEC6d|$Q(NRn~Zn6T(GA<Dn-nxn^pHcc|=^?FSX!qJ)_8xO$KhJZt#BX_y&qE|IV+l55I
q4rEU&m&nWS(Mm~5P!nQL{4|pq!I|^^Z!X`Pt(<?gB<$@@`*1rg3va1Fa8xA@;@-Z+^WLqfpRc193N?K_py8$PI=A~FIpk6KYm>+(4gRF!r6B~uXvv0bdV)(
s_@thH()jZf#VuQ%SQ78D^NkARjT%79Z^h(7J?St(VG>Bv;sZYg^9-bRC@(+1>E8+#Pn7$W$q~0`t9qy)Ei^0tW+4<9QwEDamyrKQsiUO1YB#_S$cb0n{><Y
$)Z3rEvHS#eX9AJBv=C{HPcObBCL7$WP8SbkWG=v3h_Gl9sV45_Pxo>5OJ^jwxmRm&7i(E=RxcNr*=<4UP<K?1KrIB@7WR@i<~ofS~qp|PgBDzf;!&A5O(-F
`O2b|{2Ymy&YgAfy8ix<J8yTigg0mBAaAxu$)!6C>-)EZiMBGHjVT?@<svl}elgg3$%YV$&U?mI$t<!E(C#fE$qM@(3yv<^mE&`lrK@lS)&9jhjDIln@qi+e
U#w}?*BQNVo?VXkrRW|I4N2K#PR~85E)mXhqJ?a)(8)~7gGMy1imTT;8yn?VdeBvsc+xps1XT9_Dkc{|;^8yzQz)iz6yD_G?B<afX|{rZbB1&BvuQA??>8H-
4n<~Y<v!4xSU6a2QX}N*lC-@}m;dyCU1xh-VA12ZI$JLwQYUohA}@QkBP(?T<zT(}1<sFB0;eUvU3-v_#-e;%r;G3Xr6cBwy7EV=9sANSCgAUJzrCsT&w=Bj
(-kiSb%O1Z2^utzIhL6lS5O>!<NFIbNE$PpiTYllj=nT_2Run7b&gA7{_2MhP^SXhHx)G3>W=EJY*iAkYa=b_&q39o0;Qz!ht2-7-kRY%S;wZP$F1(i3#}+V
m8Ir0Ysy$p4W_P6DOETyxnl;to3>QkyIH!!F_Ge0h!oSs|EUBMOi1SS$TOnqK6}YD|0Ko7j)B=4O6|~rX}bxsn44go)M5`eX~^lL_|NP_0V7p)aPV`Ai8s;-
EnA~-1(!sfMf0K=8q<b+Z*kBX@V2e<ea7EgtS&bT1{z0{++G2Z!LAHB-QSAtc{#)iWEEqF^zw0#G0;5vmL#}^&$9|}<+I+13Wv5{YBEbmE7A#BS@<|@K?!sM
%Q!@y?TXXe)8q|sdz{V)dn|dX$Z;b;AHjE4N<CJ%GBMa7C!!LuHLgp>hVSt(hHLIR$#^+NB#W8`L#Xd5MzZjbie1@{JXb!NqdTE4dtdpFvyz>EOg`F$FHL*^
hMH%KAnx=WLol2ReSGLargPn$oD*};LtVD+d>~?v41B1T{_N4q+AAt>e*8hKeCb@hxXUygRA;?cHMr&GEa9fN&*`4idrs{IRkXx;9ELyBH&;BAj1j1|P5Y^<
co=KbBp|JmA97g!8uF;v59!DMSMPx>(DzPSo(_e+$L12#Yj!ct7gup2mcd5>*M-SvSBx!R40g%fC|~_3QiW0AvYX+o&->AJozbN7w8X<&l?CoM!IqK&rJ^-C
#63P>&9|D^!u$wyNoP2@Wb!W*9TVC>|HA#32TKE2wAZ^w=WE!JqZVoB-VxmEP_eqaI=^_wXxWyzrs?cI_PJ*62nAoRjS5o-mS4(e3X|tdyV*G}7?;2H{AWX)
tzk@cP0qCk4NgqSY0!MKcfXkvsn1xF)`EW8TeM9;mn=DY&!S>GUcYuXCXgz|+K^1#%xRFcta%EzE>EI@JyB(%D)OCvP8(zSB>GhPz?SR*Bl!TAlDUJuY=PYi
0K?Pirk5A}FVYFpp=z9|*c5Yo#V2a|nkmp>Z4DJ_U&i#jG=5VOJY%_0^SxJ&YR>LubksJ6s&2TPleyQ3(-hwp53!Ai`wNgAjVL#0LwwkNryiK<6c?jmG;T8G
5jJ8eraXwD^!hyiGY=<Z(88&{`@WXoSZMV>m9%@-mr7%t<H}Lk(;gC)<!oCm7P{WM)oQ%x13F^I1*klXB8n(CzVwLMKawihkg+FnCUN4VOBxI1@KIM^@$N`R
cEw@={fulQ7?-Q}zcwDI%3iYc#}>_yg3ix|TrP8oD1@d!Ow-iEdGaPX;VjQHXuh{79Ksb-mb)pyb&&k82I7B95*rxj3NGftdB0o2Df~DK!>D-}_8)#RLjp#C
WVTg@ug&3qOJQt?@YhtoV7&p@F`APZ6RCwwFxL_PGLaFxS@rkvG*nhwC~3k!AXYcjneb2)R<`O-t?NR&t-~&lck@czz&zEei$hn~>ZfXphm|&4L+!p+*6fBb
syK00T7Gx8<j&aOd%z8L0E_t}6)Nb=r;f2kd(P#qBRQru8zdTR@w#M7M>5B}HG-(rm&0sT%imrdZkGe%7u&c`9#8ir_hZl&^H7V<G^6qwM92;yZ(!8oZ6RPb
2edtA4?DH9T5(5dzCBpYG?9H%!!*sFrgR(I=&kZ*4QJWY^#lc<kthw7=im65eMH4Q!r}*A2U)3uIrBk*%P^FY|6w!17a%$TB*Pwny#hUISKHauWwp=icEl!f
xsVJFgS=j*g)!#!+>o7XJgBdozAjzEv}Oy-9rZh9_Qx+bw&eMa*S?MxMSl4FZ)UVp(Y%sOJOck+ErgHgMJ9#umQB@nkwsdhP9cDR0gflTHou2QL`9JiX#1+k
bqE5v2Z0i#Rq=!>*}MyI_PBEtb&HE1bJiQp&1_EC{{Zjad0ZTZRQN~b-cVm$l;B{J$TY<`v-095NE{KhGHsldr=#xd4A5)I0+0ClHhs2Ngax%|A=wo$_M|j|
Zo+0psP&mjXp$}b;Af&QXNMG1jr%<G5Q<qYfM^RNJQUr#X>YT4`_OHPh56RR7Ad4(G{TKJEH~1kV(9Ku2T0|=y-#+VR~lBEO@_(yPt$VkbSq($eyPP^^0wvG
Q#$!J1KKKYzm4ZcliV{+T9kLFDLk{A#|>Fk07k$>Lmd?Kj9R9d$m2_j_cJSd95Ss;#RZH~r8a$8YU<Zhq4sAZqu?;IE&?l-3$jPV@rTn;L~eshKWH-z-d7tg
ULVxYBg#6wjM_-D{9gAb$c*Y(LrJ@rTV5eiRG#44G0>oaejM6lKQOh4g?jMwSeiU__}GJKWwc!qLLJ{RU0-<R>-N2B6>4}PIoxUJRnpA~tBM<0!||lLaXvTv
oeBQolKfmEb3J5A_#&PLQ4hM?C(?aQ8mBmLs|rqflcGKcKDGH%L{~4W$9v6R3#K7mXAf)%>ipN6$MJV)FBE`9c10hoCg*G~zdjeud5Sjv&0OQguUim|T@R(~
<M~s)ZV|n`VYm`Ia9&gk2XPOMY(zQzE3ho3(P0SFG}$weCl7UrD%0C}@u8<;(CHpzXPYV}*UH(5KbvntR}!hpr;_l8QKW(u7T06}l%9S!OlRNb2#7s6h?I;p
KZ+Rxj#bP9mF$wI4z`U{8xDdhwtM&NMTX&j|CcSF;}H<K2%zv?B0(`&cHmN*vH3PvFNDbhXF!ph;S-=EfTFNDqPA0BQT(ly3mEklTs-q#xd(F~;pBMv1&s9J
31S9@x9474?^)kBNjmGD*1R=B3*#!!a%<`PdIHMbLg#hV21fuCd-F=)SB0(uSmS}C=Km2QVPDhJu+K>#sWH3tJ2wcHJ<QFI)A?r+s{=8(_GdT)so*NgGh>Vl
d?(ri8^)VAkN7@ope?X%!{0nU8&E=sB>YU(D;nC;N2pPs?kl(m(!d2(c_+T!mM<oL<_QQq7cQL&8Xp(G^~I!?P7-d)!Y2t|Cy{Ecj8Qj{YQHlt9i_nH3YqT$
@^$MWqOm_on~FLj6*mEs3-jw}v-H;X-HPTQE!FC;7o=i+p8Z_|)BUAk8nctt*hcTUQhXYECRC3rj5v4;D}nF-khCannOky@onT2A%pHKGT4%bC<Dfnrc|GcI
7J5i2<F!U$sFt_X?15vj<+}7h{M8;HndlIl7u(^xi;|2E8peF-E!^!2Pm!Cr9NZswztlfEe%*{B%Wl-{{EXfgZ)Y;XRt%x@;9?^FOvv(nY^0$il+&vPSOd8C
Zzir$hR!y*CM0Y<3K-Lx*jS*@J9XHtUeM-T+-_;f%WCB%Mi}YwHqBqeISgRKyJ5_z{@(XN5z`esCTP^#48op(X>eOn#|1FB?|hlU4sW1^>wHHRTq@O8vem<5
QGBcwTIi3>D%8hwt}a!z<JxYpdQC1J36Aq`>e@@3@&m&g4B}iw)}>XuqUlSrV9<~o!n&2qOx)&<!oF(TmezRwRFOVSj|7$}^`fbTrHNyanBau28@e%uM-{NO
4<QTJ57%XhsJbXkMSv<`X8ttPVbg=2M7bf^qiVc~QjIZ0+&wWayotma!p<9N!Gga9O-TZWWJ3`w*~)C^G}Lkt1-&;63?ZN9(ik7edK*wOvq*kHT*AZef~i};
A4Enp9pCqM%KL`(GpK^a0|3$Q<oqExy_vP4wTs=1u?L_)lAofTqWBvTJnp+rU!uwBPCww<mL?4iWmvf7?BoH1?$CdhS?u*4Ohk?rn#)x4jMwkeXm2)AjdIyE
x|AGtOXsF&ws_NmY_J=wM!8Ibn+3wQCGtbuOougwJhUrKgQNVQ4EZSD!8xeR%ujvx`@g*tOzbD|Up-KuYlm0_K(qO9)d?OeT<UzB27J@v-0--r1XW0Wp}!L=
Jn!xQ7Bv~jWXE!v3RZZ7nE!yLAD5v`0qu(8<jk}ufcScAqm;UP<?5s6^Jve3{~Oh_ipND9#WHA_f#?e5ANA51kFKI4#50q#5}x?2pKm;yDvA?GV}c}}u!Y`I
CiSDpWlYKT5OKa7gRQo8-fiUdr(VaNmc-|OBIp5-yDb{*vom=2(hxT#obqlD#znhjI~RUo7rlPF5lyFFQ}k~#U14Qn0tPSbN&z!2@#1zxRaJEj$P{^;Wyvbh
Bt|1r3)kn0<LCcNr@p}eCOcI616Rwqe!3lQgem!vQ^3X~=5o@MI%2V-8qmlCLH&>V%g--x-=A|_9RVtocTPe#3v}iI2M_444}TpM;<+c>))#i-GGkm7n~hU~
tR((QZb9A{rToEfd=xL|2wz5}Ev0f8!v%Dnu6{{Zz7{3>A$mYX(G{7j1_)&5Yo<B*sA{prCAYR*oRm%{moj)H_L>Z{|CY0H8h56eBfZ|jhT`~BVGgFNQhtfU
A>aGsS#m2C7Y1g!#V``&m-!B(M@LwPTJ|j!Gd8Z#Z7HI_DAmKRxoy*@@XmYB^U0|sqJJ7`ELK;~#4IY>Q7>j#0x-NLo48!cF0i(+$dT+ZbLH9><?T(E9%dQQ
Ajg5(z&i?yUPwjoC!DLO(|%XtYOb&g3=Fr80+91_VtTSThMZrcBRZt3GHFl>J4#rg739dSh5uAR$UuMk=A4@}@zmR6OlnohPO90~6cn6VwnS1#w5zypqbnp=
2?*6qBH~qCj#`IuM!}(Mj}>V?TK{)XpoXPl3-bgI7d=0QR}7|DeXGtf$+n|Vcv~f`6%=>ZhOZ*Rbu2hD#@RH_h-R?_5NjmWP-x^005E_J&^~g{^9!bsV*0sl
-)S)N1<0vPTvn639eU<b1pIdt&EX$^yp%|dme$`x{Rwj!<OOlLKJBTf1qx|Mtn;U$dL8F$Yg0cE#5YwePJ}W^n&Sb7@cRlxa(!MuOEh$I{Lzxz*MqAlUvAf_
2@jG01B7Sb56hd4;Dls$I3Se#e@Yey3UmW8SK;K?w^O=kL~@-^hvyutzi>CR5Kp)lM}UaD{4rA6?xt}a+1d4a{8`hdOa9d2JeZQil)S||{I;~JkQyiKO~(@i
K;-G*yLGcl*$zI@t}x#sG{)tE1-`Tv4g5{bT<qf*X`b_vGo4HWp1%GsOs8N7;k3{WWq+pQF6hxMO)}rMupA=q*RlO)55AnJX=5lRR(9BnWDEBB8<NkpY7>*M
Xh_=q+08$a^SB|jnrV?F_A~^OBP+)NmCmG?3!a@$+Bw&5b-e9jnyv?#n6(%kw_lxTzBAj$9`{i>18ZPVZtCL=*O)nKiG6K*p)cspkc-d{UR8M}$tTW86vf}D
LmqIo8yl1=t?)@bYM)<@4~h~@N#DdAHjoAq@M4-JL2&=lz0#rUGxBtniysYzyLpvYF-f*k)`A2~sJOf?tf4mFL}b1DtF2FG611y@MEqu(s#*ynZw4w{lP10I
2{`~<XjPjNhhS#XNEz2K#a+Ru$hc-t*z(=gV{N<4r3uAbLB6mgKbVr>CBv{{x*1mS0;*{c+x!3iq9{f{N8sS?2q{M<8ke;h|E?li5x(|>3gLds4)TQ?YGxb?
vsK;)8C0i8UV<DLvBGUW9SVK@eRdRn-NVbh{r?8Ec%-{MJ9Y2G;|-QA3gf^q6V$=4VfUwx4T|6B2NAaJ_Ju63j&id8{el~Lb;GTvEAP@CRc2`UkX$~QhG#}w
@t^_*NS&2X58)#4F=?laiZZ<%Wo?VEPZS07>B$dElY?P({WgTIoyeBRo4kQAvM|O@gCvj(h40u*VOHrmlViy{2?f6G3fudzk2I&VW`=V6%NSFw>ZxaT%3TPo
ARne^5qpVGKEk05F=-vpEDbXzaRmfm4h1$d!2xQy9Xz<+O-ij0!44#iebeWWQ|yd&b%9LyUNo=;-5l7abKZK23mc9sHme>!@M^i=TE+Ll_NLncsrOM+o*kpc
R3UTzsVArTujqKtvU@*dBV+A@Lr04vl%WM*=w0<Kfrx5|kaP2sADG~9ao_E7;7h>`=h?CvgqP;Q<5dZp?QcnIDhgW{Ga~dkSp)tewn1uC+K0&xS|uWHbq%lC
$F`BTcyPajxGgS!`X%i2h4(!v`4Ur#H8`z`TvU{dnheU`$2VRISJB^Jp&P4eqG{Cx?qT<^5z0E5-w23eng2?GHE3$cIMLEFb=9j70Js*2+Ec4A#J#2D@$8t{
rCS25&6N<A5WD^QDmm+hdKO5h#fvqF9_8pClB;<9%qzpiDXo{0sTO1k2HhfEYYcw#Fq@M6?S5N;^r(Z=)K#_9{9cu4-O#M45XZ@^i3(1TIV>=dilv~U-5SFc
&loLM#Ins}O;FPUGraR^RBh2Us68dx-~e%d5*J}5tm~8X5lVnjc8Pj^#GIIq@YY%dPbD3<0NvPKcTBRy2r)-A=LmP?6rs>_odf(50CxRkK*Lm2hO+|b(#jtf
Ho;(2O`LcUKlVM=0#L*561mPUT2HT|v;dIRCf?ypiY<DmE^pQ&kAzvbD+G^zV)O3z#i0nA(JYSCQ6vCNFKq=(pgv;^W}lGe_f4J|Vgp@9pRZ$kXNCG~nrMU4
ug;A5Q}Q}*yWdTZ%9?&jdmKw=Y1$JNaZ_gBkSq|;3Sj_gZy->|wkVdc>$|NlkirnmBO>FPQo?~cSv@F9Rid$mO<oPJ+P!C+H51^^QIi$Ifdn_FTpdtlQ?9u2
!RSJGn62WsfIp%Q0ZfVx^UgFVU}NCmWPdC^j~&`IR<XSaa6XkEUF5b6jG!t`!l$3u27>-t0n;R?QlDuG0A+s9X70C{6=K+Np-)$UH5<?{3TZ$x1^+@zPw##N
VH)luWsZj*JF~~*g|ZO76)CFNKFX^IYO*H3UQ7FHT9#0snjzuBP>a93<?e~TeztDn8Y+%z+HV|>gV|Ci`@tln5H6nsNQWD!FkL(D)r{|;$H%W9r4|{ho$$80
Rc{D2(w^vR`1<eGE0ioy)?uw5{**VbuBGh(Q8i>zTp!J}l03pbSZ8BNLQ=AzfB)k#LpD~*up)?>Bpz2-zU-^g-`aP^!RQ%Hb(FeB&)^_Y5#Xt*q1(%`CQSQQ
>T{93<CrVev(6d?Q#lW03{pOT46Bv1HDqniKmqLR`-rMuyqTIP-ww>_{VKLnZ$YKD)b%RGK-k`%*&D&7T7LeBB&{Hc_a(oe1zf2(Cvl|bzYOA1{)`O~HLpz7
UOc)A!&?PM`v2OBUs8f{6QoGnKdDi~bFCVVXVXub?JAcJytGOl3EN-qcu*YoZM@w~PLtLnq4_O}PMBDr5YmSMbHs9!&w^8eQ4Uhdlv$=bJ^VQKf*I<*7<^T1
xe_8UrJ-@sR57*?Y$+e<#><C6L*by_gH?W&UBu<ea2QwNN3ro;#5mh_q8GY<68+Mey5qw;0tm1Wcm2F0yY@RU>LLNDsYSi!p5bb78X_+`ouOPY+nDzl$??6Q
D?{W~XuWz5S}A`HcDTHgb7SzVN@@U&Iy@@hVc?P<v-WJemQIV{d8rA@Ud!w4D@_w?S}Z^{W-B433*!}sD0)&`xib0C8xzQ&I1qhUa01|2sRyN5IqgaW9Eq^%
%YB%0Ino&xR{P`*i!r%|ZFAgXmnpemW(7$|Gr-Kzlmv#B7p_*b!6e<>#Vx`@C=2m7--aevdfv^7tW8sgXDyDt5SJK-umJ{I38B4Wn;-kXH9+CN(q~IyY__C?
&e&G;Hy!1Kg~WqbarLG=!wZgI2COUo0;9zopzS-?@I+prDz|3Kt(+820EIC(gD_5Sdp8MltW3#i)RA>s&S63@l{@asOTNr~i?dtTVoOUi*!!+GUq-rLxL{{a
c09t#pRlCyNV}5f6WXSrqM&x)@aJfmI_-RXKZ-zc6}Q?X_2s;mW3f5PK-%@qtjE7gHd&||kd2%%YpJ{`i%J4aki{>!()SsIj9^^`$!<z%K&w_;E@%;hq+^XH
RVH<4D*lwDv<+I$z!j-&c^yDbdU?J7UlVvciA<|gJDY0c1CaijP+6F@Fr!DC0CrH($tLkOoJyKqXLCt>pGF;GvEQN{5rpMtdXQDHKiXEUaARZ!drQAMNhoOM
1jaMGP7u5c1ta<$mbP7CBl)k<LGUxm>R*xdy>d}q*Jof-8z-ZesWN0}`*vIhc3G45nxUjI_34ehW(+rcrPjOS!OH15u&JgF$}T`Z5t)ggJ8;S)9QANoF2jzU
H(vCJMNkPH01qPTGo)MrJf<-d*m%HNC1_pi;X6MzFM`;h=gkA*G5-t=jY4vszTB_{;%c+LcPs_-h}NHlC_FQ~qP_M*sLNN9mKVeM>6!nlf$Fv^>ywUE>B!rO
#b|pZ%0bHRqW;?-bjD*jvaf>gKQ2E|)ezcN;pGlmz*6!`@XNyTYcJ$=wW<jWln5&~LmW|&;&@5fyOT-nc5@Cgd36;Y>b$4&zv-%~*{R96F^Qufdg=x#<IHjI
xjoPGR2i}kaJ(to8%ywQ@6DIGbkE38{m4_z=!cP_7Spns!`9qIR5iJHxJ;Qy6P>*Of2fk@$my=ylYPI?GhsV6(iY1VnQMHY>2VE)=v!V4Uwwdc*5AT&x7?{{
dg8q2-AKmVvhh4vFssZ*b>EJYB=H34eT_w2TiXVum#MtFLS){*UQrVZqsz@rRzIND#Ia5m9%ftJUjHq55bY)Li!CWS&0i11fPoIQd{1Krm(H<#WVBVxIJti6
;BlBtDdb*@UD=uFaA5^2w1MAFEqW$vLhnB-N8GDONV0o>0<8vZ9!XIzE50=a8Ul|TQ*s+?j)bxdS}^(8@|#Q6jwb_;b5uhdOts9^;D@wJmza)zeAYmiQQdla
y|Q)M7VdZgV+r<dt7$3NeRH^~)kutZf-+kIR0RIbT_tt8+p;PiG7jR4ecuvec*3W%xNcMqfFWEh=DP~$sF#w%zXDfcQTIq*y6gj-+qqEb&Zup+a*U5rQ)vVo
*tLS+(!S3_jE7tB2=YhM-%nB71@#dTCtpx?e%Z{HyGs;)B;aHf*NjpUPs}4G3lv{H5`S@0Nm+qe<Yy8L%WHg)ohPY;PIrJA5!|+;8zn-#4=AV;m-*Zu5Dau_
ll;Z5a1C$KA2>y~d%WQ}&&g`1^=cf91I~ikjxMmnPvYdO?=^FN12KxcOLc?gZ*Hj8Bf)nHoM9?J6URfIzy_`NV=2jEc3lEw4Yjylm^K2aB3DgqVU?-%O#>qX
BwofhYCJ|O%>)#-(C+Hk0|WRWpz$lw;QUp3{#_&h-c?&kT@O;1T;+)bX_`V}$*jh5ekTEhxeGUGM!Yb;T;fB-<>B`6*G--lZ-$c~V|}1$dFrw}uOmo2#D~gV
lH0m%gizW_CStw>*#$1Hw}Xl)>h*;w)tkugcBtA_tx(rs7<WjCypGU74PY$UV>k66osjEB=a`=XLa_rX3(jt390*||d?Eh;zS07Y);Z0f#PhpvA}|L7jw>tz
RRRwKEqsISXd)F_3b~6KRtZ~hxam!X4}!9bqlQX8Z1-|49srk=1Ui*yZgDb5a*nZ1FWlHgDsIEdjfllaKhq(vy$mZNg#Db3I2HDqD7L_=qn?0`hVD&S9_X`l
)+B#3%xSEVhB&Qx`d4fyH&ru>*a=vgnR+?ImMS1>S3s|~-fH9^ttpJjxP@(Vs{eF&P(96e+v1T9Yk5SqM(<&xAhNrA1hG8mmtIs$3fJd@TvPiIQ1vLRLnj@A
ZwVe7$L3rB^`b8DrLPY;Vsrh5^iu?uado#PC-qvZUnP}aS7StK2UDd1a(d+K)&BgN*@0ZhBiCyFUqBt~0lj~)1enTAH!nP5S@zWtmx=*wgK+d!%2n&fIzQVZ
cHkn`%{u6jch97Fkd#tt6*!25IAT0S$$Wrm=NM1XnYWxeAnGUlrIU6U7onitcEN6`iJ{i~sva@c=&(5M;~00HN#-86D9JSoYHXNhq^lx#O_55f&R&8nIvHb{
qQ&>%C8L=3)Gh*0c#gJ|3FQSn;_lM?d1n?&e~>XYz3hM?jh(@iPhX*hhVWC0N(M;aVfSS<);)lM>$q(fEdKx4|Gt!EmC*%dR@w`P2ij%?zjmx_P+U6A*M<ge
qv|Uk^3`f4EN760k_a>A0@o0I<J(B1F+HXi5)HGu3d(Wd<z{zivXv8M?SjR`WBVUGx<*BbW0R@}NJs0?c06ItDD=S8ge)BQvNl9JKpP^d2@I;jCnGi@twndz
?ULGwS_pQs07t4R()p>8-*C?4o3X<g8IbnoY%Pv}*<8oKfa1oprbX@mp;%8vVl9qali+*ySGV`c)d#6?YQzoI*D&zbV|heB{%{%kpAw{Y6gFUwkSR`=-O%bA
bX#sdApzr6wCyrpv*LwP0Km`IOUDqv*HqNth^3n~j0W4Cc}vTR<;-zV%(*C~w-Ix_qO|#c9j_+PTmYBMzw5NUC2R4)sboX6RyeCY>_p)j8_J!Vnw)Z$<vTYz
aDs|h2)=(PKYn6db<s_56(7$<3xl9LRR>pm$kcy~Fj+XgS8nZz-mRvYSM})cbA@y^?9F$rk9{`C{4lE&JM}l}O1ryuzcJ~d<RH&kv0w~xag5o`IyFiakhF-2
pPy0gCV{-YK(oplMlcc4NRX%F*Lb!eF`&Nq;}CNwnu?lN`2*|d5U9TX^1IaTPh#-34O1=a8L3cc+Cy-ojC0y)vh}SeUEylwr0u*Q^Q{Dhpv*ijOX1*WrC-3|
2?lHA-Mk{!ej>GMvK|sgDq3lOqY})$^k|?n-{XmCw=zK8SyWlN(q}v)JQK`vK~ok-Vz|B$l@x?C!s0n>WJZtNxP<TExkTrHjB#XWqj%G6yBNr6!LGc-L6Uq2
;mDetbERVAb@*)mRe=D!Csj5HjSs{?B~!SepvSaxyt0BuP-aZeCWu#*hOuSQ;@;-~+2>LL`WELOL$eCDqM~zt26jd8j<jmwi*<S`pbeK}%i?NnZOV3;e{C?&
(Aqk7lwS^ERA@M6ElsBbqriOEg#y{LIMUGGBzAWRUAt-Z23jIJhqMCCTQtL-Ld4>q7U}97Q6l&ShWG`uN!W6n5yRkW!b1=N%y6Hfh>i60lsS1n^3UHMJ1|Lg
AG>5!ZJGWNIvZ8;`9dMxy3JNPPG`IMZvcH{M{M3Da7fpz-C2unZgpb|b;B$dN5?yJZTSyn7{%m60Zz|pIB~}<XJ52nX!&jh39CD>sw%D^>FwF>Q5<vWa3-;u
hb%R-!$kqqM9w0F5l`%myF&ioGF4((O40(K(DFBwj-D}9@Od}oP~RgsmN(L**;do*_4>-qEziG-_4Ol?2PYQTFHjNdHdbHjBNZlwL8Z)Y9LfVo*-1gzEJ_>|
qNYy@hx6QWUXE6II>&t8=>}y@{zZDNG)sqnWDM~cv4}+R6K$2n@=<P$d0{LCR};>GXsOEI(8+Hah_MuaUSGb&`1;XO+^L6!t`N6p1v!NVExWE_qP6m_kq>ls
zL;%w{zvZTDiS{q7}CyyS<I#Zz`vr=Z-H>PlS65W{!M-JMBfjd%9_<TKXi{<%FCL$U6Dq`$4tT8^G^RT!mlYU6~{-pf|_wdq`Dgf@+&fF=3)0-YsRTBD2bzZ
MLU}uoy4#=)u^`JTUJzC4dxxW{&JEKI$~U_Zk}5I7G=PlxMTZmSF5{}wm+DSRW<+kko&rD|5KtR_Oo}$%9Z@M%`fspX?MZo526E@Vk0Lq4~MpPQ4L1RKtCZw
w7}y}?t+5YsjCNb;&>fJer4GvZOo6&wUznmq&$K~Ngj5wLGU9(E;ZyQB#<LzFQeNC7(T0|1-BHH*>5hcOXII)l0QAMUaLusA|aX;sTo9Tn9Do!5TX_~6t9$m
iODC4*V5Yt`pkeXNF#GcoG@Ph)^Ds^p}b)bcv=r0NIHOTID|%<$I*pZsolFbHP5vZKL_ledp~vf$5g^|yh!1mV64IpUpV;i15y>f(A8y$q%!EOFr>K<siX3W
7I|0q7h{qIt8=rgNM;^aUH5u<$vZzjsqu$*8>4uXx>|s=_dm4F^4AqIQuUeN6P|?G_XH_<CksTgA6RoNq9$tnl31TGgza4{Fet-UOL58RdTA<3LRu|tD=NLg
r$LooF^%XTyD__<=mnxjXlvN6mWj2RFu%McXcOmmCbGrnoV2%zs5(?OPcH^%AJPFv%o`78!eA3_mx~nL<qr(HPBSf^{4gDeyx}OA@ZMndppK55a|)CG>O43m
yDulBLvPRlQ#y%=D0Esigy~BwL!6XhM~DF<wv{&QU+vr0J)eKNK{U1!Q~DVJ0Or02h_G{lh>JXG+UwScZh5e>;*{J*a1%!2;%PRlap01F@~r%rUSraS@H;}m
y7G#~IPBo~POIN@hVom$)YxKnMn&OsBP^v*3D?mQK>d3JdSQAU8y>KV!BjA$rg~sZr=I>g2x}FcEzJ`I`pc}Do+O>3E}2Q!G^T_fmMIMySb+ty>KwhM*!P<{
BTv#h36`FLXC*dZa!5Q>xS$`>*Pht@&4LmQqQnauabJ_pIxvs1!G!tMH3V_ocEDK4y>3T}g#Z5);-FWQ&{}h=w0uN^xqo&hf=u`+)ruadOMYXCy@11wYm?`S
)TmP_#NEr~jOlK_y=gg8siOPhD&=#z@L}P@mgBd2+`I0^Z1f*S{TX9o#l%~Jc6ar8p+sC^kCU>vbA1~++u{_vl(lHy4Or~3%MRHKQvUBMhszGDH1aElw~@2h
RbP3(YH$!a8rZinyzJoW;dD&$LeE|V&b8>gPf9^*0`cWW(IcSiH&1_wt1%yNBY6_Qakm(u7M<-PzKO>m=}wj%*HJWmTucKeL>7DPfL70%6LEqBvQ7KKX#3y{
g<RdzXe)_y5u0a3WW9g$oLUajmuu_?=_2|fKSjCoCK!t}70Vc29#NuGRFpE8`eY?|or6xxlBgg3dqG0y56@TAf_PgxEcShjoXP*F+Udp%T79+WtZsHAVZ3Ek
osC~z#o!GY0VWwu14lN2_&5gDRv4ziP{kzxWWyXw(2$1LG7wJ|z&IR+@4l}I(K<zw4-Bkhk6K5ealQU2J|YBKzI>W#)E84EXx@O$mp$J28kRsD{Cq|(0_)uL
LL@u>5v1gbUA3A9MT2$XxK<g9Ku;N;B1RUG-3jSD`yNrj*IthtwssmP>UU_5ytU4QgM_iBjL;wkZEIa27X`DRRwTvsZ9D)AU=Y7xG&{j6k;sOvGP`w9b=1eV
?mKQ17^tmZ72D&G4pzHA<p~CSe!5!z1W!LZqvbaJZx{IP`2b0tVdBQ)2C~|ySu5v4q0Lx)zq94TP`nDP&Z|bPX9<qQ)eSIXC1b9WpjkKd{S7$5=}($s+inm8
v6mVu9?)tl2|umJnud)Ia2W3_l3y;+9%3fvG5ulZ`BfXaf?c2Ocsxu>DN;|MFg6Ki(?hiip9IIhv49DOL!8*qDQZmiD+n^|Ko38x&N&aXb!V(ye?{6Y0zp<Q
!+&sbBqjVe#(?LMiD?T=wX_{yrB^7gJgYR7-o{6^SUw<Ea#9q<ntGc9f;n}kuooLA@v2f=gwKsw9LXd&=1N|ud9zXeR^NaSqOA1fF(7vKGP+q@2})uC`@tXt
?@4`lefmANO60Z*Ey$0wZSq0P;<rNfHb(12j9bqpH5uvC%Wj$WOE`bU4xbEjW1VYmoIGd%Zd#N(cR?D4g3F#_x2lgsKjEkv%ks#-Tow-7WPp52oo_BO>o495
H{eaq+aaxD2R^}FoNCa9^#)B1ZKHE1x~u|eQ48UkW_RT*LH|>{V!*O-6-v@_*=yF82*BDYRtgU3_2-S$@dr#|MxG1uclP-ym+1X7zLOe$S^LZD5C1Vu(@)8m
EG>0d;Gbi|O6FtZ?lLLiI$yCV;ZK7;p&aCVB3R;@*4FdBwv?V?DHtFblY?n=&iS|qchB*5&m=}dTmM1hvm~6Wv}&h}1KjSjr1&^F$2k^kY-pzXSRYn4oxfp7
1G;>xwFi%Eu(Hw5Cq6L|R(4`pSu}lyH_Af2*aZ{BHcP4K7#DL~IL)G1c|+U%`u;Ycv9FMWoZd1kND@MW5FATlLSlB3x^A6jk-Ik}h8OaET(pV{LaUfv#YzNo
z~2;@hf>+ZTH9Y72PpckA}EkS1IGZxCbla@a6&GSyC&_@jJMAonYcv&^+wTP8C=xd9gZP9)c#V3;S!y`KBRxAB8+e^0505!o$;2|i+u1Y?KRfWU7ey20-3a}
4_{pDUA-+M4>Xg1NXKaY6a@OGdli{SORXXb4Vr+xa7){0HZ>G1(D03eYJJx%FYQ=hw)eQTSTgr6#<t%%TQC*q#&+HwI{Dte=3W&hz)<R6=QTPen1y+kX$ggv
hmucfE~D-u3fG)v4MuH8^&1LG=&X$FzWPEG>4IqVpyRxpQx(AbcJN(<pGq!(vXC?XSr;{Il_{8Cs|uCs!bp-H)1#_YZg#V$W!o~?K(yb097u|{`_2v9TlWvR
eX*c0NPka&YxPh&sO0u<M@Mj0J|Y}CL!WDXJ(nTZ$Fg*A6y0wmbb<=sk}2Ur%1*urK*9Niadhf3A*iH^xcj~O1{m1boVS%i-Gzq0WELeaJmR9=fHebA7)#!v
Jm+UZC})qa5CHaz_W@Y9ccs?CA`pwEUfbI_bw}>wH5Tmk%L1q`y+w{jp>(VsKw2Y)`mSp_-jk7jb_b}nQ2+W;^tax!OZ(($+sD~LO+wl6UB21fA>~@H_zXg!
vtTOJEue>b`Hv24sjoqyav9r#JHqW;|DQh4{0gzwL{v5>#i~)bEWG)7clPn8Gbb0&c=1~%hMj;o^a;*XK*WWST2ym-SQx4r6V!9M#{`{$eM?rG8(r4>r=%Q8
W_k3LCW%k`a6>Bh|4g2q<mGwWA&p$u3E>~axLI5OU@b(BuhKI$C=0Q{^#Gda%2M3%jIv{bUXu{PAN4(a@s^|>SCAJvu|hh2TFXX1N-g54Qoau$*a)dcdSoS{
QMGFc|A4CKEfmU$w;B?LyC!w}KCd<rb^N;yo@wi_q_(k7$C``wwsdAFj%Ya>fgUSiY_GK9MG}K<R2vmI&W-T$ynF}>r9!_rS5&}ZQl$a6k^(syojD}}XhnJy
BP0<dULOe&2(a;2NrtQqs82B?gh6OnN)FU^daS+i$TZVpK{P5BJ;HrYM=5m}(mziqRgxk?fxV7NV&mSpGm*;Y=I69`sHbUFj#N}NM2pu=KX<@4RFP0VvA`ec
NWa>fp;(OjCQL99(HS@y(IxMCFZmpWiE)ZaA<3?MM|gC{2c#cF16|@)S65njdFu6)G?BnidrB6aC15=TXY80iw{!3vaE~S3oNWbMlfQerGzVS1<N5*Rd=$@!
pl~2moyE6n8u?dY>qM~Z$f2ThUd~(V*sR1IRFG=hiSY&3Um0iF(It2!6j<%<N?&F&Sd=ELSHS0;*o9N`I~k6}T|!RpVNVOVbA*SQMQw6ELj$pPzQR7S_Sonf
M{8DuFLgoYm;wBKGt*min7q*!ETO9(-gPKk`c11`D<-S5B0j;py6y#hfqszk9VzaFnC>cIE1K-DkGL(CsdjTRhgD$Y5dHaGt9z{cc3oEHaK`L`K_*KLbD`cv
0|S>Sq9O&NN0_5v=bL$Q^#@KFcPckJiN9GjBm6F7N<pp)|2Q+>@e3JiR{~JUZV?4&0Ts??OmxfTc7O`Pn{7l#=u(pLF6^YXFf$tn@M`bFcUJoNp%2~9E;6|7
q-x?lHo}O}2?MH@nNhKaPMz1#vzjZVKhydafq5uYxK9-xFo>2v_)jExNL<I+u2)nKIsg8>*$hIi2oE?=K7}RtBE;Z9KRRmVmudnN>V?Nv2zY-zbL1Ej_*5k?
lTI2(BxbH@#fkj0NYLc;>%*C<nILHv@Sfh8m7c6>oKT?$+^F786qi@(_ezMk=f<>&zzcScdJoWLJ`zM5KJcCw?v*h9-qT_2T<>TssF-2U6OX-LD8O)z%{*<P
bae_C5Mk0+PahQ3F>>i^jH7y91+=T=mJ>HxaBU8)P@>#0>^bmfyI*~ty>hL(x+V9M$mB>>D+Jvap3eYk4eX%c<r?t4!K8U%Hp?r%-1V=Yh|F*f!Gk&PLYnNZ
A8eph-RCht0_#)O!}znoG+-YJ+uJq!HR!hAf&)IUi${N@J8pz4QW^Oj?w<F&qI})z6<N7C`PEz<lug>oJfH7if&U6GwdE>mag*`egzoW6^KTE1T5!HZ%ShYD
)kQDO-X8&MEZE@QodY9!>g?d_>i&IET>6@fywKG8B!cV4qSja-QFA6RZ3P^R&c$3NB0aLCa4nQUouVK;Hf^?Pqt<(E`9u7G8FkVMNRIRCtLOG{4}8f;{APe*
4gyYHxUKO}M>(C+SVG7bZlN6k%VGmP=DHPeJh9^%*Ipl=?&GB3FtWj08?WnNEAl@~`F2~YmEJXuE#^Gk9A~MvaIHL=EYO|}H36X8g9{j#7b|FECy0}6UR)+8
9;8>94gG(9q?op?|8PME;6rT%zQD1z->oeIIrJlkexRxgUT|R-w)qz3gEuaF`7D3Hp4V)OptMcQE0GA&GyA%re$1fYs5muh0HwZXK7R2xP-_^y$O!iIG0et@
h>KUcXcf&yo||7SKTWbDHNbU=d1pSRTKYlH(Duxd<#1M4D)90T1sF8RoDwICIgtMnn<q(2nd0oTMXsn~yT}||3^0|-p5nOPKZNl8^xJ0mdZ`s2c=lJ)wHVe?
Nk4MQ(8tFMN?7f<eR>tyAK8R$;G=3`dKB$t%A%g|O)lNWZ>(z5N6?}OD+E5r0u>6>v^`L(9P8q0K9Hs{S<zH98M2lwa`Rwh@YusSP=P}+gyVyGT;FmTj4Wwx
q!J}ieVXm_^<wrblX?ok?LKvRJ1v>-;A&kVBBh>6q1d&c#?8Ac-+6(z+QW)Yh5aL4Jf8;dAT~?N(C4`pRrz`-@n1c#?di)BbGs!c!|^b}&JY-jfQ!yK_r3fE
-0~!;Wv3vWlRvtM`>s{p!-XeR&VG~~2ijhgLi8)1-f3x=kL9Q_W&c*Ords=w^NItqTJ{#DpjJ%8+}0IU>&H#YRNDy4qm52o)BP`WTwm&UO7>7$3vILt75PIy
TkQogPw#6=8NZvJY*oT0tYT)O=3tYC%fD5Z+5S#oC5Q4!F8Y;vh4Da%?Im|c)9KI~#%g(nOq6W>HvoiF{A;;+*X1f(_9^65rh%ti7b+^??Sum{DzScbXpu7l
2nT}G8*+q+*ka^5<%oYCUll&J2JfTOX$gQ!9eJ&X?Cha87f{LW{^je<$U-1g3LFtEcSopqkX1+2*+lb1(L%YMfxme5ze^;0$OQbm(^U}@?a(t`n}3~|CC${h
xhloX1#oxeVTej&sohP+rzUnYnOyY_C4G|pl_M^NjY3rMD+6Dq>Q)ZcrotKB3gI47X~E=H-ZlAnMwHC<I&w4$GYsO~<dT<cG6$d&b9DFGd(Lw}D1WK!KJGsK
_7$!+3wp|5BGHwt^pOOj6mq6CDOQ>K7gGmoJ!39C1I;phPutbmp&%vn48LgTK_@X?SR;02d)d=Tzu&)L;U4~iCUkg$bSDg3GUJm|*rW<#nggXSna^Gg|M#XA
U9<^ShNJKtYhyRA-pDEZ;ru}g%BI|g2q1Lizf1cp0hNH$3-H2-om1|}a6IrVtapv<55KXn2wmFV*mmW_XM~?j=@#12$x(bxExo%EKBXfHfs1i?kBSok<t8Ou
zbr}-@KIE(JOG%Dgn-DD>V(7J(m%&#0cE%@cU}Q&{yi-$WRwm31<Mo~sAJ;P(i}pg{oN_Qq5j5`0)@eQI#FzF_104<V3#`i?`Q$Nr+o=)-v96DdR7WFACE@R
g#=qvO;ZaIgW$$9E?VTJcrBNoK#O-<@lluev3C$&?|LU=cNB?uT3cV?fXtnDoQ$hFp<4=WWxhm?*rKBmzdO3~AV$^TKtf-t!`FP(&17h4t2@(xh)oJS0~)l^
N5~gzOW&yJUcK%O#rmh^wZIFzfa2Qgkb!KUcyEkeDLi82LmM8x==YVJ=c%w&_21l{ko*T80y<$&lL=Y0A}gzat+}9)aNw1^S&q8M<^5R_?Vrzfke*0l#a{@3
GwnO6ziyubaRM<oBWT7<Ik^`6^0<dV0i&6O)0m!@gIKB@TKV;Kl%(Lw(pv;0!R(gG%=klZ%BV{Ez-E^MhAPOqarrFn(3~7JJe@desV8{OKLJSekQv74OAZxw
bA7~9?}*#NmIDlCI6*F);E9*Uh|X6_5Q>SbywZDmV7>oTy{6-_7qG{kzDt)RaT?)LqM&ortpuV%C6UGoqs<=6lyZ}u)LHPx969FaL6Vh~%HBS(!zTyDFQJtM
59J=!1c=LA?3v(OKfIe$U+6nSuK(Hh>*us6GrN8<L<0F0M<86DOpL-m&$@5y8XnpUqXRP~y4hsrgeo)LIYVy&RZ$O01o9B62j{8EmF8gmxK<cnN~*i5Q&}m1
c>>AC$)_N&WP9}Y@QEL+BkKTsa5J*gYq!#U-6~t+$2>0E2r3c3Y^c>aKYlpWA+&yBSy8&WfDfZZ8%v>86iub9v;#x73LP<a$H7FyJxM+_WR*m;&F!_HJYjfc
I7>drg{EGfIrGZiV~3vA1vDASD^0CQu|3XR$O-|KZ^NTubZP#GDb%WiZ@D{R+$#Td81@R#+Rs&Z;@n3JO^jE5(4)yx@zZM%)m}$8H(Ns)jSDeLG`%DK;maa?
SBf(L%cF$ymO!&<9KyyKJufi$PTchftz($B;N6n}L)Mbw*(xICU%9GU|BXiheO|8XSYg4FU)7qH3QyFL3s6z$+qp&Wd;uh-icB&LU6ennB#qv9v{E%Nr8p3}
>i=044u$JXEm>@U2Dod@{Yp62X$b~|tXsgrOBMhI;Y&*+>CAnOhNslh=mcWCxIJFR?u;ZBZIxnAG#Nf{r~O<cC8d#?-TckVJa${bVsUie*=PM){zq(mcL-_9
1K>OSZByTC9rRp^D@~bIie`h#f_9f6vVkHlVcM_RyGYMX#0w<Up}eC^ye1kvED}{YhjcF$J=a9#u>4Bm@(cec&<L)6+a?=14#~TvQ{ZzZJ(8)G0acTMz>&JF
kH<!t`yna~wL6~l7;?9!J`CRYcsIcczHVXX=;Lp%GCvvs%X29-g0(%vy7dGv4f)=GxW^&oRX8vhzW(*|fP<X(Em&+^0FCBpQ9bcr+|LTt?`MYzU_gBe?vScg
u|mQQWq7tV8q5v4^Z2ZT<c*w)l;$X^5+MwEZYi0Qt(Zr76t0HvvI!^ZVHvT~lUm)gJpAwjO7c@^g*hJaSWlr|8H(`{E@^HESSRWfK$`dn*CS|7lmoqIja}pK
3b$5&QCwU7($2_sn9b{a`baqr<tlh8<;|<}Fp$bI6qu!|I_aoH$^}Fd%KaNI?ge=Xgk!5h8hSIY((bGK$MM5JYS!F%Khk9-bPMVkIS?Jhb@hZP9bvEROhlS5
u0&?)8BO~O`bVg;toRn32+ii(K8Va)8z}zN29&?V2ow*Uc46Uo8K<ggJ4=9@ZxAvPa|0t=e-&>36Vz!0N3XSC%7g7BE#1_P<kKjYfkbC}_(a`@7C}Iw$6)nQ
X7~go0pjv3b%<BQPG7}5bM0gk$B#W54o*BX*|U<H+J@)4enayzXIIjt(o<mBjFhQrWV}h5IWm7$Um1%XfWQY#B4?w>^aoK`{WV4K=zmpQ``|!k%68^dzLI2w
9Kdub4s{5o@eqCF8qTBd5YUWqOUjz6uh8%aB4$*7x?<J>mcj>^`(dQ+nP^+O(Q>^oy0P8XAHA`~dqdS{nG0*NKpZoIpf^ZsINVj9*DICBn1&y^nV;I7YCyyq
1x;QeBX}{19-(W4N76!yj&Doyw<%@3l5C&VE&LA$h5X#vtO`Z1TL>fgW!432ikft^x(hF5&M(whxp^Ib1OjYj@IzDpL9xzRssW`BW%qa<SYmS7b1NJ6F%l^r
GqqM@9r|a6*Iu<ea@XOvnbbH}+nGB1uPnXACj~Sx%7v)<M1sK^2G|5vuK0ay+ye-S2IEabn7di~?(ZWB-(m2)yS8%uflw_Ttier99DP*(5}d{au%%mVC%^1#
l@nBZN>;Zq_oxAQ>gbfr^9pjKy7U4(cW5<|wmJreYlgOK9<a>d;IWSASkG!j5@DgFq9V%n%G)C8GR15Y{L$}mj}VZFOHHuRsJdS{_nVzxI~IxM{9h9zE<eBE
#^=<Mo)9It?C{Y-B5=#r7a&$G^#k43m!6OPT`a*omA(4O?&qN!?|pG`+sO)mVd@)^%8LRUt)3H$4%5=TK!6KdG^Xrre2E}Q=Dw`6<NM_X{;jZo6&-{%I!XWW
^O}Wb?h)+%j7hz&bCse$mvAvW2KNqAFGs0C!s^YrW{#@lDa+!lQ##%;<3R7=)fk(<#!rjc{3Pzu6{$fbkRMjgp0Ch@0g-RBB`}%J=1SE(^tT8OL#C0wcbECh
CJ1Da=pE!|2noNjQd+H@18W2*fp}nX0EEp#Cins1Vz?=+WY>Z3(P&4vzJ@DMhqAnS=nSMnOy|Dq@d=<a)?%<6{mG+qGl$8xjAfr&FDjZyXyDddR&}9R9s&mR
sA#_*cWE=^jH-4MpuApR)M6jiG?Cj&NrSCBAm|joNZJ#|oZ(NXs(~+`-lHfWFC>2|-$g~n0_i>d`?`!N%S)#=^S{(|!#}_M&8-}FE7Q6{2fh%l*=;HijI_vc
J`2vd{dqYb68OA@Z?dPhJYO(FBs?&No66ZTjR(p&qj6v?t~|8#Ym3xs$2@J&GxUCrrJ@<(78zriqQY8Ay?U6!1Qr6Y{@vh>rn?eFq%NwVpMqu!^Q^JE0w$Tf
QkqCP{v`7$$4?lB&IOl{)Xs(_)K1kH)lN(U+FkHSu`<DS<!*>K)_mi$b#fn?%fL$<F*Qy8@hcIGl{(EvP;LjHMi}Ho#fNh{ThM*I+&*}s5$>*WefW8eypk2Q
Nx61#PC%d%EXsrhR-c!F2r*1j?w55IVX*V$#jymH@@M#qz!OCA<mjQm<Bjaat{*&$OFIyHhT?KbC%OT9X`_CHH^(RquP_ow%$qc%2lzJdoA1pa_^}>@waPZO
!oh74ibQmL_2Sg1Iudf*Ak6d<Mo`O)VM0IscB0>r8LVClZ_3z2XSlR9i6)_Om2G_y*}X1A_HM!P2t_c`Oa&?#Xi{#@+`;>65N1)xs4v#r<ym&+F$Ad2mTH{@
(zoV^L>a+JcAQEQ*eA~CRU|n~^YVPuFA2;0w0I1C|CCfA3pfB@Sg2)v9Z0(-ql6Atkj+d-H0+)aApTvG8b6_1MNLIxk8bGO0X@7f^<f|YL<%ZBCEc?JTzT0{
HxeL3KfnEa<|1&r(t8uY32wNG7l!7>)(W#OmI@>ugE}gHE#HPS0i|*_7~M*uzaO2DMJ5|X>_@bs%li0{e;6pBcT>A*(#MwPv+)wBe`mcx26UM=o~AQe_e|}<
vf%T0SAu_4<fS>^I*6Vt=eq*uu2T*eg1wPtp_a$DW|mX*ZjT25I|55eP|A=nEaFb$pUGuBban@y#C6Z1v)5Et?=|9X+G*v>4(D@4^KRCI#mQ3i2sZ?;$pIkH
8u6YwoaNkCLzc>0Iq$s*`xIxEB?|qH@Oe{;3YvXyQsMR&Xokq?%Ix9|0>PP+9LM-0n)>V%>6O2=)h=o^^fWjN1OSq^Q?qxrrm&m|nJ#$(QWK?ID=3>L?F0bP
7btpci4iNKbpQ~{Yv-3j2vtbjd>r3(KBJWvO}p^u+YZli@5dt#IX4sEcmo8AW1}T9n<61Ir@X#qyo!xHbSohm>OBtD?!SiApAY#XSz8VqARaC~{TTrlfJAVe
c&Bll26$ViImu~KGz8{`f1<IW1Ks2hxN4KAPLpVl;G{OJdkt_PP2;MLodeXP2B%_@T92$cR07gh-j*-~j;zD!9F<>w`M||n`>&kg0fi&f#(W^R?YgYqj>bc{
QlPW3k274qz)ZlpWf#?cz}CdFbj4`2K`I5cht=y7(!Lw(&6P+90St2jII=n04Em}un&doW>3WV}g`aLloPnXJu*a*O%P3K!gB9vr#Vuv=jBBa?PscDlRE1cQ
Wgqc5JP&kGQ2@`pr|z-lt<3~=fH-g?&E*x{vwa_7&XZaWo_4HZdz1!nimKUUs9&U#NcelNL>6!bq!{14xMV6RD7m5(gd=a|Tz(3&MttMffauxg6zIVizC+M8
pEJjltrTeH2a2b>mzPQ?3$p~=t%|e}#*&!ANh#Ws>ilEBW`%V`{=Z5!G9Ee~<Edl)>k9xNVU_EQ=p~D-hfTuh#vcU&4Pgs58(6u3gaNp40@{hLg0XM84|X-{
M|Lh@L2V<OTLM*=dOZpt@Jey0l<2#-PoW_)LK{cjHKC)tJI!8Mr>Twx#K7rKrl@Rox`(ONz39^lH`C*02&bR(pzQ`t{ii_-j#<sy&D#g%!G-V)-*iG&_k?0O
U~C$Xp?w3W*q~X=No9<+qAh%nK7D$EOxQBS@XV60BJUh~%C2}^WTYWLpB1*+v}GX3Uq(QAvW^*lbIMl`*_5G7+Q=rY)ziq!|GAgSH`Y3(PE<c1hWC4TuoLnQ
J6PDcR2clq@2j(z$m_Mvp68O7Gt4+i%elh-Pd{sc)U*Oz354Q%OAL$a0iZR$s@YmY)*5VZCBhw!BLoSj)DFflm9r8=eAw?Iz<b!dUt)9i+FM#t^OIr&Jp_7F
X4Pce7e{g@D)aY*82C1l!T-XvT;sIz{-Ed(nO$J4`~AVKN-$Jp{1t_4(m*DOw6j>6ThXKr0vp@8Wy5V|r=Ja~E;!05=1zrd7vWWu5&yyOOGJUXjRalm+|2rd
Q^Dvf^VlVk(DKt=ep?~*va`s8ce|>BKuGPuE8fEr*`mmnGVQZ<YGP0v7^3s>7pc9xG;)x<>Ae(hU7TfQZHi&e2~u9AQ}!Xd#T{ep*~yhBWHC!#u0XEKYyIRU
AAuYg<g)e<@;1~*kV(=B<wM!i<cy;N;Od)DZ`OxMj40qU@Fwq3s)ie|vv=4!)zbr4bKlW5le~D$s|_3PQI`3LjJqnI7j(Z9j<|9TlA2%<kX7e0#{Aft-R=u;
EZ7@l9L~HnQcUs*r3_jv1mkP<npH!D3DHjz(q%l5hk&dVRE98BS9{6F<%W8ZpgK^2Bz$K%(7Lm?GxL{5uveFjCE}U!AC|Iu@n-4*e#$i)qSmX_K=c?G_f@Ub
o0m-F;D(X;(D~o>$-*$O%Up~pvt0|HkF2$G;b8oh&0%Y|)J}YFJzBQw;(oBw*;?g=roT5&1LhIhwufvVj8azSwGfL^#?GNHy0`9L|913H+9#Dfn=y@eN4U-I
XyzNXUHIc1`fB@V9;m~(VHwI~syMC;!Df*({$k{Ve13xAA?GF(h&}Eec$<3N&{C{sJXCW!lq?_Gl6QMOX&J*Kqa<-4eY>N8r7zL6_JBoSlDQphTNQ<)&#~wQ
!S-8QU+qC;q^p-TVzgw!e<zni>S{!=iuuT41YG%e;WVd3nrVzwNAElz;P>A}$hLrqu=9>;YKX)B9F*B7mtFo4nVFodz8_$<!|N_RtQ32dF7*q$Z>!zK;F4~L
^icxf<8Dk@_Oe$X-2o-u_R#}zErF(_Xe`|;)kpy`q(3qYx+z&Ew|uxq9M!wN^IZNFm3j%qfl}xW<R90<40^$Se4FZ^0jbcB1#}&w+<?*bVt*NNr!SY5<#=pH
Y)4+4Zpv&XgUA^#Io-BOm!-Q1B#-E`O)VRo*zo~$i<|7xpICo;<K5<T0~Db$a|;5=9@cBz^z`hSMoi&)w$kgFpyMAMn-Z94XA}>}aHrFnc+tdH!m3}a;E_q`
C^Z_TbMn-ba5}s>Y|Ii6DXWet4IbaeP{&y)*7bEckkJOB?*_`k*q7YkJQtCL4n)Xo$q1j(PE6MdwlL^B8=1cRar9$}jwf(>D(;uVM_t}8$t0T?5uTPZ!D-6p
f20kRVj}I|22n|<uB;%BLp<>zwkry`^LC1WrFh2Hp=G`qMTh%x{cdkTQw}`V7U~_SBs>-CLR&hZ5GSjGi16mdKHUdxmFCR-;*xMix<D$2L`$z7GYU%59em)!
>lsy3M5HS>**QeChVV|SgyUyNiP|h|E4;BJ#jG85*pd^ObPQnT%WOb$u%u5H3Yun>;|s;Zc!z(NYxa4S)^_8Iym~b+-y(&`V*@wLIs(OidO3xs>OYovH)Z|N
)wf{Wg4dmrWs!DU7!t}ws=oZk=?|B_3cb$I=(s+C6Y_|>cnSWobI3$tpZ+r*3rorg(+SkfQVm{St1Bj)R-``Da%>f|d_2+Ldx;$IjI+@&i{0z4mZ=tZ%AhE&
8=U$!R4`xPDI;?)I^5@5e14TD9d#-r4Bh6zyUacU|B1gAy(27!Dg7@6%_Z%XQ)?KOX(O3sDLHwtpzYYAeeE6nSYs7**)mL9e2gSFxT`xwgl+jap7`&7NtiG1
^>2H(8U3I3s(&h+P-FY4F!f&(kp_1!`UquqBg=ZNPHoHt+oo|F3BLMVw(2)`wtqxddoJFx20dG7)9E;HkrCGu<7Et{uYcu1>K8*p;;OEZA-%2jYX+E2IlBaE
M3lHgaz$rnaOH@8M7^BEu%TO-L^(CXIkGaA+a8nq>{1Rzc$3+;paR$jUv<Y}@Is{LRNlRZ$l^)eq=BAf4A0@!2Fo`^0t(y!{+{F$r7)-<oXeq3y6<%|8%RX2
_<Kq&VukOf9Wjruy~cN1uwB+wrI!Mv(ERDNhRJ)qldScdb*;mUaN4Xrf%r1hk4<9BwXhikeeQN|%Qr?P#T#1A?)p_<JH$#Kwzx&G>v}o;f78ga6*!pQR4nPK
L8J)Hb;cJ<cHzFWGF+}ElJSP(PP}+JS>S0T`-V^xIaixg>(1jon^pW1Iy;ai;r&7aKQ2WhClO+MNJ#$s^a$U;AdB=()jz^_JdIdmkHz9*o5je#+csZ)T{G5`
q$BQfV6#xh;v=y3mGvXt`0jHR9xHQV`3asRLu~#j=@9bN`YLwJx%keMlQ-`*CIAQ#O_H1nPd6$aLVP|6!DlSX$ss(4y~1^<C`rpFOESxim2>07_Ctn?hiSsk
2SejaT6JBVAm7;J+|$4L$}oFdN_q$%|7h~^%x?tK!YX1BeX@69S3R`C^ou;mm$DY1E4AF6tLycML4G)&UIreSppmmaC(MpDdQq@zb*sC26Mli@Qx9eEJ#x6)
`!j4KfhlI7D|Yf&Oa_ws1vB}0eoQ&rt7eribFGP~y)xa;d+F7D>M7P=!yxZxafN*jOIP2y_;e^S(4+}wimQdi)g``l&&~8BiLVv4`-P2YY5F~Vfj}hTmXNJP
Q;k$jV!{z&7_<M5U)HL*n*TgcBh~V&#vgTx>T030n;7gc=m~nA|1FNcg64YCsu0B*srIEn@r^kb-PpWq_L4Sd#|s1Qk-{Gb7Uay<e=_TH4FY1wk+bj=e^6-}
yKY>`c`k~bzH+HIPV?3k(`ul?BWn>2<M{YZm*+tR?$C~GBRw$-y`FBYpbAOUGtl3BDoC+Vs6yV-4tefony+JcaBo-?AF{jmsmfKN72?URBAMur`}Z1y&SFqw
OxdjJxGL<p1o8NL2~o+9j*5X&bvV}Wj^}LijQDU}b#@n+I(3#a>Oh3SJ(ufCBebh|kIjLv%Pior@)fmrdWAd->O~r)c=-#@gQRF-X4I^hxALbl!B5ONJ+bEL
zG3Gx*!DQIJYMq1#cft&hSsU4NfBDbI2`duQ@?GQ<Diy|z-m+X<#}UjgOtJXX(aYT{VS+;z68S*C4xRHsB_K%@K+jZ1`U;%F0Kvs<>K;deyJImuw;=YwtU#3
8G#;WG6LxW2MrDFzd2U*(Vke!#av%_8}f^f=@ys{^XBt~YXg-}Bhcmg7qXiIgP)F2KZP5~L0UI6TorHpvbBMY5?K;v9Je$kJ3I`MD<wT)kE^7Qaw3&o6-%p*
qh8es<5Kje019|+1M54e0`vc##`U7!?purjI#bsec?5BOsa2nM7GZKH?Cr{_xZ>=!aL{ti^ABtLOD6g=*l*%h4k-{CUUrO*gzeXxWkOp`<-u5;F>SJi>-(o&
BG$wGa_p(lx=6*auaT|^@SRi0;9?DU?3zwwZn2bM7aLwtkmc9P2K{_0zNj*ch;3w7AhA<0iG1`T05zb-!vLc4j1jM_@(XiGgh4$v4ak9pw^4WEP^OL$P7D^#
T0CcKP9X@SaPg{vv$}m*o+X!zIuc+L8Ue&gylNk5%MO8kVt#!!OtJso!>+*L3%XKM4)UK}DSQpa<)AB@rF?W{$|&wxBpQelp2Ffq9%LT7Mu2c%0oa`9L-n0|
Bm+*23<h&Baf9s>|7x7F1Fr!h?zvv(1LDp~_%M^we@&B|Wl5_?%x)Pylin>@SLCs5g9kMZK*je(a#nEUO>o?Yr;TK(pDt<^xx2C_Ycp6aemKIQp{8#pY4)!(
HeYcN<~Gfob{YP3bwTTKqE}{`{*tAzuc!hjv7qp96@$X&n$y)j(Z?EkeB2QFn!6wN8qu54Yyd2hWtQe5f0DUwe_X@j2rbD`V~*pej!n*^x_FPOW5C@C%5aI^
yv+Ea_H2NHfmB$*K2;qLFv`n{4o+Bb?xvE@jJ)b9mi9P@7nVk1bNJgzi}5cec2KA_ZtC3$v|4*#*b_MkBYo*>(R#T{9aDdFA=Ku<6W?6M%5CiewrALLEhP4G
zghh%TK&E#A09suL@hv!(^wFJ;GgGXMl&ZS@<X${3*FWUTiU7%eipnuz*x6taCZC3#a|Q|AF3;3H<oICSwruZEFGt<7=2!fP<~;%<-R0XKk!<J*wu(ELd7~e
Sh9`0roL7K+$?dXgN3q9&AKQc136wJv##9ct6g)LL$i!^nWYi(Vt^`Kv1Nl^QlqJX!)(=sAf8p+JEQy$>uBD2$R)*@3N5(iM*ob9t`CHm@sBS&0`Com7+&U1
2CRKFh2T)h73sjiUnVS5nwdE4311_({|m3Vm1wo`U(26YNSUVn$v$dqL1N#@r#R{kZc3?0E@%c_VpOGGvbpN<{gBncHd#+j<*{$g*H}U%pW{Ra@&^|MdFUST
+Pp#Buw7Y<9IH0f`E!-?wkS$KsMe9~CJma&lPLl}1mx~qa%QLpO&JbK&<l?hWg>p)K$%zUo$F_&>o)vAtWp|i4bl@g4S#0#<r+(946<3EL6wakTc>0pfCSA$
){V$$NW|J*zC&}Wn8dl}#m`D>rWnu!3#C^{cFAA>vV$af6V|VW_;)<7V0?E!akIu$fW*m*y$EA$zq`GYPSj348S5_$Lh0~mMyl4I*lXaZV;9`=gHdQ>N48b4
kh9_;ARz_hhG9=dx*{AUt7>%$?{u>B#|Xw6w8G;XjIjT1|5pc4Y8e9j82``;L^Iy#s$s&X^ZRYV->GVeTHEMYAmn%TXK0LtUFPoMr5#0u?vkLmTF?-<SR{31
_-@e4|4EE7jD@hIN1sqCv<T3ASDGLm%4*%RApgPap+uPDimn`e9M98-QpE#25o1b^s(XQmby~w4se>6do5Wz3>d;<cr01068#sPMd&0xLpxyT5%Yx0PS4VXJ
TC?hgD86))4xj`rqRM+WZ2xP07qa%#lf~Ie!`kT`62}Nqpqi@BkJHth)?!seWDbU8%<nf)Ml~~>l+=67xk<l5vA^5I4H2Li?3*%c<Rw()`$2`-X4P}SY8ASD
xIl5b69#mDL30NzFh-{}n6PX3Ts411m=ZRfz$rJ!Ch`Uo)a-~VGZo!Z*_WAvOpxZW$zn;c3@SCHt*jywRdFLQ_RK1=sDg)yC`X>HYd>QhGSq2%tH6=e(z6)Y
mYpJ6-Bd0|yI6!g)j`5az+XvA34crd3c;-D7&yRbxf#j$KMS7y#K^t<WYMA==GGRm4}ZXUTq>!@7s2WempE@qJOx55LLp<&8TbHK=%H-995NWNR$&m=+d@0d
B?d9)B{Hj-FhCHP$QlO`#|z8df~rPnb#Fi5$-B>U1G7^M4e1ny|I~sKxySSo=$@6z=P<Shk5Hqn#A0Hu1G?}Wi5&LrWc$&A_VG{_-svYNG(W2X$J+zWRVHta
gTP)+66EH<xp&OLNe@X+6-9Lh({n?*i^T#~)O7RlYa^nF@AA}kDPRC|JpHZtsDUc%WIZk+j-FdTVv4&8yP3a=^QtetV?b$`HNoD3)ZyQy0x{I`5M=QA_=2B-
FhCkv(idEH@+5LW0i@X`LY{id5Ej-K;sOtTIR4RB87A#WBI*m-@4T54Y|Cs;<6jx4`~YWQ0I~-)NOU{{Nn2Q@`7AJlc~?ZwW5f_(2@c-6u*!6eGT;LnJGaSO
fc^DaI==Nrl4b0XGpYYl5b>ktzUWp-xr;5|bs}Evgt;a`cYeVr#70$RbO*Ua{OP@u$A%kNz1uDy^)#Z#PM-6|)dV*{p`<&CE&GU=6V*mw@wbbXXzgDSD9%QC
!NyijdDAIeX2N^)gS{G78aPOQlUSu#^jbyDyEfa=YBo7ei%YK6t^{N)>7qZLmz?{MhkC@$KF_7+tw>6S%)qzVu>aVf2Rma~+JUc_owz7M+^$7+os1}H62JY=
kcoZ$nD(>1j?wOtV?;+m$GKFZQHy(-Kdo*09{Nhg6(lF$Yc;=qpYpHZL2Os>idq<(8)LO9mL#TQJ))~TP_DvgtOj@5D@;=+nG@F>rVNHVyG`%m=)4TR>!<nf
M3e|70$GZA%;*qViwy@)>c_J@I>%bAV}7PSL;1iHXg(bSDx0wqyy5Xc)9f#jyO9%LklWqgN;TOZ5L16r0AFw!p6=t0oSfQ16%I4q^8~$E?BPl@>*P?4Niw+A
Tg1Z+WM=7;12}lBC%BSbwae&*6G?WgN@!GES}%2f7fOY7Q+1(c40%M;_O~lVR#6RV!~7eS$Q3=AqzSP}=Y--9)V83cSG}l8JG%=5?bs>&McD97uGq_$J5_z)
uVB3q0LtFU=w3^F1AurF*rz@0HH7o^hN)gU3Fq!!{?NDo1e^7StY3Yxb&6_K>)`~RmeSa3lV1WVhr;-|+~<NB`IIxVGjC-&xlDMyH6|n^p{@}smna73oBbT{
Em>-=AqKK}BAT6|%XTe3stmR|Qu%F9G=*YPIu3YTXK1?L7U?mlsFN4n4)S?o^)j)vZ2}JOd0p96n#s|NfDC-mZlE?sH|bFBkIx6HO;rGz1^EIaQ0K>$+d#5e
fh(_*+#r%4<3c`qu@o!nLFg1SajoysZ9@37)fls^WK@T*0&S;`D_s@6Ta24ZMm4~WKwj0Rs$8|9m=EQ~0ngwOCM#j)9ik}RP{k<KalhOOWCz@m7<-JrVC$^5
yh>cKGpvlRr!r!>c;~ve4X9semk@;tPOr|U+DXy9G7R-*oPdgprTg~nHSYuLCIr#Vyy{{moEm02a(Tos>?D1=J)Yu&opm0*GeoXC=9L;G29uZu`TF1zCHN|8
Ypj9$D=n<riJ>tW=YaW~>M9H=00JyAO*6|vX_mB{kFQ^?8<DzllDk>4ZBK$>Yej=X37h{zXNnrTm+%IPvgx*x`q}DXcNA28DvZ6K*2+DGUx`_-j(Qg4Eu3l!
;#^xJ=Gpqu6GrHl2(9u^T(*ovQ^D}4xUZh5EV*$^;EbeLlSylZHsK-s8?(E4XvmWaGHX#Tf~Utg8s7#_l<ikXnZ!!5t#>Ny`o?(D^L7=0ugGYokVC_FkU_pi
(jj5(i00|?RQ3efn}v_p<IfzCr)JSFs>zA-UcO_ESmg}pb=wthV>l+ZhFC1X-c>vGwbHT<|82-)EDekC!|NpftT$D+djnz!g$ay~UdVoS+4W3q+-$bKl)VX9
c`4y-h769Q`)l#ZVev}-&XJ8qH;8Y1LL^iponM^OK8t9?0gSc1B)~i2Ld-$lW?PJCX*polBGG*PeTn(WZE|8pc?13yl94a!e}{iw`+@G=pQ>+b1@hvoLbz)8
E2-ThfnYr;XQPM{6mD~H4r7`y5Tt~))9kenf_Iyli)>64{hr}c#Ypx)!aJikkfgDsQrQ7!Etm*IB)mt{LA(l7T@R}VlnHIRZqkuI2ua5J?JZ$V`LOagNdiK8
^KQ_fhCEKH!E$N_If^tDH{c~CUe;>LA*PUkgGhWL>fN}sk3FVk3DHGylxe<Gk!!<r+qm=B?)2CKEQ`(XA_MFx>t=dpHY^f>=Z4}GYJgsC;~id$FB7lD1k#Rm
9<s~5=-PEKeDoqRMfqjxO>aYMSp%Wk@%vuAXYFaMz-Q})NHem-x!CfEv)HF`=RK&a<VqZKM2saJT{LbHjws$;e%PKRPz0PPLw^oX*i{6gs<X7+tG!0iaumAz
evNQE<|r{aa(p+{7olaQ;YOwD3L=1AM*;>|77&&sHDvcI{6l)*ei7$#)q|yoBk69fYX=2y$<ff$F?L)qF<e@Y1dJ)5ZQAC5PLgogV6f~N%F_@|>hl-X$$0Lu
e&}RcIdU4Qq%u^&a&Vq?B+e92s9e`q)6DPR6!&ppc_o{gFLVzub5l5heYt}C0iGDWV?h?9d^wzUTgw|X1aN3vt}vrVilDdC3g?G`6A>%^QCw1x?2p_mV=pmC
M<=KQhn;D-e&t>?*_%Xfz8Ma#aG%0~;A&~;Ywn4uS-Kz+c=8{N^Z>xvWks=$C|0&#6Ox@j4H@hTRL(+#cn@h-`&qPf`9-z++bkzMFiZCdpbA{@*j?$<MGVHA
;8+FZFX01v+4CPpi83e2X^1U+veYuhQ#4xk9_ib6^84ThLWA6EMf^$4$k$XI52QAQw`D!mn57Yg)q(IJBzC|92{u;apX=mYfFaVpPN|aH9PudtlH&v{Dp-H2
lp@d05Z>-BCLgML%e?I)isSJ_!YuGMaD1}`Xr%!fI<CF&D!9r{aVrnG)?uIMSaPGXV#)(7AYb3JRJyJx{Ro_akiCDc)9~ZQ>6b*TJ|RZ2oj6$Sa}|nJc%*LE
-SQ*a)vIWi0Cdumr|0(RZ`?`E?|Q$lfeGU_&Q<x};_wMj@Eq2O`43<tP5&L!{9;mYC3fLv<T&}U|1(rbnuaein2Aw(8+pZe8jRzT_q@O`!`RG2UEhS&2}a1|
eg%kB0M}->y*e%6dISlC*LJU$hl6mhQY)gqPw2IIIhF^nS7VD4lV?o?CyP=OF8&O85sS5a;^9Ie2C`Xfpy|!HIh@}{N~+hTAq}KPC6XZ2EY46b@H`6LQIUA~
5YiW1zwYaSZO(gn2jNL@H9KI6RsCJ;vY{au^^-NHj8RlU5CLkelY6}E5K)NHbAaq%M#*9DS<z7qswC>b`N9Jga4L^&v7AjMt!Q!~Cqw_yXJx}DC%}$Whnj~-
{{w9N$4rkyyl{}lEghy3#?fu-i6mx&0Oqu~muk6cr`Gk*_9kOS5qW8w)qSJQZ$1;FWl`dIogvwXNv_Sp(~?2DY(U#UC6FW(@jf_q3FqP6V+Y~GAz3EI5Nsub
UmqCZ9&md!yxU+cN;~uckHN$%_YNEIDB(U|PyZf}vr_M|CF!u=W)+Bqw-g_F5nbw%UkCyRO5Y{zNFJDx1|qz0LoSdbtqftr<fYk4J)~>5iY8!7im`&W9-_UC
eN$p}*z|2aQ6usk9a-Q&5bC(32glXp_bVSF8K`>-;I`_E45OCgb=w=UA_>^(^$I4KL~gjhyIt<tW-RvCAJ}3V9CF;f7VY|HoWmyq>Vr<9I3)tYqfWzj5lMaA
W)YRKj&yydeL5~dKZW>pqNcZSE0$B7z?7CON0y}2ZbQn2O@Ui%|GeqvJEV*{a2kS~WEd84G?oIsbp2QMQ*=L=949e*^iWVme;+8x;q|;bd}ylkm<<khDLaAT
ZaX$$srHSz6EQ0!@3x2rJs&o&4l>;y1&rVpBJx-yRdzO5y#F|(&kyg@3F<;wvW&VcN|Z42fT+CO2pcDh>`{l*EzeVwSL01@xeQwMQv0>FCC1$f3qF>H5D`Kj
WD>;yU&u!|`GNqe&Fl)_BQ)&+bhXKxqB2e^1_9+Yyna-u0{0Pl<<ONv%}r}V1usA{KPe1QFra*DDu*niFZvtMaeQk;vful5zZR?o6cmIyTe|NSZWf?1NG@ZF
oG`R!Gm`q_HJ|k^|94YwI-<$JBjs{ANWQY|Ad*{9nV8mkUIwE*o2Xx`=V}h$UEJFq&spN}ei85I@sYCv0-~x5MiSJ$(?v^i<1hnUhjvG$Nql1#)n^FNhPEmS
<kF<yrw-Vy*w&6T&CRx7xbk#iSrmtUKDud@2JAx(7j|X~&(Sm19w(mH{daJNn!r^@m-931xv&m7X~tITgVdM9dsQB0JzW`#K;-QsjA-$xn=p(ecD(0H%*@3~
8N*r|T!0#0zC{3ASx{B?0%`3_J1(`T03>*IOyrZ?WJ8>SI?aIot^G{jFDHUCy|oGcUzN`5pl3Ke*AyOI8!}pQKHx%+i{<E*Eiy1y+va_8q%eMgi?~=2^N*F?
hDTXp8JDs@jbZ4(m{jVynLk&qm&J%`EZ9PS%EIW^y_|oENlxSY_ptC!6K~xeLPv@bsf;K!XzA9m$qa_JuK=?UHWiZ=5?108JAU!<E)aThk~4X!1?c$)N9D}t
b9FGU(mQx8b3#A01+`1%leU{Q7z#tEOea7?`H7XY&`7oh7_y9LV|}@nly08Q^aKN9mN^>qgK72$K0Xw&C)2J(8S3x*_<HYo&2mH44T0Kuy?#U!f0L>sE9S;9
{dxLfe))H^tLP{t89;^Z;HpAS_J{2L$J)<@*DLL;91-u_JWXZaKu!gY&&XVZXFR<&VanK9N~-*P`9Uzxkp$vpL8t0&3R54W;#Y><m30CxdE8GRo3SRdWvsaj
%!Ji7ot?`UefJ|sb@m$0fKsfez?ThNDnj7MpjtkWjA<sp{|cE5Pko?CTwby+1wK-eCw_-)^cvyMKHGH#UOdbjDDMAa6Ey5~{)GG&hxwMVoWNRHyFD1P!ef~)
I;w<3ZJ%OBgF=baGB>nr%T{ittM5O&r;_vkR|;q?H}bx*Adp01UV_nA67;_>oZ?enu7(O_fn(=CJwQeY4D~V7XI&{2Get$`(jfrRlH`o|46<2pyHo8ES3DEu
sqLR$?YnBGfOjK+ACRa<0Q#(y^L2jRpkohkI_Sd=vtSXnK@JE^dd<uf`p(OgYMXtogin~+s0f07u2XVYHP@iO_ok!EZ&Gnxn^x=9%fM1t3HRt3ywtTXV0p#s
YH#i?UfW7DG_wx%G)=j|Pc{hC&WBDQYuFU-v-=^@E)eT7LTOA*$UmqsNea+<Z2x4UqvYlWY|ERlsArTrq|nPc$?gMTvIJZ!*Crh}N6y0IyK)NNhh&Y=otR@W
lgCzHGsx^lP92Oo#_=XsI%<fq6^Qx5=k2bolTnjqg;LjAqD>x`I0X*0h-3o_lb^nmsABi7(E5Eu;*V^f)jkv@;UeiXu13Ok!qSq(1q)>Df-|BtY25nm0;@n%
QJ~x(Z`KJQ=1XP!*!xdzN{gFi+4_v7SO7Nb@+~@)YDU2lV(zx-SLKY};~zR|**)(b0JrcW+4o4hRB3-Sl_~Gw&)^s*RX3IgFkd!&7k&$m`Dg;Ph)5=pULBDe
x<$;plr9=F_-6q_46cZQ7-;V9ADKbd5ya#{rj22&*;~JGdC?(!dhI7vv28)S-hEK$<xVM~V1pU+%ETG2G=V!>-UwUm#iosM75hIknMgMtgma}i>!BfFnvI9t
#2P6ou(S^!|E$WRO~G4Vg22*<Tvzvkb(oN@_DKoAq+3fb?{?Zck)*;A$B5PjW(P(UXPwZBe(g5{z7nCGHcvL`Qg3bOWcOFzmLkZl)>bx;NvQ1YW*Ye$Yl#I!
QCNI$F;ON@J*Yjp(Oh#z$Nv~TapEnN8Wgr0;!sm4Bw}wja!T;FRM8BI0JFb8WL2><8H-uH_O9(EqAJyirym>fc*~-sLFrd8XWtyP+}ep5nD1_%mwM`hF*l#5
d5m0H_SZJ0re^b=Pa8ReRVoSy3MVshXaT^+w9F!z`{k==v-#N&$nRZY*)bbt-*{SrI@Y6VtJN0B%Llhp2I|Q%&|&RnUGUIu6*9+@@Cyb_1gD{aDYF?hyO4v=
#W#uJYe6$!<$RReXfZEz6aB!z)Fj{4mp&7*5azNNSwhQJ$c2bC+-U%tCpsmEaHfDHB!BjaVv$!YX0i`#|6<~M^Hk;QxGxF;-Xl?s|3I20!|}Tr2toyGmlr+I
7!dmR&;N~83<row`pmOHIDwH0C<0Xl=_^MnK=dE$h?i^T>Z&+1Zc1(}m14RRtkeGui)(1&wb?*Gjl#jQ7XCX3sFvtxF)>$bF_iEtT%BpRXc~<#rT@_^eL8vl
G9hS0s9P@rFFJv}4=1dOgDdj>=jj%94JDcpV$`#t?VYqsVMg?y%0XWTYeR|8#VqophU&BQU&<~UqUDgSd+7uM*5sLE+C80^+}Rq3<*%{hzD9c4bG<}3(FFWA
!ks@!bFXv_?RCh^V?~MJ>r+yBo+-aXYJ@(FB09RF*rrxO`iB*%E~LZdjJ5NI!Cc+J4cFDUKYKTM4iPy?nEZ$geF*$xP{5Ns@^6o98A_^h9`T$1+tLRYdkc`g
SwQdn2YC%*4$<08m6$nT;G6v$mA4_6;S}E7qu!&~jdk0}I|+JP5eo`$1kaZMUXcPO8m>TJ6HM**XInJCCNCn8BC4Eup#?4i765FB!CsAT=d)_Cqb0J(y$JvG
S^YH2erM%QZ2!tk!6=*`T?x1gPR`WDZMOEt&bC(bnvwQ!`gHFsDQei9;&Rz-Hn7pQ7DDzOnTw({ALv1Vf@AtI$sXHV48O$y;qkcUHnmVZzuW$98nZBOGEofR
K~DL@>fm1uL%2Lg_D^z#5Ei?#_frySj1qcq^hd<}zxH=5o-&t^DY<f|UB5%=_d<j+L%2U_jJ)Wb6GlqbS({D^hg#=?HXwrE@8&vbiA8=Ss4VLLcBR4zD#CUP
gw}XsMP?rX<lru~iON>x8pP5cJ+v;MYyHBimXWPVQ*{?acvm=%i7D#)uta=LJ<3X{Xv+P&G;b-0IvP9#3;l(r;+>1aLE?{0l&2P(@{rvG6O}8b%1K6WM4`n^
ZFDWq2ZhfT^Z?%|)<(&aK_v&_^X#nULpvKa6vxdRdXAC1R`sB(W+*^51)cq9mlP|#K7B}xJD0R^clJl{8$VkcQ)x)Q5*82qF{FDX;)i%rS&FXAQ0S*(y0Bqs
N^BMOv8V~YMQ~jwW|5kRg8P$(sfM7mOGd4+c|g=5$#ftsuZ#+&t<A3vh?tTr={AR%dtt6jz*8J+T2$=*g&b%M7+jihX=TAP2jr#V^XVi-VEW)n`c;skCS5%V
Oq_uy{UgG(3CK4opu&e-{^@ZMk{u6-?W{Ey=DdUW8#Hw`s_XmQ2L`u!L10*e6I(CXnWm2PzYhQ!Bts_L7&;lQn`JBEh)6w(Yh88776#~>^BS%frGfr*^f{Lp
IhCck&hieVRY$f14^yTp=M0$#C~-^I1@DYY`sxDQ8=X18sM|+fkDHXo(cxF(nlr&p6P2f<2xv{gw;P-@D8iHvj=l;4W=TSLpU;`nl6*<fATZ^Jr>o0!g%wZ@
Il9`GRmmL59Q!HhP17zupH&v76ZGC@5m%>Ng6si3w|q{@aVF#R+eB_Eh=f|W=HGA!VVZoBxc%$B*WJ^`GbKx$1~$0|SIgc)5Hu(K+SYahN*3_vX5A3*@Oa&v
5GFPr4g#YlFwQf_C{6nk!!3#K0H5++WNu)cIP;W=hJMA1-=ck4Nk|_e9&;I^WS$0}M)%NNdpoST)l)|zY>*gsN;8FFG1<9>f*ymEYij<w1F^^Q`J5k7f=??W
^?EsxJ9Djg&k%L1znn(g)N;bqV9!;Vgfu?*7K%hF7U*j@GaOK(*-GhaSjET0wcDWaYPvjEiE7d1Ej2#|t5+7rY*H{}^*{oK4BWlu<iIjb!S2bq%_HtRX%w3e
)OAEG6R;OGKQro^=5SlK$dA$Im|0uwnLoWws3zh&!;RQkIm}T^0?Pr{oQ`cX7VM>I|Eg?-BG<t2Tp}!u<-(HG_4+Rqt{ihBjlq9&{eZv_UChx3zk8+^VQ;@6
VPU7Jxd};J+M;wwpdTx_gW<<55Y_BbR}H_xiG|oEDJ>Ow7ecKF4UcHJil`%UW>Kbk8;tY-@M+xf(vXS*paC?=J0EW!s7BBKnZesGcH$}OtP4@Yea{&GW{_~B
$IZ$bSPdlRh}Q<Ej(iz!hHchJ+rg*EL2pKavO`>OD(^`D(At~$uwQgC@y>S>@tn9LX~U`Iv_Pwt+(qs6HJl}`IVuXl$jgfo=jr?&15>}<q~CbVHusa|LbiKF
`be(jTd`>NQw?AseL41>c9WDFIOF_GZETq6DLJb;f;ddn?gE}WV^ZtCXji}jRAzt_mt!A?_?@26FQ}W;{8ibVLYoIa+xN8zA4Ibce-+3otKV1&^*npivG=*>
cA}^nt|ilPxkdyLm`%#0om@nd*IET(RI*2OS<`jO1?RHDZr+6=D$~EL0LY5^_bm<9j@NP0Jc=h0AncI4MblkCP{AUnF1KqOaQXBG890A+7Wv0X*d|S3sGZkE
M2R1sF?RRNiujRFD<U1#Op~+}_qolaxS_U@hI$2P<pJ;9zC*s>T0u*o2nmS!7BGSSxvm$gn<Jci+H^}R{Z$`G*pws7?p`CCTd)>z@g_=QpBY{TDTlZF&h~4A
(K-ignU|+YU<@aL;|H8WhiRFBggoS4eldCs<<>+UgP>7SS3C@pA#t~j!f+EM|9KOAyO!69AduQQ&vAem;^X=B3)Un?<YT1(i8vvFt|V&6vm_(>HX{`7imRp4
`QB;J%!Q)daJ74py8yt$a{qm8@O?QCM&fH9@_nbwnoKqoIuQ88_8_;q+H>$^5q1qBiD0*a^iSO*S~!e$s|R9gAt8zwGz2e)MItLr+(lE;JTf0eIN(Z)00+$Q
O&v!D<#tb2E3eRf{|6V|u=T)=b>*@c^3sapPpzL;$K-yqKyl;ts{i1?ndDD1>X}SfE&&}d)Z@Yi4^?9b7}4qPzUvShxLM&EHD?miRRpDka#AU2`S||;B`zDW
;)+z_u1tCllZtfI$^G;(gP|Zlk;T(QsGf~a1~TiS(9~S6zsrI94-aF;&T@#tmO~q$>P22dHI!r#TL0zN*3d|1>K2z<h%|rU@uy2bl;jT)HN}Xen^0OC!Ibpt
lYoW<RfX5%Zm|k1U4T(Q2X_Jc9vV*uD=d@><BQu{Z*aCG<<v5IQFRPN@g<<S$Yf0KjFPaE`xr5`qEc9?37cLiocw~XI})9I1NkwnZQ=cP3s2*3cW<DqJX{$z
e>X%<d3xU|Q2;27nR|cWPf@+^kJd#!Uk7P%9u}>nrqz)k=m;h8YfbKhDwliakGj@3>|(P(L<4!pCi%zP?D;NKz!(}XP!n)$`=#+AJ$s%1ylX$^u9EfsT*xIm
kW5uFxU?9Hy8<p6N}^7wv6g8&NixMUHV_P?G~NQX<>I|qrTI%o%Xr`Q>~bSnr&4jXV|I4UV*o05qdpo2=!U4o?;%_@>*g!f!_AwuM*D9`yTU1kRifcGa+lPG
@*8qIdf@oluWrFXP;W1?A{;_@S!St{V)RDig?c=V$=@QgS#%a9ZD1Hxn!5cr`9m~380a}jl<m<kzGPb0#PYTSaeJ`WpB{*4x@OcUO4Z^He=o#gveujaZa?xw
7`gm)AwnFs$+!9<z2w?!I`2drnx?pm@$oB^sz1g*yi$OCamOoU^#HF^9IZob0+{p3*lz5aUlg#e^focE&Ty8oBEl2%^09*eogvKg`rqHDz&IO=%Hu5x1?$4n
W8zKb{eaRp0cL>F=WMR<iYa7q8ug<kgqt2Zdlr%|6Vmc$m+n<20uGbua|oe=B_RZK6vw58%BtvGcZ#9XubUQVg_N(yPRqs86X(O$c5t;3J(wh#&s2&_yw={=
Ra_PSWUoGtC)x&VYR;1}7l<Z(*BjVUNJ<JE%2X_r#pMsbg$4whFXCrF5n<BJVckf}9c*Y>7yFSp+eVSb$AD#I_Z=W0P|dxz=xCvLqh=NCt!#pWiH*~<27p~?
r>~ka*Il!aqksSHglDfxLtHPw6jp8w$d5(>p^{z1N<Cecv=V|9yYJ84_nw7PNub@(kWULoykU7=K9Pp3D^C7&_<D0x^TD5RcHaCggyQtm@#yXAdtRGEj!Xdy
4J@b0&idMyi#9FDIKn?bnRF3t^LXPtf}tAF-Oa4AL17A{eH?h1CstA``rt+I)2+IO*{eximi7(w&I!Eg4Zn-5c$FC5p)0p^3$D#*_!n%6YeEos#)_=;i?!5`
#jHXGyM|$%t^7^ekVK6&>~;p^PGG>D5WpN|2<Pt9O}rlrx?d{5Fx*UkA-+m^WUe)GLLl6X&i~3lFr;f^enlLgkP#FvceL8<rMY?$J4IHo8bcPPw&93F`5(XQ
JdgQva0Vw!rxABL!*|Wb@S*R7kyC(-A!9k-S8@YWY`yj<><>N}ax`?6=2u@xfeTLJH_Q<M48#db<ieE6X2eo3)|zqPV%#jYkWCoJ?H6Hn>!-Xz{zGn>zJdtp
;Z-9}E_VVwW^SEE0J3>`#R%!E;x2(y3t3%{(Nri0TK>)YR4sq%{&wz;mOk}cWXo-r*?C-m<f&%+d4S%%L3N6-whp6kr(I{yo9=JP)}q16q&td@1kkF#AW3oH
#z%Qn-_Mpc3vqz^5|WP8r9gxfrY&;Y8>ErB?%$93e%*b2S`t;A;YSJqaOPs>x2``TGmI~%Xa9rgupj=RJgXZ<=z&w!p43av@CdL@tMOgfMs?Gd6ma1VBWyJl
eJF#){P-B~)kr&N$P@NhleK^bq#SO_NN+c3`GmPwO{PkdnS>pt03K+9!M+=r;&&VF7D9|mRT@`qwyQq;l(}^L!;n@6!$2-9Xk{sZdE{v71!to?(r2*rC%x$F
S+h=fz{8d~sH@Bo#h9NLbJ+D#>(&?RQMjMk4}>K&4@6Ni-Vy}}WnS!*5H(r^;6l+ke(4i1McMlS0v(EI`zsxTlItRYt7b=x#n_!@^Wd6dLFy-#BcAyTXJ!3?
+CboA1V7O{&x)02J(@?FBGoeWv9en>P1`(8^96>gl?*<H7a5AI`E#T<#y1IQZ!_v0BeivQvXfQI^yu$7zruGXtPd;6*-SDZlW(u>D~K*ukl?m|=nFAEfF{=Z
OtHH>rJE>}@3wavFC*P~Jc4Tqb9nFSpNhMm!>Rlo_iEcUYzOUb-v;pRp^q9=n2*^)mz+AE7(r%3a>hDBM&$Sx7*&r2#4FiO9pou)B?|*-z*r7uZxb*`YI#!i
!U69d17rJJ$g#}0Y`<3RH+Vnm4~OEsWM`-Fb{g3eG=Ff8i5IpY=Fr9W^su?g6Uth}0KO9k5f;vxu@74pgXec|0n`skfkW*PaJiVJR=?y^Nr|^_(x++Owg>m9
I&_GEIm;hj5YzNxf0YpJ9+vXMZ!*2YI0U5Ss>xz%nK~*1=(-q(cisN5(NA=B*)`L9gptY39}ac7rOiGfU=lJ{i8JSZZx4SZi=8+5K0Hf%wYHcH408y<Tfr!`
5)zk=NjWMqcxobL&j1@MJG0*6<kaCJBeWfHBa-A%miF9;gxr;$b;nk!zhQ8jP7@mRsGf9mK|tLewi7^)n8$7GIz#}b)chJ%VH-T!n?eG;<MR70`2;O^v+Um{
Vc_|N8zzUvgGy2l#CY7rUOo4+y39Ce7x9jv-l7G3Q{u}9mu!b24y4x=v;jo97-JSwkgR&gI4AfdKU`PCBQ_Z=rOMe{(Uredq?&MH9PYb0yq&s`U7cEajPmmw
tjCu^vn_&-{PC=tTSa8mon=8zjRSnkJ3Rmo*Cbsz!5}=3zlR3NKQihfR%G+JA`QwuEoq!X17iHP7JGPE_U<d(-GO3^Y(oS2VDcpuXln;AyDQ;K3=|v8-&?wv
mU$4jdTh7MzIINb{)dS&rtnGJvxRbRklh08+Edq2i4{a@t#S=!0w5{ijgsaW4W_fKFg_d#>6>hMD*yb?xu3dqPIiRyEVmXx1VZmPSs@x72+E~YZgL_RtR0#3
nuZGCnbR^nvojzVaR%KR`2V^kxnq{^*l@$XiJ9t&-^E<A81EV|-I^rY62#rCwI%S)$=b}z;88B7AR>RMV8C%>hlU+++XBaiAeU~vdtIm9)0<fyKzs^&NzgG4
+>eoS2KF?+*L)7$k)dWmfL@^QI7c{U7ZxrFcaB2@aK7h0KJz*sZHPZWgtDq@$E==ToVW>*TcOYvTH_I_?k*?6ok=<6l!2|t_+Ytbdl^cS!fZpi#RDVcSZAlW
nAHdGGjHPPU+{$shyUdDu`BLaa4kV?xVr0}$oprpEBtf`FGxl$HxXWlo_KfXpjuE21?qTb{`RmzenZDG9smw)9pK`o_3VoMA~~E@8{HK=J11%>&ASEemZR!z
k)HGe^ubs_TG=Q@Qfm9p-NkTp7%)ZV#UiaG-5_d0@%ZA+nh5_4_Q~f%CKVo!BbLyBcdEr&qyTF~4djLg;l{+={vLMBQu^Vte$}3I`6BXDr|wPd-?w*4MOOPX
Yc+P^`npsd{5bTG)h!`*w2b!1SbOr4*p%skh~L$$oQ657frFXgWCFusoI^msr~K7qtl(1jZGX8_m0_~d>(NsXbG#RTY9%##ECG*55Hwi@o#imjO|k%#7@a*p
k;Vo@<E?1ep8%vh+g&5xDQx&vCu7THlO@2wl%|<?5myp2)U*g!oKd`jvJT{ed$qtFi+;U3Wn3RlY5S(-b^E&3sEP+V03h#3R=+Cq<A!UU!s*_-OKb2>Ivl4}
g+sS$;-aUwws5g{elyu`kl7G@V`Up2h9YA0=`}qq_6|v_#uKE&tzO14Ldn=$YblMSh+!o~GRMP#vVL-`!J{VVDz4pl%OcE;U<kd8k@IA_CQgIr>AXE`v8DC(
N0}1>4bWqA+m|6%OlVM{<22G<kg7EUc~XHsn?g@qImO6X#*?2Rp4G4QE`vDMy67`!nRkQ+^G^cvu#tvv;AK&v6DBDTKGRkw9ls<aG*zQ1m45SkYW!YL$fFy7
_6xDfR$RZPJ<a?wdJIHRu6jrnDI+6KmfMR8)<yU%d)W1rd<{JVgrT4BasvBlf?yj?n5R1YLqrq0pK4#lm?BV&)SxHx*YFDXQq4%8O;YcqiQJdKudsVcv<O!k
wfyYaQ}E*E(&O(&_yPAaEJ@lilAHM!x>kb_rIhQSOCFSpBP-02KgcqqHtiswRjb7t_ybC8HmSHE6_*rLVYacgG))lG&O^Lzr(ge7k`%9KObo%OZc{mfcogGt
9NrWYaPS??N_rnWVaWN)dBf<WT?%V01ZyeE%kqJ3B_)x{q^ol0xURbLIz8|QotjJ030ZFmvud<7_JQ~|H8RJk;j9^6LwKS$%h<N1fDJ&bITZzyhGBLRm*Zo$
XPy0c7-mV_-%o|ZE%Xj@oJ;_di^j3bFl<5yH%G7=1{H;p3&aPYehVb9NS!FU5VvXsb35_-vP#1a6)@ijJ+;R$bQk{#LGEh+*=v;Wt|i0*OVB`(XI3l*?!}*Z
hUcG<GrTY?>m-`VS7_G7_xQGAAwS^sNAoDnfjV`2$59g!zvs{E4VA`@Tt@zY0-S_}t}a1UrN-Rq6YQU;Ni6G{PY~rj9A;u6l~}XM7QIUAtg_qyT~uM8XforX
f!tQkLBs<N)I`x<RB!6jX33>qrXWnk;?mfLMS(JTB>_9OhQ;I<r@&2a+jVQHI6)$j(Igh8Iuh6RfWwg8LqnuVMXgjVNUZG^fFKAr(*33Z(6Q}&%7L1O-PcFq
9#Q_-2Q_~}Gd|7cQ^2+tKqhq~L<v&2Lc&xsEU~<u%4J2o314Tx1JPHh`q?H`S=Y}`LasHt(m^^O_3yTZxIS#6#jfq>Dg?Q%utIian|8-<u(te<iC>c~d`Icr
O4+@y<zi>w_r*_hh>mMfvg{}(^c@JK7gmCZbG>!}Wt-ix&@h~tnP*2gj4%@sFs&lJok~F@Ld;;L#<uSz2xvf)?mk5)b1l|ZbkY}qS}^>5zoUIbgmy>^3p4A)
fL3WRAihVZv*%;Xp%$D=@iL~Q^KC9=QnTQqYLR-4O{!BC3*V%{|LvjD1p!FfHEsEQQj8+$G9$~sgBx6l_D&Uflm<SPg+n)Q@5?AjWy3btm7kyZ@g+C9^V+uO
@j>2NRDcQTM#-q8pfrWgGo%e~U8x|wYwMNHgn~eCTnch$5a3-}-el<S@rpe($!N3Zc^?pRkEi%|^}sDnp1*B(o(v(j<`_?qd_21&^CJ0SkN>by`z|%qS_Y~-
snIQImBCn1IT}L}H}+H6`0%n5h*oS>elTm(nV>khfqoX1C84c)3y?Fv&jcCUiVCky6%$@03<K(rZd%4d)lukHb!F0aM)B;8(5gq92RUxEbUllX$tqls6TncE
(=w!+JOC<#-t>A9(TueQY1hREvAJgm{LSOa+m*S1mIxM#l`Le%(6MY}-Ohs{Y>mrU-B(j?{@cN*@E{kI4)1WZZ^rz^xT9>K^s%I&hKl0}zIF}#F;X5fZ<vZ@
0V7VgV97^#y`=nH#d+z)JX1(t`9X_}LgCsZrhV5(JDEy5%qXbF6p6NB%{cRKbTp$}lQ)>!i2{Di-HA~lCMI(;f#cVK>WMY<0A~3mk@>*>?<|b-a8g4InwSOH
QT>yPvaqQmO}Uad4zqVvyo*U$(xIy>8s?f1&{>nP5J*<@oeV*K*f{d@J)DW>NgSAAT@2YXAbuS1j>_wovzFqfZok$rqM*8WW<9b)cYbN*cvM3Q%Fd|xqFx11
#Buej@czh%PLlTaPsVJ&w~~{&f!m}IG!3?{=F08bkYTWcL6E#Y)&DwI+(XDTppR~fa~UAAcjnv<T?Cn}^U_PV5fiD$H&)-%Jo2ut)kATg-j@POZ4r}&vu`9c
1uF55f{K48GCjOl)xWQJKIib_CGe7S;&m>C5AUceG-_*Fc)5CEd|R1k?e-&m|8d`SQ*X~SnoZ3B9ngm~81rT^+_MntlP9$0xi8>qM;i&=+l5K=&Qvzf*zELC
ZWkvs2buqdECSR&0ZtLTR0tby;y&j+Mu2=n+Iint86QLCmGuLa=sf%IbYC*mDU+Yxb`=D^73EPt&}$a{k&~}qmd(qfAFUzMy*f`B$Dqm3u%9OT7AJw1EgtZx
#hA>;Pw}GQRg&_Ofy(hOj}iNy<3q=r2Pt9+$QC7+2MBe_o80TQcWY9fQ}Kq6(64t1qYK2VvF2clou%2-V6V6vc&k>m0Cb5~8|km@)?^Ica|01^k(WS}*u5%y
60A8%3I-B)BWEGX3sP%!C+H6Yue*~2U)!;2!KSb5=MaryM*B>Ut0v!)L-xO|_FNFA8!H^iJy3WEf4o`Ssh$-k=>3wue`2*|c^*hq`Q3uc;w1T&<F1RWBjHS_
up%XbZOz~ydl>VWp80dcG)5Q}^2bCmM^mY36Oi!;o$b_#(44h(F#Y-N2G5I=*(q3i0>i||72OeAydxVShZC)UnERj<KpWog1iDlVgd8~309!|4T#+>mH*e$;
;68~;w_%L|T}~K)s$kGLIC1g;4J%762=hX+>x@AA1*>>q=ds)O>|;k-8#6=!iMmJM92bxwDOSu9{{q+5pTY_shmDf~%o~MTYgNW!`khnTtgp(S9nqcyXxCXQ
`wwfE&)|LbO+O8-uPdfSsRYJjSbIq#_*Rq+pjFoo)oFtsFWt>{J>exhyr%7@P!$=CB45aF^Z6+4;WAZrbEE0wwVNdURzDeHQq*|$?mUoK-%6x}c_RafJ1z8O
|8aOMO);23%W%oCPtBNGH5k9K_M>U2zf2V635uKHnb;-FXmDvdeq;AhN=Clr8)#JM-n2y8w{_4eBTIiR8_SCqa!BiH*~(dim7LNVDZP7842yCuWK~`Dzecr<
A`9$r8beD9HcD@e$$LwmEe1M#-4j_6MW0T8kP<>RSZA+}nH#ZI^ukrAxiKIr`+ZYp3OlcAc~Ym?Fg;aw&du&lqWQJ(lrOMMdvyqbg$YoY5wFMx<bWY8BQ7>K
0zgySP${A{fTzoyVI?c+n`%uCgnGa<ovf!--Ds9~im?zFQ1J|{8UgpYzw9^kR)h24K~FhKjwl~mO<GBa9J#?q-PB7kwi#;+up5YzXJf@-=@{4!k5;O;69&PO
i;8uvEu4}lefP$bCb6XrU&9?V!DnChy1!yxjz6Vt()0spkoA(XVCFSsH`GNMi&daceqoH8WiaKA@i=ni=^J;$jCc^%)|GM6(g?4XMFjcJFXc8I;Pl1@<Q&!-
M}XdS=ZV|ZXzd5%u;>OZ{=m4|JBgUI%yW6AW|y_9llzV*sR6H@1TjlJ%0NZr_;W(U?tcmcr|H2lE9<`wTonBx-c^?2#@y&c#((dl$#KjbFK8We$Yp7vUcL3=
VeIqx;MHae1)6!U%J3Z~RflYy5Jj0Hs!!u)ky+V$Xtf=E-N&{Q{~4pS`n=_u(PQFMQjQm{&IX5*(36G%ZximG`h-waBF_O9B*+9GDj=`QDtBOO4-R|jU`#Wz
IP*!{o_^M@32>nb$Q7=kr_^x&>6<3_gynJqv`HzptTGf7y(rXY+D%jx1$HL12D@NdE=f+UBbY9#GtC~(Y=iVixC6WjRP{||sTchgtPF(uh?ah(kIZL7F`)cn
g(Ii=5|$noK<WdII!>PF*w29iC#W8CE>r;GUr_G0g|x<n)$S|Eu3gl9Pt{BoAO0aV55U$2QUwsxOmnVEW~?w)ZmrT)K@Q*n;`nX3A|z)4nSXH`di1*{Jwa58
Nwfg+HhAM=!6vtfeds5nl2`{6%W>yNP(0A1j6G2CyKs9%MEI=M%;2;rIsTR1zH5RRC=#JFja&q;Fq5|xYv2;H>h>jm*ud2r^C|VCysI2nU#X-<OY-!GD1(BG
9{{kM%zF>VZOwh91-lK5H5!YmuC87Z8_J{ZSpXTfvlaoU?>j3=G}D%t3I-*358&*ogzVE%<XQq4t1Hgg7c(r!b`g)|Gv*AF5p2ixM%+}trfgtBazqDGncvSV
%+8rwqlifm0v6guPt;kocWNjc7J%J++d^}LU>ew1-!u(hH^vtXiMG}9fX4EJK$FvbD5s_log&*skf$fNVx(C%AEmsVwL)-5J^KGQd3^JaPA1}FJcG;BD2^ZG
6KTzwXo~i+x5?mOf0GZ7v3cL`^&_n~4hUPVK=-5LFzrQ?+i+)=WqTo#-TklWZ>9ozJ}SB{DGN_pEVfGe*$VK?TJ53W!HPZp&66c#5qpZE15JV>;EIyH!GJlU
>_pI3-fgx-OP8g)xCmm}B{WNe114m_wrAsN6d3qXW}Ie|>-qGYat^tzZrs=e%TB1Jq4znvi~<b@>@q217_P~8e?A_*m3QHiYr8}z=&||IN0~;d;#Ouj)lq;w
bYCOI#kuGD0=fjsBzD(s6VQi>0fzS#>=~@t8d_;28FgL%rBFvI8~e|Uo3y8KxeRNaf*W>!4doj*bwcsdi}#YIMB!!Z(sK~$6Yb#z6SZ-It!e1BvY-_q4|W{7
>+^?JCr{ej#xu~Ftp6+!yA)EWo+yVCqG22Rk|X984uLzbZRlvtH?Bv)k(}A=rWamp7u=6B{>~cP?N=4|3#)mu7V?cg1k#3RWREw%9yJK>^Hnfy2f}3il_wGD
do6O`tY%??>jIZC98AUUVOd{xElq%}M)8Fq1F#mjQv#NgrysC;-NjY$Bu9~p?^jBIPn>uaJvqJFN_kE_u%Z4?^99a;gfy=PM_nF-oJxG;oY1{rSt=%R2}qg|
M1&y<zzVHENyc<X?8F1$icDCjy${|>-J>HWMNeY&JpoG+&U;coaXjr@t{|xds5>GGlBNhi04Px9=|1;TUr?o&Gpwo?fHqy?7kVM*KOBrUAu1t6U2-ZG0s?Se
$5Hr>Oa%2JnJqD3QK=!ZOd#>c$a0;C0X?>H9;ZV)en>>i;3odOZJSlK#6Px+qj<m$4;_J@C7sDKJ^dBRBci;A+ekHcu9)6o>Ie`HdH%k;mf$XNlVl4I?$Y#S
)OE?q>t)K=2w_M5QInGH#}<QSr~Tb!&@Ymg#P~~!@s)u-{KMun8=SL%^*?!mx3>#0VTKr(X$s3D(vzbd=gtfC!2NqLQ46xXOYj%jP7@4JOfEb5{k@}qA=h86
%wvac=<`JvGj+1gVe>fDz>f56sVoQPOa!qU%Sc}iV+)yyY66F%C|B|2aqVj-2~*Ui`~LYVl0e)~__IprkJJDUuN9?J$amEyoT`P!dO@_DPwm7iR^&PBJ2X7T
h-@1aJ={&{M{AWi^|CylB>m@z6qb2=^PSty)1ds~>K?pYg?-7_oIWnS7285lxG)uLg9XJQrjvL&7f0`AI3>=-ckQ({qCV8V7llW~X;KgUluc4P&|dv;`-^SW
Ab{y-xU`1$)zL(X`iOv8<Z%Otkep-gr?XVmK#>M;JD7eafpfC@AIjm0v3i)i)X*MDZ_2;1!mwC@uSe2S)QBYtF*QIL%f6H0g}^K=K<e#YL2H1aBu>mi5LZGd
;i;HHT=_SLZ-sK?wLnKM?WDmthHE>|@H6if?4YIK#=Qc>7MT%+5UX5cuuDGEUgmb}3+i6$skdiH36^xL&|Vfia_im>hC%iGi@^B#!IZOcep=NsK%In{S+ml^
et%wsV-YzBbmtN3wGsT<37MAIVc{={z5M_RxyFsyxRqk*R_JMpR6dU<Iwk#pZ#kC4n%(}y4{8n)`#Apk%I&JB(pBrpPX@1xqM`P!rw->80=Dydl$xRBOhy_Y
vAaiuy?q8*+8c`9!Cx8ED%x!0m2}z-M6TlEG`&J8GVUiOMBFn##us2ord$59r@X2xSH4?&+81D!?wJg+3a~IiGbS)O5M}*|v|>O!<;Jr0`2nHMsG)#lm`T_X
UpOWgcydxu*@D-4t9?@c9g?2u7Jo?MRP&=QRsI`%4g0)2kVCA2k$E6EmT;h_sPe4!;Vug?1Z1ids*P#(@$Q-Zt~qAtm1u10xw%WyIN{$*UY8jOE_65t#MDA~
CWCw-0}l>}0pA;e^F{=4u@w+4LfvhNNwwTZ%^o<o>QU!~yJI=qHdGAhV`CXp)Qn-h!2Ua7<9|2rJr`RcKC`1_?_KrZsmDRD+`C1Qxv!EvxKc4s2=Q!PWL!0B
Z9_yV+bMQd=yxl8D#DUx9IqzsnG+Ff3(n0-wO5X>11gTypfc|T$P5@Z|D`T0elL+e!~$V!%xcn!R3eb>JxTl7{K$yMLcv38A{wX8Isi?AR^<XAt-+jovIK<l
prcR;#`j~sQu`iERheeq{Jdv=_YEJNv&N3oVD8f_fG7*jWIwsvn;obqI+sVz-aOxPJ)ar3uRug8#orFzM-(SZ@LvuY3>&2G_xLXuHkYjDMMOKKB2)2&w;n=c
4vG5vduJEhUkL12A6A9NKl1V&hWW_JCwM7di=7;tSP;z%IE2;(UrF(ufOW%44ptX~#a%-g$YzU;Xz@ZQ-q^((dFc&d)k55IfpotS-E_J;aD)53;;!Ni{f8$`
*+WsBM2H6`xZNG(Ml17~n}tSR26Og)04_9n05KVg$~I(oAfW>rr^@C59RUQlnb4<UzVV*XpCJ5LcXB|n9F#v3A~qeaqYUmUya9VvIqAj|@LE3kL&jq4GUF<c
x<jVcRn4s4&G_WTQiKtK$lx-t(8TNai#_oQEK}N(pL|XcMEWsYZKw3LUsF+4;r7kckq1}VLw_-I;*;=^p~ik)UvHdWN+@C0xQ^)U;X^$-C2)@XJPQL&|71uY
ymZl8ckC+vK0FX+?x#NLq9rsIod=D2s*;sFxpmvO^dD_Y(K9(q@YH%gOg;srXY-V!t>V*`$Y=9?HJKyqxn0mvIPzCE_T@)eeSe)K063wkaMi2dDCG%pDaKXW
kSb<(<~Q~H$Ya243%zjw_qlJ*F)MzvNS>21!oC~4aM1DKMHhBP#fED32UN@o0VIf#{W+4$-6u%qHolK@k3aOStD(*`RTljh&Ydn%^XvV;5~R+P$i`O`@yd;1
d4beL-5HH|+(Z`^u#tI)*LG{}9}tRjr^8q}LDbDFypTYdJI4jHg97DKwgH|8H3y%L>*P=%A?BeQM7qqe`jASOv87(<opa0~8(6MCl4uBPQrSIPfKB~uk9Q%*
%YDJOPrjtCyPt4HSFm^TPu!dfhg$B&oM2{^O)f-nj6q~1)uAqGM*)!&PKAUftj#@q*CBk%ed;fg+@2ksC$;+qBg7`>7D06o@ROO@B~y8)Do(4dIH^Qae|t}|
h_})#<Xhb{)!XXOc*#(g8=ChrKD!BoaCG&RBuKBRNB<EEK)=(Eh*&*af_py1Mq{Q7Zm*TtwmQ(G#LpY>(0Fq6-hZQT-L2G1o_2fjrRIpB9u*vKsC1n%P0lR(
Sos@xpitgt?LP0R-kWQibOqX%BH{U~X7S9KJ22Dm2s<+Qb`EvkXc~7qFAL@kzU|${VnxOC1}pP%$KV`^R-4QxUTuhA!p+&;^ec>eu}?0`KHmT>mWiQ6ez<hS
v-ehRQWL3Vx}nxJqu>$*%~+iTu}|rBZ~;M-K2O~DoUfD5qIcHBmJ#M`%8WZ|-3i3?Q2^gX!u8H=%t{TDYVLKozo1fKs{~zRmCuK(HDUKnZMTCvl)75nlNIUg
*c7I729JT5gW&}1^TCYt=4OJy)p>al^t(hj_o5M9H;B23H&+mdU$r`&=cxt#cq#ivDW1F5`6af>E4T~~Vb9=rEXkjW?KB%=84sPTuwx&<W?f*Zo1<fp$cx|u
x__>vk|_#-;4{=}JRUM+aRe5aaeQE_seEDKJi^+7rbO{e1tBz@+|UAV#t&@IW0CF$4o(JqyaSw+My?FQ)A&ynz`%><*1zczNRL*&Cgn&_czpGU=zmdl2$W+`
aPt}I=J+4vW{!>BAIbc{9(XrF-j85k#dn)Jy1gEIDrP@dl@5xlRT_*}@(nn7G5*OIUP+u$L$p&e_MN8~aAOe2iBjk4d2<F+3adOygA<>k39G{)5Ub=@3O>jr
T@$=RmP8^71>CQMXN<5#!6u&$fT^lE{aEy60C$pOHnYfm^O>V{gM%7+8QAR9@hmZx7dr2CjxJpYVIFWwi7no;S$!u>i&DR1j<V&Vl8F4tR^xGPgnsou<0Vg*
sL|fovVUC&s`Bo;=d#T=Hv=0P1Jrr^1pA-4dt-xUuFw%0TL0klH8NXdikiKB?qgDH&c9!v195CQDLgTaGE;&INvDTp;g@q)p}iN%mL-CPt_Vwbifkved#&vC
-DAIklS;yHADsw~Pc2m**fZ<zzI+-5JxGXW6Atc0S~kQqp=Li8&&&w$Y%qC1SGSK`rY|l?gjJidwr8WJ3VhiP&TwM;!ty{M7{!Dk^MJ)reUnvVeQ>)WA#yNA
_Yug~r1<Djo)F2=c~kv+D@3WFss<UB3HqUhpOLY-1pwd2JsYixpS!6A`EAqDIK%5oVsn&;SYk)TY!HV~ej5|9!(tX<3xs>F9Aac~*{Va39E&zKWP3LJIj2O>
o$-qJEYa%JOMqylzPt5BE{a+wf(3ZA(&iZ7T@B1w&pqtI+18J5VGMUKMDtON6O9n48k6t$|Nk4!f$H%h*W{ZZe@4I)yE++7{2Tb5ffz)$ELtbjWb!2boba}H
lw}zsOz1wO+V4Lw;3MV+p&{xRr#!;EbSJ%P%zeB<y|~H@eyD#BoNi@^80KL-1kjOy*gymkE6-<(td92(L*e~jky?OAW3^G1dc(xJ2JMV+92>jw%*pb>4ascl
9NCsouJ_Wh#Fx-9YL`i;_I%nm0fJH&kyW4vfNA0(A~Qp~^`G_90p<5^3n&kI=HmI;mbtX*ltJ%)h2pUQGLnH23{sfRfVs&OpLT~frESCHRaP^t4BfT*JZxKZ
iiPtP3opFg*`inGCk089+;Q1-<#$ef{~xW*s*K~t;k=0!W1tjA+K+EEmUZAo({wd0)WCb~$r{<qwtr{UoN+=d+7fkyPq#T#q(@4E0_;nQU#CN_^fZjb071Ij
^pdf6n%jUx%x5<7ViI3guqX>B#L5y!(n<Q}6R!hq^WBj=$%>EqdWyF$&W=>?Xzt=VZNj%!Ma3({NnH{9YElp&A(PAsJTuReNEDfxybGMVHw{Im{+e76NioFA
4Z56~rbA&iq2W1%Nr?((<S-g<T*zdQsNZcdbz71jKQ@hBAuDq;=J?z5%Dy%;#Qz+Z#UWowLk_T(&GZi0u1f|gG)lS@lsIz{f#Z1)T4bBs20nLbiXli4{sT7c
j`}y_iU>uGF2iDr{IiS5)h=T*kf3YiO(q5AP@gc)k{<F>EG0(VehU$KK%_3Ap*^zVww53U3b8rytR+W7!CM-DcyT|(py3Q1MVY~4;Yfe5P@wsVYFAW|bea0`
rOGq>Gp(dt^dAK)-tR5jCfAUj=|()Ixe9zE<ImJaq*TpcpbYDkJT$tb66U7cg_9s`1WBlp9tTe?6h+pPvjXbcPGKbK_-GTSC<|kRD_94^$_$ubq7?{fio}rL
U;2BfOjmraofTMah*b#?WKTYdIy*vyss*sPc2_x42QCayUg5TKRAg6LJyClie{gEES7@~I$`EGVyA9jf)IC3q5iFfCQ94y@uwMWE($f`k+u_8eRnb(dj7jzc
@Cv=whbqY7pD(s;f?%@Z=DXg!Pr4-gh8<^fwzU{3@zkM{fpfbGkgMilVQ~i&U4g7cwFF#~s={;15}W+?_+J;Iwm>Iu{M~XIh62ccdxMS~4S%4N?;3Y_N+kxb
;4Z{H$p_|h#S5rO^6%W3LkBO)^%;K1nyA0)9D`j|D0|DKyQiuo-+6Ro+n<}z)E#yYq4%O;;dWXgsNq^6F9MUvUF$T`TEool#t=plIP$2cJ@ed)t-2|6ks^-n
dlu_>-=yzf1n-)KhI{PVq|r)};U`>FqPw(mZZTLU7F!4vm4;S-gjDA0J5>zOSp)f6ygsy(<07Q^3lD~a4m=}=C)0XPA@@p3ugl1mc-*&CB&5h(xm==Of}dyf
?o42+IIImAtHP}N;~jUJ?<JbIJMBkP?+h_yMtPhLwT!mY0Oa!sYpX+|vZEd%ek0u^!^s^|!xh4;++Jz>2HZR3)Z5CBWeo9GP-dOpl{9nRE?r(S=O)GV0_VtS
{ti0v@~C!a^|}IL+*oO3D%<5K4LBA=n1pKPOTA~l|A6wu0kIRudWU+X+KAOGP*44J*#gsr%#{mjR40nhcY5DgChes&QnAQIqwS|{)&38gLCC9n9!ej3!E`V$
P^E#4s6)@wRL7bX`vC1r7a`+2DiepEtny&t;}zomVQ8q2;aPCfAR=)j2xJKlWm7G0>+ULyL#7vXAUUt%a(7$^7TvU*6us?fS^V3-r5j@tLpUhkiel9`d&?C`
I>2*B>#-sP*Z*^9DdH7Ry(`jIWAr=g&ddRH7GbK`_am)bP*q_jVcXKehLN4P*33$VB;-R|h<s(PtiDz1q}JSMms%3Co5Ce$K&`>g9R?`(Lxr>b{+`BMFl0||
5w3CkS$#hS|LZ5FlhteHZ^h>dYvn`K7a~Le<I1powzOs7tE#;0vJoV8DYzm*b}pqJSO}!*j)l-jx4(|%Rq(<{gestPdd*?(2L!)v<-wYsm?~+FFyzm)(wn#_
B~0n@e-Eavz`~Ovi%REufN&jHZhtb&bdbz}Wi2>t)Mf{x6%y~fgX%XIC}+mO=|r@Z*G*GhQu7jt2~g?w;4W*%Z4)|BrBj*&RF&Lk@(!lwbg`Pfn_9{9rzp~;
fj>yjWBq1P1ZN~Pclx-c?YyQnUUnne_<I>jPTMyfS2J^%hw)#}6cA$mlM<&AxBg}&J4m(n>O)7itehq12FUIkG^c-@qJpTh=H(xBH+BxJD7+_5jR_|@q%7$d
_|y&}C5h3-1cdsTN>17j4kUiH3w+)0{iKSni$&&gi{Fl0_{^iM_-SFwk{2BHIA@BsKQ47~(sqifO!NEFF2H%JWux0o2+KNnBRuC~snk)~Tf`<8ote*`o*y#K
EH{UB`gbYYN+m?x5Dou76|9x9d;5EEM(PT5Dtt(W?2CAdZW?d5s-9X1g{ZdTgtA>(fp;Go%pD9at#29kkwX2n3x~%jq-=r%8T{If3MmKFu$Ck)W$W$N%}2X+
I&1DHAnCnzp9`ZaxqTkio#Q0F6RSW<F4>3mo1Z%(aDfxI#$*skL^#Mu4)jbVE+wgA;_OUmI+n$TQ)*m8soW{~*F`inT%$1ffF{s7)>(SvHE<!qaJ;0(=GZvM
5)&wiR$S-6)h_ombp}ht$s*RZ!X@6jrPDUEglxX0SE7)xfsCPJniv0M52gFgVFj4?5Ct;6H!q6fc8cbuqupl##kx-G6N?dFP|!4?!C%^nN<HOBZ$h6BS5ep^
IilkCjw4&7#i1VVhG3)Z-%7qv51*RD>`c|_4*Zsb4_y-k<t=LS|8O1GPt2d?hy}R2&^k?3uOcQA^qVGPmcDc}EpHe)_;RSNH0FUJw3KkQXIQ0$XZYlVK`r>X
tpsOUohjv9+jC9G2h0n4S=IU`8}x+rMI7$FU;YO39x>&<|JVnd00D4+f@D=y;zf1$f#%4#zBseSbY5H=qPeUSr(i@#cY~yPriZ|zFF(g0ZAS^6>JLe^%U~<+
bqH;yrVgAKOw!-KSp%@es_Na9<W0lRX-k;gqa8~{1BfnZCb~<y%ntCW&6fWtmTZudLkH;SYh98~pPM?Gra}6bE2Cl64m=yFbH;0JV*ho%6)SJ;YM($WH>&Dp
p>9QkG5zBGN7Qj7Sgro&yCc^EfBWF~*ES(@17d^jPEB)fyaHF~A{crZEXM_tawL8O0Z2-LI>Jp?EJRXU63i*yRSD{u^!Lf+7CGUGa!<7jtt}IPYmd=MhK>+0
I+?(VdVXoj_^Gb|4kekOIb5A~zlFWMLW3zeMwfQWR&OO`pl9mKm&}7^2V6n^{YHr*MY=w;IdIgZBXicZK7iVvV;yR;*vpGrN(bp64+K!sBOO@ebm~>tc1na%
#|JP0UM%bbGq_JC(%<+dX0TOQT1?^xVLlIES3zFVQy^fLG*lJ8-eusp`&Ej&bG9>pFBYRV`jy%+7b;S)@`Yz{Y?N~;s(LM_8=<}XTYAnkh0o6dZ>srM!vOm|
ucXeB!z7kOpf}v4{20*2TjH@^<e)!u8S}WFQpwTt1FO4)AL?W=D@8RWVQXl8M~YSNqctuzGQ82*41+}Sw>GwO4S39y{a~UK^}V^A{p5e-ks!Y-i;z*d#daaJ
2(L0_dnyrw$`1L-QBGS<q~DO-7bgl)`j}6@*MDiVV(^%OT+mPxHK}nh>=FFNXa42bCVF)(3u{zoa?H_nl<3&q|9&QvTWnxV35C)lvpqP+T)x<Fymi*bSE&hE
#fr4kGl)$I7XT|uN_*}$R-Wq{aEvm>a4;l}jlCZ*8Ai>V7J(YkneRIvE2blI;vEG<!seQfb)-6~Y-f3o5v#9hAi{8>)C;qC(tQe_{65ud{!Ypi6JO7Gt2s(!
@UR0Lp{S2sjD~c3xoIx;Q~$Dy^pIdnAfp2J?gwd9?SGY~WH%>t%bWRZaq@Zz_{e``YD*HtfEBObNr8niWeK|kV58$@C<MY)hwR1hH{%aEhz7CPa63oG9pS~E
VR5}Ya&_i?+F{57xwd0uq;xSYvz9mOf1SlE7N`j?Z~W%JA;E_ZS8ED>YeX}|MEShE4{`TiGR}o*2tWC7HE-E!WY$Th@m90`@SVSST)UqKw#Wu7>}kr_3y8A)
dyFGqpMWCg48~3TMkg^b&5e^V+^hM|^^LzHd~q<)8svmvk`b=F%Q|iy%A<lb4ne}P65Gvc-~fEM;`L-{`9P79P|Wq5jHw?30ZL0LZ%&Ds?2>u0D9@e=x^O&&
=fURf9};MaAQ2;=the4-*D64DM*?is!sxgZh+PX2zkpRGdEdI|efruPV5?Ffwp2{3a)_$$fTr9{2T^W+OK@O5cabF>I@AUI-O@*xKY5aiNMrQAksF!cmiNnB
Zr|i8t~jHH0N&=d3v||&j>oy(>BKUVkOl^ZmJ%GS#YFDU^CgL|1YyZ+_MG20E&1IuLaVvTS3=z_!y&~uJ*g6YM(<)0%GIN+gxyUFQLvMRW+(CcemanoAAbTj
oRs6_(tRr!A(ixdIqK-k4ma{WF*D~yUG~YPmD^O+;+o{9CeJ!^yQlB`m)5$owRy{R3vMU!X_U~lo-ak|g<T+r9yddHl|wBelF$oLS>A@+ip_&T_Aap=_8-VC
?R90@ClCASJyhD_H<TfwK5SB^k0C(rjD?{1jNjxgf+)K3RS8-ieZdZseVXW(if-I+hfy6uHD*?JKBdS4{g_Z0IsU;+PUBdrhF(x9ROM?%i|K-FVsPJy=jOZy
EP;IUGM86WFfj9ASchzzgbCmd$dS-9Qi$`5h@6n}4lLG4S}~MaFDvbkn_UI6Mqab?X7)-8SDPF-q^Nf$@+>R9?dT1I>F5(npZ+Trz)S;vg*GDGGIC2B-AEpQ
8C?1Tz4}j?#A&&v_XPO}bE$3Mk?3t$T8X@ju2iz;(HpvlW)PsX2beyd=<^Wy@nqK+apQ@8QqszLTh*;-+EJ!$6-wD?^&Svp#QP@01x0U(L5Io>rzo|l-3T%n
AML6^N=COV?>>}CLzlSdqel`*;#lxGsu=c)PoOCyWEr~Uw;UjWn~&Vv7Y4e_l;uB=niX00q+qebSpVpuq&BFY+jA9eDMDwRuUqri#mtdRkA=;jM*5L)A=_T5
EXR02cxoe2sQc-lY959Wn#I<%z5dYnSpPBAk7}YcRpIfoe}OY6K2#CN!gFUgU{n*qMQE3=?J#V(UJ)3TRWDI?GV7lPb;;hK3zr7;8sf=k3jCMiA(oW4;Hi<%
-)PXDry4joMtD4X5mi}ziAS$}L7<k$ND)IBkd2QuhN=dHHCeY$KKrboyGU_Bn<`LU@M_^cn}r^j!#E?zg*ecQ?+*d0HI~VEtIAvcG$7jN8%OT^G^<yw7@S-y
2{5~L;fvOp-Cth-EWXswho<td67YDnG{B7@o!&)=se^!!Rb2i+13A@$uWq;;2m9<FvOdI6K>-sVAvxn+UF*zhzG%9Lm$Ci}rQQG6ZXm@L42?{Fpr{S%3#bd$
MrHfF4|54KR5t>leS<hEb!KCNLiBEcor642Z1u&_G1G^Sx7d|(b)e&mZgV}Nc>6Na!O5Hekkg0lPfA%g1w@i3l1rt_D&vW;{8-!mzJjr2ms<g9|Dce6G6C_%
%wQa4BV?;@YrRq*s7&_XGKsC9g5c?&9z<IxltEY0wOuh?D)_hMAIS&BE#<z}G`3~b>9BcBh!=P&p!*AJ-}@an)zw3$H#mBdLGSYw<bQWTy3I=%fw#4SLc&7-
H+C|XAzNOA4%Yvwx1`Ok9iLxumNdA0!+QmOa)P;_*e`nWHM(UQpyx_Bg|23l0;gSo5Q6+l&?8FKuGgY#XJyOLy16xZ$D@64rcR;)l4Yy405MFXORboKNnAeS
0d4RtmDd!V8p<TDM-g&ta(;^RbQb7>QljS3yNW-KIJi^t*j`c~?_(n5!H$ekHUQ(Yb#T=o5Qfz6=(gxFRi{sbvj>D|am<a{Kd;z_X#aD~B;*CM8w97uvH{SP
_p*pS5B{RF@!e&AMvjdep9#y>Je!QtpsK$E&0Vu%%zjausTG4)&_&j4t<|@LEFzzcg#sM^o1R8Uu)Hknvf^PIR-Jn^PC=msb2R2zZF(5zlYVP}*hT%_n3Z6L
g<M5q>Hyj$XMM~S*+SPZ%ku;gUE!-|bQdr_v`E>|F?JuSuxo?H*)TZJ)+RDeQ(bNjH(yu;UlO3;jO3l995z(d*;)BDn;v0ID)v*3)LLY%?uMs%$sc1O;vI#U
OV<LIL$}ot{{d&#nUAK3Ujs3OE_qZg#*!eoMUu+zFWwM0`OBC^St=FAUTGqsLx05{9iZ#+`<c!g&KA3cx0iXDcMyP1rk`bXZg(_{+97{HeuRrqeGGpW9MVaM
)}vt}YF_}oZ(ubi?8iJFyf3BK>lSRnpygOvtD)@(p6!uO?1*cw6as##JHHC(Dngxz`>;djPzPQ}2>b(!pww)Ad*BUH%>`sz`IO+5zS)Y2+rzF=WxT(+!+JB&
KfOOvfBl&9%t?P+y{Eg3pqJ~{Jlg$b-OCD7z}9MSKYF`r*DkxiAfUt|qdT4Rq@sHjC^h`B$y2nN(Yf|qy_Pz|Ao-*7?^iracH^~$1G>nC?s@`iD75q=tzvgJ
ECrYU(8HDDt%QRsL@&8n!MS+ljezI$KgR5X(cHaz3#c?`@I2beUWa5sPX5YUUeQ9D@*e^=psaM|3Vh{8JRGGAw)#kcOj^$E;qv5r)F}`n?ZE`FW)GTkDE?`x
YTr%I0lp8Vsc(-@_m`B%rd5)K2^HzE#8aHx3)*U|-HT9K9k59;9y%2xgxBJFCO*I~;DZw_d=Y^E!5Ai)+36@EQtg}vyivnAm$!P!J&j#<x-U(*l9@cn6Fx<)
@}#iw1)1SGZyj>LZ#W3(fZpaP&+1I#o-S=YQ9ynW9G}Jm9?JE_94d-ixQl`nqQI0S>{ZMCoPxrnVZSbGe28mPixw(p?Pda+lL#P=PtL`pw-Ap*!)bAQ-EkHo
A?QfcYZvuDawy%i%z+!_0#8MXzowQSjlqT5laDi)#>Vg)lx-)toqTB4^)|U@6!!A_+v^OV+ko_%UBcKR2&5baQtcDg9pq0EB1r+k4our4`mE8HP;mS?*;=!(
_T4{s)KpqejjbjNN$@dx%-_EW+aAgYA?njP*$T;6gTIh{jBkkR&MwfDkaQ&)#!Q&k4FYKG{JYsM^{4fr{n-JLNuQ<0r;>1>)x+h9Ikb-rOoEY~PS4BCcyqA3
l<WOK)~cz&m)$$keG%Y<fGP!v!Vk(%i0`R~!}fDP=TwT9#!7;?Y|*;0p*ruzBG(PKOc|{k9DAH@p=*>Svmo^85K)te0_C$1^a?7>eIoXGXfGWIq#GD>3u^lf
H`uJ>wgx@%SW%;0NlfjF`Ls^PDsKe`t1L;mqtaRQ4ROvHuD74r(Y4)MF7D{n^AEO0PYF>GSR@q%y+iMCY@SJlnnp{-Gq1hL5<vNeLL6|zCBHL5PCaYaExch0
hi{-S<A|#`n98GrvOX+*qvV@R^8EClgD_$(&5NiQsLoB_k)buMMl+(gNV}Ro8zN8(RAbe*LHQw2@Rq!7oU_REGWEW4T8tOo+8ObWgrgtdSBw*pxG;sT-VjRP
!;z)@q9TcnD3DX<6EbL(6El%Agy2mtFry<4N+PPJ=VzPj#Tt@SSw#%Q(WV>y=6I@R24OvgQHnzj&&SA;G2cIvsYA;ake7cGhUC*2@h?x+tEgce{fzhfn37qE
v-IRs5-nH7!#=pY6<921441l50L$h9vy;>!wmSg}JUXMqsW3>|Exd%pF88t(ZWfgDB*Zgpz-^X21iAKU%iv5z?51^l9ZFqvJfk}c8t~RP<zr(4jZ^kdQryn(
NhRMn&nqK_AeCxJ_Yf{q4ZMvM;(DcL-Duqv&cq_Qj3k9Q$z<vs$#<02wWv&t)DY~E4F(%5K;d;*yS%bMT^sg%lcK$62M_FmrA^*NVd&k27tE7s{Cd0Gz@%6^
NA$+<h`9T&Ym2NTzdDUjHJqa<r18vXrbeLQ6(FBVd9YUN8~gZz^T7R9iM~QZuYJiCVw<d$z-}1xo?WPBfd2#0i>o?pEo5zz$4uz&BgL@+Dcx6WhY+{DDeFzq
8JH2)l|!PW>Jx#&=S3Uty(tp{ei?p|`7QgQe9s&RH-R8I8v2R9o2B;i&>5`OL-Vge+Txx#7obR4>tkh#nOIOlL%PFH?TvE=;26tXvnmQ;-->^B2vb-NY(A=o
KxKzYgi+ED?T;rcM7QzcH{9Jd1=a6!d068(%(sTEmf;w(Ss7FevmQGzGg;t~n_|G<T1eUZ;S^)zra+NdqhjQ{$(80aGDYOYy;Jx_&?3>y3*c_!mlyR0;)a*{
4X^J*0Sl{(_47$_*m+A{ji|fri<TMIR%I)*E3N0kRF%ag6Eek&H+6FXr@LQUtGFT$qrzq!P66J4ewN8SUXk1jC%?Ji9*3~*Bm(0b`X&gN$N-7G@dRGOUOaFf
l=a0;65F`{3IE@yy>{-&_gO>2(}sNjyT3JorrBFgS+X6nLFY*<w+xZ_H^TTRo9JbjdxM16j}q;M^|P>P+plS$n?FoQrh;>%@kA?jC+@zt+pj_S?zdfqz(;8a
7q2s=%<Bwq(lTnG(U)&CK=Dhj^+X3d38S_e!6sPP6_W+OmNRYhZboS)v0Biyvc^nk|8uB`8`g;8>0qANL;h}qv?W5`Ehv4`(-NCkZJcT(#xP%%UxG@dq?YZ>
U}48@bk)Mx{UR;nbH~bx8XcOmy&TiP39G+Y`-e<t!%%FtC!p%7sUST`67@zpfr_|DoW>d$f4KHk;&x1GM6woD>Akm8=2g+GHvh(oG05qn_Rg4UHywe$^3pY;
^EfKgJ1NNvVaKSWO7dnn%h!r)*K@<8HpRU9B|jn+T7E>LOsloaZqS@epKSkTDM+00HC<rMCP?o|;>)!v#QKD1Zn{yN*&F==4Sl!HVmEk}KZ33bgEv)Bes~-e
-0blhm3xJ^5cQ#=NDtyn>9ihFjpV7B{=f=fMil6L;sS&fJ22F$prt`xhchr?p{kYC`o{`Jmz(wm<GFeNd4g7w+a5OO(~sMQo;Hf2g1PKvDEyqCvXTE@Csulr
DZb-S*p&PxU{BrCN_aS_kfO>zvFNJkQTX%7O=}HTLWR!(Oa)#K@*slF(CBIDN;{vWg(Jv9?{`xrZN^%<gmmtnrBi-4sES#JK;0{@R{u+1ukvDVx1vR?gdS)V
N;wf*@1mV(e%!7XPkW9t-m(Ieg9pa+>Ot>;3!3LDncjSj{pNn1G-s8I_NgH4L%BDQR<OUnNGh~!o}0oz%9$y<J)=Yu&Qxi>NiYub=eTXsxP!m~&>2#e?Xng9
GA>k)6t`)`EdNAxa9;+)0J(Y0N6SmdS;cd<-g$+Ttc`w#BFAr`Bl0f(Yc-?M_<S9EvEGgpfV)e?X+U!eCbw?mBXbVc&NOom=bi8yqIrXK+=`R^Ao#5&{=4wi
N&ISh_*&@q=2{BE^qoaCv@>lgajQyu(_7M1Rdf1f(Bs)7Jk+VZYSkX`IhJKDZv0jwAzfSDzM~@vtWqJy{QWCN6m0e{uejJ-o+?VEw#|BZXD!&eBw%_|5X!Wn
9;!n2!3FLvTMMQhyP2XRazbyZIHsbn_ldTu(%fFq{!#YC?D$dHT^cUoIx<n3Zrm-3sDAUB7Q245UTF3o!E)WM__>~7!xz}K8G>bjV_p=dx2=PyYVF5}THq?z
6iX7Q<{Y=(Sv~6fPl3#8orH>~*r6y<M%RP=eG{S#{++O1YZ2~U*H|gl%Y!)?8n|<ze)EzS@D2DbZ8%s%egqnB*10C88eimyjDb%pgG*jox(n*;r9{&ns0Pn!
6O2z4naKFDHN6uJn`Feh;rkylqOV6ymLTH>f37j$Hoft+CJ2qZC{8dfQ~ROE4iqrVdd;fbq_T<&)RRFxVK~;*g5{R)m_qh9+AI6a!WI5Uesc;oyBA_15N4L^
;mf~*{CeI-ODm>%#A#s>zaa_y?d78Wtswx5362Aebep>qr3K|RG$cSv-;%%W9eg*L2n4}}bTk!o*scY56zjG+?k}DrpZr6_EH4N)H?pDw$Te0@Lz9H66xn21
gl4guZkVI2>Lc%BTMxke#5X=htNlALCI6I$iUU@w%5gn~X1<Z;j-F%?k|wE5=d9H2o$LS8dxvZiMmG&bpG8FNJ%#i|y+D2z$&x|@pISvNZf?~Pu^BLXqSF96
{#~1!NHXZVs1yt}BI1nuM^)lMoQEuyoRQ1$ND56@hWW`~UqwaRKHm$RvnAb__xONv%#SFgZl(GzTZh?uN|GLRQkmmCAgQyd-%uv_569li?qBf!s)3ly$TFtO
2Uq0xjmn5hmBQpDJKMA1UE}mt59^;UW5RT8ff>1_JRbp(74DE|aRGBDUinz&{OIkiAxPGUjYcq2kuj{1{ZE6DyrAf$l{WJIgM{3r3rAE)JJ`dzI{$DN@Twrs
J>bOQ(@n_C5-`)$8)eO#!QXt;QjEM7QFx-4r(S#v@qjT)pP1R5hb-+1Zd$E8)u(*zUCl(gB3UEIhi{ZYZbWgIc4Pdj@{n{}CU%p&^SDLxwbl(x#d=G!pdmS&
(2f;JvZMF$!zVj9lR19w0(Nb{2AAYbew@PSEn27W!wa8-T7knKp%Sux&Go6BP^b2X5EH9i!1<gcp@`ujV!S)a^H}qF1#ldC5fccPO7+rPhgtQq^m8o2(+CyN
<_PV2y@~#+u@)qsud!OjqL{ZBNf7D(lm(2pVUvpGfn^p^63u8;l{UoT7FZn)8dwpU)J;IC=&R5ABCvWFHd(kc0hdV$)O2`;TySbG=*1yjw&TMv_tfqJTbO)=
+-O{l78#SO+x0-bJQ7j?Hn!sAegAdb+SV!-*|hPXd2#8i{R+$XI60W=c7lcE^-7u!I9%EDDko3OP)pKi-q*)$toUl~VigUXK)17UJ5iNAsVn?Pz?BC9fAUi<
Z`>Xb9$cpWUADy^nT}VpztRQLp^DiYSTx-RAS=Gr?T0qDdxCVVO%Td4W(h0C2C3x{ywu9u<Sj3}PAc9K3s&#5xaP5eS9{4o&7eA)|FGL^Q@&ihK|n2OwlMpX
_i)@cEl@e~N)?HsU+{Hvsb^9>2|H6qa3EJL?m&61F~>8W`FBOPlwp05^y(<0-G<-@5ncXE9zaKo4TPs7G%O-${VXL?+y{OkK^0N>U{12ku9X@MoM=GQXGl0%
esb^ji-=Gf@2=^nBDFXdBu8TrfsX4BAV(*q|Hyuu`bmHn*qXY7DgYRu@3JN!5NIFnZuCX3N%1M2av+cz;29Ev*TVl1(5A@sziBwMI7Q8>gQMk0Edy4<q?x@!
rJ)%hZ|yW%P)G8A1@$#P*j4XGDidu)-g&ayHir)@n2Nj+aqfOI$I{8Ra8{K2eKjtuPVRRB>;uD5JB6=H2yph(jyVIG%d#htM1_JIU&^D@7x067Q8#|_QJb-p
Q2?($s(H=X+wz%Dz-L28*8vtY{nNGH3#6?Y0bWjeZgyijXGHA_0Qi<spi01myGFG1`si+P>;yQo2uwxZqj-jxORBsOBN{M|!oOC$XtCgMMwX+<DOAd{c&~MI
JcU3giyzd6@ae`&uxRETp)$c2&$isD+N9L1Xb_Cp)}7R%-h8*><`S_s_MKYv98Q=xx!&JyoTu*oQg&);;2QBXNt50E5+WBM@a`2+a*Zx1f0zTpM1{CN2_ww!
UBJ0*Pj$8wjgIgKwEKj$`CXZm<t;LjzWV@c1f3Ct9-5k0t|9_k&$30~zbm=Pttyoueezb6daOGjx#K$*gV|83P5BE`asC+v<?+Hc_6K@Uh`X--UT%j_Rv``=
uA++O-4B%OWDh%)DubXt$Uaa<5gUhELZf~&%wp)^4SP~SdN&Tf7L~gsRy^o5=y72*_pAyIyV7d^tEe7d7eEbpHK{Oti$)aQqMNGOg2F?w%;{3}_?2dgF%pUF
5gwSU1xSEhNVma}nj0`LxEshGMj{PIx2D3kP2YvO$O^SE#?{Yi0U15+*4(6;MkRef0I_>$!>SF<h!&jwIDY^~-bSaqr)tuF-&wv+|H$2!;uB+X9~!>cW5`a~
jB)Ka-;o-7P42pVD(JA#Vx%d46gNE1JEC9quNIbJ(~>RQJ&Xv-4EG42q#(hP<f6eN<}q~W3Cae&adP0+5++8Szt<d(rL09I$yd=$Mj5)&(6I=B`l%~XB`bCW
HfqV>eaKcJtdbE9xX@BTPGZfa1Ii=Gra4KVHavo81{=iIJj#6^+x#e1d7G!Vksj@XUtxw}#h0x=%zp=m!QAGeDSR<*YBK?$bTIXjc;pg>wROi$ZZlr+$dGGg
*$4UIDgcSINn<MB){iAB%6_5#1A&=cHJZtg(JwZ<8J#+jG`v%$h@%&*`o&}l@4rdi4nG&Hgj{7V2m6}(mVgA7Rx(bqM<*ZDbFMe?BN(JSO6CUXu)sUx%*-h-
{A76|pjCjhSrUZ8sfmty+=|7#K?w_7#Y)E->?(5+k!Qhp3>OIo$Zfn7@dAM-(x$L2a_d*P53C)0lsm%ZXj?pbQ+;C#h0HqUVU(Ai26$5^A~)NFzpfS95~PI5
mLCiMKovJ=1@Ou4On76yE;2Sq3dkD?xMqE1=CxCN<1#)fmhKHHL$GE@*&{}P*DSkdg-%Gi(h_46@}EKlF>sKqS-)q~9zUR;TZFPs1*e_kYb^+mRt6n=e*_BC
H*V2g?b4N@$sYf*oN}-;*~R>atRO+KF#Ke3o$1xA(qDof6{Quat?KKoJMBWf^BKJm2(n|E<oh8lt5wfH#|SVKEbEF<Vog*F((MP__10>-fRD$e?hR#HpC--}
W5}t}x2xbSRAr}u`U*^Dx=-I*z$|F{Hn`RZ--@<nw<?mllo|RTT}*yQ$dG*e@~Yv^N1Dj6SJE6f-mVwpcN}0jzR`V-_p88GtYJIOY#W|rtVQgIkI*KKlpY;Y
Zf<0eJy`VxZ+&}&;zN!Wy#|6zz4wWo)&vWQP9ho<`XW&An2V?B_1e-W@_W>gxL;z{7{V^%#Fh*M#(pM#lA0eiRgyIg4hgufd?MBNWjAeur*az9m{yempQmTO
NpPrj=LJRi5-B;E*EL&Mx^1x&jxF7A?`{7$inU61x{PU(E)%|%ArkzWf$${Eci;>Tfy@1;>Cl`0!QCzb0v<e@&0kqVMg{_~mmzje0=7%>b+W59Dh`c_!e`=7
_Q2fF*Bd^&9FNb<HHF#b>y!;rxRoy5u+5d^8}S4%N9^Z7EtPcIiLg;|7(!+)LWN*0&tru-u7q&uQh8=q{<obmd6@yfiGkd7^T&~#Cm*j;Yt?FaejTwoAlg+-
HqG*aRe)Is%1D)%B_;n~S8wZbJxy2X;=Z$->L_8~dwZOu8==D@Ia*shnHGgX3FEYLWG@YgoYYa49F1hxRbei!d$1fg_;>R(z&9Ee5Me7rwXIu>iNHQtk5_x-
e&V1*KaqXbKD1Oe2;al-rdf(f9XPQFWy(AFFC$RuuX9pB8sg}3&L<K6pS=l&b)}l?+x=f+mB|>F`mo2N$wjmCMsP>H7u^HA-j?<K$@J^N_vyLYGi)X`f&{);
P!4yB)X<(PlO*_Bt4V8JVu{nZ(@-0oh8^ZI3++ez6r?r#to$?tnQ*jtax;JAM>(mTDEoqPt0Hwni%l!RI!^?DF8H#CVgDi?YNC>DBs)S#>EEgM8c*2r#aZ=s
ZFIFiMVL2$P)9${`fwOHYsb5&%lg}v+!cqjZJs7>$~otWyyr{;`^j~b_JIwmk6yF$y06b=b;Jyts%s#m!a6>Dn{k=kfU0c@5n?6S@J<%$e_7_gs@rnYwbw}o
mh6#}p&Y>*QVghWKJFxnSd1hcnM8*BZLzMVs9M!A5$?M}MQQ{0i4KPs<M`7`nykLQii#vAI8iG_{EMlBa6L-|>s`SkS|ZwwpNHYcz#$vDs-0Asij>Ut8swwY
SC@(%hWN(p#Z@^cCdA;;bTABo(CDVK8FT`Cr0qI#e*rGU&Vb_d(-GUDEEf2%gIAl|L!es0hk`b3xV%|<=eI-uo3go_jZh{h-s|x>4BLg(&(<oRPbWog=yJR9
Ad1w^X&w*b3vN54ph4j4sL=Q!ELDygH}zo0prbr7ywV;=k^1KsGC;&zq9Pq;Zmhb`CKh*hx%OBvFYV#QOINp2EG-}VnId*E-g&HKt8KghqW4;9X`!36d~7Ww
k)V+bo<;TtWd#q;c`{e9K@mz(CJkT4l~ycYF%G;#h((RJhkRYq1w4#rC~|ZajYVWeI8IX`DPK`2uRjy9l%r1?RC`|Z5mEx_;-H+^lCFrepeClT3Z%aNV6HOP
j@`7QNe)}Be=o*&GxqM`Xinb%8PZve#lu^{oh;8~qPH3Ym+bH1LKRk!#r7WKsI?R<1qi+9_`^pQ>4)kdC-=4lz<{%jhGiE>u@F~Nj1#z}9Jx74&0xX2_5xJj
@g+2LVh;LQ<8jQ%&mGeMIkr0wz>ATE-YmjDf3LUIES=6laeAC21!(45b(4!&+K}|=zZeLE7(FJ~>`k&BlBjMKTvd^oI-g=sa{hzjl?11i^p;zSUXM1pKRcL7
^<o8>db5VrOn(3dO@^e1Qr$(8NWOz6qW(tNdJNqTK>vg9cU4BgNA#Kgl!jj1<kWE?)eq_TH@^n)ZU-gDZ48pIX}oKV{QN@jvn+ey&)&Rc3%d8>G}^Jcd0<k<
eM_PRsvVk#=^Su{Eu0wCOi3&)$AA;`Q1bgC)U7h;osBK57A`2V2LDYmfqm5`Y%dGO!;DsyNTO5odzmsfxSh6S=nqTLDH_MGl7*%I`Bjp?qhlL}g?kpXds9=H
WJG02gETiFJULjoE>*sp1bABzsR7zw>r*Uv76cf)!9k#hv|U56bjp6kwYMBzzjoBa(+c{a9fI5k0og23)f?02x9r4aSc~)Jm^!JvGRR?34Fn}atru&;0s{3@
x5yDzsHjB5X<5g2Pa)#7l_sO7MlB?_Nw1d|8~81Y7-U?=b*HjB_!V!t3hEGcm4^#<cUln4qx6kswms3GtT+=Y#<2U#F%_dYc{XvnMxV2A6lP`-by_<397&`y
Z$fW$2hz$3)G@Ms_9@fT{wA~}m36icUpzliSEIZqC1PTC0*S7uz+hYM3eu_PC-4dUOgw-_%T&jpw257GpJ+HhX5TRhGW0v*qkt98yFq~dP~ecwgpzP>K-0EM
T)&bNe-woB$}x*TY&(2}<!<z&s81xcA7W=mb%VOBKO`)?@mg`v6~9m9rO&%28d<?O2Htk!3{YrK!gqe6`t*u^;WfbH`^nA8z6rllqz}MJ*Em=>Ia^B;0i8=s
NhJRO(Jo-?B&5yL$%1A3{q_$a?-1!iKTT~LZ=Lx1rp!~vCkCjG0|Ef#NF+vq%%=oxL%@3_-s2yJKM#2p;*H1;S`qtB_uWRkJ43`_=n(3iOQ-<brsM)_;xoSt
H7PE~)EXUx7jIBKsILE=xiQ@Q6;wQxgz`Q${UkopA|V-A&UlwffKei5{21$b8t8?@RSAWpx$}zZVS7Qne$!~xy~g3Rq=K5%32P{LAilQ(I~g^3@Fbo|O(kEp
-lJ9IKZtPtC{y_jy^h@J=*~<IgjI_t+3Y;JuXE)@4YQ=Zo#?_5-mZ}5a<kr_s=$J)rLmGmgJxiFDkzK|emA45E@akBua)lE@FAp?{j{ezK&9Brf(fg9qK&{~
LHadfYsN}NhU4+pRVCo8n96_MOxdsA((gs&y7W@DU`%6bq!K|Gh&Z=}bhWu6Q32m|b0L$PApe3>xL1{k?|yb+#nYatkI6t-$oY5$7Wuq9K`oQ(0JcdO=Jc-;
w{cZ7_kredKBECMxakx-1Dqf|GAM6@N()lnNAj(pssFmFcaZ&&v=htD5VgxX4iOWaMZnKU5PQ!23Lzsv(Tj$Stlwl`5NQ3X5IGpfVFj}ECA`^9vF3F_pQSTn
V?AM(f&UURjZpDvpvUiUyCIejhv&sC)x-M&e_2%xm!OtoU4GwpJqBMfO-*}C_I>74rL^?iS!iee315|ExAc6W`40F#qspGo%-1XBph}pCxLQ$m9&P`?xCG7G
+0dfkUVs0HjwCFq0!6qKWdo<B99;kVh}ORO-r|~(0{X?oP-59g#15Z6rDKT_?)$;FL2n>{BI>X5#rYlxAy6vKGYm(gYp+TjahlmzLxz>IQ*Aax>rUR@HWJGH
yFe=tX5EghoSR9Op)1s|amSXtj%QxkG2O;G(mS9QA%1>y(?V^Ja2xz@`>ZP#meJA^oJMQRrWEswwvJZc-`XK3Oc%bwikq@D!drHf;|Z=>!sf6xU`d4?PZYIu
E|rvmTc5N{(f<}xAh0#L5T_e~ldc)PWMnQXKnxxX*&_P6p{=@5A{V202~RMh1-Y85fSqouk@pxd*>heHVf54Bp7_>qlsg?zzqhl6eV@iT`gFyMdp%tZsQv1o
e2vxwM?31I@!w5wj0LPpJshbH=-#Az&$&HlI@hT*GWK1LkA3ReZX;9TOJw0Sbd#b#XvOSmb5jONdN)%x440ypJV`3#E{^tVQahk?k_d&LQ3`Oi%l%_BCZwLt
FO|4{)S4m|z78G6{-UMYxka<6x(P^zYCrK%-#Yq65@qif(lSBM#lCqaPFP3>Zn1FZvVSN)MkM_&TBY@udc?qg$66GKQw16pb}w9Bm~1vVB%4ymkb57;ZsrA+
COD!l>5(i^{(c1hCO}O2yb3#nk7%kAzAFg|a>u?g8<AxYSZ}GO;KSgBjxqn<i8X?2frn$W<xrt_9Jn|wI)k@zuN0a(f)x^vf5&u|eB}xbf`|QNNBsqGC1A|V
Rd{YTNs}&@Y>JE?IN}r48`jS2fGs?!0hEaWEvsv36B`Yb!NAQjl;i4anPoq|xVaW9H2JB^O`8@{<ju0|_q{~Za#GG)wI4>e-kLTiSUvufh+tVA%Y}J<7pNB5
#$497LM+gsxXG4uh@6|(UGRNr%(q6-At|!Zdk*xWFZIKs+olJnN23R$?lYWNEP33PaH)>XqNtEqe@)@gM#v36fzwiH&1@WfvMc%X1P|~)<ABw@JRtx9tb9|W
1yKgwW7%Hg#UqK>NUbNGwv`gaXlofZv?MlZ)b_$#vs2p6jeY(Vi$~MAlZwmxSc99jnlG@GLKXF-qhLmV_V$oqimIl55tZ=cf2pQ7{cvp=?gV`!;Fb$(v{>BS
KigtWRbd!JTW=h!Z@JiUvU}A3t{G}>?d`Mps?kFXg=u8ME0Bw4JrT*dzbf1xH~z4fWJ90=k|x|_!3NM@)#37~wj!jeZj=ucJX?1~;}8s;SqX^=kfeWFQOZ^Y
y`ssMRm2jNja&UQjd+^pF%|;qJrOkbz`*!z-PPg7cc_-+(8<@IMZZ&TW;AL%&_o|oVwp?Vs|E62)PJniKnW))WiA?y17rRJz|;p}?b}$)_#QSAbX=h;PSA!*
zW{3;tK#y#0p<otV3p>t!fdf8Zi1ylU(Kw{+V005vIfl4<vfUUzb)KaH&~4y(VlHA@3WKY^p~G&ir8b_eS0fTWTxu3=A^{Qw803(Sube$^=sv3c=ed$&!tY|
10LQtI&q{oeepYPM<YD=m+cc_rEjY;2<Q-&`XJT;65YTkzN4>0_%DW6i^l6`EjZNd57tb|B-wREkH^zNZj!gaJU8jd#tK7!&6ssoJ3L7AO;rB0Ci?#Pj{lGi
V^7C#h#?wUS1wd%24oULjzN6Ku4x#Mv+d_N22%%yB*5d8IG;>II$DLf?G0u|Z~|%;hKCZ7XN9JC=N{WS8u2oZ@CXcvn(ra`sHuwdA&(ABX-u2VQ+WKLS?v2B
9ijlYd&C`5xp79w8^PAjKiNK$TI(~~2|<==BhXK0bCWw_B^hFSpUd@ET4)pu-%;DLwkjKp%uq#Jf=)b}W9eUMfn401Pe9Gla6qGosO;b+yr$bV6Rlj}{1Cf`
DTq$i0ukZ0i$K4!UO7A!{>xw%$pX3?ov=lf{Jg~Z04auR_(~-5c>TQ5GBgMM|Ngh3J^%)(K+zv9@4>P9Fd3R_QcrX&M%gS8iEEogTh*(>V$XtQl1$i$sAiho
MFkv#?&O`5I~K?S6d%&P4sl9~%_;3{o^Dg&y8$u6wFH?9UxMfRiK*ju{RHAAj7zM%+?jteHRu>H{OJZ|`Zrx80E@bMqIr--|H#+l+uZ$D<{MH^!7;w_%lmDI
Mysr9VMA<a-D2(>)sxEi;wip0!xyTA|FR91PXomgF?$o73{%-=1y<{V@H}ssKL5lb18iL}*II_%32nSB#JNz@72pzmgUCBU=HTi4ingS<I*>|if_leZm1niS
Vtutz=ko=rX>TPDnPe>Wy~_HoADs-{-15IvONvn34%gDeSZpoGJ2)biZ{6{KMpzxa#p--~CxGs=m~55E9d1SPkVr8VQ7l;}vBk0e;@NZzC|GzCmIikC;aLlI
!V{*cu{cTQX9p=>9}>5miveNUP0VM_NU*RHoj~U;Zk!B?GY}2Aot5T&h!e|DM!2QVsoLQ#W)HCY2<MwPRgULZp}9$UTBS;=pY>BDUP!T%=F5~`c%QFhEkA-2
Ra>^gqy*h%-1Rf~tn7NvD(A@GS_P{JseqQ;UAHpO{PtD0f&k7{`w+>Cvm$OwK1_#_lrDr=&2#&uF8rLALwC>&c%Y{jj}R@j!LlS3D(0%ln#!^3f#=V^3as-m
#iyvKMe{>7=fi#<`zo?eeo8_ci?#$BMsa^pt4eev>_99w?k8OIg69Li=uuOruzB2Pn`-T4V66+P;`nqBUjyij70Ef(VOggMdm|-tD7oEl7$erTOYnXrH79UG
a_%54g~%ndiO1g0?2@`RZ7dgwtypN3**hkDLNb(aqY^zWJ7wo<n=Tj1EbU?_22lFHDMi{&gr13iGQyt_lhP>gV?C_~X*k8G;gCR@fdUaIA;+ToB*Urue@!86
Spv8qwII90nMS+5tYE1o8-{&0)v?-N5h}-JAj$aH>ucd#t*W0amKW9VP~Fs|(6dYPksX<-z(=PMEk`f16Sfb`0szB>?DGwVcFX1Nz8(wYx(5YGM9lU9UY*b<
8?k+3=2<(NI%*qiS!h;odx>Xr5ma)HPx?ZXEmtGDMC*=xwY3E+48bwjL@JJqc{wqS<|fHA9c)H*veR^~3&_3Z1l-AfVqp|z@A?G(?rhwTGFSw8U>VH)rKeLr
MoWc5@YwE`$}~mCe<ZJEKfIS+MVukQq%T(B;ht@D!|6WX`SMI)pb2!-eDuS?e7f_IJ1i;pa9Tm^XdoGlcWAPIE1kTFH&U@A342lHFl*rN85{qVVNu2y55IOn
Ef!&iLkj{KW;8=4kD5eH;KJ&QGFD9-p6k8GQ__*DJvdaU(8n*V<^4s{u!(9Rh;|O!ofWL>7}e2K5XQFepZ>^3zXb?yG1tTUGN+CL4h3`>tFd2B&#c0GtdKV!
4g`2-K!0UFf2U1KyxR%Ituh{!kk<m``J(dT>%P>QpsoYeC7lu_n!c3G^eovkkzw1$zY#+>sMA|XTaAa&vz_bDD)}zq1s0lBWs<EbLZvensU)OJst`rE@{V+W
UOZEyQ|^+1n-8RoP0sq2Csr;W#;QMB=cZAjdufEmS^(shp%H0asJIo{+c!kQBG-_>yr2wQ$H%`s@6FuteEU0W8vxtygXuv<R%BwvA7gjhFFJy>_&9^!!V++M
3R$StCv;%KNv_mL%su|T>M?9c<NNDp|5@<eoP(oXvCvOA;pd_sCnIK=MO~JH7ihjj4FjU)#idrUZLrtdLPJGA#|}G+9zJVGTg0#i=)#S>6KQWG#!H~{@Un}2
dFfU&(li;o94)FC3ArVWB2zi7KAUgX>fc636&%9&ON#*^aa=4SRGI(0P<!G(chOO;oz@UkJE9S98=x=8$A*!`%9s-7El)uN$|X55WQPr2zLr``++4{qLBTFG
CY2y?KH;LmB2tu)eLg8u>MoYC+W>r8I+pK2;2*}4N2$oQ7Q8XDY@^}??ZO$qG1yJLmM(h&X_07VN@l>nx89A$v956ju})2b$c2K;Iv!+aI7nm5)LhD8K|v-Y
+iUNxV$geUP-1frDxH_8fhfNv0lp38Q=P?VG+$<lr_vEi5~Tn9*#D(_^f*lkqiTf5<#av^N;5$d^B;tH?<O*FZbx6ucpMX!!_FhmQrMiSHbLHI)kU{v*r?wA
cxvjwW>xqL75~l7tAT$0^VLfL_Q_PeCR9hbeHn#E^ZapjAP44~CD#TFtA2*l7q`r9)QPEvOLJany!4Rdn=BJ$8a->8HP()>t4VzidOJ=+yTm>HT_(LGi4iaT
5vP7;UW_D(pBjVJ>W{M~4wjxQotdr*-}0QSa+Lcqtr*|ra|}rVGJgCVEbc=l3OLQ5p_IeLOWxg-n)tZoc&iVC4t<G?uJ3)Ki5D_tx9p#91vIz8*56(1A=atb
LsWl{Sd@20aq1R~)lJL~4k=9~kK9DQqMsg5xf*e3xq!N7QG>-?Q&(FFbDf@{cBLS!t;^36n6CJH=lV$`jnCrV5-e{wU)_Dj`qDjpd)wGh(Genh0fm4+&}_lq
^vbO&Pkzt7p%v9GhgM6?sn+2;n!(r120fsjZ(ENL^Z82TsX%$51tVeB2z4>%aDdi~j1P>-s1Vz<!iueCN=bWA5ab3b3M$KqVF&c<<9OFQMC;LDF=D+%&EIPv
TX!*+uch5USxIqbDquUSU#|g-MgY-avaq?uwX2@$2`O9u1PFy46g84};Zw5-tOjAP6+Y7#;6~bh#hTfsx-2ydN8=zYqZ8}5+{61hiKmj6O+9!yY)An6W~8%V
D4Z5-ec+BW&YkHhSc4ZZlYr*9imLALol=tjAfLf_TxnC-O505x(NlLu&wYk|caXY<r_56uz8BVHe1hg9k<%Rg^h$4F*J11kL2)|v6Jj^x`Ow0UlchXDVDK<)
sJU>OHLM0$k%rRoEmR`iBhYQnx?N!!k`EvoZRHy-6op@Lfx)Z`ZN?7jHD5CD=AIWk>o=RofhG7ueycgd+U!5ZhsGI4f`^DH4-rVZP-Z{fmIzNIDUZT!EpGSB
X=ipZrNu88eI!RyYam4~v0zxo2&T?CLMBB&>Ln?gP#IkH4G-;^5KEtmOB39KlHFRb=BE%^^YXCeE?DkAw>vqWJU!@08MgG?LBnF+dOF)(*>-KQ)Rye9-P7--
6%R;n<jsyDM;9+=ux!UxL-RJo9W=5?aHn9MD+Pzaj1ByTt4v&@sS!)9&9iSGS<aE8J?VLE#yKoPz=d*F3&~K1-ojo|Mg<2UDnm7X37aJ2RI*e$B4yhP$)V&T
#jcOK>cC?z(B(?tJv0`^@{2N)3x~=`hb3rYpRUB5pi9WwXyILYblYlMTVjCKsE7QaHb}-{4GKh-|2ID|$T?$rfUz+jU7rmgf#6la61rjxMjfU7x`kyKD~4ve
{W`&TTg0<Z7_BZ3iLnjWh9{q$$J<z*`+`lPPP%!YzQZvvjiH+~{`^aAeZb?sS>T?liQa{Ah2fLpg6tSIZq*1RF=lvk=@rdE#&(zTKWSdaAt*XJRO~LjV=e8{
T_i5&g1v`dSu84?eJ}&G@*XQ8*w!uXOn`<SL$nxhYB$#hWOjZCl#W8}Rz-y{pIXI_=&5=-{#U1{r^`2eE{KJo=(H7)#`-Xqn5&9N+lXnBK5v8ejnQB?y?)zY
j|m8O>qjHOs{@)`5XX=y3r1UsiCeVM4Y^RpEOgIRp{wsWPeH!>Ir)x1tj*7Cq=CJ7+KYvzuCO9w;WM%+_1uTOwsyQ<vm;qA7Z$O54!vM(>AI9N__J$RS5bc>
^&nlW(`k;TEkW?YcUcW;5-EIIDyDEg3h|<Ol0jghh;Z^o7D2_A^M-5uFXXa&=-nhY+ye)`Lr7x*A3PdKh|>ZgykWotPq=j*)icFMSN}tt=EvQUw(C9Jxo~TI
6AKfSi}((jVHv*X;>?a`5aC%wVC~>LAtW*g85Zns#3W<Bl-PY0K_EQh{bjp~WR3|h&Oom+gPYS21AzOj!-A%1I)P;1{^VI)=6jfEw7BSBM*(CZC_!(|kkNcc
YA*~7`htXZW9xmg3!nOvywgO8RMG7OtdSh$uRF-#r>Gn)X1i^s`UzIB6OLuN;86dHC_nonXYVSx{55EE!-zL!Lm&Bc2m1W~lOf3&<GdNl16~*E@MkYyu)8&o
p&=bI<OzIW)|_%<)>iJ<$<CA*9<<wmZ*@b<?ydaYg83%VwyeOp(u?w<TA>WgH?e9gfYph40YcB_{-EI-K*5_hpr?2s%hKXM>ac<;IH`S(fLIO!OTta^#~B(4
vMxM}I$lz{QUpuTBP?NP5C~VlN!7@wF4QOmuM|lW-tx`t_cXPth|meFy^4FK?au<Uk<4)4Y;69eHc$#x&7^}UX3Xqy4}&Hdm+Feo=;UR~xj65Kk7Q#5;RfhY
ArU!x0FZ=wlz%jjt4*mF?kp&ttNG^v@4|n&`~)1^sbHWiQ&bd=8Y^z#?VN~I3u;KjmhQ-0{&a8LFs`JC<v(;vdJwIQlE9W!QXBSZ1HyvcqTucL@FY-bVFYB6
e0EZ(3tMvp%qis>iEKCraY3w_&Fky+!xo4RMc4OIoqJ2;*qC|u$?jkC!wl|QWPDQRE@Tk_a+MvfH3i5jwR4o^q+qVkcF%Fu$jtrc2ycR5Ag~Lb#|#%(tD{;w
Ee;dkEW7*1%ZKmW1gdLy7oj#{!QCp$)sbAOZ5ml5>oOgj9VyKJxZj0YvaYHMFh$|`$w(yB2fl0(qh?NE)B^JIO1js>g`N_H4r;tdaK)a1V|jx&4#Pk$b)YV#
p{B+eJDdv$)iPU>9pj=N#ZVK54&GPSniE|$00R_~?kS{96x0yqFWKr;{V9xZu|Ac7#HUmiKr3C6nc*5*N{DXfjpL+ui>MXH4ax?eU>K*?t-UUeZ^0aqBNfe-
YqQ*PJ}Z33KF~M``G_fVQy(K)YW?_#Cv|;i@3~+EticVWSjFdk_vQqLh7QikyF#7`HXV?i1K$`U?F&|BMiO9Bvj!8HGFWPifaw6W4SNdhWM+h+AJuVQ`XE6@
R#E>A{8BjBZ1mC1c&x;L0I9jDINeSYTF)e$?*Zi(L91$a@3}q2Z_iq=*s3BkZHPnP{LtRY_E=_t$XDD$kP8eq#>S;-_l>oX2TRk=3b9izw#A?3L|xVW3r-Q|
U&MTSy60Z^ne*>HSaeR;u96Js7$rpc2QAW6^9vOKUXs53;Kb}MlW5EgB=Er!*&3Ey1A=$sM9T*d_5#^Gux)a_&FmLC2tN^mk9T*{)!PmC9E4S`;&PkbxSID8
(5I0qWC?$Q`uTPdT4wvI3l82A9#X{W3g3r|fhxd4{h2q=bGf=QRIs2fyxh_R><lrOv8?c}Sek(3dnM$K|Gp^|MFgIbA$~qA%=z2GXhvnLF6jhAlXwsuFhrpm
b~ZsJGiQS`uzwcy>C{6gHnejd$W;`jVy$PZ0{bwU!SaoW^9g1=ly@`)a2mVi2ungbfh)BD`{UuvG87jzozhCLWS{gx%z;0lR^OCLM4`wP#I+(DlbYk<6lu->
#qIDEK+j8g7X%?vsN{9w8GGse)B;SM%LcB#83nHBGR+b&CjH5q)Jo{`&MSxKlKA!mR*<YcgqEDn&TyxyTP@7ImoS~y!F-oKYo5&yIf5hd-Z(wL{{}3TdN#-L
x}Ii8l<Z{1WwLHuy};a<eNlphYyRx@msOrrBDOsPo}3-~>+5X6oWwpHfAhLSRxiS|IbD}gAoI@Aklh(}ibzwHm7(EWbpY_h-kECVG}*-fZa9AoRww7=HUV}5
fZPPqxai&H;m1xBc!xp{eY}L}tj7zzLVI`Z>ATB|`Ev!As@c*smt%IDaia~fp<i@z=I-}VJt%2)xLOrZX)URz#SvrC%I0MSA>KA1Wi;5&0t8n8Nsfm0yOb#R
G0s9?>5AgU<{vrSzCPH_r1YKG<?twRFSR>8pZ%i6;t*z(`8KfQG9X#0+9Zpl_p-biePt{JAZQpj?oO6~c2<vx*wFQMom4pbSERWfw!UPrESmJ?;4sTY#-T>>
4i315LzBzmexhtYC^u9Y=f`;w2%IVFmOalV7Cs5Tff5`vIRL67t+RmU=E*ijDhMkBFG!R`%Q3Rkurm|?=3c{yDOt^BT^~1;z~2?MPjUZRqmsd+qj&2rz8>Db
<bdqg%ytpE$<mw`Klw7b!e~by3RRqhnvd3@%<wOB!(IgGhrppjU*w%^JkR&x{+@q1Hp7jrv-AyxuTPRft)d0ia%5&3%cmmWtp4RQ#t$YGOrs~qtU$V5Vjn(q
x!9|}Cx+Y5Ixi~>wXytLuS^H7my?ef-|z*tWAT8CA{}?i?y5QP&-{aCM(X1C{9hw@E}Nd!J2eHNW8V_eCgCKU>7PRt3Jz~tKpda*Ouh+spdCmOh>A`W<UGDf
`;|(^*DZA1DH3*}oCsfQ$2H#X%X|n*81ea!H@a5Kr;xL)$3#k2+j>q6Qeh`VQb0BM*-EFvLu$)9<FcP0_%)4$)pA{BM~8lG_9(Fad$=pKxuPT^LQCe6A&r%A
CAmS67WwpdJkbX)UrZZ&&NnRQ8{L&Rcu~&nehEW<c!5+F$~penJlkX*!~#UAnitZlq!CUi`lYg~<&>#FnRH;m_OHbmyM|9q`N7{B)~d8-K<`2S;?Vch$fUoH
q7IvK+GB$qZvl*=9j~NyO1t09A(lGUb#GzjkAgC4u;wY`7a}Z$8DX#*v>$s%>pPr2watrM7IXr@&@VViCjuPdG9nq0RQaj5C@kv}22wu#EZULr+B@f6?1ENt
6{ua<m8*iDqas@TIxt9y`y5m)U|?|^ntG<Z{+7ZKYL&+7BS1M_^=!+7lWn{DRZ`(DyeqGfI~V1J9D#Ql?iYwXEsdX_B*wX(YBWLLj{szW?{{Z)=T^7+UAndc
`h%y+Uem~&b<#!S_Th`Lf`M?#F`mRzUb53)=LRF_9O`bWZo6KI?-ZY^pan-qd(dw8x;P2Z!y)6~^=lCfAduQJ3hAP1m7QA#2>Squ!@*RB3-7&@wvCi$OJa%?
EwUD;Gg=_`UgU*}P7(++y-Vt)?*f{g`QxZMNJ?AmYR`3QC=L9j!Zw%(=V%e`(F4?(qJTwfc(5Yw*{8?I#?Q9^(r^O3DD140pCKE!A1=V;&22ntD9)I$5CxiJ
b0X$gLwfC(1O0j#C|nkNj9};5yBpOcdA@wj6^ejUm_3A^Pu5lq{L%~0aCudP<=FS@LAECe4(X}u+!ZO_;$aujz%{yX_A-l!SamkNLM90d_phkCClZS=)khnE
P;xLUByDQ{Ed>C^@0Qe7Pm0^T{YBuvEGEJ(0&`Dw=4iuzCek$uDNppQ94h(2fWq=zGiUFeSoC=M=>CwAhm9gmb%$LZOc;6XC5o)AiW{&>-TYD{dG062_2BS!
9RCD?8%MNrC&F^P+ZAEXF66t|d)o3{Dfd6MBmdt)r)}J`bUfT#Zx1-+EE|we6a*-Lcn&`5-!v1JikeHiP7T}#fULie*I{*)OuNBE$%~lqcnaz{u+ZN}Gd!Gc
5|m-7o4;`^xmVH@?{Nb7d{Hw2H~mqi@1nuiLO(9*pB8qZpbT1XCMIF6AfFQaxA_h3tBg&k`HOT{c8q0P`5rE}{$srlch;QPoHHv^k?n0N8@xQX5dI}{N-464
kJ}7;U`p*RnrCVulk~7m6X-)V+c-3sGoh+Ty~UC`uSgv$*#9Jk2`ldM!j>fiE?eWHdnCGgT<0s?qjWexfySPPSd$L#*|yquV&OxS`o-x{d)UR&Qb)J#bxjO{
<vw%AmaG92zQcXl3bCG`3H2*xg$f0lmf>*w0*(dU!k(&qpAm~JBS5Q~XB5;;61~!p?Za<2R;P#y6dEMtjF_Xwt|(-_BXE;_0(NhBKCSfMWi2ACvn0l!rB3j1
^veQpdK4va*z93bpV1O0Vv&oO@+WmU+<3PmPw=dBICK(g(S~TGWCN~Iq@B}Mw3D)e)hcIS^jq;$J`>rsoi1<$s=k3BXbC1>&}o;f_>l}d?{><W{qq@4b`c8x
Ki*)pH?&@w$GFX4HB74d?-XjPa9mzP8H865$&xeSt$$;}U;8Brk@8!K8lavg@`1V`!Fug-JbHDdxZ%U2$?mtA!hGK$7(n4X9Yoq{){+>mL;M=@7@JxszA{zN
QTP)#dl2XAAFpuFTL)MWYAU<kyxw63<Z^2Yo&c^c`nZh}PvRBT4Fh79gN~n@DNq4C=I9JHmFDixZdq-2F^-9oyidLU4~A3Yzj3*qEu4AS&*H8Kt6uKb!QOa}
OwJZ`YK06Vj2VuO{?v?Xh|vBE56lj4S(6@&i!|k*L)2t=ZVusU#Ax~=QKu#3UMZ(mAW3aA0pPlH7#~~}x##ox_344z5>X38&MIsTpSMnpAHe?)+ssZy@M?zw
uj@6fP>}XW9WZ^$#}#U={#&)4ixU$|6G$sf5<E{S_!`QOo8xD#qi_4Y6%`By=~{?uJH$|QoGG<uC6Stzyt|*ZDlhSTQ(Q`+d7)R5U=Tg!5V=cfXo)UK2q0)3
>KfF+ES0^9BWiQhwb>fKF;EQ&0JY|guYm#9ldN9d=!&3v9sXh{AkjNMu|G6tZ(_PK+@StAV;<L;!%&z#Z95#-yIVZ7n`@+0D?GFjOi?N69LEKA|H~J6^h=3X
sSjJ;6oVUMQG^0tT$c(*bf{GuiyTi~1qSxl>6vDVGH|W1?zaCG_zpTOef>6(y?~zjts462d{;!=8n~t|W(d@Ww;8}T1k&7NtP1E;(T&J{QDXBdhOb?)ISwUl
Hu7fXaKa|of=GEW*F$6%0(kkS%ybsf`%$IL)Li*_Ji^Uv`!_$TXExRUpeeKDQFm|6M*{3cLxz=MD4QplCUdROQ<^bYjml<wsYD~uw`2H;x2T#=3)!pg3nYaS
I*^6~?&x40BrXv8&~c7?4xJBIF<&@JDS6K!>m0O>Bss`1n{)C8?|SN`r1rFww!v4}83{0whDKyI_MV=@ZCj;+#qJ4NJt92{m6g4$HG4>)T<g-<Uc+-ahnkJG
D#gON?ulgm{Hh~I_k3hRVlke$P6riGB}wHUm?`33>@C`rOKgC1XO91WgjX{C+-YF5Cb1s7`4fy>5a3--;CV>V1gW3Cxqt#g`O$GwZTjM}Nx)0xc%8Rra&GY|
vf+cKZtg<tl95aAouax3v`xgd(vd%6DYf^HmVO8OP}(Q3TExigVg>>+X%0dU3CV&1lLL}kY3`-t;a?!rA;k9&0@3q~COwzSJbWT)jT@P2c{DF~Bu!tPV@@)p
5Lu|UdDWW>&#vK-ecL!fG{UhL2_FSub<?M<uV~8Hp!|4sam7}u0g56ZQ=l+TV2|i^BBLE+s}{1>6_K4Yh(<Etk*<5o`4SGGZ%QYuqqXIU$};F$GR1<>7Hnp4
YL&3oUh!w?hlyKmQi6Zb$<qL2999ey5?4?wJ%fRu|Djk-d$0Z8G}=kd$SjEiSMiOV3Q+qf^i^Ame3o{oVJgS5+G4`P0Vw1b6j<(4>v>7BN|V(oBXi<Wne6vb
VURAw1iUa*;gW|V95lFX8<LrmpDS<rTE+N!Le)^hG&6PY^}z(AJLpK5VSSrv$Oqkg{vo|FD>yAn_{TzL<uphWRn^yla1$>zipYq8zUoNv1lw!=mY2tsm)#BK
%VP~WfgcKC&=TvV&u+%a6P7{tJ8s4Q);sCua>K{c<>C_$4Nw1hq8waJEp5loj$4fY0mr7CF1tm}apLmr<WTy%*S#FZ!YH2iSRdnVk6PrRT$|_U<uT<K_EG)?
5DPj<T%EJhX*t*_aM>QPVgNA?_=Ur>5Bpm%gF>xiGo$W4GVkIPeB>tcRKV`Q^7$_o6@}?RDIfwJb}->z8%f4^XRUaDhu?Cz8@p8wxa=W_JI$0hHNBug_@`kg
;)uv1m0^5wtmqRHGH%AkmogRcsE&FjnzoQopc(S)#791}C>?Zi;Ym^-t=r!NesJTJBS3xY<=s0La;52$_6;#W(_#vvzPw79@sr-yWHu@jN<Ba#ig!UgyQm~1
BgM5b&?t*8f=~44i(}2)%DQi!jCsHIrV)rCY8Mz-5-VVhEa54^I(U#`?<OQV3r<!%&75%z7*_F{5+?kqdYy&Rs`7RU7Ryp*4#!>44n-*Cb3_XQ{XD1u62=!S
L@wAkHHW8jO(qshus^$1OQOoa&?N?~*P(p(^Ha|gs@nDq0#5pA*Na&aFUj6l+QNNkV(ri-f~_7?`%O_M>DT!WPFM`IA8y>DWLyOS*|}9t7TP7sr%&@zpPuLs
;NLt_#cVfUQ;IbAklEFLwCrF`+{5ltP?a7`;Tz$nM)!qAB>auC(#a<`1(fLRiotit$AiGqUUj553eGtVAQWn68W>+L!L_`mWc?YWuyC>nos?bQT8Br_dt-ww
5ehFcK6P8lg1l!vQ;fjhK}njU6)8U#>xe?&DOdu3Ju`}d96D{n4l}2%32n^%MEcGu4eb%<^D?#jfQ!ymB+8=t5X^KOZ?x<4XB++CBbg1|IYA%Hw<{<&a#VS#
=t#5LJ)*PZzn6FIC}esnk+Xn91<_x>KO-cH->GmNHLI46ZS_*<C7wMZmoHK@ZKvhFmd$kw1oa3jy9%=Vt|tdqLT#cqG&A(h4l_#spsEfZ6Obz}(oHxri!?i_
=e$d8m_23H;d4Fv<H#aR0M<UrgAKS*yHu&HAF52Tp}0uiTC~GIk(_x&?Wl4M-QxS;i#<!~oc`EIDv|8*9{9_{w6pqWxa*bJZA&c(J_^YN=t-B-!8vv8mJfz@
zk}7b^?rN)RSFjj52tU}^kwEfwUsHjQ#Hu&RV|*Jfl&FQI8#C(9W`aBLrtuo?!~-AQiscmMw7lz<}?$nad0G2L>Nc5a=UIr%34l5YXotBZ`56jpJBVu+%&gJ
y490^B&#v|!a)MRS>VNGe%|T#cSd1zsow?_3eySi6zE|--s%T;9s5zvV+Z#v;``x4-%Ft_Yo}yJJo&yaVXG0{;AekW=vN)*`h%!DhzroR<JUqdiLUTBLmEt|
og`f`z#Sb3>3NH!hkGQd5c%zd6mm9HJF!W5&$5SbAJ%{*6Ar&>1fx-OiQ>>h*_?C*pi+<2W!O@l4b$@Ca=zH?_Qll$uq^>rQ4^!e_1+pshi3>;goRWG&ES);
NwO%_b`8|9TTL@h-x7ab_T^OCTf0>(t~if$t$|p7s>ds7MfGWKB@W!@GAONbh>fn)mQyJ+&Kh%pTyb+uV{>9`f_;CniJ}-H1n5X@Cem@1n&zbml@@(OM&66q
eCV6l1PR&zs~NtX1DetoOGRB_%<k~}Zj9F$V#P}Dt2$0G<8*=JZL!k636@ckS@uR5Z4-EyrX>JOE&sg=vmI8;0Vd0e>Fte2C*h^d5i;FZ&Y#!yr_;1*=F&+^
XFUq6i5>xPF2W39cfB%>N_&xuE*>6Z&W22^mRZY!)Vrv8z5;X_v|I*nG+S8s1L=TC2sFi~@86j(!lnb;k}MDu&d%8eq!QTPc%3&vk!+<jp>oeJ6w~|gY7m+`
sakcv9^149Y712h;w{*kQbhKIqK_K2ovZCksl$f~aER)_95GPb2yk*_bEsG!7lhm)8_1c$1lSm;#S@5^4m@&Pva>S(Xw%z3oWw5Ji2k&0@G7Kc=0?Fb?J<-U
lmft5!6jA>n@cLlE<J)|`$Szdh>tqtxP_%G%JPv=<=Qc1=4^-WyV<F!OAvad!$xR+MDTY(?hVFI_T%3#LZLVHgV^C6^8*St+Vl?EC)J#$m+RmVX77A}QytqY
RQ~WTgxd{nny3}kM-~!OeQQnGP=?#SQ6~oLhj<fJYsUa$hawFixH4Nb5p&LU{5rGc+mck+&66kg(O8W2K|`bu!UIq%=Hbzjpo9vPt}>(N$<{0lZ)gX!DG=@@
F*sGG_lhPa_euKjtxf+c!qXsT-Zc$3%0sbnT61HF9|Q}=WnpEUeek(wIriI1O#D%sG()ucJ`Pm0mlfmlv31U&aR&(&m$+AAkc>0aBHWC;RU(M;Z)im8&MY$~
RqV0L@ezq|VOQnV(F!iR6mAb_ELCnE%23Je>In(e=ecQEv*$?)J)V6>p#*AsGL4NHPCR1~<tmIU(kSqYHA2jj#El0WYVov~bv=1e2O?o$5cLKbYUIF~!56}@
wc?;3Rgz%~W3iST#W~c_g7KccJ?mJKkKUg4lA}H--HSiv`l`v35^)l?KG@h}0J>FvGcSN7r7!`^>!9}iXAhx?W`xVUj_aZx>Hfi>74Q4+i007>A9S_3a+}Ky
vYiEAc=^ki@YFp!QKrxs0k)<nPjxM)59#_<E83@ZsY%C+cIsg^@}q*?x4g;7U-KKxIpM8ra0LqLKu_Aj6@Q<eN&8h$rRjC@r2@fkXl_o|bUE><kZ9i1!8W0h
pqeiF5KdXCA-wuKYO9-6H$**&kr@XSdi%Y`Bv+LmU|;f8erVVI1mRO(9dwiiM9DvcFnYaZCVjMeP28=q>WY86z1m3Sq<S5!Dt>8Ubso(oDoi6tLr9RSBr}0k
u@&*l@w?$APt(jv=lUNEwLLKH%qC}V^LH#TP40I_kYW0#sWb;{-h|AW%re|86Aq_D5<#!RSRMps4A=Joaj@ki*&ABX&fxFQ{aX=Fw3n&IE$nljcuTkZZK~44
2SBLmN+iO$Q7(5Lq{o~eS~M2F5d5wQIL<Z)J}Vq#i!eAFXSZ!U*TevvLinZd5@^&lO|4t5K2niMj8|mh=yLx_F!g}*YwykB=25*}@Ul>l5--E7QCW36#Topp
dlb_?AMIaIU#@=;l&-idy*qm3&-rdX=(X<7-Lwh&Bprd>jA1`eQANEQIA+-9q0<)xYBX>nf_5kmZdCbHhi>3pm!k|u6x_^^9H`6V^`q8_B@oqPG{v;%NbA*>
o}Nv={EV*?y5h~$!YV@lS<{%e>`zaO3(|T~idWDmp_y<S{QGWo=U`5z+A+wt<~ErRK%NWSORH$47XZXV0$IShtM`9>ovh{~Lk9cKpk>^5<l4onFse(;kce4U
(nYX_$xddWmLkX7lr*|eUl*s#?#HF28(fq{qFK%24o$*vPsOlm#NwIVSA9+;1@sqlxL*#{4jW6L&>Z=@R+US4wNb~Wu2l-0aCwbq9n-_iGhT>CD0mq!)O|r1
lA^J@A4I$`LW$o2oC)__Q<@ENFymxUkgyhG5?Ripo><mGJOgiuj4Ufvz&;Y`+%mTTpqr<TGT=0?-9d_fc9x2=rdu(g-}UD=Ac#@aDHX+0C`5hD{2zlc-ehID
dYt-bS-i9^CVUY~lkY3(_;(SN2b7u)*vY$MEslPQ^B%Wn(fBD4aoA^fF6Hi?KS!UW{_?>OFJ|P{R^yjTr{4tmv3APvylUh8o71$#eZ+hr%XB|M6V(Lz#%6t`
n5P~sA$Yfo0;Y)3!8A&tlW8SS?El(TvD(-}T@;Ie4|;aSOtI-6+^JG^(L@#~3*hltvTFh5c?*UAheFEe*pvMwqy;!VS?N~R865lIeIKl-OjPAvT~B-pcGn`0
B>VYNay^y7HV!pSFx=9;F9b~l&IOmhyxB`f=-H29&zpX@-xD;39QBK1nzgIYk(0^<Oob=!TpAJSeijYJ;+JD<b~gcF#{(z$GC;rh`MtYs({9>`{7dQG{cslu
gJIIu*SnjY-D71b&f$*h=n25hp%5<dLkBfy6(IF>B0$G#AT2MwIPKc6<7jTQ_hqsl;`cZShpw*$;F4R9b5pzxPOTg-q?Ubaj}+Ga)NU>7iT^MvwBy5ge@~+E
%9Tt4XQ8OTo<WtbUDCICdFa@;Z;v8YngV#9MD2Q12%WP;er48<3jZ@mptu7{qQPKMsk=baEjxkjKkNn$e7%W8?e7U0MzhbYDV5}GB#_6UV7yD$l8(WbZypib
X;7={47P9eBf|t%W`(OosSQzdz}*zrG`T5)JdMhR&YMi(haFj2AavDU9$e6^U&-^bU+O=0)>(OS@#4d~cpIQ=Ekh&Dan|-k;WVBbRIxP&|HFErTmn=2p!cQe
CoXSL!GKuetIfYSW$oj;=!RhFUPB>I%VH(=2Y_DdatQOgL^sQWd1*-Qt-(Y0E(^O1OxEc9PJdgjCp(@t9?e=qH=JoI@TZ&BI_wv+*!l9AyfPWrk@M>I{{_Bi
+70fLKnA=R!@{b0;U1B4Yf%>X4sPEc&t?)^-xPNXybc~cN*90Jt3ZJhr_DeZdZH(y(k_{p#UlItKMQA$rET~_#gC0B?54N`^rV^oXWrX#C8XyCYP|`kSk&wB
eWYc#rfgsg5Rjivc(m<N2AJ|EEMeAsAAA+7!Fpt=qS>!#TX1~QY+sN$7(NJMJB}6+qhYs_z4FbGn1AB+OZ#aMr8&g^MqZ?T_yjm}Vwpm07G`CzW*7lenF%o5
gCeHqTGg=QMDb<&cEeHBB>(jWj=EK{8?tUSCkGs<s{!%mr2I;d8NN~}hWU>W^mkEnxyjzT!;FH}TthP_2i+nqXj9waL;$gLU#?eN!I^4yI^d92X3K5{82vw9
xA-+y+neGnPHX_hXelHeIDnj}I{&fdX+s?(7}?VdlW#n{-j7SR*gaEed;mZcOM5Pbs5@E!1JWq<wn0AN)yvBNtF@k*m+K}ZpLVh0O#YcBh_o1|l68d`S7%uU
6wRSNWerIM{zLrp=QX9Vt<+(yOtW?Vh%%34S`K1S??iA7H3cl3uWr_rwj}h~LK?5bkG53*5rcqD{1}P!0y`QyJm!vU2a#cb)Ud~8snhm8f|T@z)?Z$`afs?p
Ze}|XzH;h`J_b@jv^%%~N6AQghu7^v05p6!?LCiv>J1}gt(f{>b&hA`9rE=Fk0Y6B6t9Bq?@}xuu2N)3UNqWC2nRYEoRP<`bEn<nGE8>mTdgqBi;fJiJqw5E
N8R}=X3XB9v_%KPl$5*}G1|@`uYKSNl$Z|!%st}GxKN{`+NYQ=NEyMR3K@S!;H7c3<T86BbtBOI?-7x`!WiF=%QCopN;`-xRuy-l#UiuS)-}Og_r7tAdpEc4
!>Wk(P=5Fez_5*TEaI0DyVk%qlUCNnFfIzeYUB_%4D@)C^Z6YIr_ryVbg3UUXIAOiV#(x*+k`VPO8p7sHpT|i>qM)t-rA0~0#6&nrdH*QID8btnLdO&N_I4+
Kjvpub@z9>g1Jm1?W_p&+gUv7M406)Z-rg1`>OW!Z-cmy5738yy6Sj;t^U;R+hDY+GbbX6zcnV@pmH^*1a=SCgL56TEPljd4?_fgBp}dTnXUqCH1Q3$SWByt
Dg&rU$@Q51iy=2-`XKc!ieF<_=+k_Mw((+vJV5;}6x#TdT=7xxuNGD84u#$OltA4pG^W=SblS8=&eIc(hM$EO(!)m}^PRl+cDHl@vgCNNHR&k;iP(}1eGB#5
s}=>0;fO~l>`<dRx!kDh)`ewHbMKIV${1*tu9pmYesWB9k0z~Rw)0>5MhlK4940E9m&k^3pRIa4q46O^{VY=w2^3RaWH2IhJl&Id+CiT7!du=LauXqkV)d$U
1=*Jv;Q{l6qU=)(tkAwexef?QzH+!I?jgm(=H3a3z$9OUxvc4^idonyGWdvRuXXXH!g>sj(EwXVNf2NY9y9-5P0?acgiu$KoiUr7T)+kKdNUwYTz(Z+)$cAw
0uR_>`#=m=@VJ%p%G2T7Sa^8SJQX1$)_gG$XXL3AkSyUx3Fpu$GsE0y>qN+DEKpndp?0X(+Ia|f?1+oS6Nj9sY!D6pi7gZ*G*p^U?MUI^XZr})xr;QGF$vi-
Vh%+M3ejbIEqYwMfG!~AxVBHO1GP*%HbrLE^IRceVD;XG`jc%=i2Q-1UDKYs7nMj;1-L{s2~?*-*M_akVQ)p})k)s4jzbs++yh84zQ)>XYVn!^pkKzT9WQjl
0*S_&fOOdBkhqMw`3`Z6X8km@6+9TNj6;l^@JQ165uEb0!USAg4LMA3nqis5qlSj!r9KXwoFME$T###-HJJw>VsS$D+VsRDfukvyWX9phUJ{Hn5I?zC)oDHg
h2oK#i-^2z8PI8gF>Ii80bQELKx!)VECFXIc=|s7HY&$0AZ+LW-?jzD?l?g#d)DMT(sBwn!-sy<PqQ|g-A1Q&8*|T5EAKoRD<lV*W|z-1n?T_&@NToDPJmB_
3{s-FOCZa7ha355J7db5VhGwL@BRjMrcR2z`X+BT0unVXTi2pN_7cGX)@!-=*kqcV|C4fhKNOtPL$G`uLI>nc`o9v?VEpo!$`&+mf$4(d(}Kg(>%C5|`$R*k
Xhwe~f$zccB5JXOyN3QofmHW$ThUq~c&kmdqF99zneXae5!X)d##D{FpwF>xcAy~*hU0*ydGnD`2db=FN$jPbNz{Jl6zf>e-<7SkTjZCLRCul%@)HH!X69`%
{6qy$J%yK21BEdV6uF9kxJb~C+w~OtK<c>pxdp}zz4CVz&DMY+LJotyxLF#c&dk7jXHMT@0M+0}YAt*h+Gf@Anx4fBQ8zgMk1OJuq740_dki;qa~KY4(zk6D
JZjplt>4B#R&4zueDLUT3tw>aOo(r8XkTb}zwLZJyZbY*`=p`~lr@jW%_yb|3*wgZGoh;!zaY%Sea0Z^q7Y<Uk=|d3_}~y<=hVPH<8}`SgW6CvAWU`stZMNv
xzh4y98EiKAYwu~9tkQ-R-%5{NgpD|&O2nC`61yY7mu5KE9jLZzNHA2Q5Q1x_3-T+&R;$xD9)hBv@-S0P_3Q@F)@N{I{6#^%jDu&GoGYrKYfj>@ywSdh%|qx
M`OXofmMJTv|v(do95)wVr)<_{>H^Jh^mN@AKqjY)%7_IwpSWOde$=I3^B*{ypJ;<L`C~z(}A(x6Fj0aVUUnCd}!rm`LDHx{TI_GRRpHOeJHOCUx<zRd-0yv
fAhrBL<q?i${UCYbFyTYW>kF$Ql_D^hGDnSG38jMY-wnFC!CU8GWyShKlIYm(a|*E5@%=`j^ZyeFpI->FY=$bn_G&`lj~Eu3|#q$2Zo2SxrZL1mzZli$(C8Z
0UWxDq}=fH=MDx?hD1uMN|L>;2CEDw79@|F&X`1zDK&(hjf?i{4*fuaE(#GK^-xSvADIK;-+AM?_REAVol?G_W}nF$qv=vsjgxvF$H^VX$~tKq!_0B{WOm0F
0wu;L95lxUnk};jG^uzNnJzBhAq^|&?t#nH*^Daw&(&23Q>gXofU^(fhdY8%rTy?25Ja1hUHZCMJ2yx}HmaKA9HcA>T^RrY7SR#rY`pNQo}k{F2SrylJ6=1C
m5=pNBMXC3HBuT?3DBrF?$<<cL`Mcn)zJX;VwP(Ko{mS5N*>~ni<qVK95%BHz~_3d=7fC(2(jd|H_5=O)nVoPZmgQX`PK^o=euc$-5L>6UM_A~33>Y{`-j9@
YJpA=GQ(txw08%l9jKmIjbcf^_l)_2%%s~>F81Js<whn>O9+X@Skg<<=#txfG8s~F!@PVYI<>~0&yb19ok;5ExWcAo1UKi=E5-~Ocq=m^ALTdaOK4)K`tn*j
!e}Q#M4;_e2eiicXXb*soq#FmiiZH&rNG$fja!5A*<>xcq+APy!qglcr4okE?06uBAXS9|Qwr8Wn^BXhSi76@)@Yu5Uj=q_jWZXwf$65*TC(secETwJ#dPIL
4RH`?o=ckKyvYjDMj*NlhfRL^K95N}3nyFtzDzrQyU7?vfSisFLzdV{6UHq)%D#>!{2M&^s5LYvqhFR*>-Vxra5qz+W^k4|2qdWmJr0_r{FDqnnFu_&EP6gg
&`xN>icPu8`>rLxU#QUTsh0Z$6yO!PNJSgTymvjy$<*1Y4z$1I-t|t*aB0dUZs63?L9!({1#O--jH&L9rC-!Ls5gor+skxfzh{pmrCSPm3XsrtWBO`M<XI7u
Rc$PF^~^|GPIC)bQht%mlytxvkWVm?Kna#|QLNF3+cs)YMXc+Iuf!<Wrtgsz*`$e2EW~Ey6j1zP)ix^+%q&zcSI4aN7}QI|jt)SO$!jlM8gW@y!tE@NHinZ&
IqpRDn=j#=ajxLEPAvwN20Y*BZxC2ziy(5TYY@i>A;f(FY>!+>F1+*MU}Y+dn*XTzi>g>#*!+)Gx0N;9GP;o-5?f?OMml3ia<$s}#B{6k1f_VK$->`XKAM5P
1Kg>YNPdr7Tk`dp0(b`pQGEcJ5~A~UbV`Qu-8MKWIUM1|bI9Fv0`)5K0{AQ@vziJMmdsOy07uE$r`sx`{4A)F*vFv*YGPgcR!F*xuG76b%+BryeFig025Co6
=1IR*b*+UK4-@k0iG(<uTF=iBpiU6|5<GfeEUESB6YHvdH&Ed;UbaYL3YR;5`3ibe)Oca|BRg-(_q-Nh{?}zf!!ZZyg4(2}^}4Lh#)}#*g;hx&&CE*(@MHoH
m!ex142nP9hYtaf<Sjo+Fy`e^-INo+!@EDx`ZVj^OmlZG|FoCVUuwLb^vTcE<YrqBz&*jjB;NLA<E~#(z7a>RS*cpypxL}sta>%Bntz@Pn{Q7L`2CI(d)efv
9HT^Uno$BRK?)x~=dML8fowL5*|x%3S;t(k3xsZdMt76&MKfXLwi4ekqat6P%;fU6Gz=7LtxE=**LXAg?4ITRid1H^NL9yE`~XrCc8Lp~)Nh*{ZU`9rlTd7I
kJc2VK19x<u?gABT=%k^A@d{7tyesTZ7#*)wW+IO__S6iR^)|>*DI9N-3MM%O8d}4R{!E@g$<6mkYVvKdwWqHakmA@9y!2$ln4rU5S5`foS}UAqutYghIV6<
gYjI79=RlPdD)82%lVkgjA9=>ACHhKtIuU2I%hki&Of)m3CftV$ETht*+xEIF$7Y$95NpeBm&5wgvz|mB;R@C&J1A((!3t5ZvkE?$5?p0p5J0cI`p4~Cz^8=
gucWk?a5f8#Sv+*+{wzhj~GZKCY51PW5G*_AcR=@+X;cWt{+Iee9u$fN(9X%nA7i_ff8%$`_fJ^tYzwjb&+V~6Pj+Fu>FtT2qJZ>ZF0_NnA4^Styy|6UiuV4
tk(i3Lp0!b`|4en9_|X6qc>i@*#$kb$5l|6F}GoNF}`F%2AP)48m~{>5B=av@~Zy~Edutc!h5(={{J^8H0P-`vD7S}75ucvZZZ4vx`%viZIR{<*?u?jlGVj%
+B=O-(B5~sk(Z6rRq9J_xk+Nc9szyAq}lIG$(6?!yK0Jop+uLq=Smi|J0bFamsaz4UNz=&(nb?ME^v!*YkO@5#GSq7v7e^;XSrO*b{k&YIVT<O{<PYV(=X-A
vh%V#hpluG5kr`4^ZmLj(7g-|_ufZ5ZYlLK{aSWV`m3#O+Ly@+D_Kra^U&z<OJ#ZeScMN<<+FN5XneYg00J}DWG-0CW>rZT(NIjqY+aDEa?qMcdf?x*D?3#Q
XJ49_<|gvXAxyDDDE6(3#Ud}l*GrI@FP6_<@F?&O*#tYJe_4BH3&Uel)N6*^px?t~IImy^2hO;sI5bPGUPB!uOM-!AX7U<wl*=6X-=f$gie$+}h`;lE2gtx%
N}Q2Z?{wvVb`4*86N<c`b}XN`$XdegpR~9*;x+(RrYY*$xfMdYU(q=ii@gqJ3m}m3tm9jQlr9E$`cUgh3KVYOjs&5T(;Vg=Y6=SW32h1`X5B+2n`m%O%c?v0
fQQaS76^Dm)A9oy3-K`1krRLsiOu-fv9N8c)w7Gi1=Skp2n+(a*>O0uIR=ey<3|_L;ty`sZI6Edj1;xM=Ua5KSxWI!kd(!+fPGa8@^E3pUu~(HYvZ?F4Xn_3
GMF&(%%wXJq<5WB<Av@QpmT$HT<>dEeIH?ku=~sEU<qB3kk$RPbSY9Z9HXaQM>)>ML|9<w>Fd-#rZ1by6}Q5j63tO%BinVGhhJTKUzJ*4N-NZ@rU-<t7rtBk
^wqmM2lK6=d=y$KaAD-(Eix7|xS+;GNT}X4fROD0=YFSfx)6ABZ!P_LuO@wMH^Y!Or(r44q^;*7M=!AU=*jk~w$u#&rZJI@qp6eIZC6^R<5~j7BfEPg`Q~Tz
QGsf@Ej<HkJX}<}kE+E#f*w~HcQKCgsvq$&qZ#Z^k8Bl`-z-ywjVX&{+ow>Jb`=j1l3UAyZH6(nc(Epu$vT}WWl|LR-`$}@Cq}fkM!A?SqBTG+zYF8O-5l?h
!3B$#D5XS&a(N;NF+5d%^|mfmTf30>y=+a#*0YGG%OXSVQhyGv55=*R?KO;A{DwZCI(LX7gy_|Bxp($<tx>T2&~*8sq1=anyjs&eFUH(6i8e=gxz1wC70hO+
knN!pqETlN@gVj0HZi=%gO)7>m@l0;GmZkUOwKFXy5jU%UN*&jml43yi{v5oN~1FgoX`My$J~+RJ$(P17;j9-F-+)oZ5}`EdMY?%*)8LHT`7Msi3NaV&jO>r
`ZKkX0~$Nye8-u<K{@oG3T@iH-Ypq08tLL?(yyt;*BV9S2=7o-Ar=KxCxe2ZIt#DE$;QaJPo19X!zm@{y|CLpd11vjIJS~h3FL)I$2ZVZgPJ(QFp9vTmfm0+
HwuJ2ckIT6#FKsz9wVrE;I>Dti02Q%yjMw9uCFjXxE=f!ZM%q?GjD=kK<FiU3%7cyyj~sm(2DLlckT0vr~}`ZIJ!AWvmjSgw$T#4JvHpzK41$`D=R>{JW%=h
{dh6`aw0M(?+XbCiSMnD{NHW0zd5NlO4CV`H;IBY`3%meaznV}tB$j!fvtJtEW)M75(h>cBUv4+^Cn~kGvkUzAB40rx&v9@qzB|{pKN-eS7M4q`xP|#=Di}c
20S@~%OJ`>v5eFSzSk9w{Z|Bl32YJ_BibIB7NSm-7AJ0Rdz~*3(~8`)<H!&<)F`+H^m3Mg)76$b3N-a{Q9-WJI5prV3IfJY<oJp~n4b0=-RnvyRXYnm_4Ll@
#ZXnjAgR4i7DHil<b_9RWX9L}Jj<YbSp2t8V>RU=ozcTA4b*-V!|!El8YyA&*DEjVXws`Tp*e69_H<H786ZmLK2CrsQJ#DGdy<q#cUuG}i=Vg<_~y#YRzt0j
&C(H3%#P9AXn!Gw)qUk!fOZ^pX~(TNGHWGanKs4-d#xtCLs+Zu`oC!wmDZI%+4&TEdgOcehcSju3&Z6Cb9aFlYOeB8-q0HtXXL8@OzG#m-jvV=;^GpuZX9>g
IBIB<Kx5Q?(gwJ>&-qiD%zmK6={qWS#UIkjHY{QIr3C1BBF#NtV~DkK9Ru2?mxZ*4CDG1es~la>P*fTrTl?Az2Dhgb5(ujR;L%7$Rn)I0T5?lTlUe_U;*S+h
eghppQ5%aO*h+l?)U*h}1i+N8W)wOD)Ca(z3-B_#koy(XDF42f`2#TliQ6)(vD|fNf!K!*7cIuBi&irkD~&Xal-wt?X7>+Ep_ZLQGUYWOOQ(drU(7(&7se;x
G29(QaPHT$uJ1cp7c<2Kd(|H9f@|jR<tS5^IXHy~CQ?HO<T&Zy8F-Szvh_*LOx+mqUHR-l;j%}eE#gK4?yo@gej);GpO(nXJt)vYZ6Yv_1J+Il-HjFyREr5l
r%q#AlC;eAKys06evY0;y*OoID=NmLK5?$1s~$nQc3hdQ`pVM56QTEo7KeA6A^!{FSrB@H5}NFE?!95F_6Wyty^~(AN7{j_vF9k!m}5K1Tjc?LP4NEKyYhS7
$Z0h$U<7igB{F)p>i=d5t4_DRejaySkGgE&4Vz{E(JQ7AoyQka0oI)VkI9EJn;$ceV8k3=v{px5^PRLf3c@T(&E%2}j!gxWA|RTHjg+w3EH%4tu4BbKIMjl}
l#G7AZ;;zJw}ipH8rJchi0K~9K^K7V-ml7m6h7_Vl!*xj-=mymM5X^{WBG0_fq^NR-IjG-_le+;r6fgaodhF+p!M4l<&ofpdn?Ll`nR#f&eBg+4H0X_Ch9g_
Pr&8#9klJt469;L;H&}x|G67uCPNiHWQsdOh3sVi>=)G99DE2VmpC8GUC^G3#&B|9Z~GUoRfC(`yES0Rufc=LyLQzIF+9l<0nj0vK;of)T@=O%f)rBsPA$Vk
D2v*$Awk-S(6?PoL{baf&9`78YIR27*77c>#evA1qiZVlA^<Lj)-_6ue}tR~m>SCyCZe=(_W999RAskd6%pSJti+L;u@+ErC*o+y6*tpq808e?N0~Pjs4iDR
59YQ%b$6#-X5cOK5u|h&JEfE%QZ`NKu3P_7?40)vcOuFKJ|mvucGhds<E)%&02xpYDRO*@lKr{kK1MPH$B<r<$7Ak#@2JlbJS3R(Ovr5v4RkYgFj3~tvd)Db
Nyh~n+cji_iWAEdW`Y98hMS=3pw=!c9mN_=0^&G3O{x}?KJ8j%{PJ4*XLHqfuNqTR*Y1nC$tHiMF7UqU>V(M=+^w#WrRI2h7MBLRR7$FO9EYKfF5?>>6fHE3
G;IatK}IV$4#0i%hj#Z-aiX=pkC}x%vgL|RLzK{SdtiU+L9a%{l>VoXSfGw=P<tCDXZ!i;oD=j>+#fGBpN`>6f}R-809TE*A32z$xW1kSo#hsI^#E$p&|0&V
<U$+(f=9O%I7PIVBX!1uAG=}KC(0gihpvEA?HTV2o8w=;{EpQT^3vQ$S$~^t3z6dM*v5IcY*tVcYQKvtMoJa>H(^)`N6@@iWO<fv^jP-d^wf_S16nxnvIS=X
`F2YEM#9<kQ0?2WTuWENq>A1>B0Z2Nr%Dm4LqsfkG~cahq!pZ5YFP8*_LE*g=2AfNrrgb3VcPmFJap`tqtbC>*TS~TB%0bKXtD+W)xg8x?7_<!lJaBTdnsAf
^A}c-Ocl;xZpLLPMH3txVbWr-_2!FI1+xErbJonbNNmsfQ*wh2e*XUNa0=OUdtc2n$X#oK6fpn?Z<4FmWba(>yxFiTux#Z?3VT_*i5^o8GR{lRbk*-n|Eyo_
_d%h>m7!z_LF*|k{UGoJE693WXRPLOaJ=@+R`<ae{aWCh4x7$^FTK8&3>r+U)qb53U7Gg5?I;m->8=hb>5}z$Da5g9{^F3mny6ah_FtGJ14L2<q|G5XEK@ZD
nWfcwtWt@~5QtSf%Wm26L4G1MIQq|MTB`t+keKkpSh{auskzHt2v+e}na|J^+5oQ*d^v(}QBOPb+H;Dr@onpN2l~ITxGi>or^BfZX*tHZ+z+q6G`75XHt6Ze
K^el|JtqYU^Z<1FfD?sT-HLa4bCtv7uQ?YFx^nwgM@{eU5SSqg`?`{<mecC|dLBK%VDKm*Gw*2J6Ys4TNoFpbN!dyq17u?)XbWLqLG7TfTRjnAnzBo3H@%&y
AZ0LrR_4t>QJ~dcMZC)^$LBY+b%Ai*{FHRss`y3Tn(UtXBqqzQyI4)49({k>l09$Qx)+8)jTLEe^>eL)(p3AnL5T26qEiy#Z+wdLvNX^YFU^1jyXlc}0hLrh
NSML~dOg?wfhb0sz5a*7T5M?7KCgVZgAWefud1&$KVFx3J&m3Yc%`cdV8BK^_Ql1$^>AI<%~N*G{=NVJg0cNJK{&TO00G>{1kn2!)TslRvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/treino_v2.csv')

In [ ]:
import subprocess, sys
cmd = [sys.executable, '-u', '/tmp/nutrelle_codigo/treino_gpu.py', '--rodada', RODADA, '--saida', SAIDA, '--orcamento-h', str(ORCAMENTO_H)] + EXTRA
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for linha in p.stdout:
    print(linha, end='')
print('código de saída:', p.wait())   # não levanta erro: o que ficou pronto continua na pasta de saída

In [ ]:
import json, os
m = os.path.join(SAIDA, 'manifesto.json')
if os.path.exists(m):
    m = json.load(open(m))
    for nome, c in m['configs'].items():
        print(f"{c['estado']:14s} {nome:46s} serviço: {c.get('servico', {}).get('estado')}")
    print('paridade com o D1:', m.get('paridade_d1'))
    print('recorte pelo prato:', {k: v for k, v in (m.get('prato') or {}).items() if k in ('regra', 'consultas', 'reprovados', 'n5k_ok')})
    print('confiança:', m.get('confianca'))
    print('bases externas:', {k: v.get('situacao') for k, v in (m.get('dominios') or {}).items()})
    print('tamanho da saída:', m.get('tamanho_saida'))
print(sorted(os.listdir(SAIDA)))